# Swiss Law Citation Retrieval Pipeline — Notebook-Clean Edition

This version is structured for notebook execution with **explicit path config in the first cells**, original source cells for the pipeline stages, no CLI `__main__` blocks, and notebook-friendly runner cells.


## How this notebook is organized

- **Config cells** set paths and runtime parameters.
- **Utility/source cells** expose the original project modules without CLI entrypoints.
- **Stage runner cells** call functions sequentially the way a notebook should.
- **Pipeline and submission cells** are included at the end for one-shot orchestration or validation.
- **No notebook path-registry cell** is required; paths are set directly in config.


## Project Overview

# Swiss Law Citation Retrieval — Revised Pipeline
## BM25-First Architecture with LegalMALR-Adapted Multi-Agent System

---

## Quick Start

```powershell
# 1. Setup (one-time)
cd E:\swiss-law-pipeline
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
pip install -r requirements.txt

# 2. Build indexes (one-time, ~4-7 hours, CPU only)
python stage0_build_index.py

# 3. Run full pipeline on val set (~40-80 min, needs GPU)
python pipeline.py --split val

# 4. Generate test submission
python pipeline.py --split test

# 5. Validate submission
python submit.py --validate submissions/submission_test.csv
```

---

## Architecture Overview

```
Stage 0 (OFFLINE, one-time)          Stage 1 (GPU)
┌──────────────────────────┐         ┌──────────────────────────┐
│ BM25 index (171K+2M docs)│         │ Qwen3-32B Query Analysis │
│ Citation graph (Art↔Case) │    ──→  │ Extract explicit cites    │
│ Lookup tables             │         │ Classify law domains      │
└──────────────────────────┘         │ Generate DE search queries│
                                     └───────────┬──────────────┘
                                                 │
                                     ┌───────────▼──────────────┐
                                     │ Stage 2: MAS Retrieval   │
                                     │ Planner → Agent → BM25   │
                                     │ 2-4 iterations           │
                                     │ ~200+ candidates         │
                                     └───────────┬──────────────┘
                                                 │
                                     ┌───────────▼──────────────┐
                                     │ Stage 3: Graph Expansion │
                                     │ Art. → citing BGE cases  │
                                     │ BGE → cited Art. statutes│
                                     │ CPU only, <1s            │
                                     └───────────┬──────────────┘
                                                 │
                                     ┌───────────▼──────────────┐
                                     │ Stage 4: LLM Reranker    │
                                     │ Select relevant from pool│
                                     │ + Direct citation gen    │
                                     └───────────┬──────────────┘
                                                 │
                                     ┌───────────▼──────────────┐
                                     │ Stage 5: Verify + Score  │
                                     │ Normalize citations      │
                                     │ Confidence scoring       │
                                     │ F1 threshold tuning      │
                                     └──────────────────────────┘
```

**GPU usage**: Qwen3-32B (full bf16, ~64GB VRAM) loaded once, serves all LLM stages. Qwen3-Reranker-4B (bf16) loaded separately for Stage 4b cross-encoding.

---

## Running Each Stage Individually

Each stage reads from the previous stage's checkpoint and can be run independently.

### Stage 0: Build Indexes (CPU, one-time)
```powershell
python stage0_build_index.py              # build everything (~4-7 hours)
python stage0_build_index.py --bm25-only  # just BM25 index (~3-5 hours)
python stage0_build_index.py --graph-only # just citation graph (~1-2 hours)
python stage0_build_index.py --lookup-only # just lookup tables (~1 minute)
```
**Outputs**: `index/bm25_v2_index.pkl`, `index/citation_graph.pkl`, `index/citation_lookup.pkl`
**Expects**: ~16GB RAM for BM25 build. CPU only.

### Stage 1: Query Analysis (GPU, ~7 min)
```powershell
python stage1_query_analysis.py                           # val set
python stage1_query_analysis.py --split test              # test set
python stage1_query_analysis.py --query "A claimant..."   # single query
```
**Outputs**: `checkpoints/stage1_{split}.json`
**Expects**: GPU with Qwen3-32B (bf16).
**Per query**: Extracts explicit citations (regex), classifies legal domains,
identifies legal issues, generates 3-5 German search queries.

### Stage 2: Multi-Agent Sparse Retrieval (GPU+CPU, ~20 min)
```powershell
python stage2_mas_retrieval.py                    # val set
python stage2_mas_retrieval.py --split test        # test set
python stage2_mas_retrieval.py --max-iter 3       # fewer iterations (faster)
```
**Outputs**: `checkpoints/stage2_{split}.json`
**Expects**: BM25 index + Qwen3-4B + Stage 1 checkpoint.
**Per query**: Runs 2-4 iterations of Planner → Agent → BM25 loop.
Agents: Rewrite (EN→DE), Supplement (implicit conditions), Decompose (sub-issues),
Supportive (procedural), CrossRef (constitutional). Each generates DE queries → BM25.
Expected: ~200+ candidates per query.

### Stage 3: Citation Graph Expansion (CPU, ~30 sec)
```powershell
python stage3_graph_expansion.py                  # val set
python stage3_graph_expansion.py --split test     # test set
```
**Outputs**: `checkpoints/stage3_{split}.json`
**Expects**: Citation graph + Stage 2 checkpoint.
**Per query**: For each statute found, adds top 5 citing court decisions.
For each court decision, adds all cited statutes.
Expected: +5-10% recall on case law citations.

### Stage 4: LLM Reranker + Direct Gen (GPU, ~10 min)
```powershell
python stage4_llm_reranker.py                     # val set
python stage4_llm_reranker.py --split test        # test set
python stage4_llm_reranker.py --skip-direct-gen   # skip memory-based generation
```
**Outputs**: `checkpoints/stage4_{split}.json`
**Expects**: Qwen3-4B + Stage 1 + Stage 3 checkpoints.
**Per query**: LLM selects relevant citations from expanded pool (precision),
then generates additional citations from memory (recall supplement).
No chain-of-thought in output (LegalMALR finding).

### Stage 5: Verify + Score + Threshold (CPU, ~30 sec)
```powershell
python stage5_verify_and_score.py                  # val: tune threshold + submit
python stage5_verify_and_score.py --split test     # test: apply threshold + submit
python stage5_verify_and_score.py --threshold 0.20 # override threshold
```
**Outputs**: `submissions/submission_{split}.csv`, `checkpoints/stage5_{split}.json`
**Expects**: Lookup tables + Stage 4 checkpoint.
**Operations**:
  - 5A: Normalize citations, check existence, drop hallucinations
  - 5B: Confidence score = weighted sum of source signals
  - 5C: Sweep thresholds 0.05-0.95 to maximize macro-F1 (val only)
  - Expected optimal threshold: 0.15-0.30 (low = inclusive)

---

## Key Design Decisions

### Why BM25, not Dense Retrieval?
Your empirical results: BM25 R@50=0.73 vs Dense R@100=0.24.
Three reasons:
1. **Exact citation matching**: Queries contain "Art. 221 Abs. 1 StPO" — BM25 matches exactly
2. **Cross-lingual gap**: EN queries vs DE corpus — dense models lose precision
3. **High recall@50 needed**: Val queries need 10-47 citations each

### Why Multi-Agent System (MAS)?
LegalMALR shows that diverse query reformulations dramatically improve recall.
Each agent targets a different gap:
- Rewrite: vocabulary mismatch (EN→DE legal terms)
- Decompose: multi-issue queries need independent sub-queries
- Supportive: procedural articles missed by topic-focused retrieval
- CrossRef: constitutional provisions and general clauses

### Why Citation Graph?
40% of val citations are case law, but BM25 struggles with case law retrieval
(case numbers are sparse signals). The graph provides mechanical lookup:
Art. → which BGE decisions cite it → add those to pool.

---

## Expected Performance

| Component                      | Estimated Recall |
|-------------------------------|-----------------|
| BM25 baseline (R@50)          | 0.73            |
| + MAS multi-agent reformulation | +0.08-0.12      |
| + Citation graph expansion    | +0.05-0.10      |
| + Explicit citation extraction| +0.02-0.05      |
| + LLM direct generation       | +0.03-0.05      |
| **Combined recall estimate**  | **0.85-0.92**   |
| After LLM reranking (precision)| 0.65-0.75      |
| **Estimated F1**              | **0.70-0.80**   |

---

## File Structure

```
swiss-law-pipeline/
├── README.md                          # This file
├── requirements.txt                   # Python dependencies
├── pipeline.py                        # Master orchestrator (all stages)
├── submit.py                          # Competition submission generator
│
├── stage0_build_index.py              # STAGE 0: Offline index building
├── stage1_query_analysis.py           # STAGE 1: LLM query analysis
├── stage2_mas_retrieval.py            # STAGE 2: Multi-agent sparse retrieval
├── stage3_graph_expansion.py          # STAGE 3: Citation graph expansion
├── stage4_llm_reranker.py             # STAGE 4: LLM reranking + direct gen
├── stage5_verify_and_score.py         # STAGE 5: Verify + score + threshold
│
├── data/
│   ├── data_paths.py                  # Central path configuration
│   └── __init__.py
│
├── agent/
│   ├── llm_backend.py                 # Qwen3-4B singleton loader
│   ├── verifier.py                    # Citation normalization + verification
│   ├── __init__.py
│   └── prompts/
│       └── mas_prompts.py             # Stage 2 MAS agent prompts
│       # Stage 1 system prompt and Stage 4 reranker/direct-gen prompts are inlined
│
├── retrieval/
│   ├── sparse_retriever.py            # BM25 search (primary engine)
│   ├── graph_retriever.py             # Citation graph traversal
│   ├── explicit_citations.py          # Regex citation extraction
│   └── __init__.py
│
├── scoring/
│   ├── confidence.py                  # Composite scoring + F1 optimization
│   └── __init__.py
│
├── indexing/
│   ├── build_bm25_index.py            # BM25 index builder
│   ├── build_citation_graph.py        # Citation graph builder
│   ├── build_lookup_tables.py         # Normalization table builder
│   └── __init__.py
│
├── configs/
│   └── default.json                   # Pipeline configuration
│
├── index/                             # Built indexes (gitignored)
├── checkpoints/                       # Per-stage outputs (gitignored)
├── submissions/                       # Final CSVs (gitignored)
└── models/                            # Local model files (gitignored)
```

---

## Troubleshooting

**Out of GPU memory**: Qwen3-32B bf16 needs ~64GB VRAM (H100 80GB recommended). Close other GPU processes and ensure no other large models are loaded.

**BM25 index takes too long**: The 2.47M court rows are the bottleneck.
Ensure you have ~16GB RAM free. Consider reducing COURT_TEXT_LIMIT in
`indexing/build_bm25_index.py`.

**Low F1 on val**: Check each stage's checkpoint to diagnose:
1. Stage 1: Are DE queries reasonable? Are domains correctly classified?
2. Stage 2: Is the candidate pool large enough (>100)? Check retrieval_log.
3. Stage 3: Are graph expansions finding new case law?
4. Stage 4: Is the reranker too aggressive (filtering too many)?
5. Stage 5: Try lowering the threshold (e.g., --threshold 0.10).

**Resuming after errors**: All stages save incrementally. Just re-run the
same command and it will skip already-processed queries.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Config

These cells define the notebook runtime explicitly. Set the actual folders here first, then run the stage cells below.

The notebook uses visible path variables like `PROJECT_ROOT`, `RAW_DATA_DIR`, `INDEX_DIR`, `CHECKPOINTS_DIR`, and `SUBMISSIONS_DIR`. You do not need a separate notebook cell that imports or reloads a path registry.


In [2]:
! pip install -qU deep-translator rank-bm25 scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 94.0 MB/s eta 0:00:00


In [3]:
# HF cache must be set BEFORE `from transformers import ...` — otherwise
# the models re-download on Kaggle/Colab every run.
import os as _os_pre
from pathlib import Path as _Path_pre
try:
    _proj_pre = _Path_pre(root_dir).resolve()
except NameError:
    _proj_pre = _Path_pre('/content/drive/MyDrive/swiss_law').resolve()
_models_pre = (_proj_pre / 'model').resolve()
_models_pre.mkdir(parents=True, exist_ok=True)
_os_pre.environ.setdefault('HF_HOME', str(_models_pre))
_os_pre.environ.setdefault('HUGGINGFACE_HUB_CACHE', str(_models_pre))
_os_pre.environ.setdefault('TRANSFORMERS_CACHE', str(_models_pre))

# ── Standard library ──────────────────────────────────────────────
from __future__ import annotations
import argparse
import csv
import gc
import importlib
import json
import math
import os
import pickle
import random
import re
import sys
import threading
import time
import traceback
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import combinations
from pathlib import Path
from typing import Any

# ── Third-party ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
from deep_translator import GoogleTranslator
from rank_bm25 import BM25Okapi
from scipy import sparse as sp_sparse
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Base Directories ─────────────────────────────────────────────
try:
    PROJECT_ROOT = Path(root_dir).resolve()
except NameError:
    PROJECT_ROOT = Path('/content/drive/MyDrive/swiss_law').resolve()

INDEX_DIR = (PROJECT_ROOT / "index").resolve()
RAW_DATA_DIR = (PROJECT_ROOT / "data").resolve()
CHECKPOINTS_DIR = (PROJECT_ROOT / "checkpoints").resolve()
SUBMISSIONS_DIR = (PROJECT_ROOT / "submissions").resolve()

# ── Data paths (Kaggle layout) ────────────────────────────────────
# Dataset 1: law-db (index artifacts + corpus + KB)
LAW_DB       = INDEX_DIR

# Dataset 2: competition CSV files
COMP_DATA    = RAW_DATA_DIR

# ── Raw data files ───────────────────────────────────────────────
LAW_CORPUS_PARQUET     = LAW_DB / "corpus.parquet"
KB_JSONL               = LAW_DB / "laws_knowledge_base.jsonl"
COURT_CSV              = COMP_DATA / "court_considerations.csv"
LAWS_DE_CSV            = COMP_DATA / "laws_de.csv"
TRAIN_CSV              = COMP_DATA / "train.csv"
VAL_CSV                = COMP_DATA / "val.csv"
TEST_CSV               = COMP_DATA / "test.csv"
SAMPLE_SUBMISSION_CSV  = COMP_DATA / "sample_submission.csv"

# ── Index artifacts (pre-built, in law-db) ───────────────────────
STATUTORY_BM25_PKL     = LAW_DB / "bm25_v2_index.pkl"
STATUTORY_BM25_IDS     = LAW_DB / "bm25_v2_ids.pkl"
CITATION_GRAPH_PKL     = LAW_DB / "citation_graph.pkl"
LOOKUP_PKL             = LAW_DB / "citation_lookup.pkl"
CITATION_SIGNAL_LOOKUP_PKL = LAW_DB / "citation_signal_lookup.pkl"
REFERENCE_GRAPH_V2_PKL = LAW_DB / "reference_graph_v2.pkl"
GOLD_COCITATION_PRIOR_TRAIN_PKL = LAW_DB / "gold_cocitation_prior_train.pkl"
GOLD_COCITATION_PRIOR_TRAINVAL_PKL = LAW_DB / "gold_cocitation_prior_trainval.pkl"
STATUTE_TO_NUMBERED_PKL  = LAW_DB / "statute_to_numbered.pkl"
NUMBERED_TO_STATUTES_PKL = LAW_DB / "numbered_to_statutes.pkl"
NUMBERED_TO_BGE_PKL      = LAW_DB / "numbered_to_bge.pkl"
NUMBERED_COCITATION_PKL  = LAW_DB / "numbered_cocitation.pkl"

# BM25 chunked parts directory
BM25_PARTS_DIR         = LAW_DB / "bm25_v2_index_parts" / "bm25_v2_index_parts"

# ── Writable output dirs (Kaggle /kaggle/working/) ───────────────
OUT_DIR                = SUBMISSIONS_DIR
CHECKPOINTS_DIR        = CHECKPOINTS_DIR
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────
# QWEN3_32B_PATH is now defined in the config cell and used globally.
QWEN3_32B_HF_ID        = "Qwen/Qwen3-32B"

# ── Legacy aliases (some cells may still reference these) ────────
QUERY_TRANSLATIONS_JSON_SRC = LAW_DB / "query_translations_trainval.json"  # read-only source
QUERY_TRANSLATIONS_JSON     = CHECKPOINTS_DIR / "query_translations_trainval.json"  # writable copy
TOKEN_LAW_FREQ         = LAW_DB / "token_law_freq.json"
CITATION_SIGNAL_SCHEMA_JSON = LAW_DB / "citation_signal_schema_v2.json"
STATUTE_SIGNALS_JSONL  = LAW_DB / "statute_signals.jsonl"
CASE_SIGNALS_JSONL     = LAW_DB / "case_signals.jsonl"

# Copy translations cache to writable location if not already there
import shutil
if QUERY_TRANSLATIONS_JSON_SRC.exists() and not QUERY_TRANSLATIONS_JSON.exists():
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(QUERY_TRANSLATIONS_JSON_SRC, QUERY_TRANSLATIONS_JSON)


## Utility Functions (inlined from project modules)

The following cells contain all project module code inlined for self-sufficiency.


In [4]:
# =================================================================
# UTILS: retrieval/bm25_artifact
# =================================================================

ARTIFACT_FORMAT = "chunked_bm25_v1"

try:
    BM25_CHUNK_SIZE = max(1000, int(os.getenv("SWISS_BM25_CHUNK_SIZE", "10000")))
except Exception:
    BM25_CHUNK_SIZE = 10000


def get_bm25_artifact_dir(index_path: Path) -> Path:
    return index_path.with_name(f"{index_path.stem}_parts")


class ChunkedBM25Okapi:
    """
    BM25 Okapi scorer backed by a scipy CSC sparse term-frequency matrix.

    At init time the list-of-dicts representation is converted into a sparse
    matrix (one-time cost, ~30-60s for 2M docs).  After that, get_scores()
    runs entirely in vectorised NumPy/SciPy -- no Python loops over docs.

    Same formula, same output as the dict-based version; just faster.
    """

    def __init__(
        self,
        *,
        doc_freqs: list[dict[str, int]],
        doc_len: np.ndarray,
        idf: dict[str, float],
        avgdl: float,
        k1: float,
        b: float,
        epsilon: float,
    ):
        self.idf = idf
        self.avgdl = float(avgdl)
        self.k1 = float(k1)
        self.b = float(b)
        self.epsilon = float(epsilon)
        self.corpus_size = len(doc_freqs)
        self.doc_len = np.asarray(doc_len, dtype=np.float32)
        self._norm = self.k1 * (1.0 - self.b + self.b * self.doc_len / self.avgdl)

        # ---- Build vocab + sparse TF matrix (CSC for fast column slicing) ----
        print("  Converting BM25 doc_freqs to sparse matrix ...")
        t0 = time.time()
        vocab: dict[str, int] = {}
        rows, cols, data = [], [], []
        for doc_idx, doc in enumerate(doc_freqs):
            for token, freq in doc.items():
                col = vocab.get(token)
                if col is None:
                    col = len(vocab)
                    vocab[token] = col
                rows.append(doc_idx)
                cols.append(col)
                data.append(freq)

        self._vocab = vocab
        self._tf_csc = sp_sparse.csc_matrix(
            (np.array(data, dtype=np.float32),
             (np.array(rows, dtype=np.int32),
              np.array(cols, dtype=np.int32))),
            shape=(self.corpus_size, len(vocab)),
        )
        # Free the original dicts -- sparse matrix is the source of truth now
        del doc_freqs, rows, cols, data
        elapsed = time.time() - t0
        nnz = self._tf_csc.nnz
        n_terms = len(vocab)
        print(f"    Sparse matrix: {self.corpus_size:,} docs x {n_terms:,} terms "
              f"({nnz:,} non-zero) in {elapsed:.1f}s")

    def get_scores(self, query):
        """Score all docs for a tokenised query. Fully vectorised."""
        # Collect query tokens that exist in vocab
        q_indices = []
        q_idfs = []
        for q in query:
            col = self._vocab.get(q)
            if col is None:
                continue
            idf_val = self.idf.get(q)
            if not idf_val:
                continue
            q_indices.append(col)
            q_idfs.append(idf_val)

        if not q_indices:
            return np.zeros(self.corpus_size, dtype=np.float32)

        # Extract TF sub-matrix for all query tokens at once: (n_docs, n_query_tokens)
        tf_sub = self._tf_csc[:, q_indices].toarray()   # dense, float32
        idf_vec = np.array(q_idfs, dtype=np.float32)    # (n_query_tokens,)

        # BM25 Okapi formula -- fully vectorised across docs AND tokens
        # score = sum_over_tokens[ idf(q) * tf(q,d) * (k1+1) / (tf(q,d) + norm(d)) ]
        numerator = tf_sub * (self.k1 + 1.0)
        denominator = tf_sub + self._norm[:, np.newaxis]
        return (numerator / denominator * idf_vec).sum(axis=1).astype(np.float32)

    def get_batch_scores(self, query, doc_ids):
        """Score a subset of docs. Used by continuous BM25 scoring."""
        doc_ids = list(doc_ids)
        q_indices = []
        q_idfs = []
        for q in query:
            col = self._vocab.get(q)
            if col is None:
                continue
            idf_val = self.idf.get(q)
            if not idf_val:
                continue
            q_indices.append(col)
            q_idfs.append(idf_val)

        if not q_indices:
            return [0.0] * len(doc_ids)

        tf_sub = self._tf_csc[doc_ids][:, q_indices].toarray()
        idf_vec = np.array(q_idfs, dtype=np.float32)
        norm = self._norm[doc_ids]
        numerator = tf_sub * (self.k1 + 1.0)
        denominator = tf_sub + norm[:, np.newaxis]
        scores = (numerator / denominator * idf_vec).sum(axis=1)
        return scores.tolist()



def load_bm25_artifact(index_path: Path):
    index_path = Path(index_path)
    with open(index_path, "rb") as f:
        obj = pickle.load(f)

    if not isinstance(obj, dict) or obj.get("format") != ARTIFACT_FORMAT:
        return obj

    artifact_dir = index_path.parent / obj["artifact_dir"]
    # Handle nested directory structure (e.g. Kaggle datasets)
    # If doc_len.npy isn't here, check one level deeper (same-named subdir)
    if not (artifact_dir / obj["doc_len_file"]).exists():
        for sub in artifact_dir.iterdir():
            if sub.is_dir() and (sub / obj["doc_len_file"]).exists():
                artifact_dir = sub
                break


    doc_len = np.load(artifact_dir / obj["doc_len_file"], allow_pickle=False)
    with open(artifact_dir / obj["idf_file"], "rb") as f:
        idf = pickle.load(f)

    doc_freqs: list[dict[str, int]] = []
    for chunk_idx in tqdm(
        range(obj["n_chunks"]),
        total=obj["n_chunks"],
        desc="  Loading BM25 chunks",
        ncols=80,
        ascii=True,
    ):
        chunk_path = artifact_dir / f"{obj['doc_freq_prefix']}{chunk_idx:05d}{obj['doc_freq_suffix']}"
        with open(chunk_path, "rb") as f:
            chunk = pickle.load(f)
        doc_freqs.extend(chunk)

    return ChunkedBM25Okapi(
        doc_freqs=doc_freqs,
        doc_len=doc_len,
        idf=idf,
        avgdl=obj["avgdl"],
        k1=obj["k1"],
        b=obj["b"],
        epsilon=obj["epsilon"],
    )

In [5]:
# =================================================================
# UTILS: scoring/confidence
# =================================================================

DEFAULT_WEIGHTS = {
    "explicit_from_query":    0.40,
    "bm25_top10":             0.15,   # binary: in BM25 top-10 yes/no
    "bm25_initial":           0.10,
    "citation_graph":         0.10,
    "llm_reranker_tier3":     0.25,
    "llm_reranker":           0.15,
    "llm_direct_gen":         0.05,
    "llm_stage1":             0.05,
    "llm_stage1_procedural":  0.05,
    "co_citation":            0.03,
}

# Additional continuous BM25 gradient (on top of binary signals)
# Lower weight than before — this adds ranking within same-signal groups
# without inflating all scores and pushing the threshold too high.
BM25_CONTINUOUS_WEIGHT = 0.10

# Cross-encoder weight (Strategy C): continuous P(yes) score × this weight
# replaces binary llm_reranker/tier3 signals. Validated at 0.18 on all 10
# val queries: 0.5400 → 0.6520 macro-F1.
CROSS_ENCODER_WEIGHT = 0.18

# Signals that count as "independent" for the multi-signal bonus
_INDEPENDENT_SIGNALS = {
    "explicit_from_query", "bm25_top10", "bm25_initial",
    "citation_graph", "llm_reranker", "llm_reranker_tier3",
    "llm_direct_gen", "llm_stage1",
}

MULTI_SIGNAL_BONUS = 0.10  # added when 3+ independent signals agree


def score_citation(cite: str, sources: dict[str, set[str]],
                   weights: dict[str, float] | None = None,
                   bm25_scores: dict[str, float] | None = None) -> float:
    """
    Compute composite confidence score for a single citation.

    Scoring approach: additive weights + continuous BM25 score + graduated
    multi-signal bonus. The key discriminators are:
      1. NUMBER of independent signals (gold avg 3-4, noise avg 1-2)
      2. Continuous BM25 relevance score (creates ranking gradient within
         same-signal-count groups — this is what separates 2-signal gold
         from 2-signal noise)

    Args:
        cite:           Citation string.
        sources:        Dict mapping source_name -> set of citations from that source.
        weights:        Override default weights.
        bm25_scores:    Dict mapping citation -> normalized BM25 score (0-1).
                        If provided, replaces binary bm25_top10 with continuous signal.

    Returns:
        Float score in [0, 1].
    """
    w = weights if weights is not None else DEFAULT_WEIGHTS

    score = 0.0
    active_signals = set()

    for source_name, weight in w.items():
        if cite in sources.get(source_name, set()):
            # If citation has tier3, don't also add the base llm_reranker weight
            if source_name == "llm_reranker" and cite in sources.get("llm_reranker_tier3", set()):
                continue  # tier3 supersedes base reranker
            score += weight
            active_signals.add(source_name)

    # Also add MAS agent signals as bm25_initial weight if they contributed
    for mas_source in ["mas_rewrite", "mas_supplement", "mas_decompose",
                       "mas_supportive", "mas_crossref"]:
        if cite in sources.get(mas_source, set()):
            if "bm25_initial" not in active_signals:
                score += w.get("bm25_initial", 0.10)
                active_signals.add("bm25_initial")
            break

    # Continuous BM25 score: normalized per-query BM25 relevance (0.0 to 0.20)
    # This creates the ranking gradient that binary signals can't provide.
    # A gold citation at BM25 rank #2 gets ~0.18, while noise at rank #40 gets ~0.05.
    if bm25_scores is not None:
        bm25_norm = bm25_scores.get(cite, 0.0)
        if bm25_norm > 0:
            score += bm25_norm * BM25_CONTINUOUS_WEIGHT
            active_signals.add("bm25_hit")

    # Graduated multi-signal bonus: the more independent signals agree,
    # the more likely this is a true gold citation.
    independent_count = len(active_signals & _INDEPENDENT_SIGNALS)
    if independent_count >= 4:
        score += 0.20  # very high confidence
    elif independent_count >= 3:
        score += 0.10  # high confidence
    elif independent_count >= 2:
        score += 0.05  # moderate confidence

    return min(score, 1.0)


def score_all_citations(
    all_citations: list[str],
    sources: dict[str, set[str]],
    weights: dict[str, float] | None = None,
    bm25_scores: dict[str, float] | None = None,
) -> list[tuple[str, float]]:
    """
    Score all citations and return sorted (citation, score) list.
    """
    scored = [
        (cite, score_citation(cite, sources, weights, bm25_scores=bm25_scores))
        for cite in all_citations
    ]
    return sorted(scored, key=lambda x: -x[1])


def compute_f1(predicted: list[str], gold: set[str]) -> float:
    """Compute F1 for a single query (case-insensitive matching)."""
    if not predicted and not gold:
        return 1.0
    if not predicted or not gold:
        return 0.0
    gold_lower = {c.lower() for c in gold}
    pred_lower = {c.lower() for c in predicted}
    tp = len(pred_lower & gold_lower)
    p = tp / len(pred_lower)
    r = tp / len(gold_lower)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def optimize_threshold(
    scored_predictions: dict[str, list[tuple[str, float]]],
    val_gold: dict[str, set[str]],
    threshold_range: tuple[float, float, float] = (0.05, 0.95, 0.01),
) -> tuple[float, float]:
    """
    Sweep thresholds on val set to find the one maximizing macro-F1.
    """
    lo, hi, step = threshold_range
    thresholds = np.arange(lo, hi + step, step)

    best_f1, best_thresh = 0.0, lo

    for thresh in thresholds:
        f1s = []
        for qid, scored in scored_predictions.items():
            preds = [c for c, s in scored if s >= thresh]
            gold = val_gold.get(qid, set())
            if gold:
                f1s.append(compute_f1(preds, gold))

        macro = sum(f1s) / len(f1s) if f1s else 0.0
        if macro > best_f1:
            best_f1 = macro
            best_thresh = float(thresh)

    return best_thresh, best_f1

In [6]:
# =================================================================
# UTILS: retrieval/explicit_citations
# =================================================================

# Common alternate abbreviations seen in multilingual Swiss citations.
# We normalize these to the corpus' canonical abbreviations so downstream
# verification and retrieval can keep them instead of dropping them.
_ALT_ABBREV_TO_CANON = {
    "CC": "ZGB",
    "CO": "OR",
    "LP": "SchKG",
    "CPC": "ZPO",
    "LDIP": "IPRG",
    "LFors": "GestG",
    "CPP": "StPO",
    "CP": "StGB",
    "LTF": "BGG",
    "PA": "VwVG",
}

_SINGLE_STAT_PATTERN = re.compile(
    r"Art\.\s+"
    r"(\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?)"
    r"\s+(?:of\s+the\s+)?"
    r"([A-Z][A-Za-z0-9]{1,14})",
    re.UNICODE,
)

_MULTI_STAT_PATTERN = re.compile(
    r"Art\.\s+"
    r"("
    r"\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?"
    r"(?:\s*(?:,|and|und|or|oder)\s*"
    r"\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?)+"
    r")"
    r"\s+(?:of\s+the\s+)?"
    r"([A-Z][A-Za-z0-9]{1,14})",
    re.UNICODE,
)

_ARTICLE_PART_SPLIT = re.compile(r"\s*(?:,|and|und|or|oder)\s*", re.UNICODE)


def _canon_law_abbrev(abbrev: str) -> str:
    return _ALT_ABBREV_TO_CANON.get(abbrev, abbrev)


def _norm_statute_ref(article_part: str, law_abbrev: str) -> str:
    article_part = re.sub(r"\s+", " ", article_part.strip())
    law_abbrev = _canon_law_abbrev(law_abbrev.strip())
    return f"Art. {article_part} {law_abbrev}".strip()


def extract_explicit_citations(query_text: str) -> set[str]:
    """
    Extract all citation strings from query text using regex patterns.

    Returns a set of citation strings derived from literal references in the
    query text. We lightly normalize whitespace and common multilingual law
    abbreviations (e.g. CC -> ZGB, CO -> OR), and expand shared-law patterns
    such as "Art. 38 and 39 CO" into individual citations.
    """
    citations = set()

    # ── Shared-law statutory refs: Art. 38 and 39 CO -> 2 citations ────────
    for m in _MULTI_STAT_PATTERN.finditer(query_text):
        raw_articles, raw_law = m.group(1), m.group(2)
        for art_part in _ARTICLE_PART_SPLIT.split(raw_articles):
            art_part = art_part.strip()
            if art_part:
                citations.add(_norm_statute_ref(art_part, raw_law))

    # ── Statutory: Art. X [Abs. Y [lit. z]] LAW ──────────────────────────
    # Matches: Art. 221 Abs. 1 lit. b StPO, Art. 44 ATSG, Art. 125 of the CC
    for m in _SINGLE_STAT_PATTERN.finditer(query_text):
        citations.add(_norm_statute_ref(m.group(1), m.group(2)))

    # ── BGE decisions: BGE XXX II/III/IV/V YYY E. Z.Z ───────────────────
    # Matches: BGE 137 IV 122 E. 6.2
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"BGE\s+\d+\s+[IVX]+\s+\d+\s+E\.\s+[\d.]+",
            query_text,
        )
    )

    # ── BGE decisions without Erwägung: BGE 137 IV 122 ──────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"BGE\s+\d+\s+[IVX]+\s+\d+",
            query_text,
        )
    )

    # ── Numbered cases: 1B_210/2023 E. 4.1 ──────────────────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"\d+[A-Z]_\d+/\d{4}\s+E\.\s+[\d.]+",
            query_text,
        )
    )

    # ── Numbered cases without Erwägung: 1B_210/2023 ────────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"\d+[A-Z]_\d+/\d{4}",
            query_text,
        )
    )

    return citations

In [7]:
# =================================================================
# UTILS: agent/prompts/mas_prompts
# =================================================================

# ─── PLANNER AGENT ──────────────────────────────────────────────────────────
# Decides which agent to invoke next, or whether to terminate.
PLANNER_PROMPT = """You are a Swiss law retrieval planner. You coordinate a team of
specialist agents to find ALL relevant legal provisions for a given scenario.

You have already retrieved some candidate provisions. Your job is to decide
what to do next to improve recall (find more relevant provisions).

Available agents:
1. REWRITE — rewrites colloquial English into precise German legal terminology
2. SUPPLEMENT — makes implicit legal conditions explicit (thresholds, standing, procedural)
3. DECOMPOSE — splits complex multi-issue queries into focused sub-queries
4. SUPPORTIVE — generates queries for procedural, interpretive, and auxiliary provisions
5. CROSSREF — generates queries for constitutional and cross-referenced articles

Based on the current state, decide:
- Which agent would most likely find NEW relevant provisions not yet in the pool?
- Or should we TERMINATE because further searching is unlikely to help?

Output JSON only:
{
  "reasoning": "Brief explanation of what's missing and why this agent helps",
  "action": "REWRITE" | "SUPPLEMENT" | "DECOMPOSE" | "SUPPORTIVE" | "CROSSREF" | "TERMINATE"
}
"""

# ─── SINGLE-ELEMENT REWRITE AGENT ──────────────────────────────────────────
# Rewrites colloquial EN terms into precise DE legal terminology for BM25.
REWRITE_PROMPT = """You are a Swiss law terminology expert. Given an English legal scenario,
generate precise German search queries using the exact legal terms that would appear
in Swiss statutory articles and court decisions.

CRITICAL: Your queries will be used for BM25 keyword search, so use the EXACT German
legal terms (Fachbegriffe) that appear in the law text, not paraphrases.

Examples of good translations:
- "pre-trial detention" → "Untersuchungshaft"
- "collusion risk" → "Kollusionsgefahr"
- "disability insurance" → "Invalidenversicherung"
- "medical expert opinion" → "medizinisches Gutachten"
- "right to be heard" → "rechtliches Gehör"
- "proportionality" → "Verhältnismässigkeit"

Generate 3-5 focused German search queries. Each query should:
- Target a SPECIFIC legal concept from the scenario
- Use 2-4 precise German legal terms
- Include relevant law abbreviations (StPO, ATSG, BGG, etc.)

Output JSON:
{
  "queries": ["Untersuchungshaft Kollusionsgefahr StPO", "Haftprüfung Verhältnismässigkeit", ...]
}
"""

# ─── SUPPLEMENTARY-ELEMENT AGENT ───────────────────────────────────────────
# Makes implicit legal conditions explicit.
SUPPLEMENT_PROMPT = """You are a Swiss law expert who identifies IMPLICIT legal requirements.

Given a legal scenario, identify conditions, thresholds, and requirements that are
implied but not explicitly stated. For each, generate a German search query.

Think about:
- Standing requirements (Legitimation, Beschwerdebefugnis)
- Jurisdictional prerequisites (örtliche/sachliche Zuständigkeit)
- Time limits and deadlines (Fristen, Verwirkung)
- Burden of proof rules (Beweislast)
- Exhaustion of remedies requirements
- Threshold amounts or severity levels
- Formal requirements (Schriftlichkeit, Begründungspflicht)

Generate 2-4 German search queries targeting these implicit requirements.

Output JSON:
{
  "implicit_issues": [
    {"issue": "...", "query": "Beschwerdelegitimation BGG Strafverfahren"}
  ]
}
"""

# ─── MULTI-ELEMENT DECOMPOSITION AGENT ─────────────────────────────────────
# Splits complex queries into focused sub-queries.
DECOMPOSE_PROMPT = """You are a Swiss law analyst who breaks down complex legal scenarios
into individual legal questions (Rechtsfragen).

A typical exam scenario may involve 3-8 distinct legal issues, each requiring
different statutory articles and case law. Your job is to identify each issue
and create a focused German search query for it.

For each sub-issue, generate ONE precise German search query targeting:
- The specific legal provision(s) that govern this issue
- The relevant law abbreviation

Example decomposition:
  Scenario about wrongful arrest + appeal + costs
  → "Untersuchungshaft Voraussetzungen Art 221 StPO"
  → "Beschwerde gegen Haftanordnung Art 222 StPO"
  → "Kostenregelung Strafverfahren Art 428 StPO"
  → "Beschwerdefrist BGG Strafverfahren"

Generate 3-6 sub-queries covering ALL distinct legal issues.

Output JSON:
{
  "sub_issues": [
    {"issue": "...", "query": "..."}
  ]
}
"""

# ─── SUPPORTIVE-LAW AGENT ─────────────────────────────────────────────────
# Targets procedural, interpretive, and auxiliary provisions.
SUPPORTIVE_PROMPT = """You are a Swiss procedural law specialist. Given a legal scenario
and the substantive provisions already found, identify SUPPORTING provisions that
a well-prepared exam candidate must also cite.

These are often missed but always required:
1. PROCEDURAL provisions:
   - Which court has jurisdiction? (BGG 72-89 for Federal Tribunal)
   - What is the appeal mechanism? (Beschwerde in Strafsachen, Zivilsachen, öff. Recht)
   - Filing deadlines (BGG 100 Abs. 1: 30 Tage)
   - Cost allocation (BGG 65-68, ZPO 106-107)
   - Legal aid (unentgeltliche Rechtspflege, BGG 64)

2. INTERPRETIVE provisions:
   - Definition articles (e.g., StGB 110 Definitionen)
   - General clauses (OR 2 Treu und Glauben, ZGB 4 Richterliches Ermessen)

3. AUXILIARY provisions:
   - Transitional provisions (Übergangsrecht)
   - Scope of application articles

Generate 2-4 German search queries for supporting provisions.

Output JSON:
{
  "supporting_queries": [
    {"type": "procedural", "query": "Beschwerde Bundesgericht Strafsachen BGG 78"},
    {"type": "cost", "query": "Gerichtskosten Parteientschädigung BGG 65 66"}
  ]
}
"""

# ─── CROSS-REFERENCE AGENT ────────────────────────────────────────────────
# NEW agent not in LegalMALR — targets constitutional and cross-referenced articles.
CROSSREF_PROMPT = """You are a Swiss constitutional and cross-reference law expert.
Given a legal scenario, identify:

1. CONSTITUTIONAL provisions (BV articles) that underlie the specific rules:
   - Art. 9 BV (Willkürverbot / prohibition of arbitrariness)
   - Art. 29 BV (Verfahrensgarantien / procedural guarantees)
   - Art. 32 BV (Unschuldsvermutung / presumption of innocence)
   - Art. 10 BV (Recht auf persönliche Freiheit)
   - Art. 13 BV (Schutz der Privatsphäre)
   - Art. 36 BV (Einschränkung von Grundrechten)

2. GENERAL PROVISIONS that apply alongside specific rules:
   - ATSG provisions alongside IVG/UVG/AVIG
   - OR general part alongside specific contracts
   - ZGB Einleitungsartikel (ZGB 1-10)

3. CROSS-REFERENCED articles:
   - If Art. X refers to Art. Y, both should be cited
   - "sinngemäss anwendbar" references

Generate 2-4 German search queries for these cross-references.

Output JSON:
{
  "crossref_queries": [
    {"type": "constitutional", "query": "Willkürverbot Art 9 BV Grundrechtseingriff"},
    {"type": "general", "query": "ATSG Allgemeiner Teil Invalidenversicherung"}
  ]
}
"""

# Map agent names to their prompts
AGENT_PROMPTS = {
    "PLANNER":    PLANNER_PROMPT,
    "REWRITE":    REWRITE_PROMPT,
    "SUPPLEMENT":  SUPPLEMENT_PROMPT,
    "DECOMPOSE":  DECOMPOSE_PROMPT,
    "SUPPORTIVE": SUPPORTIVE_PROMPT,
    "CROSSREF":   CROSSREF_PROMPT,
}

In [8]:
# =================================================================
# UTILS: retrieval/citation_graph (runtime interface only)
# =================================================================

class CitationGraph:
    """
    Runtime interface: direct citation graph + co-citation graph.
    All loaded from a single citation_graph.pkl.
    """

    def __init__(self, graph_pkl: Path = CITATION_GRAPH_PKL):
        with open(graph_pkl, "rb") as f:
            data = pickle.load(f)
        self.article_to_cases    = data["article_to_cases"]
        self.case_to_articles    = data["case_to_articles"]
        self.case_to_cases       = data.get("case_to_cases", {})
        self.article_corpus_freq = data.get("article_corpus_freq", {})
        self.cocitation          = data.get("cocitation", {})
        self.citation_doc_freq   = data.get("citation_doc_freq", {})

    # ── Direct citation lookups ───────────────────────────────────────────────

    def get_citing_cases(self, art_citation: str, top_n: int = 50) -> list[str]:
        """Article → top citing cases. Fuzzy: strips Abs./lit. if exact fails."""
        key   = art_citation.strip()
        cases = self.article_to_cases.get(key, {})
        if not cases:
            parent = re.sub(r"\s+Abs\.\s+\d+", "", key)
            parent = re.sub(r"\s+lit\.\s+\w+", "", parent)
            if parent != key:
                cases = self.article_to_cases.get(parent, {})
        return [c for c, _ in sorted(cases.items(), key=lambda x: -x[1])[:top_n]] if cases else []

    def get_cited_articles(self, case_citation: str) -> list[str]:
        """Case → articles it cites. Fuzzy: strips E. suffix if exact fails."""
        key    = case_citation.strip()
        result = self.case_to_articles.get(key, [])
        if not result:
            base = re.sub(r"\s+E\.\s+[\d.]+$", "", key)
            if base != key:
                result = self.case_to_articles.get(base, [])
        return result

    def get_related_cases(self, case_citation: str, top_n: int = 20) -> list[str]:
        """Case → other cases it references (case-to-case graph)."""
        key    = case_citation.strip()
        others = self.case_to_cases.get(key, {})
        if not others:
            base = re.sub(r"\s+E\.\s+[\d.]+$", "", key)
            if base != key:
                others = self.case_to_cases.get(base, {})
        return [c for c, _ in sorted(others.items(), key=lambda x: -x[1])[:top_n]]

    def get_article_importance(self, art_citation: str) -> int:
        return self.article_corpus_freq.get(art_citation.strip(), 0)

    # ── Co-citation lookups ───────────────────────────────────────────────────

    def get_co_cited(self, citation: str, top_n: int = 10,
                     min_count: int = 5) -> list[tuple[str, int]]:
        """
        Return top citations that co-occur with this one in court texts.
        Returns [(companion, count), ...] sorted descending by count.
        """
        key       = citation.strip()
        neighbors = self.cocitation.get(key, {})
        filtered  = [(c, n) for c, n in neighbors.items() if n >= min_count]
        return sorted(filtered, key=lambda x: -x[1])[:top_n]

    def get_co_cited_pmi(self, citation: str, top_n: int = 10,
                         total_rows: int = 2_476_315) -> list[tuple[str, float]]:
        """
        Co-citations ranked by PMI — emphasises pairs that are specifically
        associated, not just ubiquitous (e.g. Art. 66 BGG appears everywhere
        but has low PMI with rare articles).
        PMI(A,B) = log( P(A,B) / P(A) / P(B) )
        """
        key      = citation.strip()
        freq_a   = max(self.citation_doc_freq.get(key, 1), 1)
        neighbors = self.cocitation.get(key, {})
        scored   = []
        for companion, co_count in neighbors.items():
            freq_b = max(self.citation_doc_freq.get(companion, 1), 1)
            pmi    = math.log(max(co_count * total_rows / (freq_a * freq_b), 1e-10))
            scored.append((companion, pmi))
        return sorted(scored, key=lambda x: -x[1])[:top_n]

    def expand_pool_cocitation(self, candidate_pool: list[str],
                               top_n_per_cite: int = 5,
                               min_count: int = 10) -> set[str]:
        """
        Expand a candidate pool using co-citation.
        For each citation, add its top co-cited companions.
        """
        expanded = set(candidate_pool)
        for cite in candidate_pool:
            for companion, _ in self.get_co_cited(cite, top_n=top_n_per_cite,
                                                   min_count=min_count):
                expanded.add(companion)
        return expanded


# =============================================================================


In [9]:
# =================================================================
# UTILS: indexing/build_lookup_tables
# =================================================================


_ART_RE = re.compile(r"^Art\.\s+(\d+)(?:\s+Abs\.\s+(\d+))?(?:\s+lit\.\s+(\w+))?\s+(.+)$")


def normalize_citation(c: str) -> str:
    c = re.sub(r"\s+", " ", str(c or "").strip())
    c = re.sub(r"Art\.\s*",  "Art. ",  c)
    c = re.sub(r"Abs\.\s*",  "Abs. ",  c)
    c = re.sub(r"lit\.\s*",  "lit. ",  c)
    c = re.sub(r"Ziff\.\s*", "Ziff. ", c)
    return c.strip()



class CitationLookup:
    """Runtime interface for citation lookup tables."""

    def __init__(self, lookup_pkl: Path = LOOKUP_PKL):
        with open(lookup_pkl, "rb") as f:
            data = pickle.load(f)
        self.citation_set      = data["citation_set"]
        self.lower_map         = data["citation_lower_map"]
        self.article_to_abs    = data["article_to_abs"]
        self.abs_to_article    = data["abs_to_article"]
        self.sr_to_abbrev      = data["sr_to_abbrev"]
        self.abbrev_to_sr      = data["abbrev_to_sr"]

    def exists(self, cite: str) -> bool:
        return cite in self.citation_set or cite.lower() in self.lower_map

    def normalize(self, cite: str) -> str | None:
        """Return canonical form of citation, or None if not found."""
        if cite in self.citation_set:
            return cite
        return self.lower_map.get(cite.lower())

    def expand_to_abs(self, article_cite: str) -> list[str]:
        """Art. 975 ZGB -> [Art. 975 Abs. 1 ZGB, Art. 975 Abs. 2 ZGB, ...]"""
        return self.article_to_abs.get(article_cite, [article_cite])

    def collapse_to_article(self, abs_cite: str) -> str:
        """Art. 975 Abs. 1 ZGB -> Art. 975 ZGB"""
        return self.abs_to_article.get(abs_cite, abs_cite)

In [10]:
# =================================================================
# UTILS: retrieval/sparse_retriever
# =================================================================

_TOKEN_RE = re.compile(r"[^\w\d]+", re.UNICODE)


def tokenise(text: str) -> list[str]:
    """Tokenize text for BM25 queries. Keeps short tokens if numeric (article numbers)."""
    if not text:
        return []
    return [t for t in _TOKEN_RE.split(text.lower()) if len(t) >= 2 or t.isdigit()]


class SparseRetriever:
    """
    Wraps the combined BM25 index (law + court).
    Exposes search_statutory(), search_caselaw(), search_combined().
    """

    def __init__(self):
        print("  Loading combined BM25 index ...")
        self._bm25 = load_bm25_artifact(STATUTORY_BM25_PKL)

        with open(STATUTORY_BM25_IDS, "rb") as f:
            meta = pickle.load(f)

        self._ids:     list[str] = meta["citation_canon"]
        self._n_law:   int       = meta["n_law"]
        self._n_court: int       = meta["n_court"]
        print(f"    {self._n_law:,} law  +  {self._n_court:,} court  =  {len(self._ids):,} total")

        # Build id->index mapping for fast rank lookup
        self._id_to_idx: dict[str, int] = {c: i for i, c in enumerate(self._ids)}

        # PMI token->law map for query enhancement
        self._token_law_freq: dict = {}
        self._token_total:    dict = {}
        if TOKEN_LAW_FREQ.exists():
            with open(TOKEN_LAW_FREQ, encoding="utf-8") as f:
                raw = json.load(f)
            self._token_law_freq = {t: v for t, v in raw.items()}
            self._token_total    = {t: sum(v.values()) for t, v in self._token_law_freq.items()}
            print(f"    PMI token->law map: {len(self._token_law_freq):,} tokens")

    # ------------------------------------------------------------------
    # Public search API
    # ------------------------------------------------------------------

    def search_statutory(self, queries: list[str], top_k: int = 100) -> list[tuple[str, float]]:
        """Search law articles only. Returns [(citation, score), ...]."""
        return self._search(queries, top_k, law_only=True)

    def search_caselaw(self, queries: list[str], top_k: int = 100) -> list[tuple[str, float]]:
        """Search court considerations only. Returns [(citation, score), ...]."""
        return self._search(queries, top_k, court_only=True)

    def search_combined(self, queries: list[str], top_k: int = 200) -> list[tuple[str, float]]:
        """Search across the full combined index (law + court)."""
        return self._search(queries, top_k)

    def get_bm25_rank(self, query_tokens: list[str], citation: str) -> int:
        """Get the BM25 rank of a specific citation for given query tokens. -1 if not found."""
        idx = self._id_to_idx.get(citation, -1)
        if idx < 0:
            return -1
        scores = self._bm25.get_scores(query_tokens)
        rank = (scores > scores[idx]).sum() + 1
        return int(rank)

    # ------------------------------------------------------------------
    # Internal
    # ------------------------------------------------------------------

    def _enhance_tokens(self, tokens: list[str], law_boost: int = 10,
                        top_n_laws: int = 5) -> list[str]:
        """Append predicted law abbreviations (PMI) to query tokens."""
        if not self._token_law_freq:
            return tokens

        scores: dict[str, float] = {}
        for t in set(tokens):
            if t not in self._token_law_freq:
                continue
            idf = self._bm25.idf.get(t, 0.0)
            if idf < 1.0:
                continue
            total = self._token_total.get(t, 1)
            for ab, cnt in self._token_law_freq[t].items():
                scores[ab] = scores.get(ab, 0.0) + (cnt / total) * idf

        predicted = sorted(scores.items(), key=lambda x: -x[1])[:top_n_laws]
        enhanced = list(dict.fromkeys(tokens))
        for ab, _ in predicted:
            enhanced.extend([ab] * law_boost)
        return enhanced

    def _search(self, queries: list[str], top_k: int,
                law_only: bool = False, court_only: bool = False) -> list[tuple[str, float]]:
        """Core search: runs each query, takes max score per doc across queries."""
        scores: dict[str, float] = {}

        for q in queries:
            toks = tokenise(q)
            if not toks:
                continue
            toks = self._enhance_tokens(toks)

            raw = self._bm25.get_scores(toks)

            # Restrict to relevant slice
            if law_only:
                raw[self._n_law:] = 0.0
            elif court_only:
                raw[:self._n_law] = 0.0

            k = min(top_k, int((raw > 0).sum()), len(raw))
            if k == 0:
                continue
            top_idx = np.argpartition(raw, -k)[-k:]
            for i in top_idx:
                if raw[i] <= 0:
                    continue
                c = self._ids[i]
                scores[c] = max(scores.get(c, 0.0), float(raw[i]))

        return sorted(scores.items(), key=lambda x: -x[1])[:top_k]

In [11]:
# =================================================================
# UTILS: retrieval/graph_retriever
# =================================================================

class GraphRetriever:
    def __init__(self, graph_pkl: Path = CITATION_GRAPH_PKL):
        print("  Loading citation graph ...")
        self.graph = CitationGraph(graph_pkl)
        print(f"    {len(self.graph.article_to_cases):,} articles with case links")
        print(f"    {len(self.graph.case_to_articles):,} cases with article links")

    def articles_to_cases(
        self,
        article_citations: list[str],
        top_k_per_article: int = 20,
    ) -> list[tuple[str, float]]:
        """
        Given statutory articles, return citing case law.
        Score = sum of (1/rank) across articles (RRF-style).
        """
        scores: dict[str, float] = {}
        for art in article_citations:
            cases = self.graph.get_citing_cases(art, top_n=top_k_per_article)
            for rank, case in enumerate(cases, 1):
                scores[case] = scores.get(case, 0.0) + 1.0 / (60 + rank)
        return sorted(scores.items(), key=lambda x: -x[1])

    def cases_to_articles(self, case_citations: list[str]) -> list[str]:
        """Given case citations, return statutory articles they cite."""
        found: set[str] = set()
        for case in case_citations:
            arts = self.graph.get_cited_articles(case)
            found.update(arts)
        return list(found)

    def bidirectional_expand(
        self,
        seed_articles: list[str],
        seed_cases: list[str],
        depth: int = 1,
    ) -> tuple[list[tuple[str, float]], list[str]]:
        """
        Expand seeds bidirectionally up to `depth` hops.
        Returns (expanded_cases_with_scores, expanded_articles).
        """
        all_cases:    dict[str, float] = {}
        all_articles: set[str]         = set(seed_articles)

        current_articles = list(seed_articles)
        current_cases    = list(seed_cases)

        for _ in range(depth):
            new_cases = self.articles_to_cases(current_articles)
            for c, s in new_cases:
                all_cases[c] = max(all_cases.get(c, 0.0), s)

            new_articles = self.cases_to_articles(
                current_cases + [c for c, _ in new_cases]
            )
            all_articles.update(new_articles)

            current_articles = new_articles
            current_cases    = [c for c, _ in new_cases[:20]]

        return (
            sorted(all_cases.items(), key=lambda x: -x[1]),
            list(all_articles),
        )

In [12]:
# =================================================================
# UTILS: agent/llm_backend
# =================================================================

# Singleton model instance
_MODEL = None
_TOKENIZER = None


def get_model_and_tokenizer():
    """
    Lazy-load Qwen3-32B in full bf16 precision (no quantization).
    Returns (model, tokenizer). Cached after first call.

    VRAM usage: ~64 GB on H100 80GB.
    """
    global _MODEL, _TOKENIZER
    if _MODEL is not None:
        return _MODEL, _TOKENIZER

    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

    model_path = str(QWEN3_32B_PATH) if QWEN3_32B_PATH.exists() else QWEN3_32B_HF_ID
    print(f"  Loading Qwen3-32B (bf16, no quantization) from: {model_path}")

    _TOKENIZER = AutoTokenizer.from_pretrained(model_path)
    if _TOKENIZER.pad_token is None:
        _TOKENIZER.pad_token = _TOKENIZER.eos_token
        _TOKENIZER.padding_side = "left"

    _MODEL = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    _MODEL.eval()
    print(f"  Qwen3-32B loaded (bf16). VRAM: ~64GB")
    return _MODEL, _TOKENIZER


def generate(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 2048,
    temperature: float = 0.6,
    top_k: int = 20,
    top_p: float = 0.95,
    enable_thinking: bool = False,
) -> str:
    """
    Generate a response from Qwen3-32B.

    Args:
        system_prompt: System message (role definition).
        user_prompt:   User message (the actual query/task).
        max_new_tokens: Max generation length.
        temperature:   Sampling temperature (0.6 = focused but not greedy).
        top_k/top_p:   Nucleus sampling params.
        enable_thinking: If True, allows <think> blocks for chain-of-thought.
                        Stage 1 uses True, Stage 4 reranker uses False.

    Returns:
        Generated text string (with <think> blocks stripped if enable_thinking=False).
    """

    model, tokenizer = get_model_and_tokenizer()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        out[0][input_len:],
        skip_special_tokens=True,
    )

    # Explicitly free GPU tensors so VRAM doesn't fragment across calls.
    del inputs, out
    torch.cuda.empty_cache()

    return generated



def generate_batch(
    system_prompt: str,
    user_prompts: list[str],
    max_new_tokens: int = 2048,
    temperature: float = 0.6,
    top_k: int = 20,
    top_p: float = 0.95,
    enable_thinking: bool = False,
) -> list[str]:
    """
    Generate responses for multiple prompts in a single batched forward pass.
    Much faster than calling generate() in a loop on H100.

    Args:
        system_prompt: Shared system message for all prompts.
        user_prompts:  List of user messages (batch size = len(user_prompts)).

    Returns:
        List of generated text strings, one per input prompt.
    """
    if not user_prompts:
        return []

    model, tokenizer = get_model_and_tokenizer()

    # Build chat-templated texts for each prompt
    texts = []
    for user_prompt in user_prompts:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ]
        texts.append(tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        ))

    # Tokenize with left-padding for batch generation
    tokenizer.padding_side = "left"
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    input_lens = (inputs["attention_mask"] != 0).sum(dim=1)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode each sequence, stripping its own input portion
    results = []
    for i in range(len(user_prompts)):
        start = input_lens[i]
        generated = tokenizer.decode(
            outputs[i][start:],
            skip_special_tokens=True,
        )
        results.append(generated)

    del inputs, outputs
    torch.cuda.empty_cache()

    return results



def generate_batch_multi_system(
    system_prompts: list[str],
    user_prompts: list[str],
    max_new_tokens: int = 2048,
    temperature: float = 0.6,
    top_k: int = 20,
    top_p: float = 0.95,
    enable_thinking: bool = False,
) -> list[str]:
    """
    Generate responses for multiple prompts where each has its OWN system prompt.
    Used by stage 1 where BM25 probe context differs per query.
    """
    assert len(system_prompts) == len(user_prompts)
    if not user_prompts:
        return []

    model, tokenizer = get_model_and_tokenizer()

    texts = []
    for sys_p, user_p in zip(system_prompts, user_prompts):
        messages = [
            {"role": "system", "content": sys_p},
            {"role": "user",   "content": user_p},
        ]
        texts.append(tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        ))

    tokenizer.padding_side = "left"
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    input_lens = (inputs["attention_mask"] != 0).sum(dim=1)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
        )

    results = []
    for i in range(len(user_prompts)):
        start = input_lens[i]
        generated = tokenizer.decode(
            outputs[i][start:],
            skip_special_tokens=True,
        )
        results.append(generated)

    del inputs, outputs
    torch.cuda.empty_cache()

    return results


def generate_json_batch_multi_system(
    system_prompts: list[str],
    user_prompts: list[str],
    max_new_tokens: int = 2048,
    enable_thinking: bool = False,
) -> list[dict]:
    """
    Generate and parse JSON from multiple prompts, each with its own system prompt.
    Falls back to sequential on OOM.
    """
    try:
        raw_texts = generate_batch_multi_system(
            system_prompts, user_prompts,
            max_new_tokens=max_new_tokens,
            enable_thinking=enable_thinking,
        )
    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "CUDA" in str(e):
            print(f"  [OOM] Batch of {len(user_prompts)} failed -- falling back to sequential")
            torch.cuda.empty_cache()
            return [
                generate_json(sp, up, max_new_tokens, enable_thinking)
                for sp, up in zip(system_prompts, user_prompts)
            ]
        raise
    return [parse_json_response(raw) for raw in raw_texts]


def generate_json_batch(
    system_prompt: str,
    user_prompts: list[str],
    max_new_tokens: int = 2048,
    enable_thinking: bool = False,
) -> list[dict]:
    """
    Generate and parse JSON from multiple prompts in one batched call.
    Returns a list of dicts (one per prompt). Failed parses return {}.
    """
    try:
        raw_texts = generate_batch(
            system_prompt, user_prompts,
            max_new_tokens=max_new_tokens,
            enable_thinking=enable_thinking,
        )
    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "CUDA" in str(e):
            print(f"  [OOM] Batch of {len(user_prompts)} failed -- falling back to sequential")
            torch.cuda.empty_cache()
            return [generate_json(system_prompt, p, max_new_tokens, enable_thinking) for p in user_prompts]
        raise
    return [parse_json_response(raw) for raw in raw_texts]


def generate_json(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 2048,
    enable_thinking: bool = False,
) -> dict:
    """
    Generate and parse JSON from Qwen3-32B.
    Handles ```json ... ``` wrapping and malformed JSON gracefully.
    On CUDA OOM: clears cache and retries once with halved max_new_tokens.
    """
    try:
        raw = generate(system_prompt, user_prompt, max_new_tokens=max_new_tokens,
                       enable_thinking=enable_thinking)
    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "CUDA" in str(e):
            print(f"\n  [OOM] CUDA out of memory — clearing cache and retrying with {max_new_tokens // 2} tokens...")
            torch.cuda.empty_cache()
            try:
                raw = generate(system_prompt, user_prompt,
                               max_new_tokens=max_new_tokens // 2,
                               enable_thinking=enable_thinking)
            except RuntimeError as e2:
                print(f"  [OOM] Retry also failed: {e2}")
                torch.cuda.empty_cache()
                return {}
        else:
            raise
    return parse_json_response(raw)


def parse_json_response(raw_text: str) -> dict:
    """Extract JSON from LLM response, handling markdown wrapping."""
    raw_text = raw_text.strip()

    # Strip ```json ... ``` wrapping
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw_text, re.DOTALL)
    if m:
        raw_text = m.group(1)
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        # Fallback: find any JSON object
        m2 = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if m2:
            try:
                return json.loads(m2.group(0))
            except json.JSONDecodeError:
                pass
    return {}




In [13]:
# =================================================================
# UTILS: agent/verifier
# =================================================================

# Corpus uses UPPERCASE abbreviations but gold standard uses mixed-case.
# This map converts corpus forms to gold-compatible forms.
_ABBREV_CASING = {
    'STGB': 'StGB', 'STPO': 'StPO', 'STBOG': 'StBOG', 'JSTG': 'JStG',
    'JSTPO': 'JStPO', 'SCHKG': 'SchKG', 'BANKG': 'BankG', 'BANKV': 'BankV',
    'ASYLG': 'AsylG', 'VWVG': 'VwVG', 'VSTG': 'VStG', 'VSTV': 'VStV',
    'BETMG': 'BetmG', 'GSCHG': 'GSchG', 'GSCHV': 'GSchV', 'MSCHG': 'MSchG',
    'MSCHV': 'MSchV', 'STHG': 'StHG', 'PARLG': 'ParlG', 'SEBG': 'SebG',
    'FINFRAG': 'FinfraG', 'FINFRAV': 'FinfraV', 'CHEMG': 'ChemG',
    'FAMZG': 'FamZG', 'ELEG': 'EleG', 'GWG': 'GwG', 'EPG': 'EpG',
    'EPV': 'EpV', 'STAHIG': 'StAhiG', 'STROMVG': 'StromVG',
    'STROMVV': 'StromVV', 'VSTRR': 'VStrR', 'HREGV': 'HRegV',
    'GEOIG': 'GeoIG', 'WAG': 'WaG', 'PATG': 'PatG', 'PRSG': 'PrSG',
    'DESG': 'DesG', 'ARG': 'ArG', 'TWWV': 'TwwV', 'PRHG': 'PrHG',
}


def _fix_abbrev_casing(c: str) -> str:
    """Convert corpus UPPERCASE abbreviations to gold-compatible mixed-case."""
    if not c.startswith("Art."):
        return c
    parts = c.split()
    if len(parts) >= 3:
        last = parts[-1]
        fixed = _ABBREV_CASING.get(last, last)
        if fixed != last:
            parts[-1] = fixed
            return " ".join(parts)
    return c


def normalize_citation(c: str) -> str:
    """Normalize whitespace, abbreviation spacing, and casing in a citation string."""
    c = re.sub(r"\s+",    " ",     str(c or "").strip())
    c = re.sub(r"Art\.\s*",  "Art. ",  c)
    c = re.sub(r"Abs\.\s*",  "Abs. ",  c)
    c = re.sub(r"lit\.\s*",  "lit. ",  c)
    c = re.sub(r"Ziff\.\s*", "Ziff. ", c)
    c = re.sub(r"BGE\s+",    "BGE ",   c)
    c = re.sub(r"\bE\.\s*",  "E. ",    c)
    c = _fix_abbrev_casing(c.strip())
    return c.strip()


def is_law_citation(c: str) -> bool:
    return bool(re.match(r"^Art\.\s+\d", c.strip()))


def is_case_citation(c: str) -> bool:
    return bool(
        re.match(r"^BGE\s+\d+", c.strip()) or
        re.match(r"^\d+[A-Z]_\d+/\d{4}", c.strip())
    )


class Verifier:
    def __init__(self, lookup_pkl: Path = LOOKUP_PKL, caselaw_ids: set[str] | None = None):
        self.lookup = CitationLookup(lookup_pkl)
        self.caselaw_ids = caselaw_ids or set()

    def verify_and_normalize(
        self,
        citations: list[str],
        expand_to_abs: bool = True,
    ) -> tuple[list[str], list[str]]:
        """
        Normalize and verify a list of citations.

        Returns (verified, dropped):
          verified = citations that passed verification
          dropped  = citations that failed (hallucinations / format errors)
        """
        verified = []
        dropped  = []

        for raw in citations:
            c = normalize_citation(raw)
            if not c:
                continue

            if is_law_citation(c):
                canon = self.lookup.normalize(c)
                if canon:
                    verified.append(_fix_abbrev_casing(canon))
                elif expand_to_abs:
                    expanded = self.lookup.expand_to_abs(c)
                    if expanded and expanded != [c]:
                        verified.extend(_fix_abbrev_casing(e) for e in expanded)
                    else:
                        # Try stripping lit./Ziff. to find parent Abs. or base Art.
                        # E.g. "Art. 221 Abs. 1 lit. b StPO" -> try "Art. 221 Abs. 1 StPO"
                        #      then "Art. 221 StPO"
                        kept = False
                        m = re.match(r"(Art\.\s+\d+[a-z]?)\s+(Abs\.\s+\d+\s*)?(lit\.\s+\w+\s*)?(Ziff\.\s+\d+\s?)?(.*)", c)
                        if m:
                            law_part = m.group(5).strip() if m.group(5) else ""
                            # Try Abs. level first (strip lit./Ziff.)
                            if m.group(2):
                                abs_cite = f"{m.group(1)} {m.group(2).strip()} {law_part}".strip()
                                if self.lookup.normalize(abs_cite):
                                    verified.append(c)
                                    kept = True
                            # Try base article (strip everything)
                            if not kept:
                                base = f"{m.group(1)} {law_part}".strip()
                                if self.lookup.normalize(base):
                                    verified.append(c)
                                    kept = True
                        if not kept:
                            dropped.append(c)
                else:
                    dropped.append(c)

            elif is_case_citation(c):
                if not self.caselaw_ids or c in self.caselaw_ids:
                    verified.append(c)
                else:
                    # Lenient: keep if format looks valid (Swiss court citation patterns)
                    if re.match(r"^BGE\s+\d{2,3}\s+[IVX]+\s+\d+(\s+E\.\s+[\d\.]+)?(\s+S\.\s+\d+)?$", c):
                        verified.append(c)
                    elif re.match(r"^\d+[A-Z]_\d+/\d{4}(\s+\d{2}\.\d{2}\.\d{4})?(\s+E\.\s+[\d\.]+)?$", c):
                        verified.append(c)
                    else:
                        dropped.append(c)
            else:
                dropped.append(c)

        # Deduplicate preserving order
        seen = set()
        final = []
        for c in verified:
            if c not in seen:
                seen.add(c)
                final.append(c)

        return final, dropped

In [14]:
# =================================================================
# UTILS: stage5_verify_and_score
# =================================================================



def _compute_bm25_scores(
    qid: str,
    query_text: str,
    all_citations: list[str],
    stage1_analysis: dict,
    sparse: SparseRetriever,
    verbose: bool = False,
) -> dict[str, float]:
    """
    Compute per-query normalized BM25 scores for candidate citations ONLY.

    FAST approach:
      1. Combine all query texts into one merged token list (no PMI enhancement)
      2. Call bm25.get_scores() ONCE (single pass over 2.15M docs)
      3. Look up only the indices of our candidate citations
      4. Normalize to [0, 1]

    Skipping PMI enhancement (which adds 50+ tokens) makes this ~6x faster.
    Using a single combined call instead of per-query calls makes it ~5x faster.
    Net: ~30x faster than the naive approach.
    """

    # Gather query texts: original + top DE translations
    de_queries = stage1_analysis.get("search_queries_de", [])[:3]
    en_queries = [query_text] + stage1_analysis.get("search_queries_en", [])[:2]
    all_queries = en_queries + de_queries

    # Merge all query tokens into one set (deduplicated), NO PMI enhancement
    merged_tokens = []
    seen = set()
    for q in all_queries:
        for t in tokenise(q):
            if t not in seen:
                merged_tokens.append(t)
                seen.add(t)

    if not merged_tokens:
        return {}

    if verbose:
        print(f"      {len(all_queries)} queries → {len(merged_tokens)} unique tokens")

    # Pre-resolve citation indices
    cite_indices: dict[str, int] = {}
    for cite in all_citations:
        idx = sparse._id_to_idx.get(cite, -1)
        if idx >= 0:
            cite_indices[cite] = idx
        else:
            cl = cite.lower()
            for canon_cite, canon_idx in sparse._id_to_idx.items():
                if canon_cite.lower() == cl:
                    cite_indices[cite] = canon_idx
                    break

    if not cite_indices:
        return {}

    # Single BM25 call with merged tokens — one pass over 2.15M docs
    if verbose:
        print(f"      Scoring {len(cite_indices)} citations (single BM25 call)...")

    raw = sparse._bm25.get_scores(merged_tokens)

    # Extract only our candidate citations' scores
    cite_names = list(cite_indices.keys())
    idx_array = np.array([cite_indices[c] for c in cite_names])
    scores = raw[idx_array]

    if verbose:
        print(f"      BM25 call done")

    # Normalize by max score → [0, 1]
    max_val = scores.max()
    if max_val <= 0:
        return {}

    normalized = {}
    for i, cite in enumerate(cite_names):
        if scores[i] > 0:
            normalized[cite] = float(scores[i] / max_val)

    return normalized


def _compute_and_cache_all_bm25_scores(
    split: str,
    stage4_results: dict,
    stage1_results: dict,
    query_texts: dict[str, str],
    sparse: SparseRetriever,
) -> dict[str, dict[str, float]]:
    """
    Compute BM25 scores for ALL queries and cache to disk.
    On subsequent runs, loads from cache instantly.

    Returns dict: qid -> {citation: normalized_score}
    """
    cache_path = CHECKPOINTS_DIR / f"bm25_scores_{split}.json"

    # Try loading from cache
    if cache_path.exists():
        print(f"  Loading cached BM25 scores from {cache_path}")
        t0 = time.time()
        with open(cache_path, encoding="utf-8") as f:
            cached = json.load(f)
        print(f"  Loaded {len(cached)} queries in {time.time()-t0:.1f}s")
        # Verify cache covers all queries
        missing = set(stage4_results.keys()) - set(cached.keys())
        if not missing:
            return cached
        print(f"  Cache missing {len(missing)} queries, computing them...")
    else:
        cached = {}

    # Compute missing queries
    total = len(stage4_results)
    for i, (qid, stage4) in enumerate(stage4_results.items()):
        if qid in cached:
            continue
        t0 = time.time()
        bm25_scores = _compute_bm25_scores(
            qid, query_texts.get(qid, ""),
            stage4.get("all_citations", []),
            stage1_results.get(qid, {}),
            sparse,
            verbose=False,
        )
        elapsed = time.time() - t0
        cached[qid] = bm25_scores
        n_scored = len(bm25_scores)
        n_total = len(stage4.get("all_citations", []))
        print(f"    [{i+1}/{total}] {qid}: {n_scored}/{n_total} scored in {elapsed:.1f}s")

        # Save incrementally
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(cached, f, ensure_ascii=False)

    print(f"  Cached to {cache_path}")
    return cached


def verify_and_score_single(
    stage4_result: dict,
    verifier: Verifier | None,
    confidence_weights: dict[str, float] | None = None,
    bm25_scores: dict[str, float] | None = None,
) -> dict:
    """
    Verify, normalize, and score all citations for a single query.

    Returns {
        'scored': [(citation, score), ...],  # sorted by score desc
        'verified': [...],
        'dropped': [...],
        'stats': {...}
    }
    """
    all_citations = stage4_result.get("all_citations", [])
    sources_raw = stage4_result.get("sources", {})
    reranked_tiers = stage4_result.get("reranked_tiers", {})

    # Convert source lists to sets
    source_sets: dict[str, set[str]] = {}
    for source_name in ["explicit_from_query", "bm25_top10", "bm25_initial",
                        "citation_graph", "co_citation",
                        "llm_reranker", "llm_reranker_tier3", "llm_direct_gen",
                        "llm_stage1", "llm_stage1_procedural",
                        "mas_rewrite", "mas_supplement", "mas_decompose",
                        "mas_supportive", "mas_crossref"]:
        source_sets[source_name] = set()

    for cite, cite_sources in sources_raw.items():
        for s in cite_sources:
            if s in source_sets:
                source_sets[s].add(cite)
            # Map MAS sources to generic "bm25_initial" for scoring
            if s.startswith("mas_"):
                source_sets.setdefault("bm25_initial", set()).add(cite)

    # Inject reranker tier data from Stage 4's tiered output
    # Handles both old format (tier 2/3) and new format (0-10 scores)
    for cite, score_val in reranked_tiers.items():
        if not isinstance(score_val, (int, float)):
            continue
        score_val = int(score_val)
        # Map 0-10 scores: >=8 → tier3, >=4 → tier2 (also works for old 2/3 format)
        if score_val >= 3:  # tier 3 in old format, or 3+ in new
            source_sets.setdefault("llm_reranker_tier3", set()).add(cite)
        if score_val >= 2:  # tier 2+ in old format, or 2+ in new
            source_sets.setdefault("llm_reranker", set()).add(cite)

    # 5A: Verification
    if verifier:
        verified, dropped = verifier.verify_and_normalize(all_citations)
    else:
        verified = all_citations
        dropped = []

    # Remap source sets: verifier may normalize citation strings (e.g. STPO→StPO),
    # so we need to ensure the sources dict uses the post-verification forms.
    _verified_lower_to_verified = {c.lower(): c for c in verified}
    for source_name, cite_set in source_sets.items():
        remapped = set()
        for c in cite_set:
            cl = c.lower()
            if cl in _verified_lower_to_verified:
                remapped.add(_verified_lower_to_verified[cl])
            else:
                remapped.add(c)
        source_sets[source_name] = remapped

    # Remap BM25 scores to use post-verification citation forms
    bm25_remapped = None
    if bm25_scores:
        bm25_remapped = {}
        for c, s in bm25_scores.items():
            cl = c.lower()
            if cl in _verified_lower_to_verified:
                bm25_remapped[_verified_lower_to_verified[cl]] = max(
                    bm25_remapped.get(_verified_lower_to_verified[cl], 0.0), s
                )
            else:
                bm25_remapped[c] = max(bm25_remapped.get(c, 0.0), s)

    # 5B: Confidence scoring
    scored = score_all_citations(verified, source_sets, weights=confidence_weights,
                                 bm25_scores=bm25_remapped)

    return {
        "scored": scored,
        "verified": verified,
        "dropped": dropped,
        "stats": {
            "input_citations": len(all_citations),
            "verified": len(verified),
            "dropped": len(dropped),
        },
    }


In [15]:
import os
import sys
from pathlib import Path

try:
    PROJECT_ROOT = Path(root_dir).resolve()
except NameError:
    PROJECT_ROOT = Path('/content/drive/MyDrive/swiss_law').resolve()

# Raw inputs. Update this if your competition CSV/Parquet/JSONL files live elsewhere.
RAW_DATA_DIR = (PROJECT_ROOT / "data").resolve()

# Runtime artifact/output folders
INDEX_DIR = (PROJECT_ROOT / "index").resolve()
CHECKPOINTS_DIR = (PROJECT_ROOT / "checkpoints").resolve()
SUBMISSIONS_DIR = (PROJECT_ROOT / "submissions").resolve()
MODELS_DIR = (PROJECT_ROOT / "model").resolve()

for folder in [RAW_DATA_DIR, INDEX_DIR, CHECKPOINTS_DIR, SUBMISSIONS_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Set HuggingFace Cache Directory so models are saved to Google Drive
os.environ["HF_HOME"] = str(MODELS_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODELS_DIR)

# Model locations (acquired from HuggingFace, cached in MODELS_DIR)
QWEN3_32B_HF_ID = os.environ.get("SWISS_QWEN3_32B_HF_ID", "Qwen/Qwen3-32B")
QWEN3_RERANKER_ID = os.environ.get("SWISS_QWEN3_RERANKER_ID", "Qwen/Qwen3-Reranker-4B")

# We set the path to be the HF ID. The HF library will resolve it to the cached models inside MODELS_DIR.
QWEN3_32B_PATH = Path(QWEN3_32B_HF_ID)

os.environ["SWISS_QWEN3_32B_PATH"] = str(QWEN3_32B_PATH)
os.environ["SWISS_QWEN3_32B_HF_ID"] = QWEN3_32B_HF_ID
os.environ["SWISS_QWEN3_RERANKER_ID"] = QWEN3_RERANKER_ID

# Export the same paths for project modules that import data.data_paths
os.environ["SWISS_PIPELINE_ROOT"] = str(PROJECT_ROOT)
os.environ["SWISS_DATA_DIR"] = str(RAW_DATA_DIR)
os.environ["SWISS_INDEX_DIR"] = str(INDEX_DIR)
os.environ["SWISS_MODELS_DIR"] = str(MODELS_DIR)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT    =", PROJECT_ROOT)
print("RAW_DATA_DIR    =", RAW_DATA_DIR)
print("INDEX_DIR       =", INDEX_DIR)
print("CHECKPOINTS_DIR =", CHECKPOINTS_DIR)
print("SUBMISSIONS_DIR =", SUBMISSIONS_DIR)
print("MODELS_DIR      =", MODELS_DIR)
print("HF Cache Dir    =", os.environ["HF_HOME"])
print("Working dir     =", Path.cwd())


PROJECT_ROOT    = /content/drive/MyDrive/swiss_law
RAW_DATA_DIR    = /content/drive/MyDrive/swiss_law/data
INDEX_DIR       = /content/drive/MyDrive/swiss_law/index
CHECKPOINTS_DIR = /content/drive/MyDrive/swiss_law/checkpoints
SUBMISSIONS_DIR = /content/drive/MyDrive/swiss_law/submissions
MODELS_DIR      = /content/drive/MyDrive/swiss_law/model
HF Cache Dir    = /content/drive/MyDrive/swiss_law/model
Working dir     = /content/drive/MyDrive/swiss_law


### Path behavior in this notebook

- The notebook uses the explicit path variables above.
- Original project modules remain unchanged and pick up those same folders from the environment.
- There is no separate notebook sync or reload step.


In [16]:
# Editable runtime settings for notebook execution
SPLIT = "val"                   # "train", "val", or "test"
BACKEND = "local"               # "local", "anthropic", or "openai"
STAGE3_GOLD_PRIOR_MODE = "off"  # "off", "train", or "trainval"

# Optional stage-specific knobs
STAGE2_MAX_ITERATIONS  = 4
STAGE4_SKIP_DIRECT_GEN = True   # ← default flipped: direct-gen was 60% hallucinated on val (cell 97 diag)
STAGE4B_TOP_N          = None   # None -> stage default (Stage 4b is disabled)
STAGE5_THRESHOLD       = None   # val: tuned; test: loaded from configs/threshold.json

# Stage 4: pool size for LLM scoring. Candidates outside this pool are
# now passed through (via Stage 3 multi_signal), not dropped, so recall
# is preserved even when the ranked pool is small.
STAGE4_RANK_POOL_SIZE = 150    # v2: was 80; Stage-3 now 8K wide so 80 leaves too many gold items unscored

# Stage 3: text-mine statute→BGE edges from court_considerations.csv
# the first time Stage 3 runs (~2–4 min streaming; cached to pickle).
# Fixes cell-37 diagnostic: 0 statute→BGE edges in the precomputed graph.
STAGE3_BUILD_STATUTE_TO_BGE = True
STAGE3_STATUTE_TO_BGE_TOPK  = 5
STAGE3_USE_NUMBERED_INDEXES     = True   # load & use all 4 numbered-decision indexes
STAGE3_STATUTE_TO_NUMBERED_TOPK = 3    # v2: was 5, cascades via _all_numbered      # statute → numbered cases, top-K
STAGE3_NUMBERED_TO_BGE_TOPK     = 3      # numbered case → BGE decisions, top-K
STAGE3_NUMBERED_COCITATION_TOPK = 1    # v2: was 3, too noisy (+1923/q co-cited, low precision)      # numbered case → co-cited numbered, top-K

# Stage 6: Numbered-type FP suppression and Abs-flooding collapse.
# Numbered citations (case numbers like "4A_294/2019 E. 6") had 7 TP vs
# 48 FP on val (cell 118). Require a higher BM25 quantile for them.
STAGE6_NUMBERED_BM25_QUANTILE = 0.90    # v2: was 0.75 — still let 45 FP through on val
STAGE6_NUMBERED_REQUIRE_GRAPH_SOURCE = True  # v2: Numbered must come from a non-text-mined source
STAGE6_ABS_COLLAPSE            = True    # keep one Abs variant per article unless explicit
STAGE6_ABS_MAX_VARIANTS        = 2       # allow up to this many Abs of the same Art.

print({
    "SPLIT": SPLIT,
    "BACKEND": BACKEND,
    "STAGE3_GOLD_PRIOR_MODE": STAGE3_GOLD_PRIOR_MODE,
    "STAGE4_SKIP_DIRECT_GEN": STAGE4_SKIP_DIRECT_GEN,
    "STAGE3_BUILD_STATUTE_TO_BGE": STAGE3_BUILD_STATUTE_TO_BGE,
    "STAGE6_NUMBERED_BM25_QUANTILE": STAGE6_NUMBERED_BM25_QUANTILE,
})


{'SPLIT': 'val', 'BACKEND': 'local', 'STAGE3_GOLD_PRIOR_MODE': 'off', 'STAGE4_SKIP_DIRECT_GEN': True, 'STAGE3_BUILD_STATUTE_TO_BGE': True, 'STAGE6_NUMBERED_BM25_QUANTILE': 0.9}


In [17]:
default_config = {
    "_comment": "Revised pipeline config — BM25-first, LegalMALR-adapted MAS architecture",
    "stage1_backend": "local",
    "stage1_model": None,
    "bm25_top_k": 100,
    "graph_top_k": 50,
    "mas_max_iterations": 4,
    "mas_bm25_top_k": 30,
    "reranker_backend": "local",
    "reranker_batch_size": 20,
    "confidence_weights": {
        "explicit_from_query": 0.40,
        "bm25_top10": 0.25,
        "citation_graph": 0.15,
        "llm_reranker": 0.15,
        "llm_direct_gen": 0.05,
    },
    "threshold": 0.15,
}
default_config


{'_comment': 'Revised pipeline config — BM25-first, LegalMALR-adapted MAS architecture',
 'stage1_backend': 'local',
 'stage1_model': None,
 'bm25_top_k': 100,
 'graph_top_k': 50,
 'mas_max_iterations': 4,
 'mas_bm25_top_k': 30,
 'reranker_backend': 'local',
 'reranker_batch_size': 20,
 'confidence_weights': {'explicit_from_query': 0.4,
  'bm25_top10': 0.25,
  'citation_graph': 0.15,
  'llm_reranker': 0.15,
  'llm_direct_gen': 0.05},
 'threshold': 0.15}

### `configs/default.json`

```json
{
    "_comment": "Revised pipeline config — BM25-first, LegalMALR-adapted MAS architecture",

    "bm25_top_k":      100,
    "graph_top_k":     50,

    "mas_max_iterations": 4,
    "mas_bm25_top_k":    30,

    "reranker_batch_size": 20,

    "confidence_weights": {
        "explicit_from_query": 0.40,
        "bm25_top10":          0.25,
        "citation_graph":      0.15,
        "llm_reranker":        0.15,
        "llm_direct_gen":      0.05
    },

    "threshold": 0.15
}

```

### Notebook execution helpers

These wrappers are the notebook-native entrypoints. They call the stage functions directly instead of relying on CLI `main` blocks.


In [18]:
def run_stage1(split=None, backend=None, model=None, resume=False):
    split = SPLIT if split is None else split
    backend = BACKEND if backend is None else backend
    return run_batch_stage1(split, backend=backend, model=model, resume=resume)

def run_stage2(split=None, backend=None, max_iterations=None, resume=False):
    split = SPLIT if split is None else split
    backend = BACKEND if backend is None else backend
    max_iterations = STAGE2_MAX_ITERATIONS if max_iterations is None else max_iterations
    return run_batch_stage2(split, backend=backend, max_iterations=max_iterations, resume=resume)

def run_stage3(split=None, resume=False, gold_prior_mode=None, gold_prior_rules=None):
    split = SPLIT if split is None else split
    gold_prior_mode = STAGE3_GOLD_PRIOR_MODE if gold_prior_mode is None else gold_prior_mode
    return run_batch_stage3(split, resume=resume, gold_prior_mode=gold_prior_mode, gold_prior_rules=gold_prior_rules)

def run_stage4(split=None, backend=None, skip_direct_gen=None, resume=False, pool_size=None):
    split = SPLIT if split is None else split
    backend = BACKEND if backend is None else backend
    skip_direct_gen = STAGE4_SKIP_DIRECT_GEN if skip_direct_gen is None else skip_direct_gen
    pool_size = STAGE4_RANK_POOL_SIZE if pool_size is None else pool_size
    return run_batch_stage4(split, backend=backend, skip_direct_gen=skip_direct_gen,
                            resume=resume, pool_size=pool_size)

def run_stage4b(split=None, top_n=None):
    split = SPLIT if split is None else split
    if top_n is None:
        return run_batch_stage4b(split)
    return run_batch_stage4b(split, top_n=top_n)

def run_stage5(split=None, threshold=None, confidence_weights=None):
    split = SPLIT if split is None else split
    threshold = STAGE5_THRESHOLD if threshold is None else threshold
    if threshold is None and confidence_weights is None:
        return run_batch_stage5(split)
    return run_batch_stage5(split, threshold=threshold, confidence_weights=confidence_weights)


def get_submission_path(split=None):
    split = SPLIT if split is None else split
    return SUBMISSIONS_DIR / f"submission_{split}.csv"

def get_checkpoint_path(stage_name: str, split=None):
    split = SPLIT if split is None else split
    return CHECKPOINTS_DIR / f"{stage_name}_{split}.json"

print("Notebook helper functions are ready.")


Notebook helper functions are ready.


### Runtime artifact locations
These notebook-native helpers make the output files explicit before you run the stages.


In [19]:
SUBMISSION_FILE = get_submission_path(SPLIT)
STAGE1_CHECKPOINT = get_checkpoint_path("stage1", SPLIT)
STAGE2_CHECKPOINT = get_checkpoint_path("stage2", SPLIT)
STAGE3_CHECKPOINT = get_checkpoint_path("stage3", SPLIT)
STAGE4_CHECKPOINT = get_checkpoint_path("stage4", SPLIT)
STAGE5_CHECKPOINT = get_checkpoint_path("stage5", SPLIT)

print("SUBMISSION_FILE  =", SUBMISSION_FILE)
print("STAGE1_CHECKPOINT =", STAGE1_CHECKPOINT)
print("STAGE5_CHECKPOINT =", STAGE5_CHECKPOINT)


SUBMISSION_FILE  = /content/drive/MyDrive/swiss_law/submissions/submission_val.csv
STAGE1_CHECKPOINT = /content/drive/MyDrive/swiss_law/checkpoints/stage1_val.json
STAGE5_CHECKPOINT = /content/drive/MyDrive/swiss_law/checkpoints/stage5_val.json


## Pre-run diagnostics

Seven cells. All read-only. Run these before Stage 1 — if any of them fails,
*do not* run the pipeline: the bug is below the pipeline and will cascade.

1. **Data distribution** — val/train/test citation counts and type mix; flags the train vs val discrepancy
2. **KB integrity** — laws_de size, code distribution, corpus file checks, artifact existence
3. **Gold format audit** — Abs-level / lit-level fraction (tells us what our matcher must handle)
4. **Procedural-boilerplate census** — how many BGG/ZPO procedural cites recur across queries
5. **BGE reachability** — do the val-gold BGE stems exist anywhere in court_considerations?
6. **Test-vs-val domain overlap** — does val cover the law codes test queries mention?
7. **Graph edge inventory** — statute→BGE vs statute→numbered edge counts (if artifacts exist)


In [20]:

# ══ Pre-diag 1/7: data distribution ═════════════════════════════════════
from collections import Counter
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

def _types(cs):
    t = Counter()
    for c in cs:
        c = c.strip()
        if not c: continue
        if c.startswith("BGE"): t["BGE"] += 1
        elif c.startswith("Art"): t["Law"] += 1
        elif "_" in c and "/" in c: t["Numbered"] += 1
        else: t["Other"] += 1
    return t

def _describe(name, path):
    if pd is None or not Path(path).exists():
        print(f"  {name}: missing at {path}"); return None
    df = pd.read_csv(path)
    print(f"\n[{name}] {path}  rows={len(df)}")
    if "gold_citations" in df.columns:
        cs = df["gold_citations"].fillna("").apply(lambda s: [c for c in s.split(";") if c.strip()])
        counts = cs.apply(len)
        print(f"  cites/q:  min={counts.min()} max={counts.max()} mean={counts.mean():.1f} median={counts.median():.1f}")
        allc = [c for lst in cs for c in lst]
        t = _types(allc); total = sum(t.values()) or 1
        print(f"  types: " + ", ".join(f"{k}={v} ({v/total*100:.0f}%)" for k, v in t.most_common()))
    return df

print("═" * 70); print("1/7  DATA DISTRIBUTION"); print("═" * 70)
val_df   = _describe("val",   VAL_CSV   if 'VAL_CSV'   in dir() else "data/val.csv")
train_df = _describe("train", TRAIN_CSV if 'TRAIN_CSV' in dir() else "data/train.csv")
test_df  = _describe("test",  TEST_CSV  if 'TEST_CSV'  in dir() else "data/test.csv")

if val_df is not None and train_df is not None and "gold_citations" in train_df.columns:
    v = val_df["gold_citations"].fillna("").apply(lambda s: len([c for c in s.split(";") if c.strip()])).mean()
    t = train_df["gold_citations"].fillna("").apply(lambda s: len([c for c in s.split(";") if c.strip()])).mean()
    if v > 2 * t:
        print(f"\n  ** WARNING: val ({v:.1f}/q) has >2× train ({t:.1f}/q) — NEVER tune thresholds on train")


══════════════════════════════════════════════════════════════════════
1/7  DATA DISTRIBUTION
══════════════════════════════════════════════════════════════════════

[val] /content/drive/MyDrive/swiss_law/data/val.csv  rows=10
  cites/q:  min=10 max=47 mean=25.1 median=22.0
  types: Law=149 (59%), BGE=69 (27%), Numbered=33 (13%)

[train] /content/drive/MyDrive/swiss_law/data/train.csv  rows=1139
  cites/q:  min=1 max=44 mean=4.1 median=2.0
  types: Law=4602 (99%), BGE=57 (1%)

[test] /content/drive/MyDrive/swiss_law/data/test.csv  rows=40

  ** WARNING: val (25.1/q) has >2× train (4.1/q) — NEVER tune thresholds on train


In [21]:

# ══ Pre-diag 2/7: KB integrity ══════════════════════════════════════════
# Checks the canonical KB (laws_knowledge_base.jsonl) — not laws_de.csv,
# which uses SR-number style abbreviations that won't match "StGB"/"StPO".
# Sub 37 (F1=0.058) was a KB-load failure — same root cause we guard against.
from pathlib import Path
import os, json as _json

print("═" * 70); print("2/7  KB INTEGRITY"); print("═" * 70)

laws_csv  = LAWS_CSV  if 'LAWS_CSV'  in dir() else "data/laws_de.csv"
court_csv = COURT_CSV if 'COURT_CSV' in dir() else "data/court_considerations.csv"

# Canonical KB: count citations per law_abbreviation
kb_path = KB_JSONL if 'KB_JSONL' in dir() else Path("index/laws_knowledge_base.jsonl")
kb_ok = False
if Path(kb_path).exists():
    from collections import Counter as _Counter
    by_code_kb = _Counter()
    total_kb = 0
    with open(kb_path, encoding="utf-8") as f:
        for line in f:
            try:
                rec = _json.loads(line)
            except Exception:
                continue
            total_kb += 1
            ab = rec.get("law", {}).get("law_abbreviation", "")
            if ab: by_code_kb[ab] += 1
    print(f"  KB (laws_knowledge_base.jsonl): {total_kb:,} records")
    print(f"  top codes: {dict(by_code_kb.most_common(10))}")
    EXPECTED = {"ZGB": 2000, "OR": 3000, "StGB": 500, "StPO": 500, "BGG": 200, "ZPO": 500}
    problems = []
    if total_kb < 150_000: problems.append(f"KB count {total_kb} < 150,000")
    for code_, floor in EXPECTED.items():
        got = int(by_code_kb.get(code_, 0))
        if got < floor: problems.append(f"{code_} has only {got} (<{floor})")
    if problems:
        print("  ** KB INTEGRITY FAILED:")
        for p in problems: print(f"       - {p}")
        print("     DO NOT run the pipeline until the KB is fixed.")
    else:
        print("  KB ok.")
        kb_ok = True
else:
    print(f"  ** KB_JSONL MISSING at {kb_path}")

# Also show laws_de.csv row count for reference (FYI only — its abbreviations
# are SR-style and don't match the canonical KB/BM25 format).
try:
    import pandas as pd
    if Path(laws_csv).exists():
        laws_df = pd.read_csv(laws_csv)
        print(f"  laws_de.csv: {len(laws_df):,} rows (FYI only; uses SR-style codes)")
except Exception as e:
    print(f"  laws_de.csv load skipped: {e}")

# Court corpus
if Path(court_csv).exists():
    sz = os.path.getsize(court_csv) / (1024**3)
    print(f"  court_considerations.csv: {sz:.2f} GB present")
else:
    print(f"  ** court_considerations.csv MISSING at {court_csv} — BGE recall will be 0")

# Artifacts
_index_dir = INDEX_DIR if 'INDEX_DIR' in dir() else Path("index")
for art_name, art_fname in [
    ("citation_lookup.pkl",      "citation_lookup.pkl"),
    ("bm25 index",               "bm25_v2_index.pkl"),
    ("graph (statute->cases)",   "citation_graph.pkl"),
    ("statute->BGE",             "statute_to_bge.pkl"),
    ("statute->numbered",        "statute_to_numbered.pkl"),
    ("numbered->statutes",       "numbered_to_statutes.pkl"),
    ("numbered->BGE",            "numbered_to_bge.pkl"),
    ("numbered co-citation",     "numbered_cocitation.pkl"),
]:
    art_path = _index_dir / art_fname
    ex = art_path.exists()
    print(f"  {art_name:28s}: {'present' if ex else 'MISSING'}  ({art_path})")


══════════════════════════════════════════════════════════════════════
2/7  KB INTEGRITY
══════════════════════════════════════════════════════════════════════
  KB (laws_knowledge_base.jsonl): 179,641 records
  top codes: {'EFZ': 6500, 'OR': 3667, 'EBA': 2469, 'ZGB': 2405, 'StPO': 1306, 'StGB': 1244, 'TSV': 1094, 'VTS': 1023, 'SchKG': 919, 'ZV': 874}
  KB ok.
  laws_de.csv: 175,933 rows (FYI only; uses SR-style codes)
  court_considerations.csv: 2.26 GB present
  citation_lookup.pkl         : present  (/content/drive/MyDrive/swiss_law/index/citation_lookup.pkl)
  bm25 index                  : present  (/content/drive/MyDrive/swiss_law/index/bm25_v2_index.pkl)
  graph (statute->cases)      : present  (/content/drive/MyDrive/swiss_law/index/citation_graph.pkl)
  statute->BGE                : present  (/content/drive/MyDrive/swiss_law/index/statute_to_bge.pkl)
  statute->numbered           : present  (/content/drive/MyDrive/swiss_law/index/statute_to_numbered.pkl)
  numbered->statutes   

In [22]:

# ══ Pre-diag 3/7: gold format audit (Abs / lit / bare) ══════════════════
# Tells us what fraction of gold requires Abs.-level matching. If >30%,
# a blind Art→Abs expansion will destroy precision.
import re
from collections import Counter

print("═" * 70); print("3/7  GOLD FORMAT AUDIT"); print("═" * 70)

if val_df is not None and "gold_citations" in val_df.columns:
    level = Counter()
    for s in val_df["gold_citations"].fillna(""):
        for c in s.split(";"):
            c = c.strip()
            if not c.startswith("Art"): continue
            if " lit. " in c:     level["lit"] += 1
            elif " Abs. " in c:   level["Abs"] += 1
            else:                 level["bare_Art"] += 1
    total = sum(level.values()) or 1
    print(f"  Val gold Art-citations by specificity:")
    for k, v in level.most_common():
        print(f"    {k:10s} {v:4d} ({v/total*100:.0f}%)")
    if level["Abs"] + level["lit"] > total * 0.3:
        print(f"  NOTE: {(level['Abs']+level['lit'])/total*100:.0f}% of Art cites are sub-article — Abs-level matching is critical")


══════════════════════════════════════════════════════════════════════
3/7  GOLD FORMAT AUDIT
══════════════════════════════════════════════════════════════════════
  Val gold Art-citations by specificity:
    Abs         121 (81%)
    bare_Art     28 (19%)
  NOTE: 81% of Art cites are sub-article — Abs-level matching is critical


In [23]:

# ══ Pre-diag 4/7: procedural-boilerplate census ═════════════════════════
# Implicit procedural cites (BGG admissibility, costs, standing; ZPO service)
# appear in val gold but almost never in query text. Sets the budget for a
# procedural-prior lookup.
from collections import Counter

print("═" * 70); print("4/7  PROCEDURAL BOILERPLATE CENSUS (val gold)"); print("═" * 70)

if val_df is not None and "gold_citations" in val_df.columns:
    proc_counter = Counter()
    for s in val_df["gold_citations"].fillna(""):
        for c in s.split(";"):
            c = c.strip()
            if any(code_ in c for code_ in [" BGG", " ZPO", " VwVG", " ATSG"]):
                proc_counter[c] += 1
    most = [(c, n) for c, n in proc_counter.most_common(30) if n >= 2]
    print(f"  Procedural cites appearing in ≥2 val queries: {len(most)}")
    for c, n in most[:25]:
        in_q = 0
        for _, r in val_df.iterrows():
            if c.split(" Abs")[0] in str(r.get("query", "")):
                in_q += 1
        flag = f"  (mentioned in query text: {in_q}/10)" if in_q else "  (never in query text)"
        print(f"    {c:40s} in {n:2d} queries{flag}")


══════════════════════════════════════════════════════════════════════
4/7  PROCEDURAL BOILERPLATE CENSUS (val gold)
══════════════════════════════════════════════════════════════════════
  Procedural cites appearing in ≥2 val queries: 1
    Art. 100 Abs. 1 BGG                      in  9 queries  (never in query text)


In [24]:

# ══ Pre-diag 5/7: BGE reachability in corpus ════════════════════════════
# If a val-gold BGE stem is not even in court_considerations.csv, we cannot
# recall it. Period. This one-pass scan costs 1-2 min.
import csv as _csv
from pathlib import Path

print("═" * 70); print("5/7  BGE REACHABILITY"); print("═" * 70)

court_csv = COURT_CSV if 'COURT_CSV' in dir() else "data/court_considerations.csv"

if val_df is None or not Path(court_csv).exists():
    print("  skipped (corpus or val missing)")
else:
    want = set()
    for s in val_df["gold_citations"].fillna(""):
        for c in s.split(";"):
            c = c.strip()
            if c.startswith("BGE"):
                want.add(c.split(" E.")[0].strip())  # stem: "BGE 139 I 2"
    print(f"  Unique val-gold BGE stems: {len(want)}")
    found = set()
    with open(court_csv, encoding="utf-8") as f:
        rdr = _csv.reader(f); hdr = next(rdr)
        ci = hdr.index("citation") if "citation" in hdr else 0
        for row in rdr:
            if not row: continue
            cite = row[ci] if ci < len(row) else ""
            for stem in want:
                if stem in cite:
                    found.add(stem)
    miss = want - found
    print(f"  Present in corpus: {len(found)}/{len(want)}")
    if miss:
        print(f"  UNREACHABLE (these val golds are physically impossible to recall):")
        for s in sorted(miss)[:10]:
            print(f"    - {s}")


══════════════════════════════════════════════════════════════════════
5/7  BGE REACHABILITY
══════════════════════════════════════════════════════════════════════
  Unique val-gold BGE stems: 54
  Present in corpus: 54/54


In [25]:

# ══ Pre-diag 6/7: test vs val domain overlap ════════════════════════════
# If test queries mention law codes val does not cover, val-tuned thresholds
# will not generalize.  Proxy: law-code abbreviations in the query text.
import re
from collections import Counter

print("═" * 70); print("6/7  TEST vs VAL DOMAIN OVERLAP"); print("═" * 70)

ABBREV = re.compile(r"\b(ZGB|OR|StGB|StPO|BGG|ZPO|BV|SVG|IPRG|SchKG|VwVG|ATSG|IVG|UVG|AVIG|BVG|URG|MSchG|PatG|KVG|RVOG|GwG|KG|PrHG|CC|CO|LTF|CPC|CPP)\b")

def codes_in_queries(df):
    c = Counter()
    if df is None or "query" not in df.columns: return c
    for s in df["query"].fillna(""):
        c.update(ABBREV.findall(s))
    return c

def codes_in_gold(df):
    c = Counter()
    if df is None or "gold_citations" not in df.columns: return c
    for s in df["gold_citations"].fillna(""):
        c.update(ABBREV.findall(s))
    return c

test_q  = codes_in_queries(test_df)
val_q   = codes_in_queries(val_df)
val_g   = codes_in_gold(val_df)

only_in_test = set(test_q) - set(val_q) - set(val_g)
print(f"  Law codes in test queries: {dict(test_q.most_common())}")
print(f"  Law codes in val queries:  {dict(val_q.most_common())}")
print(f"  Law codes in val gold:     {dict(val_g.most_common(15))}")
if only_in_test:
    print(f"  ** Codes in test but NOT covered by val: {sorted(only_in_test)}")
    print(f"     val-tuned thresholds may not generalize to these queries")


══════════════════════════════════════════════════════════════════════
6/7  TEST vs VAL DOMAIN OVERLAP
══════════════════════════════════════════════════════════════════════
  Law codes in test queries: {'StPO': 5, 'SVG': 4, 'PrHG': 4, 'OR': 4, 'UVG': 4, 'IPRG': 3, 'ZGB': 3, 'CO': 2, 'SchKG': 2, 'ATSG': 1, 'CC': 1, 'ZPO': 1, 'MSchG': 1}
  Law codes in val queries:  {'OR': 3, 'StPO': 2, 'ZGB': 2}
  Law codes in val gold:     {'ZGB': 39, 'StPO': 36, 'OR': 18, 'BGG': 13, 'StGB': 12, 'IVG': 10, 'ATSG': 7, 'ZPO': 4, 'BV': 3, 'IPRG': 2, 'SchKG': 1}
  ** Codes in test but NOT covered by val: ['CC', 'CO', 'MSchG', 'PrHG', 'SVG', 'UVG']
     val-tuned thresholds may not generalize to these queries


In [26]:

# ══ Pre-diag 7/7: citation graph edge inventory ═════════════════════════
# Counts edges by type. If statute→BGE edges are 0, graph expansion cannot
# produce BGE citations — which explains every prior submission having 0 BGE.
from pathlib import Path
import pickle

print("═" * 70); print("7/7  CITATION GRAPH EDGE INVENTORY"); print("═" * 70)

candidates = [
    "index/citation_graph.pkl",
    "index/citation_graph_v2.pkl",
    "index/edges.pkl",
]

graph = None; used = None
for p in candidates:
    if Path(p).exists():
        try:
            with open(p, "rb") as f:
                graph = pickle.load(f); used = p; break
        except Exception as e:
            print(f"  {p}: failed to load ({e})")

if graph is None:
    print("  No citation-graph file found.  Checked:", candidates)
else:
    print(f"  Loaded: {used}")
    # Count edges by type
    types = {"stat→BGE": 0, "stat→num": 0, "BGE→BGE": 0, "BGE→stat": 0, "other": 0}
    n_nodes = 0
    def _type(x):
        if x.startswith("Art"): return "stat"
        if x.startswith("BGE"): return "BGE"
        if "_" in x and "/" in x: return "num"
        return "other"

    def iter_edges(g):
        # Tolerate dict-of-lists, list-of-pairs, networkx
        if hasattr(g, "edges"):
            for u, v in g.edges():
                yield str(u), str(v)
        elif isinstance(g, dict):
            for u, nbrs in g.items():
                if isinstance(nbrs, (list, set, tuple)):
                    for v in nbrs: yield str(u), str(v)
                elif isinstance(nbrs, dict):
                    for v in nbrs: yield str(u), str(v)
        elif isinstance(g, list):
            for u, v in g[:5_000_000]: yield str(u), str(v)

    nset = set()
    for u, v in iter_edges(graph):
        nset.add(u); nset.add(v)
        tu, tv = _type(u), _type(v)
        if   tu == "stat" and tv == "BGE": types["stat→BGE"] += 1
        elif tu == "stat" and tv == "num": types["stat→num"] += 1
        elif tu == "BGE"  and tv == "BGE": types["BGE→BGE"]  += 1
        elif tu == "BGE"  and tv == "stat": types["BGE→stat"] += 1
        else: types["other"] += 1
    print(f"  nodes: {len(nset):,}")
    for k, v in types.items():
        flag = "  <-- KEY" if k == "stat→BGE" and v == 0 else ""
        print(f"  {k}: {v:,}{flag}")
    if types["stat→BGE"] == 0:
        print("  ** NO statute→BGE edges.  BGE recall via graph expansion is impossible.")
        print("     Fix: build text-mined statute→BGE index from court_considerations.csv")


══════════════════════════════════════════════════════════════════════
7/7  CITATION GRAPH EDGE INVENTORY
══════════════════════════════════════════════════════════════════════
  Loaded: index/citation_graph.pkl
  nodes: 2,201,708
  stat→BGE: 0  <-- KEY
  stat→num: 0
  BGE→BGE: 0
  BGE→stat: 0
  other: 3,897,617
  ** NO statute→BGE edges.  BGE recall via graph expansion is impossible.
     Fix: build text-mined statute→BGE index from court_considerations.csv


## Shared Utilities

### `agent/llm_backend.py`

In [27]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/agent/llm_backend.py"

"""
llm_backend.py -- Singleton Qwen3-32B loader for all pipeline LLM tasks.

Loads the model ONCE (~64GB (bf16) VRAM as Q4/NF4) and provides a unified
generate() interface used by:
  - Stage 1: Query analysis
  - Stage 2: MAS agent reformulations
  - Stage 4: LLM reranking + direct citation generation

# DESIGN: One model, loaded once, serves all LLM functions.
#   The model stays in GPU memory across all stages — no loading/unloading.
#
# HOW TO RUN (standalone smoke-test):
#   cd E:\\swiss-law-pipeline
#   python agent/llm_backend.py
#   python agent/llm_backend.py --prompt "List 3 Swiss criminal law articles"
"""


sys.path.insert(0, str(Path(__file__).parent.parent))

# Singleton model instance
_MODEL = None
_TOKENIZER = None


def get_model_and_tokenizer():
    """
    Lazy-load Qwen3-32B in full bf16 precision (no quantization).
    Returns (model, tokenizer). Cached after first call.

    VRAM usage: ~64 GB.
    """
    global _MODEL, _TOKENIZER
    if _MODEL is not None:
        return _MODEL, _TOKENIZER

    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

    model_path = str(QWEN3_32B_PATH) if QWEN3_32B_PATH.exists() else QWEN3_32B_HF_ID
    print(f"  Loading Qwen3-32B from: {model_path}")

    _TOKENIZER = AutoTokenizer.from_pretrained(model_path)
    _MODEL = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    _MODEL.eval()
    print(f"  Qwen3-32B loaded. VRAM: ~64GB (bf16)")
    return _MODEL, _TOKENIZER


def generate(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 2048,
    temperature: float = 0.6,
    top_k: int = 20,
    top_p: float = 0.95,
    enable_thinking: bool = False,
) -> str:
    """
    Generate a response from Qwen3-32B.

    Args:
        system_prompt: System message (role definition).
        user_prompt:   User message (the actual query/task).
        max_new_tokens: Max generation length.
        temperature:   Sampling temperature (0.6 = focused but not greedy).
        top_k/top_p:   Nucleus sampling params.
        enable_thinking: If True, allows <think> blocks for chain-of-thought.
                        Stage 1 uses True, Stage 4 reranker uses False.

    Returns:
        Generated text string (with <think> blocks stripped if enable_thinking=False).
    """
    model, tokenizer = get_model_and_tokenizer()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        out[0][input_len:],
        skip_special_tokens=True,
    )

    # Explicitly free GPU tensors so VRAM doesn't fragment across calls.
    del inputs, out
    torch.cuda.empty_cache()

    return generated


def generate_json(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 2048,
    enable_thinking: bool = False,
) -> dict:
    """
    Generate and parse JSON from Qwen3-32B.
    Handles ```json ... ``` wrapping and malformed JSON gracefully.
    On CUDA OOM: clears cache and retries once with halved max_new_tokens.
    """

    try:
        raw = generate(system_prompt, user_prompt, max_new_tokens=max_new_tokens,
                       enable_thinking=enable_thinking)
    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "CUDA" in str(e):
            print(f"\n  [OOM] CUDA out of memory — clearing cache and retrying with {max_new_tokens // 2} tokens...")
            torch.cuda.empty_cache()
            try:
                raw = generate(system_prompt, user_prompt,
                               max_new_tokens=max_new_tokens // 2,
                               enable_thinking=enable_thinking)
            except RuntimeError as e2:
                print(f"  [OOM] Retry also failed: {e2}")
                torch.cuda.empty_cache()
                return {}
        else:
            raise
    return parse_json_response(raw)


def parse_json_response(raw_text: str) -> dict:
    """Extract JSON from LLM response, handling markdown wrapping."""
    raw_text = raw_text.strip()

    # Strip ```json ... ``` wrapping
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw_text, re.DOTALL)
    if m:
        raw_text = m.group(1)
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        # Fallback: find any JSON object
        m2 = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if m2:
            try:
                return json.loads(m2.group(0))
            except json.JSONDecodeError:
                pass
    return {}




### `agent/verifier.py`

In [28]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/agent/verifier.py"

"""
verifier.py -- Citation verification and normalization.

Validates each citation against the corpus:
  - Normalize formatting (whitespace, abbreviation spacing)
  - Check existence in statutory corpus
  - Expand article-level to Abs.-level if needed
  - Drop hallucinated case citations not in corpus (with lenient fallback)

# HOW TO RUN (standalone smoke-test):
#   cd E:\\swiss-law-pipeline
#   python agent/verifier.py
#
# EXPECTS: index/citation_lookup.pkl (built by: python stage0_build_index.py)
"""


sys.path.insert(0, str(Path(__file__).parent.parent))


# Corpus uses UPPERCASE abbreviations but gold standard uses mixed-case.
# This map converts corpus forms to gold-compatible forms.
_ABBREV_CASING = {
    'STGB': 'StGB', 'STPO': 'StPO', 'STBOG': 'StBOG', 'JSTG': 'JStG',
    'JSTPO': 'JStPO', 'SCHKG': 'SchKG', 'BANKG': 'BankG', 'BANKV': 'BankV',
    'ASYLG': 'AsylG', 'VWVG': 'VwVG', 'VSTG': 'VStG', 'VSTV': 'VStV',
    'BETMG': 'BetmG', 'GSCHG': 'GSchG', 'GSCHV': 'GSchV', 'MSCHG': 'MSchG',
    'MSCHV': 'MSchV', 'STHG': 'StHG', 'PARLG': 'ParlG', 'SEBG': 'SebG',
    'FINFRAG': 'FinfraG', 'FINFRAV': 'FinfraV', 'CHEMG': 'ChemG',
    'FAMZG': 'FamZG', 'ELEG': 'EleG', 'GWG': 'GwG', 'EPG': 'EpG',
    'EPV': 'EpV', 'STAHIG': 'StAhiG', 'STROMVG': 'StromVG',
    'STROMVV': 'StromVV', 'VSTRR': 'VStrR', 'HREGV': 'HRegV',
    'GEOIG': 'GeoIG', 'WAG': 'WaG', 'PATG': 'PatG', 'PRSG': 'PrSG',
    'DESG': 'DesG', 'ARG': 'ArG', 'TWWV': 'TwwV', 'PRHG': 'PrHG',
}


def _fix_abbrev_casing(c: str) -> str:
    """Convert corpus UPPERCASE abbreviations to gold-compatible mixed-case."""
    if not c.startswith("Art."):
        return c
    parts = c.split()
    if len(parts) >= 3:
        last = parts[-1]
        fixed = _ABBREV_CASING.get(last, last)
        if fixed != last:
            parts[-1] = fixed
            return " ".join(parts)
    return c


def normalize_citation(c: str) -> str:
    """Normalize whitespace, abbreviation spacing, and casing in a citation string."""
    c = re.sub(r"\s+",    " ",     str(c or "").strip())
    c = re.sub(r"Art\.\s*",  "Art. ",  c)
    c = re.sub(r"Abs\.\s*",  "Abs. ",  c)
    c = re.sub(r"lit\.\s*",  "lit. ",  c)
    c = re.sub(r"Ziff\.\s*", "Ziff. ", c)
    c = re.sub(r"BGE\s+",    "BGE ",   c)
    c = re.sub(r"\bE\.\s*",  "E. ",    c)
    c = _fix_abbrev_casing(c.strip())
    return c.strip()


def is_law_citation(c: str) -> bool:
    return bool(re.match(r"^Art\.\s+\d", c.strip()))


def is_case_citation(c: str) -> bool:
    return bool(
        re.match(r"^BGE\s+\d+", c.strip()) or
        re.match(r"^\d+[A-Z]_\d+/\d{4}", c.strip())
    )


class Verifier:
    def __init__(self, lookup_pkl: Path = LOOKUP_PKL, caselaw_ids: set[str] | None = None):
        self.lookup = CitationLookup(lookup_pkl)
        self.caselaw_ids = caselaw_ids or set()

    def verify_and_normalize(
        self,
        citations: list[str],
        expand_to_abs: bool = True,
    ) -> tuple[list[str], list[str]]:
        """
        Normalize and verify a list of citations.

        Returns (verified, dropped):
          verified = citations that passed verification
          dropped  = citations that failed (hallucinations / format errors)
        """
        verified = []
        dropped  = []

        for raw in citations:
            c = normalize_citation(raw)
            if not c:
                continue

            if is_law_citation(c):
                canon = self.lookup.normalize(c)
                if canon:
                    verified.append(_fix_abbrev_casing(canon))
                elif expand_to_abs:
                    expanded = self.lookup.expand_to_abs(c)
                    if expanded and expanded != [c]:
                        verified.extend(_fix_abbrev_casing(e) for e in expanded)
                    else:
                        # Try stripping lit./Ziff. to find parent Abs. or base Art.
                        # E.g. "Art. 221 Abs. 1 lit. b StPO" -> try "Art. 221 Abs. 1 StPO"
                        #      then "Art. 221 StPO"
                        kept = False
                        m = re.match(r"(Art\.\s+\d+[a-z]?)\s+(Abs\.\s+\d+\s*)?(lit\.\s+\w+\s*)?(Ziff\.\s+\d+\s?)?(.*)", c)
                        if m:
                            law_part = m.group(5).strip() if m.group(5) else ""
                            # Try Abs. level first (strip lit./Ziff.)
                            if m.group(2):
                                abs_cite = f"{m.group(1)} {m.group(2).strip()} {law_part}".strip()
                                if self.lookup.normalize(abs_cite):
                                    verified.append(c)
                                    kept = True
                            # Try base article (strip everything)
                            if not kept:
                                base = f"{m.group(1)} {law_part}".strip()
                                if self.lookup.normalize(base):
                                    verified.append(c)
                                    kept = True
                        if not kept:
                            dropped.append(c)
                else:
                    dropped.append(c)

            elif is_case_citation(c):
                if not self.caselaw_ids or c in self.caselaw_ids:
                    verified.append(c)
                else:
                    # Lenient: keep if format looks valid (Swiss court citation patterns)
                    if re.match(r"^BGE\s+\d{2,3}\s+[IVX]+\s+\d+(\s+E\.\s+[\d\.]+)?(\s+S\.\s+\d+)?$", c):
                        verified.append(c)
                    elif re.match(r"^\d+[A-Z]_\d+/\d{4}(\s+\d{2}\.\d{2}\.\d{4})?(\s+E\.\s+[\d\.]+)?$", c):
                        verified.append(c)
                    else:
                        dropped.append(c)
            else:
                dropped.append(c)

        # Deduplicate preserving order
        seen = set()
        final = []
        for c in verified:
            if c not in seen:
                seen.add(c)
                final.append(c)

        return final, dropped


### `retrieval/explicit_citations.py`

In [29]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/retrieval/explicit_citations.py"

"""
explicit_citations.py -- Extract citation strings directly from query text.

Many val queries contain literal citation strings like:
  "Art. 221 Abs. 1 lit. b StPO"
  "BGE 137 IV 122 E. 6.2"
  "1B_210/2023 E. 4.1"

These are FREE RECALL — they are derived from literal query references and
should ALWAYS be included in the candidate pipeline regardless of retrieval scores.

# HOW TO RUN (standalone test):
#   cd E:\\swiss-law-pipeline
#   python retrieval/explicit_citations.py
"""



# Common alternate abbreviations seen in multilingual Swiss citations.
# We normalize these to the corpus' canonical abbreviations so downstream
# verification and retrieval can keep them instead of dropping them.
_ALT_ABBREV_TO_CANON = {
    "CC": "ZGB",
    "CO": "OR",
    "LP": "SchKG",
    "CPC": "ZPO",
    "LDIP": "IPRG",
    "LFors": "GestG",
    "CPP": "StPO",
    "CP": "StGB",
    "LTF": "BGG",
    "PA": "VwVG",
}

_SINGLE_STAT_PATTERN = re.compile(
    r"Art\.\s+"
    r"(\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?)"
    r"\s+(?:of\s+the\s+)?"
    r"([A-Z][A-Za-z0-9]{1,14})",
    re.UNICODE,
)

_MULTI_STAT_PATTERN = re.compile(
    r"Art\.\s+"
    r"("
    r"\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?"
    r"(?:\s*(?:,|and|und|or|oder)\s*"
    r"\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?)+"
    r")"
    r"\s+(?:of\s+the\s+)?"
    r"([A-Z][A-Za-z0-9]{1,14})",
    re.UNICODE,
)

_ARTICLE_PART_SPLIT = re.compile(r"\s*(?:,|and|und|or|oder)\s*", re.UNICODE)


def _canon_law_abbrev(abbrev: str) -> str:
    return _ALT_ABBREV_TO_CANON.get(abbrev, abbrev)


def _norm_statute_ref(article_part: str, law_abbrev: str) -> str:
    article_part = re.sub(r"\s+", " ", article_part.strip())
    law_abbrev = _canon_law_abbrev(law_abbrev.strip())
    return f"Art. {article_part} {law_abbrev}".strip()


def extract_explicit_citations(query_text: str) -> set[str]:
    """
    Extract all citation strings from query text using regex patterns.

    Returns a set of citation strings derived from literal references in the
    query text. We lightly normalize whitespace and common multilingual law
    abbreviations (e.g. CC -> ZGB, CO -> OR), and expand shared-law patterns
    such as "Art. 38 and 39 CO" into individual citations.
    """
    citations = set()

    # ── Shared-law statutory refs: Art. 38 and 39 CO -> 2 citations ────────
    for m in _MULTI_STAT_PATTERN.finditer(query_text):
        raw_articles, raw_law = m.group(1), m.group(2)
        for art_part in _ARTICLE_PART_SPLIT.split(raw_articles):
            art_part = art_part.strip()
            if art_part:
                citations.add(_norm_statute_ref(art_part, raw_law))

    # ── Statutory: Art. X [Abs. Y [lit. z]] LAW ──────────────────────────
    # Matches: Art. 221 Abs. 1 lit. b StPO, Art. 44 ATSG, Art. 125 of the CC
    for m in _SINGLE_STAT_PATTERN.finditer(query_text):
        citations.add(_norm_statute_ref(m.group(1), m.group(2)))

    # ── BGE decisions: BGE XXX II/III/IV/V YYY E. Z.Z ───────────────────
    # Matches: BGE 137 IV 122 E. 6.2
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"BGE\s+\d+\s+[IVX]+\s+\d+\s+E\.\s+[\d.]+",
            query_text,
        )
    )

    # ── BGE decisions without Erwägung: BGE 137 IV 122 ──────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"BGE\s+\d+\s+[IVX]+\s+\d+",
            query_text,
        )
    )

    # ── Numbered cases: 1B_210/2023 E. 4.1 ──────────────────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"\d+[A-Z]_\d+/\d{4}\s+E\.\s+[\d.]+",
            query_text,
        )
    )

    # ── Numbered cases without Erwägung: 1B_210/2023 ────────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"\d+[A-Z]_\d+/\d{4}",
            query_text,
        )
    )

    return citations


### `retrieval/graph_retriever.py`

In [30]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/retrieval/graph_retriever.py"

"""
graph_retriever.py -- Citation graph traversal retrieval.

Given candidate statutory articles -> find case law that cites them.
Given candidate case citations   -> find statutory articles they cite.

This is the CRITICAL bridge between statutory and case law retrieval.
40% of val citations are court decisions, but train data is 98.8% statutory.
The citation graph mechanically bridges this gap.

# HOW TO RUN (requires index/citation_graph.pkl):
#   cd E:\\swiss-law-pipeline
#   python retrieval/graph_retriever.py
#   python retrieval/graph_retriever.py --article "Art. 221 StPO"
#
# EXPECTS: index/citation_graph.pkl (built by: python stage0_build_index.py)
"""


sys.path.insert(0, str(Path(__file__).parent.parent))


class GraphRetriever:
    def __init__(self, graph_pkl: Path = CITATION_GRAPH_PKL):
        print("  Loading citation graph ...")
        self.graph = CitationGraph(graph_pkl)
        print(f"    {len(self.graph.article_to_cases):,} articles with case links")
        print(f"    {len(self.graph.case_to_articles):,} cases with article links")

    def articles_to_cases(
        self,
        article_citations: list[str],
        top_k_per_article: int = 20,
    ) -> list[tuple[str, float]]:
        """
        Given statutory articles, return citing case law.
        Score = sum of (1/rank) across articles (RRF-style).
        """
        scores: dict[str, float] = {}
        for art in article_citations:
            cases = self.graph.get_citing_cases(art, top_n=top_k_per_article)
            for rank, case in enumerate(cases, 1):
                scores[case] = scores.get(case, 0.0) + 1.0 / (60 + rank)
        return sorted(scores.items(), key=lambda x: -x[1])

    def cases_to_articles(self, case_citations: list[str]) -> list[str]:
        """Given case citations, return statutory articles they cite."""
        found: set[str] = set()
        for case in case_citations:
            arts = self.graph.get_cited_articles(case)
            found.update(arts)
        return list(found)

    def bidirectional_expand(
        self,
        seed_articles: list[str],
        seed_cases: list[str],
        depth: int = 1,
    ) -> tuple[list[tuple[str, float]], list[str]]:
        """
        Expand seeds bidirectionally up to `depth` hops.
        Returns (expanded_cases_with_scores, expanded_articles).
        """
        all_cases:    dict[str, float] = {}
        all_articles: set[str]         = set(seed_articles)

        current_articles = list(seed_articles)
        current_cases    = list(seed_cases)

        for _ in range(depth):
            new_cases = self.articles_to_cases(current_articles)
            for c, s in new_cases:
                all_cases[c] = max(all_cases.get(c, 0.0), s)

            new_articles = self.cases_to_articles(
                current_cases + [c for c, _ in new_cases]
            )
            all_articles.update(new_articles)

            current_articles = new_articles
            current_cases    = [c for c, _ in new_cases[:20]]

        return (
            sorted(all_cases.items(), key=lambda x: -x[1]),
            list(all_articles),
        )


### `retrieval/sparse_retriever.py`

In [31]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/retrieval/sparse_retriever.py"

"""
sparse_retriever.py -- BM25 retrieval over the combined law+court index.

This is the PRIMARY retrieval engine in the revised pipeline.
Dense retrieval is dropped entirely — BM25 gives 3x better recall (R@50=0.73
vs R@100=0.24 for dense) on this cross-lingual legal task.

Features:
  - PMI-based law abbreviation boosting
  - Multi-query union with max-score fusion
  - np.argpartition for O(n) top-K
  - Separate law/court/combined search modes

# HOW TO RUN (standalone smoke-test):
#   cd E:\\swiss-law-pipeline
#   python retrieval/sparse_retriever.py
#   python retrieval/sparse_retriever.py --query "Untersuchungshaft Kollusionsgefahr StPO"
#
# EXPECTS: index/bm25_v2_index.pkl + index/bm25_v2_ids.pkl
#   (built by: python stage0_build_index.py)
"""



sys.path.insert(0, str(Path(__file__).parent.parent))

_TOKEN_RE = re.compile(r"[^\w\d]+", re.UNICODE)


def tokenise(text: str) -> list[str]:
    """Tokenize text for BM25 queries. Keeps short tokens if numeric (article numbers)."""
    if not text:
        return []
    return [t for t in _TOKEN_RE.split(text.lower()) if len(t) >= 2 or t.isdigit()]


class SparseRetriever:
    """
    Wraps the combined BM25 index (law + court).
    Exposes search_statutory(), search_caselaw(), search_combined().
    """

    def __init__(self):
        print("  Loading combined BM25 index ...")
        self._bm25 = load_bm25_artifact(STATUTORY_BM25_PKL)

        with open(STATUTORY_BM25_IDS, "rb") as f:
            meta = pickle.load(f)

        self._ids:     list[str] = meta["citation_canon"]
        self._n_law:   int       = meta["n_law"]
        self._n_court: int       = meta["n_court"]
        print(f"    {self._n_law:,} law  +  {self._n_court:,} court  =  {len(self._ids):,} total")

        # Build id->index mapping for fast rank lookup
        self._id_to_idx: dict[str, int] = {c: i for i, c in enumerate(self._ids)}

        # PMI token->law map for query enhancement
        self._token_law_freq: dict = {}
        self._token_total:    dict = {}
        if TOKEN_LAW_FREQ.exists():
            with open(TOKEN_LAW_FREQ, encoding="utf-8") as f:
                raw = json.load(f)
            self._token_law_freq = {t: v for t, v in raw.items()}
            self._token_total    = {t: sum(v.values()) for t, v in self._token_law_freq.items()}
            print(f"    PMI token->law map: {len(self._token_law_freq):,} tokens")

    # ------------------------------------------------------------------
    # Public search API
    # ------------------------------------------------------------------

    def search_statutory(self, queries: list[str], top_k: int = 100) -> list[tuple[str, float]]:
        """Search law articles only. Returns [(citation, score), ...]."""
        return self._search(queries, top_k, law_only=True)

    def search_caselaw(self, queries: list[str], top_k: int = 100) -> list[tuple[str, float]]:
        """Search court considerations only. Returns [(citation, score), ...]."""
        return self._search(queries, top_k, court_only=True)

    def search_combined(self, queries: list[str], top_k: int = 200) -> list[tuple[str, float]]:
        """Search across the full combined index (law + court)."""
        return self._search(queries, top_k)

    def get_bm25_rank(self, query_tokens: list[str], citation: str) -> int:
        """Get the BM25 rank of a specific citation for given query tokens. -1 if not found."""
        idx = self._id_to_idx.get(citation, -1)
        if idx < 0:
            return -1
        scores = self._bm25.get_scores(query_tokens)
        rank = (scores > scores[idx]).sum() + 1
        return int(rank)

    # ------------------------------------------------------------------
    # Internal
    # ------------------------------------------------------------------

    def _enhance_tokens(self, tokens: list[str], law_boost: int = 10,
                        top_n_laws: int = 5) -> list[str]:
        """Append predicted law abbreviations (PMI) to query tokens."""
        if not self._token_law_freq:
            return tokens

        scores: dict[str, float] = {}
        for t in set(tokens):
            if t not in self._token_law_freq:
                continue
            idf = self._bm25.idf.get(t, 0.0)
            if idf < 1.0:
                continue
            total = self._token_total.get(t, 1)
            for ab, cnt in self._token_law_freq[t].items():
                scores[ab] = scores.get(ab, 0.0) + (cnt / total) * idf

        predicted = sorted(scores.items(), key=lambda x: -x[1])[:top_n_laws]
        enhanced = list(dict.fromkeys(tokens))
        for ab, _ in predicted:
            enhanced.extend([ab] * law_boost)
        return enhanced

    def _search(self, queries: list[str], top_k: int,
                law_only: bool = False, court_only: bool = False) -> list[tuple[str, float]]:
        """Core search: runs each query, takes max score per doc across queries."""
        scores: dict[str, float] = {}

        for q in queries:
            toks = tokenise(q)
            if not toks:
                continue
            toks = self._enhance_tokens(toks)

            raw = self._bm25.get_scores(toks)

            # Restrict to relevant slice
            if law_only:
                raw[self._n_law:] = 0.0
            elif court_only:
                raw[:self._n_law] = 0.0

            k = min(top_k, int((raw > 0).sum()), len(raw))
            if k == 0:
                continue
            top_idx = np.argpartition(raw, -k)[-k:]
            for i in top_idx:
                if raw[i] <= 0:
                    continue
                c = self._ids[i]
                scores[c] = max(scores.get(c, 0.0), float(raw[i]))

        return sorted(scores.items(), key=lambda x: -x[1])[:top_k]


### `scoring/confidence.py`

In [32]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/scoring/confidence.py"

"""
confidence.py -- Composite confidence scoring and F1 threshold optimization.

Revised scoring system with:
  - Signal-counting approach (how many independent sources found this citation)
  - Tiered LLM reranker scores (tier 3 = directly applicable, tier 2 = related)
  - Multi-signal bonus for citations found by 3+ independent signals
  - BM25 initial and MAS agent signals now contribute
  - **Continuous BM25 score**: per-query normalized BM25 relevance (0-1) adds
    up to 0.20, creating a ranking gradient within same-signal-count groups.
  - **Cross-encoder score** (Strategy C): when available, replaces the binary
    llm_reranker/tier3 signals with a continuous Qwen3-Reranker-4B P(yes)
    probability × 0.18. Falls back to binary signals for unscored citations.

Signal weights (additive, max ~1.0):
  - Explicit in query text:     0.40
  - BM25 continuous score:      0.00-0.20 (normalized per query)
  - BM25 initial / MAS agents:  0.10
  - Citation graph:              0.10
  - Cross-encoder score:        0.00-0.18 (replaces binary reranker when available)
  - LLM reranker tier 3:        0.25  (fallback when no CE score)
  - LLM reranker tier 2:        0.15  (fallback when no CE score)
  - LLM direct generation:      0.05
  - LLM stage1 candidate:       0.05
  - Procedural (stage1):        0.05
  - Multi-signal bonus (3+):    0.10

# HOW TO RUN (standalone unit-test):
#   cd E:\\swiss-law-pipeline
#   python scoring/confidence.py
"""



DEFAULT_WEIGHTS = {
    "explicit_from_query":    0.40,
    "bm25_top10":             0.15,   # binary: in BM25 top-10 yes/no
    "bm25_initial":           0.10,
    "citation_graph":         0.10,
    "llm_reranker_tier3":     0.25,
    "llm_reranker":           0.15,
    "llm_direct_gen":         0.05,
    "llm_stage1":             0.05,
    "llm_stage1_procedural":  0.05,
    "co_citation":            0.03,
}

# Additional continuous BM25 gradient (on top of binary signals)
# Lower weight than before — this adds ranking within same-signal groups
# without inflating all scores and pushing the threshold too high.
BM25_CONTINUOUS_WEIGHT = 0.10

# Cross-encoder weight (Strategy C): continuous P(yes) score × this weight
# replaces binary llm_reranker/tier3 signals. Validated at 0.18 on all 10
# val queries: 0.5400 → 0.6520 macro-F1.
CROSS_ENCODER_WEIGHT = 0.18

# Signals that count as "independent" for the multi-signal bonus
_INDEPENDENT_SIGNALS = {
    "explicit_from_query", "bm25_top10", "bm25_initial",
    "citation_graph", "llm_reranker", "llm_reranker_tier3",
    "llm_direct_gen", "llm_stage1",
}

MULTI_SIGNAL_BONUS = 0.10  # added when 3+ independent signals agree


def score_citation(cite: str, sources: dict[str, set[str]],
                   weights: dict[str, float] | None = None,
                   bm25_scores: dict[str, float] | None = None) -> float:
    """
    Compute composite confidence score for a single citation.

    Scoring approach: additive weights + continuous BM25 score + graduated
    multi-signal bonus. The key discriminators are:
      1. NUMBER of independent signals (gold avg 3-4, noise avg 1-2)
      2. Continuous BM25 relevance score (creates ranking gradient within
         same-signal-count groups — this is what separates 2-signal gold
         from 2-signal noise)

    Args:
        cite:           Citation string.
        sources:        Dict mapping source_name -> set of citations from that source.
        weights:        Override default weights.
        bm25_scores:    Dict mapping citation -> normalized BM25 score (0-1).
                        If provided, replaces binary bm25_top10 with continuous signal.

    Returns:
        Float score in [0, 1].
    """
    w = weights if weights is not None else DEFAULT_WEIGHTS

    score = 0.0
    active_signals = set()

    for source_name, weight in w.items():
        if cite in sources.get(source_name, set()):
            # If citation has tier3, don't also add the base llm_reranker weight
            if source_name == "llm_reranker" and cite in sources.get("llm_reranker_tier3", set()):
                continue  # tier3 supersedes base reranker
            score += weight
            active_signals.add(source_name)

    # Also add MAS agent signals as bm25_initial weight if they contributed
    for mas_source in ["mas_rewrite", "mas_supplement", "mas_decompose",
                       "mas_supportive", "mas_crossref"]:
        if cite in sources.get(mas_source, set()):
            if "bm25_initial" not in active_signals:
                score += w.get("bm25_initial", 0.10)
                active_signals.add("bm25_initial")
            break

    # Continuous BM25 score: normalized per-query BM25 relevance (0.0 to 0.20)
    # This creates the ranking gradient that binary signals can't provide.
    # A gold citation at BM25 rank #2 gets ~0.18, while noise at rank #40 gets ~0.05.
    if bm25_scores is not None:
        bm25_norm = bm25_scores.get(cite, 0.0)
        if bm25_norm > 0:
            score += bm25_norm * BM25_CONTINUOUS_WEIGHT
            active_signals.add("bm25_hit")

    # Graduated multi-signal bonus: the more independent signals agree,
    # the more likely this is a true gold citation.
    independent_count = len(active_signals & _INDEPENDENT_SIGNALS)
    if independent_count >= 4:
        score += 0.20  # very high confidence
    elif independent_count >= 3:
        score += 0.10  # high confidence
    elif independent_count >= 2:
        score += 0.05  # moderate confidence

    return min(score, 1.0)


def score_all_citations(
    all_citations: list[str],
    sources: dict[str, set[str]],
    weights: dict[str, float] | None = None,
    bm25_scores: dict[str, float] | None = None,
) -> list[tuple[str, float]]:
    """
    Score all citations and return sorted (citation, score) list.
    """
    scored = [
        (cite, score_citation(cite, sources, weights, bm25_scores=bm25_scores))
        for cite in all_citations
    ]
    return sorted(scored, key=lambda x: -x[1])


def compute_f1(predicted: list[str], gold: set[str]) -> float:
    """Compute F1 for a single query (case-insensitive matching)."""
    if not predicted and not gold:
        return 1.0
    if not predicted or not gold:
        return 0.0
    gold_lower = {c.lower() for c in gold}
    pred_lower = {c.lower() for c in predicted}
    tp = len(pred_lower & gold_lower)
    p = tp / len(pred_lower)
    r = tp / len(gold_lower)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def optimize_threshold(
    scored_predictions: dict[str, list[tuple[str, float]]],
    val_gold: dict[str, set[str]],
    threshold_range: tuple[float, float, float] = (0.05, 0.95, 0.01),
) -> tuple[float, float]:
    """
    Sweep thresholds on val set to find the one maximizing macro-F1.
    """
    lo, hi, step = threshold_range
    thresholds = np.arange(lo, hi + step, step)

    best_f1, best_thresh = 0.0, lo

    for thresh in thresholds:
        f1s = []
        for qid, scored in scored_predictions.items():
            preds = [c for c, s in scored if s >= thresh]
            gold = val_gold.get(qid, set())
            if gold:
                f1s.append(compute_f1(preds, gold))

        macro = sum(f1s) / len(f1s) if f1s else 0.0
        if macro > best_f1:
            best_f1 = macro
            best_thresh = float(thresh)

    return best_thresh, best_f1


## Stage 1 — Query Analysis

Corpus-grounded legal query understanding.

### `agent/prompts/query_analyzer_prompt.txt`

```text
You are a Swiss law examiner (Prüfungsexperte) analyzing a legal scenario.
Your task is to identify ALL relevant Swiss legal provisions and leading
cases (Leitentscheide) that a well-prepared candidate should cite.

Think step by step like a Swiss law professor:

1. CLASSIFY the legal domain(s): Civil law (ZGB/OR), Criminal law (StGB/StPO),
   Public law (BV/VwVG), Social insurance (ATSG/IVG/UVG/AVIG/BVG), International
   private law (IPRG), Debt collection (SchKG), Procedural (ZPO/StPO/BGG), etc.

2. IDENTIFY the specific legal issues (Rechtsfragen):
   - What are the main legal questions?
   - What sub-questions arise from each main question?
   - What procedural questions are relevant (standing, jurisdiction, appeal)?

3. For each issue, REASON about applicable provisions:
   - Which specific articles govern this issue?
   - Which Absatz (paragraph) is most directly relevant?
   - Are there related articles that modify, supplement, or define terms?
   - What are the leading BGE decisions (Leitentscheide) on this point?
   - Are there recent numbered decisions (e.g., 4A_xxx/20xx) on this?

4. Consider PROCEDURAL citations (often missed but always required):
   - Jurisdiction and competence articles (ZPO 1-12, StPO 1-15)
   - Procedural standing / Legitimation (BGG 76, 89, 115)
   - Appellate procedure provisions (BGG 72/78/82/113, StPO 393-428)
   - Cost and fee provisions (BGG 65-68, ZPO 106-107)
   - Time limits for appeal (BGG 100, ZPO 321)

5. Consider CROSS-REFERENCES:
   - Articles that reference other articles
   - General provisions applying alongside specific ones (e.g., OR 97 with specific liability)
   - Constitutional provisions underlying specific rules (BV 9/29/32 for fundamental rights)
   - ATSG general provisions alongside specific social insurance laws

6. Extract any EXPLICIT CITATIONS already in the query text:
   - If the query mentions "Art. 221 Abs. 1 lit. b StPO", cite it verbatim
   - These are guaranteed gold citations

7. Generate GERMAN SEARCH QUERIES: For each legal issue, create 3-5 focused
   German search queries using precise legal terminology that would appear in
   statutory articles and court decisions. These will be used for BM25 retrieval.

Output your analysis as JSON:
{
  "law_areas": ["Strafprozessrecht", "Strafrecht", ...],
  "explicit_citations": ["Art. 221 Abs. 1 StPO", ...],
  "legal_issues": [
    {
      "issue": "Conditions for pre-trial detention",
      "german_terms": ["Untersuchungshaft", "Kollusionsgefahr", "Fluchtgefahr"],
      "articles": ["Art. 221 Abs. 1 StPO", "Art. 222 StPO"],
      "cases": ["BGE 137 IV 122 E. 6.2"]
    }
  ],
  "procedural_articles": ["Art. 393 Abs. 1 StPO", "Art. 100 Abs. 1 BGG"],
  "candidate_articles": ["Art. 221 Abs. 1 StPO", "Art. 222 StPO", ...],
  "candidate_cases": ["BGE 137 IV 122 E. 6.2", ...],
  "search_queries_de": [
    "Verlängerung Untersuchungshaft Kollusionsgefahr StPO",
    "Haftentlassungsgesuch Verhältnismässigkeit",
    ...
  ],
  "search_queries_en": ["pre-trial detention collusion risk proportionality", ...]
}

```

### `stage1_query_analysis.py`

In [33]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage1_query_analysis.py"

"""
stage1_query_analysis.py -- STAGE 1: Corpus-Grounded Legal Query Understanding

DESIGN PHILOSOPHY:
  A 4B model doesn't know Swiss law. But if you SHOW it evidence from the corpus,
  it becomes dramatically more accurate. This stage builds scaffolding around the
  LLM so it effectively acts as a human lawyer analyzing the query.

ARCHITECTURE (6-step pipeline, each step feeds the next):

  Step 1: EXPLICIT EXTRACTION (regex, instant)
    Extract any citation strings verbatim from the query text.
    These are free recall — always included regardless of everything else.

  Step 2: BM25 PROBE (CPU, <1s)
    Run the raw English query against BM25 to get initial signal.
    The top-10 results tell us WHICH law areas are relevant.
    This grounds all subsequent LLM reasoning in corpus evidence.

  Step 3: DOMAIN SCAFFOLDING (CPU, instant)
    From the BM25 probe results, extract:
      - Which law abbreviations appeared (StPO, ATSG, BGG, ...)
      - Their English names from the KB (Swiss Criminal Procedure Code, ...)
      - Their German keywords from the KB
    This creates a "cheat sheet" we inject into the LLM prompt.

  Step 4: LLM ANALYSIS (GPU, ~5-10s)
    With the domain scaffolding injected, the LLM:
      - Classifies legal domains (guided by BM25 evidence, not guessing)
      - Identifies legal issues with German Fachbegriffe
      - Suggests candidate citations it knows from training
      - Generates focused German search queries
    The LLM sees: "BM25 found articles from StPO (Criminal Procedure).
    The query mentions detention. What specific legal issues apply?"

  Step 5: HyDE — HYPOTHETICAL DOCUMENT EMBEDDING (GPU, ~5-10s)
    The LLM generates what a RELEVANT German legal article WOULD say.
    This is NOT a translation — it's a hypothetical document in the style
    of a Swiss statutory article. BM25 matches this against real articles
    far better than matching a short English query.

    Example: Query "Can pre-trial detention be extended?"
    HyDE output: "Die Untersuchungshaft kann vom Zwangsmassnahmengericht
    auf Antrag der Staatsanwaltschaft verlängert werden, wenn die
    Haftgründe nach Art. 221 weiterhin bestehen und die Verhältnismässigkeit
    gewahrt bleibt..."

    This 50-word German paragraph will match BM25 articles that a 5-word
    English query never could.

  Step 6: CROSS-LINGUAL TERM EXPANSION (CPU, instant)
    From the KB, expand English concepts to German equivalents:
      - "pre-trial detention" → "Untersuchungshaft" (from KB keyword mapping)
      - "disability insurance" → "Invalidenversicherung"
    These are exact corpus-grounded mappings, not LLM guesses.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage1_query_analysis.py                              # all val queries
#   python stage1_query_analysis.py --split test                 # test queries
#   python stage1_query_analysis.py --query "A claimant was..."  # single query debug
#   python stage1_query_analysis.py --backend anthropic          # use Claude API
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage1_{split}.json
#   Per query: {
#     explicit_citations, law_areas, legal_issues, candidate_articles,
#     candidate_cases, search_queries_de, search_queries_en,
#     hyde_documents_de, bm25_probe_laws, domain_context,
#     procedural_articles, cross_lingual_terms
#   }
"""



sys.path.insert(0, str(Path(__file__).parent))


CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# OFFLINE DOMAIN KNOWLEDGE (loaded once, shared across all queries)
# =============================================================================

_DOMAIN_KNOWLEDGE = None  # Lazy-loaded singleton


def _load_domain_knowledge() -> dict:
    """
    Build domain scaffolding from the corpus KB.
    This creates the "cheat sheet" that grounds LLM reasoning.

    Loads from laws_knowledge_base.jsonl:
      - law_taxonomy:       {abbrev: {name_en, name_de, sr_number}}
      - en_to_de_terms:     {"disability insurance": "Invalidenversicherung", ...}
      - law_area_groups:    {"Criminal": ["StGB", "StPO", ...], ...}
      - keyword_index:      {"Untersuchungshaft": ["Art. 221 StPO", ...]}
      - citation_to_text:   {citation_canon: "short article text snippet"}   ← NEW
      - valid_citations:    set of all citation_canon strings in corpus       ← NEW
    """
    global _DOMAIN_KNOWLEDGE
    if _DOMAIN_KNOWLEDGE is not None:
        return _DOMAIN_KNOWLEDGE

    print("  Loading domain knowledge from KB ...")

    law_taxonomy = {}
    en_to_de_terms = {}
    keyword_to_articles = {}
    law_to_en_name = {}
    law_to_de_name = {}
    citation_to_text = {}   # Fix 1: citation_canon -> article text snippet
    valid_citations = set() # Fix 3: all known citation_canon strings

    if KB_JSONL.exists():
        with open(KB_JSONL, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except Exception:
                    continue

                canon = rec.get("citation_canon", "")
                if not canon:
                    continue

                valid_citations.add(canon)  # Fix 3

                law_info = rec.get("law", {})
                abbrev = law_info.get("law_abbreviation", "")
                name_en = law_info.get("law_name_en", "")
                # KB has no "law_name_de" — German name is in law_title_clean
                name_de = (law_info.get("law_name_de", "")
                           or law_info.get("law_title_clean", ""))
                sr = law_info.get("sr_number", "")

                if abbrev and abbrev not in law_taxonomy:
                    law_taxonomy[abbrev] = {
                        "name_en": name_en,
                        "name_de": name_de,
                        "sr_number": sr,
                    }

                if abbrev and name_en:
                    law_to_en_name[abbrev] = name_en
                if abbrev and name_de:
                    law_to_de_name[abbrev] = name_de

                # English→German term mapping from law names
                if name_en and name_de:
                    en_to_de_terms[name_en.lower()] = name_de

                # German keyword → article mapping
                kws = rec.get("keywords", {})
                for kw in (kws.get("provision_keywords_de") or []):
                    if isinstance(kw, str) and kw:
                        keyword_to_articles.setdefault(kw.lower(), []).append(canon)

                # Note: provision_keywords_en does not exist in the KB.
                # Cross-lingual terms come from law names + hardcoded dict below.

                # Fix 1: build article text snippets for BM25 probe injection
                text = (rec.get("search", {}).get("search_text_de", "")
                        or rec.get("content", {}).get("text_clean_de", "")
                        or "")
                if text:
                    citation_to_text[canon] = text[:250]

    LAW_AREA_MAP = {
        "Criminal law": ["StGB", "MStG", "BetmG", "SVG"],
        "Criminal procedure": ["StPO", "JStPO", "StBOG"],
        "Civil law": ["ZGB", "OR", "PrHG", "DSG"],
        "Civil procedure": ["ZPO"],
        "Public law": ["BV", "VwVG", "BGG", "RPG", "USG", "EnG"],
        "Social insurance": ["ATSG", "IVG", "UVG", "AVIG", "BVG", "AHV", "ELG", "KVG", "FamZG"],
        "Immigration": ["AIG", "AsylG"],
        "Debt enforcement": ["SchKG"],
        "Private international law": ["IPRG", "IRSG"],
        "Intellectual property": ["URG", "MSchG", "PatG"],
        "Federal court procedure": ["BGG", "BGerR"],
        "Tax law": ["LIFD", "LHID", "LTVA"],
    }
    law_to_areas = {}
    for area, abbrevs in LAW_AREA_MAP.items():
        for abbrev in abbrevs:
            law_to_areas.setdefault(abbrev, []).append(area)

    # Hardcoded EN→DE legal term dictionary for common query vocabulary.
    # The KB lacks provision_keywords_en, so this bridges the cross-lingual gap.
    _HARDCODED_EN_DE = {
        "pre-trial detention": "Untersuchungshaft",
        "pretrial detention": "Untersuchungshaft",
        "detention": "Haft",
        "collusion": "Kollusionsgefahr",
        "risk of flight": "Fluchtgefahr",
        "proportionality": "Verhältnismässigkeit",
        "disability insurance": "Invalidenversicherung",
        "invalidity": "Invalidität",
        "earning capacity": "Erwerbsfähigkeit",
        "incapacity for work": "Arbeitsunfähigkeit",
        "social insurance": "Sozialversicherung",
        "unemployment insurance": "Arbeitslosenversicherung",
        "accident insurance": "Unfallversicherung",
        "criminal law": "Strafrecht",
        "criminal procedure": "Strafprozessrecht",
        "civil procedure": "Zivilprozessrecht",
        "appeal": "Beschwerde",
        "cassation": "Kassation",
        "federal court": "Bundesgericht",
        "cantonal court": "Kantonsgericht",
        "statute of limitations": "Verjährung",
        "prescription": "Verjährung",
        "negligence": "Fahrlässigkeit",
        "intent": "Vorsatz",
        "self-defense": "Notwehr",
        "custody": "Sorgerecht",
        "visitation": "Besuchsrecht",
        "child welfare": "Kindeswohl",
        "child protection": "Kindesschutz",
        "divorce": "Scheidung",
        "maintenance": "Unterhalt",
        "alimony": "Unterhaltsbeitrag",
        "inheritance": "Erbrecht",
        "will": "Testament",
        "testamentary": "testamentarisch",
        "executor": "Willensvollstrecker",
        "heir": "Erbe",
        "contract": "Vertrag",
        "damages": "Schadenersatz",
        "liability": "Haftung",
        "tort": "Haftpflicht",
        "good faith": "Treu und Glauben",
        "abuse of rights": "Rechtsmissbrauch",
        "property": "Eigentum",
        "possession": "Besitz",
        "bona fide": "gutgläubig",
        "acquisition": "Erwerb",
        "mortgage": "Hypothek",
        "lease": "Miete",
        "tenancy": "Mietvertrag",
        "employment": "Arbeitsvertrag",
        "dismissal": "Kündigung",
        "termination": "Kündigung",
        "construction": "Werkvertrag",
        "mandate": "Auftrag",
        "asylum": "Asyl",
        "deportation": "Ausschaffung",
        "expulsion": "Landesverweisung",
        "residence permit": "Aufenthaltsbewilligung",
        "debt enforcement": "Schuldbetreibung",
        "bankruptcy": "Konkurs",
        "seizure": "Pfändung",
        "constitutional rights": "Grundrechte",
        "freedom of expression": "Meinungsfreiheit",
        "right to be heard": "rechtliches Gehör",
        "due process": "faires Verfahren",
        "recusal": "Ausstand",
        "legal aid": "unentgeltliche Rechtspflege",
        "power of attorney": "Vollmacht",
        "arbitration": "Schiedsgerichtsbarkeit",
        "recognition": "Anerkennung",
        "enforcement": "Vollstreckung",
        "acquittal": "Freispruch",
        "conviction": "Verurteilung",
        "sentencing": "Strafzumessung",
        "probation": "bedingte Strafe",
        "parole": "bedingte Entlassung",
        "confiscation": "Einziehung",
        "money laundering": "Geldwäscherei",
        "fraud": "Betrug",
        "theft": "Diebstahl",
        "robbery": "Raub",
        "embezzlement": "Veruntreuung",
        "forgery": "Urkundenfälschung",
        "sexual assault": "sexuelle Nötigung",
        "domestic violence": "häusliche Gewalt",
        "trafficking": "Menschenhandel",
        "tax evasion": "Steuerhinterziehung",
        "insolvency": "Zahlungsunfähigkeit",
        "surety": "Bürgschaft",
        "guarantee": "Garantie",
        "set-off": "Verrechnung",
        "assignment": "Zession",
        "unjust enrichment": "ungerechtfertigte Bereicherung",
        "servitude": "Dienstbarkeit",
        "easement": "Grunddienstbarkeit",
        "co-ownership": "Miteigentum",
        "condominium": "Stockwerkeigentum",
    }
    en_to_de_terms.update(_HARDCODED_EN_DE)

    # Common procedural articles (BGG) that appear in nearly every Federal Court case.
    # These are almost always cited but never in the candidate pool because they're
    # generic procedure, not substance. Inject them by default.
    PROCEDURAL_ARTICLES = [
        "Art. 42 BGG", "Art. 42 Abs. 1 BGG", "Art. 42 Abs. 2 BGG",
        "Art. 66 Abs. 1 BGG", "Art. 68 Abs. 2 BGG",
        "Art. 72 Abs. 1 BGG", "Art. 72 Abs. 2 BGG",
        "Art. 74 Abs. 1 BGG", "Art. 75 BGG", "Art. 75 Abs. 1 BGG",
        "Art. 76 Abs. 1 BGG",
        "Art. 82 BGG", "Art. 83 BGG",
        "Art. 90 BGG", "Art. 93 Abs. 1 BGG",
        "Art. 95 BGG", "Art. 97 Abs. 1 BGG",
        "Art. 100 Abs. 1 BGG", "Art. 105 Abs. 1 BGG", "Art. 106 Abs. 2 BGG",
        "Art. 107 Abs. 2 BGG", "Art. 113 BGG",
    ]

    _DOMAIN_KNOWLEDGE = {
        "law_taxonomy": law_taxonomy,
        "en_to_de_terms": en_to_de_terms,
        "keyword_to_articles": keyword_to_articles,
        "law_to_en_name": law_to_en_name,
        "law_to_de_name": law_to_de_name,
        "law_area_map": LAW_AREA_MAP,
        "law_to_areas": law_to_areas,
        "citation_to_text": citation_to_text,   # Fix 1
        "valid_citations": valid_citations,       # Fix 3
        "procedural_articles": PROCEDURAL_ARTICLES,
    }

    print(f"    Law abbreviations: {len(law_taxonomy):,}")
    print(f"    EN->DE term mappings: {len(en_to_de_terms):,}")
    print(f"    DE keywords indexed: {len(keyword_to_articles):,}")
    print(f"    Article texts indexed: {len(citation_to_text):,}")
    print(f"    Valid citations: {len(valid_citations):,}")
    return _DOMAIN_KNOWLEDGE


# =============================================================================
# STEP 2: BM25 PROBE — get corpus evidence before LLM runs
# =============================================================================

_SPARSE_RETRIEVER = None


def _get_sparse_retriever():
    """Lazy-load BM25 retriever (needed for probe + MAS stages)."""
    global _SPARSE_RETRIEVER
    if _SPARSE_RETRIEVER is None:
        if STATUTORY_BM25_PKL.exists():
            _SPARSE_RETRIEVER = SparseRetriever()
        else:
            print("  WARNING: BM25 index not found. Probe step skipped.")
    return _SPARSE_RETRIEVER


def bm25_probe(query_text: str, top_k: int = 20) -> dict:
    """
    Quick BM25 search to get corpus-grounded signal BEFORE the LLM runs.

    Returns:
      - top_articles: [(citation, score), ...] top statutory BM25 hits
      - top_cases: [(citation, score), ...] top case-law BM25 hits
      - detected_laws: Counter of law abbreviations in top statutory results
      - detected_law_names: {abbrev: english_name} for detected laws
      - probe_context: formatted string to inject into LLM prompt
    """
    sparse = _get_sparse_retriever()
    if sparse is None:
        return {
            "top_articles": [],
            "top_cases": [],
            "detected_laws": Counter(),
            "detected_law_names": {},
            "probe_context": "",
            "top_citations": [],
            "top_case_citations": [],
            "top_with_text": [],
        }

    # Stage 1 should ground the LLM in statutory text, not let case-law
    # dominate the initial scaffold. We still keep a separate case probe for
    # downstream recall, but the LLM sees article evidence first.
    article_results = sparse.search_statutory([query_text], top_k=top_k)
    case_results = sparse.search_caselaw([query_text], top_k=min(top_k, 10))

    # Extract law abbreviations from top results
    # BM25 IDs use uppercase (STPO, STGB) but KB uses mixed-case (StPO, StGB).
    # Normalise via upper_to_abbrev so law name lookups work correctly.
    dk = _load_domain_knowledge()
    upper_to_abbrev = {ab.upper(): ab for ab in dk["law_taxonomy"]}

    detected_laws = Counter()
    for cite, _ in article_results:
        if cite.startswith("Art."):
            parts = cite.split()
            if parts:
                raw_abbrev = parts[-1]
                abbrev = upper_to_abbrev.get(raw_abbrev.upper(), raw_abbrev)
                detected_laws[abbrev] += 1

    # Map to English names
    detected_law_names = {}
    for abbrev in detected_laws:
        en_name = dk["law_to_en_name"].get(abbrev, "")
        if en_name:
            detected_law_names[abbrev] = en_name

    # Format context string for LLM injection
    probe_lines = []
    for abbrev, count in detected_laws.most_common(10):
        en_name = detected_law_names.get(abbrev, "unknown")
        probe_lines.append(f"  - {abbrev} ({en_name}): {count} statutory hits in top-{top_k}")

    probe_context = ""
    if probe_lines:
        probe_context = (
            "BM25 CORPUS EVIDENCE (top statutory articles matching your query):\n"
            + "\n".join(probe_lines)
            + "\n\nThese law areas are likely relevant. Use this as a starting point."
        )

    # Fix 1: enrich top results with article text for LLM injection
    citation_to_text = dk["citation_to_text"]
    top_with_text = []
    for i, (cite, _) in enumerate(article_results[:10]):
        text = citation_to_text.get(cite, "")
        # strip the law title prefix (everything up to " | Art.")
        if " | Art." in text:
            text = text[text.index(" | Art.") + 3:]
        elif " | BGE" in text:
            text = text[text.index(" | BGE") + 3:]
        top_with_text.append({
            "idx": i,
            "citation": cite,
            "text": text[:200].strip() if text else "",
        })

    top_citations = [cite for cite, _ in article_results[:10]]
    top_case_citations = [cite for cite, _ in case_results[:10]]

    return {
        "top_articles": article_results[:top_k],
        "top_cases": case_results[:min(top_k, 10)],
        "detected_laws": dict(detected_laws),
        "detected_law_names": detected_law_names,
        "probe_context": probe_context,
        "top_citations": top_citations,
        "top_case_citations": top_case_citations,
        "top_with_text": top_with_text,   # Fix 1
    }


# =============================================================================
# STEP 3: DOMAIN SCAFFOLDING — build a "cheat sheet" for the LLM
# =============================================================================

def build_domain_context(detected_laws: dict, query_text: str,
                         top_with_text: list | None = None) -> str:
    """
    Build a domain context string injected into the LLM prompt.

    Fix 1: now includes actual article text snippets from BM25 probe so the
    LLM reads corpus evidence rather than recalling from training memory.
    """
    dk = _load_domain_knowledge()
    sections = []

    # 1. Fix 1: BM25 probe results WITH article text — the core grounding signal.
    #    Each entry is numbered so the LLM can reference them by index.
    if top_with_text:
        probe_lines = []
        for entry in top_with_text:
            citation = entry["citation"]
            text = entry["text"]
            snippet = f"  [{entry['idx']}] {citation}"
            if text:
                snippet += f"\n      \"{text}\""
            probe_lines.append(snippet)
        sections.append(
            "BM25 CORPUS EVIDENCE — top matching articles (THESE EXIST IN THE CORPUS):\n"
            + "\n".join(probe_lines)
            + "\n\nThese are real articles. Use their terminology in your German search queries."
        )

    # 2. Detected law areas with full names
    if detected_laws:
        law_lines = []
        for abbrev, count in sorted(detected_laws.items(), key=lambda x: -x[1])[:15]:
            en = dk["law_to_en_name"].get(abbrev, "")
            de = dk["law_to_de_name"].get(abbrev, "")
            law_lines.append(f"  {abbrev} ({count} hits): {en} / {de}")
        sections.append("DETECTED LAW AREAS:\n" + "\n".join(law_lines))

    # 3. Cross-lingual term suggestions
    query_lower = query_text.lower()
    matched_terms = []
    for en_term, de_term in dk["en_to_de_terms"].items():
        if len(en_term) > 4 and en_term in query_lower:
            matched_terms.append((en_term, de_term))
    if matched_terms:
        term_lines = [f"  \"{en}\" → \"{de}\"" for en, de in matched_terms[:10]]
        sections.append(
            "CROSS-LINGUAL TERM MAPPINGS (English → German Fachbegriff):\n"
            + "\n".join(term_lines)
        )

    # 4. Swiss law area taxonomy (always included as reference)
    tax_lines = []
    for area, abbrevs in dk["law_area_map"].items():
        tax_lines.append(f"  {area}: {', '.join(abbrevs)}")
    sections.append("SWISS LAW AREA TAXONOMY:\n" + "\n".join(tax_lines))

    return "\n\n".join(sections)


def _dedupe_keep_order(items) -> list:
    seen = set()
    out = []
    for item in items:
        if not item or item in seen:
            continue
        seen.add(item)
        out.append(item)
    return out


def _extract_law_abbrev(citation: str) -> str:
    if not isinstance(citation, str) or not citation.startswith("Art."):
        return ""
    parts = citation.rsplit(" ", 1)
    if len(parts) != 2:
        return ""
    dk = _load_domain_knowledge()
    upper_to_abbrev = {ab.upper(): ab for ab in dk["law_taxonomy"]}
    raw = parts[1].strip()
    return upper_to_abbrev.get(raw.upper(), raw)


def _derive_law_hints(detected_laws: dict, explicit_citations: list[str] | set[str]) -> list[str]:
    hints = list(detected_laws.keys())
    for citation in explicit_citations:
        abbrev = _extract_law_abbrev(citation)
        if abbrev:
            hints.append(abbrev)
    return _dedupe_keep_order(hints)


def _build_crosslingual_queries(cross_lingual_terms: dict, law_hints: list[str]) -> list[str]:
    de_terms = _dedupe_keep_order(cross_lingual_terms.values())
    queries = []

    for de_term in de_terms[:5]:
        for law_hint in law_hints[:2]:
            queries.append(f"{de_term} {law_hint}")
        queries.append(de_term)

    for i in range(min(len(de_terms), 4)):
        for j in range(i + 1, min(len(de_terms), 4)):
            queries.append(f"{de_terms[i]} {de_terms[j]}")

    return _dedupe_keep_order(q.strip() for q in queries if isinstance(q, str) and len(q.strip()) > 3)[:8]


def _normalize_law_areas(
    raw_areas: list,
    law_hints: list[str],
    cross_lingual_terms: dict,
) -> list[str]:
    dk = _load_domain_knowledge()
    law_to_areas = dk.get("law_to_areas", {})

    alias_to_area = {
        "strafrecht": "Criminal law",
        "betrug": "Criminal law",
        "strafprozessrecht": "Criminal procedure",
        "strafverfahren": "Criminal procedure",
        "zivilrecht": "Civil law",
        "familienrecht": "Civil law",
        "erbrecht": "Civil law",
        "bürgerliches recht": "Civil law",
        "privatrecht": "Civil law",
        "privatrechtsverhältnisse": "Civil law",
        "eigentumrecht": "Civil law",
        "schuldrecht": "Civil law",
        "arbeitsrecht": "Civil law",
        "gesundheitsrecht": "Civil law",
        "handelsrecht": "Civil law",
        "wirtschaftsrecht": "Civil law",
        "produkthaftung": "Civil law",
        "konsumkreditrecht": "Civil law",
        "baulaw": "Civil law",
        "schadensrecht": "Civil law",
        "zivilprozessrecht": "Civil procedure",
        "handelsprozessrecht": "Civil procedure",
        "verwaltungsrecht": "Public law",
        "bundesrecht": "Public law",
        "recht auf gehör": "Public law",
        "verhältnismässigkeit": "Public law",
        "sozialversicherung": "Social insurance",
        "sozialversicherungsrecht": "Social insurance",
        "sozialversicherungrecht": "Social insurance",
        "wohlfahrtsrecht": "Social insurance",
        "immigrationrecht": "Immigration",
        "schuldbetreibung": "Debt enforcement",
        "schuldbeklagung": "Debt enforcement",
        "konkursrecht": "Debt enforcement",
        "internationales privatrecht": "Private international law",
        "privatinternationales recht": "Private international law",
        "privat- und internationales recht": "Private international law",
        "markenrecht": "Intellectual property",
        "wettbewerbsrecht": "Intellectual property",
        "schutz vor unlauterem wettbewerb": "Intellectual property",
        "bundesgerichtsrecht": "Federal court procedure",
        "bgg": "Federal court procedure",
    }

    normalized = []
    for raw_area in raw_areas or []:
        if not isinstance(raw_area, str):
            continue
        area = raw_area.strip()
        if not area:
            continue
        if area in dk["law_area_map"]:
            normalized.append(area)
            continue

        mapped = alias_to_area.get(area.lower())
        if mapped:
            normalized.append(mapped)
            continue

        normalized.extend(law_to_areas.get(area, []))
        normalized.extend(law_to_areas.get(area.upper(), []))

    if not normalized:
        for law_hint in law_hints:
            normalized.extend(law_to_areas.get(law_hint, []))

    if not normalized and cross_lingual_terms:
        term_to_area = [
            (("pre-trial detention", "pretrial detention", "detention", "collusion",
              "risk of flight", "self-defense", "acquittal", "conviction",
              "sentencing", "probation", "parole", "confiscation",
              "money laundering", "fraud", "theft", "robbery",
              "embezzlement", "forgery", "sexual assault",
              "domestic violence", "trafficking", "tax evasion"), "Criminal law"),
            (("appeal", "constitutional rights", "right to be heard",
              "due process", "recusal", "legal aid"), "Public law"),
            (("social insurance", "disability insurance", "invalidity",
              "earning capacity", "incapacity for work",
              "unemployment insurance", "accident insurance"), "Social insurance"),
            (("asylum", "deportation", "expulsion", "residence permit"), "Immigration"),
            (("debt enforcement", "bankruptcy", "seizure"), "Debt enforcement"),
            (("arbitration", "recognition"), "Private international law"),
            (("trademark", "unfair competition"), "Intellectual property"),
            (("maintenance", "alimony", "custody", "visitation", "child welfare",
              "child protection", "divorce", "inheritance", "will", "executor",
              "heir", "contract", "damages", "liability", "tort", "good faith",
              "abuse of rights", "property", "possession", "acquisition",
              "mortgage", "lease", "tenancy", "employment", "dismissal",
              "termination", "construction", "mandate", "power of attorney",
              "guarantee", "set-off", "assignment", "unjust enrichment",
              "servitude", "easement", "co-ownership", "condominium"), "Civil law"),
        ]
        for en_term in cross_lingual_terms:
            low = en_term.lower()
            for keywords, area in term_to_area:
                if low in keywords:
                    normalized.append(area)

    return _dedupe_keep_order(normalized)[:4]


def _build_fallback_hyde(de_terms: list[str], law_hints: list[str]) -> list[str]:
    if not de_terms:
        return []
    law_hint = law_hints[0] if law_hints else "dem einschlägigen Schweizer Recht"
    term_phrase = ", ".join(de_terms[:3])
    return [
        "Die einschlägigen Bestimmungen zu "
        f"{term_phrase} regeln die Anspruchsvoraussetzungen, mögliche Einwendungen "
        f"und die gerichtliche Durchsetzung nach {law_hint}. Massgeblich sind "
        "insbesondere die materiellen Voraussetzungen, die Beweisführung und die "
        "Verhältnismässigkeit der beantragten Rechtsfolge."
    ]


# =============================================================================
# STEP 4 + 5: LLM ANALYSIS + HyDE GENERATION (single LLM call)
# =============================================================================

# Fix 2: tightened system prompt — LLM is a READER of corpus evidence, not a knowledge source.
# No candidate_articles or candidate_cases in the schema (BM25 probe provides those).
# LLM only selects relevant probe indices + generates German terminology + HyDE text.
_SYSTEM_PROMPT_TEMPLATE = """You are a Swiss law classifier with access to corpus search results.
Your job is to READ the BM25 evidence below and help retrieve more relevant articles.

{domain_context}

RULES — read carefully:
1. DO NOT invent article numbers. The corpus evidence above shows REAL articles.
2. DO NOT cite BGE cases from memory — they are unreliable. Only extract cases verbatim from the query.
3. Your value is: classifying the legal domain, extracting German Fachbegriffe, and generating BM25 search queries.

TASK: Analyze the legal scenario and output JSON with exactly these fields:

STEP 1 — CLASSIFY: Which law areas apply? Choose only from "DETECTED LAW AREAS" and "SWISS LAW AREA TAXONOMY" shown above.
Output the English taxonomy labels exactly as written in "SWISS LAW AREA TAXONOMY".

STEP 2 — IDENTIFY ISSUES: For each distinct legal issue (Rechtsfrage), list:
  - A short issue description (English)
  - The German legal terms (Fachbegriffe) that appear in Swiss statutory text for this issue.
    Base these on the article texts shown in "BM25 CORPUS EVIDENCE" above.

STEP 3 — SELECT RELEVANT PROBE RESULTS: From the numbered BM25 results [0]..[9] above,
list the indices of results relevant to the query. Example: [0, 2, 4].

STEP 4 — GERMAN SEARCH QUERIES: Generate 5 focused German BM25 keyword queries.
Use EXACT German legal terms from the article texts shown above. These queries will
be used to retrieve more articles from the same corpus.

STEP 5 — HYPOTHETICAL GERMAN ARTICLE (HyDE): Write 1-2 short paragraphs in German
describing what a relevant Swiss statutory article would say about this scenario.
Use only terminology consistent with the law areas identified in STEP 1.
Stay strictly within the identified legal domain — do not blend in unrelated areas.

STEP 6 — EXTRACT any citations written verbatim in the query text (exact string match only).

Output ONLY valid JSON, no other text:
{{
  "law_areas": ["Civil law", "Civil procedure"],
  "legal_issues": [
    {{
      "issue": "Restriction of visitation rights",
      "german_terms": ["Besuchsrecht", "persönlicher Verkehr", "Kindesinteresse", "Kindeswohl"]
    }}
  ],
  "relevant_probe_indices": [0, 1, 3],
  "search_queries_de": [
    "Besuchsrecht Einschränkung Kindesinteresse ZGB",
    "persönlicher Verkehr Kind Elternteil Gefährdung",
    "Kindesschutz Besuchsrecht Einschränkung Gericht"
  ],
  "search_queries_en": ["restriction of visitation rights child welfare ZGB"],
  "hyde_documents_de": [
    "Das Gericht kann den persönlichen Verkehr zwischen dem Kind und dem nicht sorgeberechtigten Elternteil einschränken oder untersagen, wenn das Wohl des Kindes durch den Kontakt gefährdet wird. Voraussetzung ist, dass konkrete Anhaltspunkte für eine Gefährdung bestehen."
  ],
  "explicit_citations": []
}}"""


def analyze_single_query(
    query_text: str,
    backend: str = "local",
    model: str | None = None,
) -> dict:
    """
    Run the full 6-step query understanding pipeline.
    """
    t0 = time.time()
    results = {}

    # ── Step 1: Explicit citation extraction (regex) ─────────────────────
    explicit = extract_explicit_citations(query_text)
    results["explicit_citations_regex"] = sorted(explicit)

    # ── Step 2: BM25 probe — corpus evidence ─────────────────────────────
    probe = bm25_probe(query_text, top_k=20)
    results["bm25_probe"] = {
        "detected_laws": probe["detected_laws"],
        "detected_law_names": probe["detected_law_names"],
        "top_citations": probe.get("top_citations", []),
        "top_case_citations": probe.get("top_case_citations", []),
    }

    # ── Step 3: Domain scaffolding (Fix 1: inject article texts) ─────────
    top_with_text = probe.get("top_with_text", [])
    domain_context = build_domain_context(
        probe["detected_laws"], query_text, top_with_text=top_with_text
    )
    results["domain_context"] = domain_context

    # ── Step 4 + 5: LLM analysis + HyDE (single call) ───────────────────
    # Fix 2: new prompt schema — no candidate_articles/candidate_cases
    # Fix 4: enable_thinking=False, max_new_tokens=2048 (LLM classifies, not reasons)
    system_prompt = _SYSTEM_PROMPT_TEMPLATE.format(domain_context=domain_context)

    analysis = generate_json(
        system_prompt=system_prompt,
        user_prompt=query_text,
        max_new_tokens=2048,
        enable_thinking=False,
    )

    # ── LLM FAILURE FALLBACK ─────────────────────────────────────────────
    # If LLM returned empty/broken JSON (val_010 case), build minimal analysis
    # from BM25 probe + cross-lingual terms so downstream stages aren't starved.
    _llm_failed = (
        not analysis.get("law_areas")
        and not analysis.get("search_queries_de")
        and not analysis.get("hyde_documents_de")
    )
    if _llm_failed:
        print("  WARNING: LLM returned empty analysis — using BM25 probe fallback")
        dk_fallback = _load_domain_knowledge()
        # Derive fallback signals from BM25 + explicit query citations.
        query_lower_fb = query_text.lower()
        fb_cross_lingual_terms = {}
        for en_t, de_t in dk_fallback["en_to_de_terms"].items():
            if len(en_t) > 4 and en_t in query_lower_fb:
                fb_cross_lingual_terms[en_t] = de_t
        fb_terms = _dedupe_keep_order(fb_cross_lingual_terms.values())
        law_hints = _derive_law_hints(probe["detected_laws"], explicit)
        analysis["law_areas"] = _normalize_law_areas([], law_hints, fb_cross_lingual_terms)

        fb_queries = _build_crosslingual_queries(fb_cross_lingual_terms, law_hints)
        # Also add detected law keywords from probe top texts
        for entry in top_with_text[:5]:
            text = entry.get("text", "")
            if text:
                # Use first 40 chars as a keyword query
                fb_queries.append(text[:40].strip())
        analysis["search_queries_de"] = _dedupe_keep_order(fb_queries)[:8]
        analysis["search_queries_en"] = [query_text[:100]]
        analysis["legal_issues"] = [{"issue": "See BM25 probe", "german_terms": fb_terms[:5]}]
        analysis["hyde_documents_de"] = _build_fallback_hyde(fb_terms, law_hints)

    # ── Step 6: Cross-lingual term expansion ─────────────────────────────
    dk = _load_domain_knowledge()
    cross_lingual_terms = {}
    query_lower = query_text.lower()
    for en_term, de_term in dk["en_to_de_terms"].items():
        if len(en_term) > 4 and en_term in query_lower:
            cross_lingual_terms[en_term] = de_term
    results["cross_lingual_terms"] = cross_lingual_terms

    # ── Merge explicit citations (regex + LLM) ────────────────────────────
    # Only accept LLM-added citations that actually appear verbatim in the query
    llm_explicit = {
        c for c in analysis.get("explicit_citations", [])
        if c in query_text
    }
    results["explicit_citations"] = sorted(explicit | llm_explicit)
    law_hints = _derive_law_hints(probe["detected_laws"], results["explicit_citations"])

    # ── Fix 2: build candidate_articles from probe selection + probe top ──
    # LLM selects relevant probe indices; we map back to citation strings.
    relevant_indices = analysis.get("relevant_probe_indices", [])
    selected_citations = []
    for idx in relevant_indices:
        if isinstance(idx, int) and 0 <= idx < len(top_with_text):
            selected_citations.append(top_with_text[idx]["citation"])

    # candidate_articles = LLM-selected statutory probe results + statutory top-10
    probe_top_articles = [c for c in probe.get("top_citations", []) if c.startswith("Art.")]
    probe_top_cases = [c for c in probe.get("top_case_citations", []) if not c.startswith("Art.")]
    candidate_articles = list(dict.fromkeys(selected_citations + probe_top_articles))

    # Fix 3: post-validation — strip any LLM-hallucinated citations not in corpus
    valid_citations = dk["valid_citations"]
    # Only validate law citations (Art. ...) — court citations not in KB
    def _is_valid(c: str) -> bool:
        if c.startswith("Art."):
            return c in valid_citations
        return True  # court citations pass through

    candidate_articles = [c for c in candidate_articles if c.startswith("Art.") and _is_valid(c)]
    results["candidate_articles"] = candidate_articles

    # candidate_cases come from the separate case-law BM25 probe.
    results["candidate_cases"] = probe_top_cases

    # Standard fields from LLM analysis
    for k, default in [
        ("law_areas", []),
        ("legal_issues", []),
        ("search_queries_de", []),
        ("search_queries_en", []),
        ("hyde_documents_de", []),
    ]:
        results[k] = analysis.get(k, default)

    # Fix 3: strip hallucinated law citations from explicit_citations too.
    # Regex-derived citations come from literal query references and may use
    # alternate abbreviations (e.g. CO -> OR), so we keep them here and let the
    # verifier normalize or drop them later if needed.
    # Only validate the LLM-added ones.
    results["explicit_citations"] = sorted(
        explicit | {c for c in llm_explicit if _is_valid(c)}
    )
    law_hints = _derive_law_hints(probe["detected_laws"], results["explicit_citations"])
    results["law_areas"] = _normalize_law_areas(
        results.get("law_areas", []),
        law_hints,
        cross_lingual_terms,
    )

    # BM25 probe top for downstream stages
    results["bm25_probe_top"] = probe_top_articles
    results["bm25_probe_cases"] = probe_top_cases

    # ── Procedural articles: inject common BGG articles ────────────────
    # Nearly every Federal Court case cites these. They appear in ~60% of
    # gold citations but BM25 never retrieves them (too generic).
    dk_proc = _load_domain_knowledge()
    # Filter procedural articles to those that exist in the corpus
    proc_valid = dk_proc.get("valid_citations", set())
    procedural = [a for a in dk_proc.get("procedural_articles", [])
                  if a in proc_valid]
    results["procedural_articles"] = procedural

    # ── Normalize BM25 candidate casing (STPO→StPO) ───────────────────
    upper_to_abbrev = {ab.upper(): ab for ab in dk_proc["law_taxonomy"]}
    def _norm_casing(cite: str) -> str:
        if cite.startswith("Art."):
            parts = cite.rsplit(" ", 1)
            if len(parts) == 2:
                canon_ab = upper_to_abbrev.get(parts[1].upper(), parts[1])
                return f"{parts[0]} {canon_ab}"
        return cite
    results["candidate_articles"] = [_norm_casing(c) for c in results["candidate_articles"]]
    results["candidate_cases"] = [_norm_casing(c) for c in results.get("candidate_cases", [])]

    # Generate additional German queries from cross-lingual terms
    results["search_queries_de_crosslingual"] = _build_crosslingual_queries(
        cross_lingual_terms,
        law_hints,
    )

    # Combine all German search queries for stage 2
    all_de_queries = _dedupe_keep_order(
        results["search_queries_de"]
        + results.get("hyde_documents_de", [])
        + results["search_queries_de_crosslingual"]
    )
    results["all_search_queries_de"] = all_de_queries

    elapsed = time.time() - t0
    results["analysis_time_s"] = round(elapsed, 1)

    return results


# =============================================================================
# BATCH RUNNER
# =============================================================================

def run_batch_stage1(split: str, backend: str = "local", model: str | None = None,
              resume: bool = True, batch_size: int = 5) -> dict:
    """Run Stage 1 on all queries with batched LLM inference (batch_size queries at a time)."""
    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    df = pd.read_csv(csv_map[split])

    out_path = CHECKPOINTS_DIR / f"stage1_{split}.json"

    _load_domain_knowledge()
    _get_sparse_retriever()

    # Resume from checkpoint
    results = {}
    if resume and out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            results = json.load(f)
        failed = [qid for qid, r in results.items() if "error" in r]
        for qid in failed:
            del results[qid]
        if failed:
            print(f"  Resuming: {len(results)} done, {len(failed)} failed will be re-run: {failed}")
        else:
            print(f"  Resuming: {len(results)} queries already done")

    # Collect pending queries
    pending = []
    for _, row in df.iterrows():
        qid = row["query_id"]
        if qid not in results:
            pending.append(row)

    if not pending:
        print("  All queries already done.")
        return results

    print(f"  Processing {len(pending)} queries in batches of {batch_size}...")

    for batch_start in range(0, len(pending), batch_size):
        batch_rows = pending[batch_start : batch_start + batch_size]
        actual_bs = len(batch_rows)

        print(f"\n{'='*60}")
        print(f"  Batch {batch_start // batch_size + 1}: queries {batch_start + 1}-{batch_start + actual_bs} of {len(pending)}")
        print(f"{'='*60}")

        if backend == "local":
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

        # ── STEP A: Pre-compute BM25 probes + domain context ──
        batch_precomputed = []
        for row in batch_rows:
            qid = row["query_id"]
            query = row["query"]
            print(f"  [{batch_start + len(batch_precomputed) + 1}/{len(pending)}] {qid}: pre-computing...")

            t0_pre = time.time()
            explicit = extract_explicit_citations(query)
            probe = bm25_probe(query, top_k=20)
            top_with_text = probe.get("top_with_text", [])
            domain_context = build_domain_context(
                probe["detected_laws"], query, top_with_text=top_with_text
            )
            system_prompt = _SYSTEM_PROMPT_TEMPLATE.format(domain_context=domain_context)

            batch_precomputed.append({
                "row": row,
                "qid": qid,
                "query": query,
                "explicit": explicit,
                "probe": probe,
                "top_with_text": top_with_text,
                "domain_context": domain_context,
                "system_prompt": system_prompt,
            })
            print(f"    Pre-compute done in {time.time() - t0_pre:.1f}s")

        # ── STEP B: Batched LLM call ──
        system_prompts = [p["system_prompt"] for p in batch_precomputed]
        user_prompts = [p["query"] for p in batch_precomputed]

        print(f"\n  Running batched LLM call for {actual_bs} queries...")
        t0_llm = time.time()

        if backend == "local":
            try:
                analyses = generate_json_batch_multi_system(
                    system_prompts, user_prompts,
                    max_new_tokens=2048,
                    enable_thinking=False,
                )
            except Exception as e:
                print(f"  BATCH ERROR: {e}")
                traceback.print_exc()
                print("  Falling back to sequential processing...")
                analyses = []
                for sp, up in zip(system_prompts, user_prompts):
                    try:
                        gc.collect()
                        torch.cuda.empty_cache()
                        a = generate_json(
                            system_prompt=sp,
                            user_prompt=up,
                            max_new_tokens=2048,
                            enable_thinking=False,
                        )
                    except Exception as e2:
                        print(f"    Sequential fallback also failed: {e2}")
                        a = {}
                    analyses.append(a)
        else:
            analyses = [generate_json(sp, up) for sp, up in zip(system_prompts, user_prompts)]

        print(f"  LLM batch done in {time.time() - t0_llm:.1f}s ({(time.time() - t0_llm) / actual_bs:.1f}s/query)")

        # ── STEP C: Post-process each query ──
        dk = _load_domain_knowledge()
        valid_citations = dk["valid_citations"]

        for precomp, analysis in zip(batch_precomputed, analyses):
            qid = precomp["qid"]
            query = precomp["query"]
            explicit = precomp["explicit"]
            probe = precomp["probe"]
            top_with_text = precomp["top_with_text"]

            qr = {}  # query_results

            qr["explicit_citations_regex"] = sorted(explicit)
            qr["bm25_probe"] = {
                "detected_laws": probe["detected_laws"],
                "detected_law_names": probe["detected_law_names"],
                "top_citations": probe.get("top_citations", []),
                "top_case_citations": probe.get("top_case_citations", []),
            }
            qr["domain_context"] = precomp["domain_context"]

            # LLM failure fallback
            _llm_failed = (
                not analysis.get("law_areas")
                and not analysis.get("search_queries_de")
                and not analysis.get("hyde_documents_de")
            )
            if _llm_failed:
                print(f"  WARNING [{qid}]: LLM empty -- using BM25 probe fallback")
                dk_fb = _load_domain_knowledge()
                ql_fb = query.lower()
                fb_xl = {}
                for en_t, de_t in dk_fb["en_to_de_terms"].items():
                    if len(en_t) > 4 and en_t in ql_fb:
                        fb_xl[en_t] = de_t
                fb_terms = _dedupe_keep_order(fb_xl.values())
                lh = _derive_law_hints(probe["detected_laws"], explicit)
                analysis["law_areas"] = _normalize_law_areas([], lh, fb_xl)
                fb_q = _build_crosslingual_queries(fb_xl, lh)
                for entry in top_with_text[:5]:
                    t = entry.get("text", "")
                    if t:
                        fb_q.append(t[:40].strip())
                analysis["search_queries_de"] = _dedupe_keep_order(fb_q)[:8]
                analysis["search_queries_en"] = [query[:100]]
                analysis["legal_issues"] = [{"issue": "See BM25 probe", "german_terms": fb_terms[:5]}]
                analysis["hyde_documents_de"] = _build_fallback_hyde(fb_terms, lh)

            # Cross-lingual terms
            cross_lingual_terms = {}
            ql = query.lower()
            for en_term, de_term in dk["en_to_de_terms"].items():
                if len(en_term) > 4 and en_term in ql:
                    cross_lingual_terms[en_term] = de_term
            qr["cross_lingual_terms"] = cross_lingual_terms

            # Merge explicit citations
            llm_explicit = {c for c in analysis.get("explicit_citations", []) if c in query}
            qr["explicit_citations"] = sorted(explicit | llm_explicit)
            law_hints = _derive_law_hints(probe["detected_laws"], qr["explicit_citations"])

            # Build candidate_articles
            relevant_indices = analysis.get("relevant_probe_indices", [])
            selected_citations = []
            for ri in relevant_indices:
                if isinstance(ri, int) and 0 <= ri < len(top_with_text):
                    selected_citations.append(top_with_text[ri]["citation"])

            probe_top_articles = [c for c in probe.get("top_citations", []) if c.startswith("Art.")]
            probe_top_cases = [c for c in probe.get("top_case_citations", []) if not c.startswith("Art.")]
            candidate_articles = list(dict.fromkeys(selected_citations + probe_top_articles))

            def _is_valid(c):
                return c in valid_citations if c.startswith("Art.") else True

            candidate_articles = [c for c in candidate_articles if c.startswith("Art.") and _is_valid(c)]
            qr["candidate_articles"] = candidate_articles
            qr["candidate_cases"] = probe_top_cases

            for k, default in [
                ("law_areas", []), ("legal_issues", []),
                ("search_queries_de", []), ("search_queries_en", []),
                ("hyde_documents_de", []),
            ]:
                qr[k] = analysis.get(k, default)

            qr["explicit_citations"] = sorted(explicit | {c for c in llm_explicit if _is_valid(c)})
            law_hints = _derive_law_hints(probe["detected_laws"], qr["explicit_citations"])
            qr["law_areas"] = _normalize_law_areas(qr.get("law_areas", []), law_hints, cross_lingual_terms)

            qr["bm25_probe_top"] = probe_top_articles
            qr["bm25_probe_cases"] = probe_top_cases

            dk_proc = _load_domain_knowledge()
            proc_valid = dk_proc.get("valid_citations", set())
            qr["procedural_articles"] = [a for a in dk_proc.get("procedural_articles", []) if a in proc_valid]

            upper_to_abbrev = {ab.upper(): ab for ab in dk_proc["law_taxonomy"]}
            def _norm_casing(cite):
                if cite.startswith("Art."):
                    parts = cite.rsplit(" ", 1)
                    if len(parts) == 2:
                        return f"{parts[0]} {upper_to_abbrev.get(parts[1].upper(), parts[1])}"
                return cite
            qr["candidate_articles"] = [_norm_casing(c) for c in qr["candidate_articles"]]
            qr["candidate_cases"] = [_norm_casing(c) for c in qr.get("candidate_cases", [])]

            qr["search_queries_de_crosslingual"] = _build_crosslingual_queries(cross_lingual_terms, law_hints)
            all_de = _dedupe_keep_order(
                qr["search_queries_de"] + qr.get("hyde_documents_de", []) + qr["search_queries_de_crosslingual"]
            )
            qr["all_search_queries_de"] = all_de

            results[qid] = qr

            print(f"  {qid}: laws={probe.get('detected_laws', {})}, "
                  f"explicit={len(qr['explicit_citations'])}, "
                  f"candidates={len(candidate_articles)} art + {len(probe_top_cases)} cases, "
                  f"DE queries={len(qr['search_queries_de'])}")

        # Save after each batch
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nStage 1 complete. Saved: {out_path}")
    return results



### Stage 1 runner

In [34]:
stage1_outputs = run_stage1(split=SPLIT, backend=BACKEND)

  Loading domain knowledge from KB ...
    Law abbreviations: 1,125
    EN->DE term mappings: 2,114
    DE keywords indexed: 69,837
    Article texts indexed: 171,654
    Valid citations: 171,654
  Loading combined BM25 index ...


  Loading BM25 chunks: 100%|##################| 216/216 [02:09<00:00,  1.67it/s]


  Converting BM25 doc_freqs to sparse matrix ...
    Sparse matrix: 2,156,832 docs x 822,251 terms (212,228,366 non-zero) in 34.5s
    171,654 law  +  1,985,178 court  =  2,156,832 total
  Processing 10 queries in batches of 5...

  Batch 1: queries 1-5 of 10
  [1/10] val_001: pre-computing...
    Pre-compute done in 1.3s
  [2/10] val_002: pre-computing...
    Pre-compute done in 1.8s
  [3/10] val_003: pre-computing...
    Pre-compute done in 2.2s
  [4/10] val_004: pre-computing...
    Pre-compute done in 1.5s
  [5/10] val_005: pre-computing...
    Pre-compute done in 1.9s

  Running batched LLM call for 5 queries...
  Loading Qwen3-32B from: Qwen/Qwen3-32B


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

  Qwen3-32B loaded. VRAM: ~64GB (bf16)
  LLM batch done in 763.5s (152.7s/query)
  val_001: laws={'StPO': 13, 'StGB': 1, 'StBOG': 2, 'BGG': 1, 'ZGB': 2, 'OR': 1}, explicit=1, candidates=10 art + 10 cases, DE queries=5
  val_002: laws={'ATSG': 4, 'IVG': 9, 'BGG': 1, 'StPO': 2, 'ZGB': 3, 'BV': 1}, explicit=1, candidates=10 art + 10 cases, DE queries=5
  val_003: laws={'ZGB': 4, 'StPO': 14, 'StBOG': 2}, explicit=1, candidates=10 art + 10 cases, DE queries=5
  val_004: laws={'ZGB': 10, 'BGG': 1, 'OR': 7, 'IPRG': 1, 'StPO': 1}, explicit=0, candidates=10 art + 10 cases, DE queries=5
  val_005: laws={'ZGB': 13, 'BGG': 1, 'StPO': 1, 'BG-KKE': 4, 'ATSG': 1}, explicit=0, candidates=10 art + 10 cases, DE queries=5

  Batch 2: queries 6-10 of 10
  [6/10] val_006: pre-computing...
    Pre-compute done in 2.0s
  [7/10] val_007: pre-computing...
    Pre-compute done in 2.5s
  [8/10] val_008: pre-computing...
    Pre-compute done in 1.8s
  [9/10] val_009: pre-computing...
    Pre-compute done in 1.5s


### Stage 1 diagnosis (6 cells)

1. Artifact summary (explicit/candidates/cases/queries/HyDE counts)
2. Explicit-extraction fidelity — when gold appears verbatim in query, did we catch it?
3. Candidate hallucination — are candidate_articles real citations in the KB?
4. LLM law_area routing accuracy on val — predicted vs gold law codes
5. HyDE drift — does each HyDE doc share tokens with its query?
6. Free-recall ceiling on val — what % of gold is already in Stage 1 candidates?


In [35]:

# ══ Stage 1 diag 1/6: artifact summary ═════════════════════════════════
import json
from statistics import mean

s1_path = get_checkpoint_path("stage1", SPLIT)
if not s1_path.exists():
    print(f"Stage 1 checkpoint not found: {s1_path}")
else:
    s1 = json.load(open(s1_path, encoding="utf-8"))
    print(f"Stage 1 checkpoint: {s1_path}   queries: {len(s1)}")

    def _st(name, xs):
        xs = xs or [0]
        print(f"  {name:30s} min={min(xs)} max={max(xs)} mean={mean(xs):.1f}")

    _st("explicit_citations",       [len(r.get("explicit_citations", []))   for r in s1.values()])
    _st("candidate_articles",       [len(r.get("candidate_articles", []))   for r in s1.values()])
    _st("candidate_cases",          [len(r.get("candidate_cases", []))      for r in s1.values()])
    _st("search_queries_en",        [len(r.get("search_queries_en", []))    for r in s1.values()])
    _st("search_queries_de",        [len(r.get("search_queries_de", []))    for r in s1.values()])
    _st("hyde_documents_de count",  [len(r.get("hyde_documents_de", []))    for r in s1.values()])
    _st("hyde_documents_de tokens", [len(" ".join(r.get("hyde_documents_de", [])).split()) for r in s1.values()])
    _st("law_areas",                [len(r.get("law_areas", []))            for r in s1.values()])
    _st("legal_issues",             [len(r.get("legal_issues", []))         for r in s1.values()])


Stage 1 checkpoint: /content/drive/MyDrive/swiss_law/checkpoints/stage1_val.json   queries: 10
  explicit_citations             min=0 max=3 mean=0.9
  candidate_articles             min=10 max=10 mean=10.0
  candidate_cases                min=10 max=10 mean=10.0
  search_queries_en              min=2 max=5 mean=4.0
  search_queries_de              min=5 max=5 mean=5.0
  hyde_documents_de count        min=2 max=2 mean=2.0
  hyde_documents_de tokens       min=78 max=133 mean=99.0
  law_areas                      min=1 max=3 mean=1.8
  legal_issues                   min=1 max=5 mean=2.8


In [36]:

# ══ Stage 1 diag 2/6: explicit-extraction fidelity ═════════════════════
# When a gold Art.X LAWCODE citation appears verbatim in the query text,
# explicit extraction MUST catch it.  This should be 100%.
import re, json

s1_path = get_checkpoint_path("stage1", SPLIT)
if s1_path.exists() and SPLIT == "val":
    s1 = json.load(open(s1_path, encoding="utf-8"))
    import pandas as pd
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
    ART = re.compile(r"Art\.?\s*\d+\w*(?:\s+Abs\.?\s*\d+\w*)?(?:\s+lit\.?\s*[a-z])?\s+[A-Z]{2,8}")
    hit = 0; miss = 0; misses = []
    for _, r in vdf.iterrows():
        qid = r["query_id"]; q = str(r.get("query", ""))
        gold = {c.strip() for c in str(r.get("gold_citations", "")).split(";") if c.strip()}
        extr = set(s1.get(qid, {}).get("explicit_citations", []))
        for g in gold:
            if not g.startswith("Art"): continue
            # Does query contain this cite (any paragraph-agnostic form)?
            root = re.sub(r"\s+Abs\.\s*\d+\w*", "", g)
            root = re.sub(r"\s+lit\.\s*[a-z]", "", root)
            if root in q or g in q:
                if g in extr or root in extr:
                    hit += 1
                else:
                    miss += 1
                    misses.append((qid, g))
    print(f"Explicit-extraction fidelity on val: {hit}/{hit+miss} = {hit/max(hit+miss,1)*100:.1f}%")
    if miss:
        print(f"  MISSES (gold in query text but extraction failed):")
        for qid, g in misses[:10]:
            print(f"    {qid}: {g}")


Explicit-extraction fidelity on val: 6/6 = 100.0%


In [37]:

# ══ Stage 1 diag 3/6: candidate hallucination ═════════════════════════
# candidate_articles come from the LLM.  Check whether each is real.
import json
from pathlib import Path

s1_path = get_checkpoint_path("stage1", SPLIT)
laws_csv = LAWS_CSV if 'LAWS_CSV' in dir() else "data/laws_de.csv"
if s1_path.exists() and Path(laws_csv).exists():
    import pandas as pd
    s1 = json.load(open(s1_path, encoding="utf-8"))
    laws = pd.read_csv(laws_csv)
    valid = set(laws["citation"].dropna().astype(str))
    # Also accept root-form (Art. X LAWCODE without Abs./lit.)
    roots = set()
    for c in valid:
        roots.add(c)
    total = 0; bad = 0; examples = []
    for qid, r in s1.items():
        for cand in r.get("candidate_articles", []):
            total += 1
            if cand not in valid:
                # Try root form match
                import re
                m = re.match(r"(Art\.\s*\d+\w*)\s+([A-Z]{2,8})", cand)
                if m:
                    root = f"{m.group(1)} {m.group(2)}"
                    if root in valid: continue
                bad += 1
                if len(examples) < 15: examples.append((qid, cand))
    print(f"Candidate-article validity: {total-bad}/{total} valid ({(total-bad)/max(total,1)*100:.1f}%)")
    if bad:
        print(f"  {bad} hallucinated examples:")
        for qid, c in examples:
            print(f"    {qid}: {c}")


Candidate-article validity: 100/100 valid (100.0%)


In [38]:

# ══ Stage 1 diag 4/6: law_area routing accuracy ════════════════════════
# Compares Stage 1's ROUTING SIGNAL to gold law codes. The routing signal
# is sourced from (a) bm25_probe.detected_laws, (b) explicit citations,
# (c) law_areas expanded through the taxonomy. The previous version only
# regex-matched uppercase in law_areas — but law_areas stores taxonomy
# labels like "Criminal procedure", so pred was always empty (false alarm).
import json, re
from collections import Counter

# Keep in sync with stage1_query_analysis.LAW_AREA_MAP
_LAW_AREA_MAP = {
    "Criminal law":                ["StGB", "MStG", "BetmG", "SVG"],
    "Criminal procedure":          ["StPO", "JStPO", "StBOG"],
    "Civil law":                   ["ZGB", "OR", "PrHG", "DSG"],
    "Civil procedure":             ["ZPO"],
    "Public law":                  ["BV", "VwVG", "BGG", "RPG", "USG", "EnG"],
    "Social insurance":            ["ATSG", "IVG", "UVG", "AVIG", "BVG", "AHV", "ELG", "KVG", "FamZG"],
    "Immigration":                 ["AIG", "AsylG"],
    "Debt enforcement":            ["SchKG"],
    "Private international law":   ["IPRG", "IRSG"],
    "Intellectual property":       ["URG", "MSchG", "PatG"],
    "Federal court procedure":     ["BGG", "BGerR"],
    "Tax law":                     ["LIFD", "LHID", "LTVA"],
}
# BGG is nearly always cited in Federal Court judgments (admissibility),
# include by default so routing reflects Stage 1's procedural_articles inject.
_ALWAYS_ROUTE = {"BGG"}

_CITE_CODE = re.compile(r" ([A-Z][A-Za-z]{1,7})$")
_AREA_FALLBACK = re.compile(r"\b([A-Z]{2,8})\b")

def _codes_from_stage1(rec):
    codes = set(_ALWAYS_ROUTE)
    probe = rec.get("bm25_probe", {})
    for code in probe.get("detected_laws", {}).keys():
        codes.add(code)
    for cite in rec.get("explicit_citations", []):
        m = _CITE_CODE.search(cite)
        if m: codes.add(m.group(1))
    for area in rec.get("law_areas", []) or []:
        if isinstance(area, str):
            codes.update(_LAW_AREA_MAP.get(area, []))
            codes.update(_AREA_FALLBACK.findall(area))
    return codes

s1_path = get_checkpoint_path("stage1", SPLIT)
if s1_path.exists() and SPLIT == "val":
    import pandas as pd
    s1 = json.load(open(s1_path, encoding="utf-8"))
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
    per_q_p, per_q_r = [], []
    for _, r in vdf.iterrows():
        qid = r["query_id"]
        gold_codes = set()
        for c in str(r.get("gold_citations", "")).split(";"):
            c = c.strip()
            if c.startswith("Art"):
                m = _CITE_CODE.search(c)
                if m: gold_codes.add(m.group(1))
        pred_codes = _codes_from_stage1(s1.get(qid, {}))
        if not gold_codes: continue
        tp = len(pred_codes & gold_codes)
        p = tp / len(pred_codes) if pred_codes else 0.0
        r_ = tp / len(gold_codes) if gold_codes else 0.0
        per_q_p.append(p); per_q_r.append(r_)
        if r_ < 1.0:
            miss = gold_codes - pred_codes
            print(f"  {qid}: pred={sorted(pred_codes)}  miss={sorted(miss)}")
    if per_q_p:
        print(f"\n  Routing precision (mean): {sum(per_q_p)/len(per_q_p):.3f}")
        print(f"  Routing recall    (mean): {sum(per_q_r)/len(per_q_r):.3f}")


  val_003: pred=['BGG', 'JStPO', 'StBOG', 'StPO', 'ZGB']  miss=['BV']
  val_007: pred=['BGG', 'BV', 'CPLR', 'DSG', 'IPRG', 'IRSG', 'OR', 'PrHG', 'StPO', 'ZGB', 'ZPO']  miss=['StGB']

  Routing precision (mean): 0.361
  Routing recall    (mean): 0.955


In [39]:

# ══ Stage 1 diag 5/6: HyDE drift ═══════════════════════════════════════
# Proper cross-lingual drift check: the HyDE doc is German, the query is
# English — token overlap can't work directly. Instead we use Stage 1's
# own cross_lingual_terms (EN→DE) mapping as the expected overlap set,
# plus a broader EN→DE seed dictionary for queries where xl is thin.
import json, re
from pathlib import Path

_FALLBACK_SEEDS = {
    "detention":      ["Haft", "Inhaftierung"],
    "criminal":       ["Straf", "strafrechtlich"],
    "civil":          ["Zivil", "zivilrechtlich"],
    "contract":       ["Vertrag"],
    "insurance":      ["Versicherung"],
    "tenancy":        ["Miet"], "landlord": ["Vermiet"], "tenant": ["Mieter"],
    "family":         ["Familie"], "marriage": ["Ehe"], "divorce": ["Scheidung"],
    "inheritance":    ["Erb"], "heir": ["Erbe"],
    "liability":      ["Haftung", "Haftpflicht"],
    "damage":         ["Schad"], "damages": ["Schaden"],
    "copyright":      ["Urheber"], "trademark": ["Marke"], "patent": ["Patent"],
    "employment":     ["Arbeit"], "dismissal": ["Kündig"],
    "debt":           ["Schuld"], "bankruptcy": ["Konkurs"],
    "appeal":         ["Beschwerde"],
    "disability":     ["Invalid"],
    "invalidity":     ["Invalid"],
    "child":          ["Kind"],
    "custody":        ["Sorge"],
    "visitation":     ["Besuch"],
    "negligence":     ["Fahrlässig"],
    "asylum":         ["Asyl"],
    "property":       ["Eigentum"],
    "fraud":          ["Betrug"],
    "theft":          ["Diebstahl"],
    "tax":            ["Steuer"],
    "court":          ["Gericht"],
    "police":         ["Polizei"],
    "witness":        ["Zeug"],
    "evidence":       ["Beweis"],
    "prosecution":    ["Strafverfolg", "Staatsanwalt"],
    "judge":          ["Richter"],
    "parties":        ["Partei"],
    "settlement":     ["Vergleich"],
    "work":           ["Arbeit"],
    "accident":       ["Unfall"],
    "pension":        ["Rente"],
    "benefit":        ["Leistung"],
    "driver":         ["Fahrer", "Lenker"],
    "traffic":        ["Verkehr"],
    "vehicle":        ["Fahrzeug"],
    "licence":        ["Führerschein", "Bewilligung"],
    "license":        ["Führerschein", "Bewilligung"],
}

s1_path = get_checkpoint_path("stage1", SPLIT)
if s1_path.exists():
    s1 = json.load(open(s1_path, encoding="utf-8"))
    import pandas as pd
    vdf_path = VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv"
    q_map = {}
    if Path(vdf_path).exists():
        vdf = pd.read_csv(vdf_path)
        q_map = {row["query_id"]: str(row["query"]) for _, row in vdf.iterrows()}

    drift = 0; ok = 0
    for qid, r in s1.items():
        docs = r.get("hyde_documents_de", [])
        if not docs: continue
        hyde_low = " ".join(docs).lower()

        # Source 1: Stage 1 cross_lingual_terms — the primary expected overlap
        xl = r.get("cross_lingual_terms", {}) or {}
        matched_xl = sum(1 for de in xl.values() if isinstance(de, str) and de.lower() in hyde_low)

        # Source 2: fallback seed map for queries where xl is thin
        q = q_map.get(qid, "").lower()
        matched_seed = 0
        if q:
            for en, de_list in _FALLBACK_SEEDS.items():
                if en in q and any(de.lower() in hyde_low for de in de_list):
                    matched_seed += 1

        total = matched_xl + matched_seed
        if total < 2:
            drift += 1
            print(f"  {qid}: LIKELY DRIFT  (xl_matched={matched_xl}, seed_matched={matched_seed}, xl_size={len(xl)})")
        else:
            ok += 1
    print(f"\n  HyDE drift: {drift}/{drift+ok} queries suspicious")


  val_004: LIKELY DRIFT  (xl_matched=0, seed_matched=0, xl_size=3)
  val_007: LIKELY DRIFT  (xl_matched=0, seed_matched=1, xl_size=4)
  val_008: LIKELY DRIFT  (xl_matched=0, seed_matched=0, xl_size=3)

  HyDE drift: 3/10 queries suspicious


In [40]:

# ══ Stage 1 diag 6/6: free-recall ceiling on val ═══════════════════════
# What fraction of val gold is ALREADY in Stage 1 outputs (before any
# retrieval happens)?  This is the ceiling you get from Stage 1 alone.
import json

s1_path = get_checkpoint_path("stage1", SPLIT)
if s1_path.exists() and SPLIT == "val":
    import pandas as pd
    s1 = json.load(open(s1_path, encoding="utf-8"))
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
    gold = {r["query_id"]: {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
            for _, r in vdf.iterrows()}
    per_type_hit = {"Law":[0,0], "BGE":[0,0], "Numbered":[0,0]}
    per_q = []
    for qid, r in s1.items():
        g = gold.get(qid, set())
        s = set(r.get("explicit_citations", [])) | set(r.get("candidate_articles", [])) | set(r.get("candidate_cases", []))
        per_q.append((qid, len(s & g), len(g)))
        for c in g:
            k = "BGE" if c.startswith("BGE") else ("Law" if c.startswith("Art") else "Numbered")
            per_type_hit[k][1] += 1
            if c in s: per_type_hit[k][0] += 1
    tot_h = sum(h for _, h, _ in per_q); tot_g = sum(g for _, _, g in per_q)
    print(f"  Free-recall ceiling Stage 1: {tot_h}/{tot_g} = {tot_h/max(tot_g,1)*100:.1f}%")
    for k, (h, t) in per_type_hit.items():
        if t: print(f"    {k}: {h}/{t} = {h/t*100:.1f}%")
    print(f"\n  Per-query:")
    for qid, h, g in per_q:
        print(f"    {qid}: {h}/{g} ({h/max(g,1)*100:.0f}%)")


  Free-recall ceiling Stage 1: 126/251 = 50.2%
    Law: 82/149 = 55.0%
    BGE: 33/69 = 47.8%
    Numbered: 11/33 = 33.3%

  Per-query:
    val_001: 20/42 (48%)
    val_002: 20/36 (56%)
    val_003: 17/47 (36%)
    val_004: 6/10 (60%)
    val_005: 7/11 (64%)
    val_006: 9/18 (50%)
    val_007: 11/19 (58%)
    val_008: 12/29 (41%)
    val_009: 11/14 (79%)
    val_010: 13/25 (52%)


## Stage 2 — Multi-Agent Sparse Retrieval

MAS retrieval over the indexed corpus.

### `agent/prompts/mas_prompts.py`

```python
"""
mas_prompts.py -- System prompts for the Multi-Agent Sparse retrieval system.

Adapted from LegalMALR Table 2 for Swiss law with BM25 retrieval.
All agents use the SAME Qwen3-4B model — only the system prompt differs.

Each agent outputs German search queries that will be fed to BM25.
"""

# ─── PLANNER AGENT ──────────────────────────────────────────────────────────
# Decides which agent to invoke next, or whether to terminate.
PLANNER_PROMPT = """You are a Swiss law retrieval planner. You coordinate a team of
specialist agents to find ALL relevant legal provisions for a given scenario.

You have already retrieved some candidate provisions. Your job is to decide
what to do next to improve recall (find more relevant provisions).

Available agents:
1. REWRITE — rewrites colloquial English into precise German legal terminology
2. SUPPLEMENT — makes implicit legal conditions explicit (thresholds, standing, procedural)
3. DECOMPOSE — splits complex multi-issue queries into focused sub-queries
4. SUPPORTIVE — generates queries for procedural, interpretive, and auxiliary provisions
5. CROSSREF — generates queries for constitutional and cross-referenced articles

Based on the current state, decide:
- Which agent would most likely find NEW relevant provisions not yet in the pool?
- Or should we TERMINATE because further searching is unlikely to help?

Output JSON only:
{
  "reasoning": "Brief explanation of what's missing and why this agent helps",
  "action": "REWRITE" | "SUPPLEMENT" | "DECOMPOSE" | "SUPPORTIVE" | "CROSSREF" | "TERMINATE"
}
"""

# ─── SINGLE-ELEMENT REWRITE AGENT ──────────────────────────────────────────
# Rewrites colloquial EN terms into precise DE legal terminology for BM25.
REWRITE_PROMPT = """You are a Swiss law terminology expert. Given an English legal scenario,
generate precise German search queries using the exact legal terms that would appear
in Swiss statutory articles and court decisions.

CRITICAL: Your queries will be used for BM25 keyword search, so use the EXACT German
legal terms (Fachbegriffe) that appear in the law text, not paraphrases.

Examples of good translations:
- "pre-trial detention" → "Untersuchungshaft"
- "collusion risk" → "Kollusionsgefahr"
- "disability insurance" → "Invalidenversicherung"
- "medical expert opinion" → "medizinisches Gutachten"
- "right to be heard" → "rechtliches Gehör"
- "proportionality" → "Verhältnismässigkeit"

Generate 3-5 focused German search queries. Each query should:
- Target a SPECIFIC legal concept from the scenario
- Use 2-4 precise German legal terms
- Include relevant law abbreviations (StPO, ATSG, BGG, etc.)

Output JSON:
{
  "queries": ["Untersuchungshaft Kollusionsgefahr StPO", "Haftprüfung Verhältnismässigkeit", ...]
}
"""

# ─── SUPPLEMENTARY-ELEMENT AGENT ───────────────────────────────────────────
# Makes implicit legal conditions explicit.
SUPPLEMENT_PROMPT = """You are a Swiss law expert who identifies IMPLICIT legal requirements.

Given a legal scenario, identify conditions, thresholds, and requirements that are
implied but not explicitly stated. For each, generate a German search query.

Think about:
- Standing requirements (Legitimation, Beschwerdebefugnis)
- Jurisdictional prerequisites (örtliche/sachliche Zuständigkeit)
- Time limits and deadlines (Fristen, Verwirkung)
- Burden of proof rules (Beweislast)
- Exhaustion of remedies requirements
- Threshold amounts or severity levels
- Formal requirements (Schriftlichkeit, Begründungspflicht)

Generate 2-4 German search queries targeting these implicit requirements.

Output JSON:
{
  "implicit_issues": [
    {"issue": "...", "query": "Beschwerdelegitimation BGG Strafverfahren"}
  ]
}
"""

# ─── MULTI-ELEMENT DECOMPOSITION AGENT ─────────────────────────────────────
# Splits complex queries into focused sub-queries.
DECOMPOSE_PROMPT = """You are a Swiss law analyst who breaks down complex legal scenarios
into individual legal questions (Rechtsfragen).

A typical exam scenario may involve 3-8 distinct legal issues, each requiring
different statutory articles and case law. Your job is to identify each issue
and create a focused German search query for it.

For each sub-issue, generate ONE precise German search query targeting:
- The specific legal provision(s) that govern this issue
- The relevant law abbreviation

Example decomposition:
  Scenario about wrongful arrest + appeal + costs
  → "Untersuchungshaft Voraussetzungen Art 221 StPO"
  → "Beschwerde gegen Haftanordnung Art 222 StPO"
  → "Kostenregelung Strafverfahren Art 428 StPO"
  → "Beschwerdefrist BGG Strafverfahren"

Generate 3-6 sub-queries covering ALL distinct legal issues.

Output JSON:
{
  "sub_issues": [
    {"issue": "...", "query": "..."}
  ]
}
"""

# ─── SUPPORTIVE-LAW AGENT ─────────────────────────────────────────────────
# Targets procedural, interpretive, and auxiliary provisions.
SUPPORTIVE_PROMPT = """You are a Swiss procedural law specialist. Given a legal scenario
and the substantive provisions already found, identify SUPPORTING provisions that
a well-prepared exam candidate must also cite.

These are often missed but always required:
1. PROCEDURAL provisions:
   - Which court has jurisdiction? (BGG 72-89 for Federal Tribunal)
   - What is the appeal mechanism? (Beschwerde in Strafsachen, Zivilsachen, öff. Recht)
   - Filing deadlines (BGG 100 Abs. 1: 30 Tage)
   - Cost allocation (BGG 65-68, ZPO 106-107)
   - Legal aid (unentgeltliche Rechtspflege, BGG 64)

2. INTERPRETIVE provisions:
   - Definition articles (e.g., StGB 110 Definitionen)
   - General clauses (OR 2 Treu und Glauben, ZGB 4 Richterliches Ermessen)

3. AUXILIARY provisions:
   - Transitional provisions (Übergangsrecht)
   - Scope of application articles

Generate 2-4 German search queries for supporting provisions.

Output JSON:
{
  "supporting_queries": [
    {"type": "procedural", "query": "Beschwerde Bundesgericht Strafsachen BGG 78"},
    {"type": "cost", "query": "Gerichtskosten Parteientschädigung BGG 65 66"}
  ]
}
"""

# ─── CROSS-REFERENCE AGENT ────────────────────────────────────────────────
# NEW agent not in LegalMALR — targets constitutional and cross-referenced articles.
CROSSREF_PROMPT = """You are a Swiss constitutional and cross-reference law expert.
Given a legal scenario, identify:

1. CONSTITUTIONAL provisions (BV articles) that underlie the specific rules:
   - Art. 9 BV (Willkürverbot / prohibition of arbitrariness)
   - Art. 29 BV (Verfahrensgarantien / procedural guarantees)
   - Art. 32 BV (Unschuldsvermutung / presumption of innocence)
   - Art. 10 BV (Recht auf persönliche Freiheit)
   - Art. 13 BV (Schutz der Privatsphäre)
   - Art. 36 BV (Einschränkung von Grundrechten)

2. GENERAL PROVISIONS that apply alongside specific rules:
   - ATSG provisions alongside IVG/UVG/AVIG
   - OR general part alongside specific contracts
   - ZGB Einleitungsartikel (ZGB 1-10)

3. CROSS-REFERENCED articles:
   - If Art. X refers to Art. Y, both should be cited
   - "sinngemäss anwendbar" references

Generate 2-4 German search queries for these cross-references.

Output JSON:
{
  "crossref_queries": [
    {"type": "constitutional", "query": "Willkürverbot Art 9 BV Grundrechtseingriff"},
    {"type": "general", "query": "ATSG Allgemeiner Teil Invalidenversicherung"}
  ]
}
"""

# Map agent names to their prompts
AGENT_PROMPTS = {
    "PLANNER":    PLANNER_PROMPT,
    "REWRITE":    REWRITE_PROMPT,
    "SUPPLEMENT":  SUPPLEMENT_PROMPT,
    "DECOMPOSE":  DECOMPOSE_PROMPT,
    "SUPPORTIVE": SUPPORTIVE_PROMPT,
    "CROSSREF":   CROSSREF_PROMPT,
}

```

### `stage2_mas_retrieval.py`

In [41]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage2_mas_retrieval.py"

"""
stage2_mas_retrieval.py -- STAGE 2: Multi-Agent Sparse Retrieval (LegalMALR-Adapted)

This is the CORE INNOVATION of the revised pipeline, directly adapted from
LegalMALR's MAS architecture but using BM25 instead of dense retrieval.

Architecture:
  - Planner Agent:       Decides which specialist to invoke next, or terminate
  - Rewrite Agent:       EN colloquial -> precise DE legal terminology
  - Supplement Agent:    Makes implicit conditions explicit
  - Decompose Agent:     Splits multi-issue queries into sub-queries
  - Supportive Agent:    Targets procedural/auxiliary provisions
  - Cross-Reference Agent: Finds constitutional + cross-referenced articles

Each agent generates German search queries -> BM25 retrieval -> merge into pool.
The loop runs 2-4 iterations, stopping when no new candidates are found.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage2_mas_retrieval.py                              # all val queries
#   python stage2_mas_retrieval.py --split test                 # test queries
#   python stage2_mas_retrieval.py --query "A claimant was..."  # single query
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   index/bm25_v2_index.pkl   (from stage0)
#   index/bm25_v2_ids.pkl     (from stage0)
#   checkpoints/stage1_{split}.json  (from stage1)
#   GPU: Qwen3-32B loaded (~64 GB VRAM)
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage2_{split}.json
#   Per query: {query_id: {candidate_pool: [...], retrieval_log: [...]}}
#
# ─── EXPECTED TIMING ────────────────────────────────────────────────────────
#   ~20-40 seconds per query (2-4 iterations × LLM + BM25)
#   ~40 queries × 30s = ~20 min for full val set
"""



sys.path.insert(0, str(Path(__file__).parent))


CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

FORCE_FIRST_ACTION = os.getenv("SWISS_STAGE2_FORCE_FIRST_ACTION", "").strip().upper()
if FORCE_FIRST_ACTION not in {"REWRITE", "SUPPLEMENT", "DECOMPOSE", "SUPPORTIVE", "CROSSREF"}:
    FORCE_FIRST_ACTION = ""


def mas_retrieve_single(
    query_text: str,
    stage1_analysis: dict,
    sparse_retriever,
    llm_backend,
    max_iterations: int = 4,
    bm25_top_k: int = 30,
) -> dict:
    """
    Run MAS iterative retrieval for a single query.

    Returns {
        'candidate_pool': set of citation strings,
        'retrieval_log':  list of iteration records,
        'sources': dict mapping citation -> set of source names
    }
    """
    candidate_pool: set[str] = set()
    sources: dict[str, set[str]] = {}  # cite -> {'explicit', 'bm25_initial', 'mas_rewrite', ...}
    bm25_scores: dict[str, float] = {}  # cite -> max BM25 score across all queries

    # Build abbreviation normalization map (STPO->StPO, STGB->StGB, etc.)
    _upper_to_canon: dict[str, str] = {}
    try:
        with open(KB_JSONL, encoding="utf-8") as _f:
            for _line in _f:
                _rec = _json.loads(_line)
                _ab = (_rec.get("law") or {}).get("law_abbreviation") or ""
                if _ab:
                    _upper_to_canon[_ab.upper()] = _ab
    except Exception:
        pass

    def _norm(cite: str) -> str:
        """Normalize BM25 citation casing: 'Art. 222 STPO' -> 'Art. 222 StPO'."""
        if not cite.startswith("Art.") or not _upper_to_canon:
            return cite
        parts = cite.rsplit(" ", 1)
        if len(parts) == 2:
            return f"{parts[0]} {_upper_to_canon.get(parts[1].upper(), parts[1])}"
        return cite

    def add_candidates(cites, source_name):
        for c, score in cites:
            c = _norm(c)
            candidate_pool.add(c)
            if c not in sources:
                sources[c] = set()
            sources[c].add(source_name)
            # Track max BM25 score per citation
            if score > 0:
                bm25_scores[c] = max(bm25_scores.get(c, 0.0), score)

    retrieval_log = []

    # ── Initial: explicit citations from query ───────────────────────────
    explicit = set(stage1_analysis.get("explicit_citations", []))
    for c in explicit:
        candidate_pool.add(c)
        sources[c] = {"explicit_from_query"}

    # ── Initial: BM25 with original query + all DE queries from Stage 1 ──
    de_queries = stage1_analysis.get("search_queries_de", [])
    hyde_docs = stage1_analysis.get("hyde_documents_de", [])
    xl_queries = stage1_analysis.get("search_queries_de_crosslingual", [])
    en_queries = [query_text] + stage1_analysis.get("search_queries_en", [])
    # Include HyDE docs and cross-lingual queries in initial retrieval —
    # these were computed in Stage 1 but previously unused until MAS loop.
    all_initial_queries = en_queries + de_queries + hyde_docs + xl_queries

    for q in all_initial_queries:
        results = sparse_retriever.search_combined([q], top_k=50)
        add_candidates(results, "bm25_initial")

    # Track top-10 for confidence scoring
    initial_results = sparse_retriever.search_combined(all_initial_queries, top_k=10)
    bm25_top10 = {c for c, _ in initial_results}
    for c in bm25_top10:
        if c in sources:
            sources[c].add("bm25_top10")

    retrieval_log.append({
        "iteration": 0,
        "action": "INITIAL",
        "queries": all_initial_queries,
        "new_candidates": len(candidate_pool),
        "total_pool": len(candidate_pool),
    })

    # ── LLM-suggested candidates from Stage 1 ───────────────────────────
    for c in stage1_analysis.get("candidate_articles", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1")
    for c in stage1_analysis.get("candidate_cases", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1")
    for c in stage1_analysis.get("procedural_articles", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1_procedural")

    # ── MAS Iterative Loop ───────────────────────────────────────────────
    # Optional force-schedule for the first iteration. Leave unset by default so
    # the planner can pick the best first specialist for the query.
    _forced_actions = {1: FORCE_FIRST_ACTION} if FORCE_FIRST_ACTION else {}

    for iteration in range(1, max_iterations + 1):
        prev_size = len(candidate_pool)

        # Check if this iteration has a forced action
        action = _forced_actions.get(iteration)
        planner_response = {"reasoning": f"forced {action}", "action": action} if action else None

        if not action:
            # Planner decides next action
            planner_context = (
                f"SCENARIO:\n{query_text}\n\n"
                f"LEGAL ANALYSIS:\n{json.dumps(stage1_analysis.get('legal_issues', []), indent=2)}\n\n"
                f"CURRENT CANDIDATE POOL: {len(candidate_pool)} citations\n"
                f"ITERATION: {iteration}/{max_iterations}\n"
                f"PREVIOUS ACTIONS: {[r['action'] for r in retrieval_log]}\n\n"
                f"CURRENT CANDIDATE LIST:\n{json.dumps(sorted(candidate_pool), ensure_ascii=False)}"
            )

            planner_response = llm_backend.generate_json(
                system_prompt=AGENT_PROMPTS["PLANNER"],
                user_prompt=planner_context,
                max_new_tokens=256,
                enable_thinking=False,
            )

            action = str(planner_response.get("action", "TERMINATE")).strip().upper()

        if action == "TERMINATE" or action not in AGENT_PROMPTS:
            retrieval_log.append({
                "iteration": iteration,
                "action": "TERMINATE",
                "reasoning": planner_response.get("reasoning", ""),
                "total_pool": len(candidate_pool),
            })
            break

        # Selected agent generates reformulations
        agent_context = (
            f"SCENARIO:\n{query_text}\n\n"
            f"LEGAL ANALYSIS:\n{json.dumps(stage1_analysis.get('legal_issues', []), indent=2)}\n\n"
            f"ALREADY FOUND ({len(candidate_pool)} citations):\n"
            f"{json.dumps(sorted(candidate_pool), ensure_ascii=False)}\n\n"
            f"Generate search queries to find provisions NOT YET in the pool."
        )

        agent_response = llm_backend.generate_json(
            system_prompt=AGENT_PROMPTS[action],
            user_prompt=agent_context,
            max_new_tokens=512,
            enable_thinking=False,
        )

        # Extract queries from agent response (format varies by agent)
        reformulated_queries = _extract_queries(agent_response)

        if not reformulated_queries:
            print(
                f"    Iter {iteration}: {action} returned 0 queries "
                f"(response keys: {sorted(agent_response.keys()) if isinstance(agent_response, dict) else type(agent_response).__name__})"
            )

        # Each reformulation triggers BM25 retrieval
        new_count = 0
        for rq in reformulated_queries:
            results = sparse_retriever.search_combined([rq], top_k=bm25_top_k)
            for c, _ in results:
                if c not in candidate_pool:
                    new_count += 1
            add_candidates(results, f"mas_{action.lower()}")

        retrieval_log.append({
            "iteration": iteration,
            "action": action,
            "reasoning": planner_response.get("reasoning", ""),
            "queries": reformulated_queries,
            "new_candidates": new_count,
            "total_pool": len(candidate_pool),
        })

        print(f"    Iter {iteration}: {action} → {len(reformulated_queries)} queries → +{new_count} new ({len(candidate_pool)} total)")

        # Early termination: no new candidates found
        if len(candidate_pool) == prev_size:
            retrieval_log.append({
                "iteration": iteration + 1,
                "action": "TERMINATE (no new candidates)",
                "total_pool": len(candidate_pool),
            })
            break

    return {
        "candidate_pool": sorted(candidate_pool),
        "sources": {c: sorted(s) for c, s in sources.items()},
        "bm25_scores": bm25_scores,  # cite -> max BM25 score (for Stage 5 scoring)
        "retrieval_log": retrieval_log,
    }


def _extract_queries(agent_response: dict) -> list[str]:
    """Extract search queries from common agent JSON response shapes."""
    if not isinstance(agent_response, dict):
        return []

    queries: list[str] = []

    def _push(value):
        if isinstance(value, str):
            queries.append(value)
            return
        if isinstance(value, dict):
            for key in ("query", "search_query", "search", "text"):
                if isinstance(value.get(key), str):
                    queries.append(value[key])
                    return
            return
        if isinstance(value, (list, tuple)):
            for item in value:
                _push(item)

    for key in (
        "queries",
        "sub_issues",
        "implicit_issues",
        "supporting_queries",
        "crossref_queries",
        "search_queries",
        "retrieval_queries",
    ):
        _push(agent_response.get(key))

    seen = set()
    cleaned = []
    for q in queries:
        q = q.strip()
        if len(q) <= 3 or q in seen:
            continue
        seen.add(q)
        cleaned.append(q)
    return cleaned



# ── Shared abbreviation normalization map ─────────────────────────────────
_UPPER_TO_CANON: dict[str, str] = {}
def _build_upper_to_canon():
    """Build STPO->StPO mapping once (called before batched retrieval)."""
    global _UPPER_TO_CANON
    if _UPPER_TO_CANON:
        return _UPPER_TO_CANON
    try:
        with open(KB_JSONL, encoding="utf-8") as _f:
            for _line in _f:
                _rec = _json.loads(_line)
                _ab = (_rec.get("law") or {}).get("law_abbreviation") or ""
                if _ab:
                    _UPPER_TO_CANON[_ab.upper()] = _ab
    except Exception:
        pass
    return _UPPER_TO_CANON

def _norm_cite(cite: str) -> str:
    """Normalize BM25 citation casing using shared map."""
    if not cite.startswith("Art.") or not _UPPER_TO_CANON:
        return cite
    parts = cite.rsplit(" ", 1)
    if len(parts) == 2:
        return f"{parts[0]} {_UPPER_TO_CANON.get(parts[1].upper(), parts[1])}"
    return cite


STAGE2_BATCH_SIZE = 3  # Process 3 queries at a time


def _init_query_state(query_text, stage1_analysis, sparse_retriever):
    """Run initial BM25 retrieval for one query. Returns per-query state dict."""
    candidate_pool = set()
    sources = {}
    bm25_scores = {}
    retrieval_log = []

    def add_cands(cites, source_name):
        for c, score in cites:
            c = _norm_cite(c)
            candidate_pool.add(c)
            sources.setdefault(c, set()).add(source_name)
            if score > 0:
                bm25_scores[c] = max(bm25_scores.get(c, 0.0), score)

    # Explicit citations from query
    explicit = set(stage1_analysis.get("explicit_citations", []))
    for c in explicit:
        candidate_pool.add(c)
        sources[c] = {"explicit_from_query"}

    # BM25 with original query + all DE queries from Stage 1
    de_queries = stage1_analysis.get("search_queries_de", [])
    hyde_docs = stage1_analysis.get("hyde_documents_de", [])
    xl_queries = stage1_analysis.get("search_queries_de_crosslingual", [])
    en_queries = [query_text] + stage1_analysis.get("search_queries_en", [])
    all_initial_queries = en_queries + de_queries + hyde_docs + xl_queries

    for q in all_initial_queries:
        results = sparse_retriever.search_combined([q], top_k=50)
        add_cands(results, "bm25_initial")

    # Track top-10 for confidence scoring
    initial_results = sparse_retriever.search_combined(all_initial_queries, top_k=10)
    bm25_top10 = {c for c, _ in initial_results}
    for c in bm25_top10:
        if c in sources:
            sources[c].add("bm25_top10")

    retrieval_log.append({
        "iteration": 0, "action": "INITIAL",
        "queries": all_initial_queries,
        "new_candidates": len(candidate_pool),
        "total_pool": len(candidate_pool),
    })

    # LLM-suggested candidates from Stage 1
    for c in stage1_analysis.get("candidate_articles", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1")
    for c in stage1_analysis.get("candidate_cases", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1")
    for c in stage1_analysis.get("procedural_articles", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1_procedural")

    return {
        "candidate_pool": candidate_pool,
        "sources": sources,
        "bm25_scores": bm25_scores,
        "retrieval_log": retrieval_log,
        "terminated": False,
    }


def mas_retrieve_batch(
    query_texts: list[str],
    stage1_analyses: list[dict],
    sparse_retriever,
    max_iterations: int = 4,
    bm25_top_k: int = 30,
) -> list[dict]:
    """
    Run MAS iterative retrieval for N queries in lockstep.
    Batches LLM calls (planner + agent) across all active queries.
    Returns list of result dicts (same format as mas_retrieve_single).
    """
    n = len(query_texts)
    _build_upper_to_canon()

    # Phase A: initial BM25 retrieval for all queries (CPU)
    states = []
    for i in range(n):
        print(f"    [{i+1}/{n}] Initial BM25 retrieval...")
        states.append(_init_query_state(
            query_texts[i], stage1_analyses[i], sparse_retriever
        ))

    _forced_actions = {1: FORCE_FIRST_ACTION} if FORCE_FIRST_ACTION else {}

    # Phase B: MAS iterative loop (batched LLM calls)
    for iteration in range(1, max_iterations + 1):
        active_indices = [i for i in range(n) if not states[i]["terminated"]]
        if not active_indices:
            break

        prev_sizes = {i: len(states[i]["candidate_pool"]) for i in active_indices}

        # ── Step 1: Batch planner calls for all active queries ────────
        forced_action = _forced_actions.get(iteration)
        if forced_action:
            # All queries get the forced action
            actions = {i: forced_action for i in active_indices}
            planner_responses = {i: {"reasoning": f"forced {forced_action}", "action": forced_action}
                                 for i in active_indices}
        else:
            planner_prompts = []
            planner_indices = []
            for i in active_indices:
                st = states[i]
                ctx = (
                    f"SCENARIO:\n{query_texts[i]}\n\n"
                    f"LEGAL ANALYSIS:\n{json.dumps(stage1_analyses[i].get('legal_issues', []), indent=2)}\n\n"
                    f"CURRENT CANDIDATE POOL: {len(st['candidate_pool'])} citations\n"
                    f"ITERATION: {iteration}/{max_iterations}\n"
                    f"PREVIOUS ACTIONS: {[r['action'] for r in st['retrieval_log']]}\n\n"
                    f"CURRENT CANDIDATE LIST:\n{json.dumps(sorted(st['candidate_pool']), ensure_ascii=False)}"
                )
                planner_prompts.append(ctx)
                planner_indices.append(i)

            # Single batched LLM call for all planners (same system prompt)
            print(f"  Iter {iteration}: batched planner call for {len(planner_indices)} queries")
            batch_responses = generate_json_batch(
                system_prompt=AGENT_PROMPTS["PLANNER"],
                user_prompts=planner_prompts,
                max_new_tokens=256,
                enable_thinking=False,
            )

            actions = {}
            planner_responses = {}
            for idx_pos, i in enumerate(planner_indices):
                resp = batch_responses[idx_pos]
                act = str(resp.get("action", "TERMINATE")).strip().upper()
                actions[i] = act
                planner_responses[i] = resp

        # Mark terminated queries
        for i in list(active_indices):
            if actions[i] == "TERMINATE" or actions[i] not in AGENT_PROMPTS:
                states[i]["retrieval_log"].append({
                    "iteration": iteration,
                    "action": "TERMINATE",
                    "reasoning": planner_responses[i].get("reasoning", ""),
                    "total_pool": len(states[i]["candidate_pool"]),
                })
                states[i]["terminated"] = True

        # ── Step 2: Batch agent calls for remaining active queries ────
        agent_indices = [i for i in active_indices if not states[i]["terminated"]]
        if not agent_indices:
            continue

        agent_system_prompts = []
        agent_user_prompts = []
        for i in agent_indices:
            st = states[i]
            agent_system_prompts.append(AGENT_PROMPTS[actions[i]])
            agent_user_prompts.append(
                f"SCENARIO:\n{query_texts[i]}\n\n"
                f"LEGAL ANALYSIS:\n{json.dumps(stage1_analyses[i].get('legal_issues', []), indent=2)}\n\n"
                f"ALREADY FOUND ({len(st['candidate_pool'])} citations):\n"
                f"{json.dumps(sorted(st['candidate_pool']), ensure_ascii=False)}\n\n"
                f"Generate search queries to find provisions NOT YET in the pool."
            )

        # Single batched LLM call for all agents (different system prompts per action)
        action_summary = ", ".join(f"Q{i}={actions[i]}" for i in agent_indices)
        print(f"  Iter {iteration}: batched agent call [{action_summary}]")
        agent_responses = generate_json_batch_multi_system(
            system_prompts=agent_system_prompts,
            user_prompts=agent_user_prompts,
            max_new_tokens=512,
            enable_thinking=False,
        )

        # ── Step 3: BM25 retrieval per query (CPU) ───────────────────
        for idx_pos, i in enumerate(agent_indices):
            st = states[i]
            action = actions[i]
            reformulated_queries = _extract_queries(agent_responses[idx_pos])

            if not reformulated_queries:
                resp = agent_responses[idx_pos]
                print(
                    f"    Q{i} Iter {iteration}: {action} returned 0 queries "
                    f"(keys: {sorted(resp.keys()) if isinstance(resp, dict) else type(resp).__name__})"
                )

            new_count = 0
            for rq in reformulated_queries:
                results = sparse_retriever.search_combined([rq], top_k=bm25_top_k)
                for c, _ in results:
                    c_n = _norm_cite(c)
                    if c_n not in st["candidate_pool"]:
                        new_count += 1
                for c, score in results:
                    c_n = _norm_cite(c)
                    st["candidate_pool"].add(c_n)
                    st["sources"].setdefault(c_n, set()).add(f"mas_{action.lower()}")
                    if score > 0:
                        st["bm25_scores"][c_n] = max(st["bm25_scores"].get(c_n, 0.0), score)

            st["retrieval_log"].append({
                "iteration": iteration,
                "action": action,
                "reasoning": planner_responses[i].get("reasoning", ""),
                "queries": reformulated_queries,
                "new_candidates": new_count,
                "total_pool": len(st["candidate_pool"]),
            })

            print(f"    Q{i} Iter {iteration}: {action} -> {len(reformulated_queries)} queries -> +{new_count} new ({len(st['candidate_pool'])} total)")

            # Early termination if no new candidates
            if len(st["candidate_pool"]) == prev_sizes[i]:
                st["retrieval_log"].append({
                    "iteration": iteration + 1,
                    "action": "TERMINATE (no new candidates)",
                    "total_pool": len(st["candidate_pool"]),
                })
                st["terminated"] = True

    # Finalize results
    results = []
    for st in states:
        results.append({
            "candidate_pool": sorted(st["candidate_pool"]),
            "sources": {c: sorted(s) for c, s in st["sources"].items()},
            "bm25_scores": st["bm25_scores"],
            "retrieval_log": st["retrieval_log"],
        })
    return results


def run_batch_stage2(split: str, backend: str = "local", max_iterations: int = 4,
              resume: bool = True) -> dict:
    """Run Stage 2 on all queries in a split, processing STAGE2_BATCH_SIZE queries at a time."""
    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    df = pd.read_csv(csv_map[split])

    # Load Stage 1 results
    stage1_path = CHECKPOINTS_DIR / f"stage1_{split}.json"
    if not stage1_path.exists():
        print(f"ERROR: Stage 1 results not found: {stage1_path}")
        print("Run first: python stage1_query_analysis.py")
        sys.exit(1)
    with open(stage1_path, encoding="utf-8") as f:
        stage1_results = json.load(f)

    # Load BM25 retriever
    print("Loading BM25 retriever...")
    sparse = SparseRetriever()

    # Load LLM (stays in memory)
    if backend == "local":
        print("Loading Qwen3-32B...")
        get_model_and_tokenizer()

    out_path = CHECKPOINTS_DIR / f"stage2_{split}.json"
    results = {}
    if resume and out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            results = json.load(f)
        print(f"  Resuming: {len(results)} queries already done")

    # Collect pending queries
    pending = []
    for _, row in df.iterrows():
        qid = row["query_id"]
        if qid not in results:
            pending.append((qid, row["query"], stage1_results.get(qid, {})))

    print(f"  {len(pending)} queries to process in batches of {STAGE2_BATCH_SIZE}")

    # Process in batches
    for batch_start in range(0, len(pending), STAGE2_BATCH_SIZE):
        batch = pending[batch_start:batch_start + STAGE2_BATCH_SIZE]
        batch_qids = [b[0] for b in batch]
        batch_queries = [b[1] for b in batch]
        batch_analyses = [b[2] for b in batch]

        done_so_far = len(results)
        print(f"\n{'='*60}")
        print(f"Batch {batch_start // STAGE2_BATCH_SIZE + 1}: "
              f"queries {done_so_far + 1}-{done_so_far + len(batch)} / {done_so_far + len(pending) - batch_start}")
        for qi, qid in enumerate(batch_qids):
            print(f"  Q{qi}: {qid}")
        print(f"{'='*60}")

        # Clear VRAM before each batch
        if backend == "local":
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

        t0 = time.time()
        try:
            batch_results = mas_retrieve_batch(
                query_texts=batch_queries,
                stage1_analyses=batch_analyses,
                sparse_retriever=sparse,
                max_iterations=max_iterations,
            )
        except Exception as e:
            print(f"  BATCH ERROR: {e} -- falling back to sequential")
            batch_results = []
            for qi in range(len(batch)):
                try:
                    r = mas_retrieve_single(
                        query_text=batch_queries[qi],
                        stage1_analysis=batch_analyses[qi],
                        sparse_retriever=sparse,
                        llm_backend=sys.modules[__name__],
                        max_iterations=max_iterations,
                    )
                except Exception as e2:
                    print(f"    ERROR on {batch_qids[qi]}: {e2}")
                    r = {"candidate_pool": [], "sources": {}, "bm25_scores": {}, "retrieval_log": [{"error": str(e2)}]}
                batch_results.append(r)

        elapsed = time.time() - t0

        for qi, qid in enumerate(batch_qids):
            results[qid] = batch_results[qi]
            n_cands = len(batch_results[qi]["candidate_pool"])
            print(f"  {qid}: {n_cands} candidates")

        print(f"  Batch done in {elapsed:.1f}s")

        # Save after each batch
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nStage 2 complete. Saved: {out_path}")
    return results


### Stage 2 runner

In [42]:
stage2_outputs = run_stage2(split=SPLIT, backend=BACKEND, max_iterations=STAGE2_MAX_ITERATIONS)

Loading BM25 retriever...
  Loading combined BM25 index ...


  Loading BM25 chunks: 100%|##################| 216/216 [01:30<00:00,  2.39it/s]


  Converting BM25 doc_freqs to sparse matrix ...
    Sparse matrix: 2,156,832 docs x 822,251 terms (212,228,366 non-zero) in 36.2s
    171,654 law  +  1,985,178 court  =  2,156,832 total
Loading Qwen3-32B...
  10 queries to process in batches of 3

Batch 1: queries 1-3 / 10
  Q0: val_001
  Q1: val_002
  Q2: val_003
    [1/3] Initial BM25 retrieval...
    [2/3] Initial BM25 retrieval...
    [3/3] Initial BM25 retrieval...
  Iter 1: batched planner call for 3 queries
  Iter 1: batched agent call [Q0=REWRITE, Q1=SUPPORTIVE, Q2=SUPPORTIVE]
    Q0 Iter 1: REWRITE -> 5 queries -> +97 new (780 total)
    Q1 Iter 1: SUPPORTIVE -> 4 queries -> +116 new (896 total)
    Q2 Iter 1: SUPPORTIVE -> 4 queries -> +96 new (776 total)
  Iter 2: batched planner call for 3 queries
  Iter 2: batched agent call [Q0=SUPPORTIVE, Q1=SUPPORTIVE, Q2=SUPPLEMENT]
    Q0 Iter 2: SUPPORTIVE -> 4 queries -> +80 new (860 total)
    Q1 Iter 2: SUPPORTIVE -> 4 queries -> +116 new (1012 total)
    Q2 Iter 2: SUPPLEMENT ->

### Stage 2 diagnosis (6 cells)

1. Candidate counts + type breakdown
2. Val recall per type (Law / BGE / Numbered)
3. R@{10,50,100,500} curves on val
4. Single-query vs multi-query ablation (BM25 recall — tests your dilution hypothesis)
5. Dead query variants (generated queries that never contribute a unique hit)
6. Candidate duplication across variants (high overlap → candidates for RRF fusion)


In [43]:

# ══ Stage 2 diag 1/6: candidate counts + type breakdown ════════════════
# Stage 2 stores candidates under "candidate_pool" (sorted list).
import json
from collections import Counter
from statistics import mean

s2_path = get_checkpoint_path("stage2", SPLIT)
if not s2_path.exists():
    print(f"not found: {s2_path}")
else:
    s2 = json.load(open(s2_path, encoding="utf-8"))

    def _cands(rec):
        # Primary key used by Stage 2
        v = rec.get("candidate_pool")
        if isinstance(v, list): return v
        # Fallback for older checkpoint formats
        for k in ("bm25_candidates", "candidates", "all_candidates", "retrieved"):
            v = rec.get(k)
            if isinstance(v, list): return v
            if isinstance(v, dict):
                flat = [x for lst in v.values() if isinstance(lst, list) for x in lst]
                if flat: return flat
        return []

    counts = [len(_cands(r)) for r in s2.values()]
    print(f"Stage 2 checkpoint: {s2_path}   queries: {len(s2)}")
    print(f"  candidates/q: min={min(counts)} max={max(counts)} mean={mean(counts):.1f}")
    t = Counter()
    for r in s2.values():
        for c in _cands(r):
            if c.startswith("BGE"): t["BGE"] += 1
            elif c.startswith("Art"): t["Law"] += 1
            elif "_" in c and "/" in c: t["Numbered"] += 1
            else: t["Other"] += 1
    total = sum(t.values()) or 1
    print("  by type: " + ", ".join(f"{k}={v} ({v/total*100:.0f}%)" for k, v in t.most_common()))


Stage 2 checkpoint: /content/drive/MyDrive/swiss_law/checkpoints/stage2_val.json   queries: 10
  candidates/q: min=976 max=1430 mean=1183.5
  by type: Numbered=6947 (59%), Law=3258 (28%), BGE=833 (7%), Other=797 (7%)


In [44]:

# ══ Stage 2 diag 2/6: val recall per type ══════════════════════════════
import json

s2_path = get_checkpoint_path("stage2", SPLIT)
if s2_path.exists() and SPLIT == "val":
    import pandas as pd
    s2 = json.load(open(s2_path, encoding="utf-8"))
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")

    def _cands(rec):
        v = rec.get("candidate_pool")
        if isinstance(v, list): return v
        for k in ("bm25_candidates", "candidates", "all_candidates"):
            v = rec.get(k)
            if isinstance(v, list): return v
            if isinstance(v, dict):
                return [x for lst in v.values() if isinstance(lst, list) for x in lst]
        return []

    per_t = {"Law":[0,0], "BGE":[0,0], "Numbered":[0,0]}
    for _, r in vdf.iterrows():
        gold = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
        c = set(_cands(s2.get(r["query_id"], {})))
        for cite in gold:
            k = "BGE" if cite.startswith("BGE") else ("Law" if cite.startswith("Art") else "Numbered")
            per_t[k][1] += 1
            if cite in c: per_t[k][0] += 1
    for k, (h, t_) in per_t.items():
        if t_:
            flag = "  <-- CRITICAL" if k == "BGE" and h/t_ < 0.3 else ""
            print(f"  {k}: {h}/{t_} = {h/t_*100:.1f}%{flag}")


  Law: 117/149 = 78.5%
  BGE: 48/69 = 69.6%
  Numbered: 26/33 = 78.8%


In [45]:

# ══ Stage 2 diag 3/6: R@k curves on val ════════════════════════════════
# candidate_pool is unordered; we approximate ranking by bm25_scores (desc).
import json

s2_path = get_checkpoint_path("stage2", SPLIT)
if s2_path.exists() and SPLIT == "val":
    import pandas as pd
    s2 = json.load(open(s2_path, encoding="utf-8"))
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")

    def _ordered(rec):
        # Rank by bm25_scores descending; unscored items go last
        pool = rec.get("candidate_pool")
        if not isinstance(pool, list):
            for k in ("bm25_ranked", "ranked_candidates", "bm25_candidates", "candidates"):
                v = rec.get(k)
                if isinstance(v, list): return v
                if isinstance(v, dict):
                    return [x for lst in v.values() if isinstance(lst, list) for x in lst]
            return []
        scores = rec.get("bm25_scores", {}) or {}
        return sorted(pool, key=lambda c: scores.get(c, 0.0), reverse=True)

    Ks = [10, 50, 100, 200, 500, 1000]
    recall_at = {k: {"Law":[0,0], "BGE":[0,0], "Numbered":[0,0]} for k in Ks}
    for _, r in vdf.iterrows():
        gold = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
        ranked = _ordered(s2.get(r["query_id"], {}))
        for K in Ks:
            topk = set(ranked[:K])
            for c in gold:
                t = "BGE" if c.startswith("BGE") else ("Law" if c.startswith("Art") else "Numbered")
                recall_at[K][t][1] += 1
                if c in topk: recall_at[K][t][0] += 1

    print("  K    Law          BGE          Numbered")
    for K in Ks:
        ps = []
        for t in ["Law", "BGE", "Numbered"]:
            h, tot = recall_at[K][t]
            ps.append(f"{h}/{tot}={h/max(tot,1)*100:4.1f}%" if tot else "     -    ")
        print(f"  {K:4d} {ps[0]:12s} {ps[1]:12s} {ps[2]:12s}")


  K    Law          BGE          Numbered
    10 66/149=44.3% 17/69=24.6%  2/33= 6.1%  
    50 86/149=57.7% 42/69=60.9%  22/33=66.7% 
   100 86/149=57.7% 42/69=60.9%  22/33=66.7% 
   200 99/149=66.4% 46/69=66.7%  25/33=75.8% 
   500 101/149=67.8% 46/69=66.7%  25/33=75.8% 
  1000 106/149=71.1% 47/69=68.1%  26/33=78.8% 


In [46]:

# ══ Stage 2 diag 4/6: single- vs multi-query ablation ══════════════════
# Uses the notebook's already-loaded BM25 singleton (_get_sparse_retriever)
# rather than re-importing retrieval module (not importable in this env).
import json, re

s1_path = get_checkpoint_path("stage1", SPLIT)
if SPLIT != "val" or not s1_path.exists():
    print("  skipped (not val split or no Stage 1 checkpoint)")
else:
    # Re-use the sparse retriever that was already loaded by Stage 1.
    SR = _get_sparse_retriever()
    if SR is None:
        print("  skipped (SparseRetriever not loaded — run Stage 1 first)")
    else:
        import pandas as pd
        s1 = json.load(open(s1_path, encoding="utf-8"))
        vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")

        def _bm25_recall(query_fn, K=100):
            per = {"Law":[0,0], "BGE":[0,0], "Numbered":[0,0]}
            for _, r in vdf.iterrows():
                qid = r["query_id"]
                gold = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
                cands = set()
                for q in query_fn(qid):
                    try:
                        results = SR.search_combined([q], top_k=K)
                        cands.update(c for c, _ in results)
                    except Exception:
                        pass
                for c in gold:
                    t = "BGE" if c.startswith("BGE") else ("Law" if c.startswith("Art") else "Numbered")
                    per[t][1] += 1
                    if c in cands: per[t][0] += 1
            return per

        def pretty(name, per):
            parts = [f"{t}: {h}/{tot}={h/max(tot,1)*100:.1f}%"
                     for t, (h, tot) in per.items() if tot]
            print(f"  {name:<40s} {', '.join(parts)}")

        try:
            q_map = {row["query_id"]: str(row["query"]) for _, row in vdf.iterrows()}
            pretty("EN only (raw query)",
                   _bm25_recall(lambda qid: [q_map[qid]]))
            pretty("Single DE query",
                   _bm25_recall(lambda qid: (s1.get(qid, {}).get("search_queries_de", []) or [""])[:1]))
            pretty("All DE queries (≤10)",
                   _bm25_recall(lambda qid:  s1.get(qid, {}).get("search_queries_de", [])[:10]))
            pretty("All DE + HyDE",
                   _bm25_recall(lambda qid:  s1.get(qid, {}).get("search_queries_de", [])[:10]
                                           + s1.get(qid, {}).get("hyde_documents_de", [])))
        except Exception as e:
            print(f"  ablation failed: {e}")


  EN only (raw query)                      Law: 59/149=39.6%, BGE: 45/69=65.2%, Numbered: 25/33=75.8%
  Single DE query                          Law: 5/149=3.4%, BGE: 0/69=0.0%, Numbered: 3/33=9.1%
  All DE queries (≤10)                     Law: 20/149=13.4%, BGE: 1/69=1.4%, Numbered: 3/33=9.1%
  All DE + HyDE                            Law: 21/149=14.1%, BGE: 2/69=2.9%, Numbered: 5/33=15.2%


In [47]:

# ══ Stage 2 diag 5/6: dead query variants ══════════════════════════════
# Uses the already-loaded BM25 singleton; no fresh module import needed.
import json

s1_path = get_checkpoint_path("stage1", SPLIT)
if SPLIT != "val" or not s1_path.exists():
    print("  skipped (not val split or no Stage 1 checkpoint)")
else:
    SR = _get_sparse_retriever()
    if SR is None:
        print("  skipped (SparseRetriever not loaded)")
    else:
        import pandas as pd
        from collections import defaultdict
        s1 = json.load(open(s1_path, encoding="utf-8"))
        vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
        variant_hits = defaultdict(lambda: {"hits": 0, "runs": 0})
        for _, r in vdf.iterrows():
            qid = r["query_id"]
            gold = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
            rec = s1.get(qid, {})
            variants = [("EN_raw", str(r.get("query", "")))]
            for i, q in enumerate(rec.get("search_queries_en", [])): variants.append((f"EN_{i}", q))
            for i, q in enumerate(rec.get("search_queries_de", [])): variants.append((f"DE_{i}", q))
            for i, q in enumerate(rec.get("hyde_documents_de",  [])): variants.append((f"HYDE_{i}", q))
            for name, q in variants:
                try:
                    hits = set(c for c, _ in SR.search_combined([q], top_k=50))
                except Exception:
                    hits = set()
                variant_hits[name]["runs"] += 1
                variant_hits[name]["hits"] += len(hits & gold)
        print("  variant         runs   gold_hits   avg/run")
        for name in sorted(variant_hits):
            d = variant_hits[name]
            avg = d["hits"] / max(d["runs"], 1)
            flag = "  <-- DEAD" if avg < 0.5 and d["runs"] >= 5 else ""
            print(f"  {name:14s} {d['runs']:4d}   {d['hits']:5d}       {avg:.2f}{flag}")


  variant         runs   gold_hits   avg/run
  DE_0             10       6       0.60
  DE_1             10       5       0.50
  DE_2             10       7       0.70
  DE_3             10       5       0.50
  DE_4             10       1       0.10  <-- DEAD
  EN_0             10      75       7.50
  EN_1             10      51       5.10
  EN_2              9      31       3.44
  EN_3              6       1       0.17  <-- DEAD
  EN_4              5      16       3.20
  EN_raw           10     121       12.10
  HYDE_0           10       3       0.30  <-- DEAD
  HYDE_1           10       1       0.10  <-- DEAD


In [48]:

# ══ Stage 2 diag 6/6: MAS action source breakdown ══════════════════════
# Stage 2 stores {citation: [sources]} in "sources" field.
# Summarises how many citations came uniquely from each MAS action type.
import json
from collections import Counter, defaultdict

s2_path = get_checkpoint_path("stage2", SPLIT)
if not s2_path.exists():
    print(f"not found: {s2_path}")
else:
    s2 = json.load(open(s2_path, encoding="utf-8"))
    action_totals = Counter()
    action_unique = Counter()   # cite appeared in only 1 source
    overlap_sizes = []

    for qid, rec in s2.items():
        sources = rec.get("sources", {})
        if not sources: continue
        # Count contributions per action label
        for cite, src_list in sources.items():
            for src in src_list:
                action_totals[src] += 1
        # Fraction of cites reachable only from one action
        single_src = sum(1 for v in sources.values() if len(v) == 1)
        multi_src  = sum(1 for v in sources.values() if len(v) > 1)
        total = single_src + multi_src
        overlap_sizes.append(multi_src / max(total, 1))

    if action_totals:
        print(f"  Queries processed: {len(s2)}")
        print(f"  Avg overlap fraction (cite reached by >1 action): "
              f"{sum(overlap_sizes)/max(len(overlap_sizes),1):.2f}")
        print(f"  (low = actions are complementary → each iteration adds unique value)")
        print()
        print("  Action source          Total citations")
        for src, cnt in action_totals.most_common():
            print(f"    {src:25s} {cnt:6d}")
    else:
        print("  (no sources data in checkpoint)")


  Queries processed: 10
  Avg overlap fraction (cite reached by >1 action): 0.07
  (low = actions are complementary → each iteration adds unique value)

  Action source          Total citations
    bm25_initial                7513
    mas_supportive              3199
    mas_crossref                 934
    mas_rewrite                  351
    mas_supplement               267
    llm_stage1                   200
    llm_stage1_procedural        200
    mas_decompose                178
    bm25_top10                    82
    explicit_from_query            9


### Statute→BGE edge builder (fixes 0-BGE-graph-recall gap)


In [49]:
# [NEW] Index loader: statute/numbered co-occurrence indexes
#
# Loads 5 pre-built pickles from INDEX_DIR (= law_db/ on Kaggle/Drive):
#   statute_to_bge.pkl        statute -> BGE decisions  (3,840 statutes)
#   statute_to_numbered.pkl   statute -> numbered cases (4,991 statutes)
#   numbered_to_statutes.pkl  numbered case -> statutes (80,569 cases)
#   numbered_to_bge.pkl       numbered case -> BGE      (67,513 cases)
#   numbered_cocitation.pkl   numbered case -> co-cited numbered (81,709 cases)
#
# All built locally from court_considerations.csv (build_statute_to_bge.py +
# build_numbered_indexes.py) and uploaded to Google Drive / Kaggle law-db dataset
# alongside the other index artifacts. Subsequent runs load instantly from pickle.
import pickle as _idx_pk

def _load_index(path, label):
    if path.exists():
        with open(path, "rb") as _f:
            _idx = _idx_pk.load(_f)
        print(f"  {label}: {len(_idx):,} entries loaded from {path.name}")
        return _idx
    print(f"  {label}: NOT FOUND at {path} — expansion channel disabled")
    return {}

print("Loading co-occurrence indexes...")
STATUTE_TO_BGE      = _load_index(INDEX_DIR / "statute_to_bge.pkl",       "statute->BGE")
STATUTE_TO_NUMBERED = _load_index(INDEX_DIR / "statute_to_numbered.pkl",   "statute->Numbered") if STAGE3_USE_NUMBERED_INDEXES else {}
NUMBERED_TO_STATUTES = _load_index(INDEX_DIR / "numbered_to_statutes.pkl", "numbered->statutes") if STAGE3_USE_NUMBERED_INDEXES else {}
NUMBERED_TO_BGE     = _load_index(INDEX_DIR / "numbered_to_bge.pkl",       "numbered->BGE")     if STAGE3_USE_NUMBERED_INDEXES else {}
NUMBERED_COCITATION = _load_index(INDEX_DIR / "numbered_cocitation.pkl",   "numbered->co-cited") if STAGE3_USE_NUMBERED_INDEXES else {}
print("Done.")


Loading co-occurrence indexes...
  statute->BGE: 3,840 entries loaded from statute_to_bge.pkl
  statute->Numbered: 4,991 entries loaded from statute_to_numbered.pkl
  numbered->statutes: 80,569 entries loaded from numbered_to_statutes.pkl
  numbered->BGE: 67,513 entries loaded from numbered_to_bge.pkl
  numbered->co-cited: 81,709 entries loaded from numbered_cocitation.pkl
Done.


## Stage 3 — Citation Graph Expansion

Graph-based candidate expansion.

### `stage3_graph_expansion.py`

In [50]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage3_graph_expansion.py"

"""
stage3_graph_expansion.py -- STAGE 3: Citation Graph Traversal + Expansion

After MAS produces the candidate pool, expand via the citation graph:
  - Statute found → find citing court decisions (top 5 per article)
  - Court decision found → find cited statutes

This is the RECALL MULTIPLIER for case law:
  - 59.4% of val citations are statutory, 40.6% are case law
  - Train data is 98.8% statutory → model can't learn case law patterns
  - The citation graph bridges this gap mechanically

CPU only, runs in <1 second per query.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage3_graph_expansion.py                    # all val queries
#   python stage3_graph_expansion.py --split test       # test queries
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   index/citation_graph.pkl       (from stage0)
#   checkpoints/stage2_{split}.json (from stage2)
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage3_{split}.json
#   Per query: {query_id: {expanded_pool: [...], sources: {...},
#               graph_stats: {articles_expanded, cases_found, ...}}}
#
# ─── EXPECTED TIMING ────────────────────────────────────────────────────────
#   <1 second per query, ~40 queries = ~30 seconds total
"""



sys.path.insert(0, str(Path(__file__).parent))


CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)


def is_law_citation(c: str) -> bool:
    return bool(re.match(r"^Art\.\s+\d", c.strip()))


def is_case_citation(c: str) -> bool:
    return bool(re.match(r"^BGE\s+\d+", c.strip()) or re.match(r"^\d+[A-Z]_\d+/\d{4}", c.strip()))


def _extract_base_article(cite: str) -> str | None:
    """'Art. 221 Abs. 1 lit. b StPO' -> 'Art. 221 StPO'"""
    m = re.match(r"(Art\.\s+\d+[a-z]*)\s+(?:Abs\.|lit\.).*?\s+(\S+)$", cite, re.IGNORECASE)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    m = re.match(r"(Art\.\s+\d+[a-z]*)\s+(\S+)$", cite, re.IGNORECASE)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    return None


# Sources considered "strong" for targeted sibling expansion
_STRONG_SOURCES = {
    "explicit_from_query", "bm25_top10", "bm25_initial",
    "llm_stage1", "llm_stage1_procedural",
    "mas_rewrite", "mas_supplement", "mas_decompose",
    "mas_supportive", "mas_crossref",
}

_GOLD_PRIOR_RULES = {
    "law-law": {"min_support": 3, "min_npmi": 0.45, "top_n": 3},
    "law-court": {"min_support": 2, "min_npmi": 0.70, "top_n": 1},
    "court-court": {"min_support": 3, "min_npmi": 0.90, "top_n": 1},
}


def expand_single_query(
    candidate_pool: list[str],
    sources: dict[str, list[str]],
    graph_retriever,
    gold_prior=None,
    gold_prior_rules: dict[str, dict[str, float | int]] | None = None,
    top_cases_per_article: int = 3,
    top_related_cases: int = 2,
    a2a_lower: dict[str, list[str]] | None = None,
    statute_to_bge: dict[str, list[str]] | None = None,
    statute_to_bge_topk: int = 5,
    statute_to_numbered: dict[str, list[str]] | None = None,
    statute_to_numbered_topk: int = 5,
    numbered_to_statutes: dict[str, list[str]] | None = None,
    numbered_to_bge: dict[str, list[str]] | None = None,
    numbered_to_bge_topk: int = 3,
    numbered_cocitation: dict[str, list[str]] | None = None,
    numbered_cocitation_topk: int = 3,
) -> dict:
    """
    Expand a candidate pool via the citation graph.

    Three expansion directions:
      1. Statute  → top citing court decisions   (article_to_cases)
      2. Case     → all cited statutes            (case_to_articles)
      3. Case     → top related cases             (case_to_cases)  ← NEW

    Returns {
        'expanded_pool': list of all citations (original + graph-found),
        'sources': updated sources dict,
        'graph_stats': {...}
    }
    """
    expanded = set(candidate_pool)
    src = {c: set(s) for c, s in sources.items()}

    articles_expanded      = 0
    cases_found            = 0
    reverse_articles_found = 0
    related_cases_found    = 0

    for cite in list(candidate_pool):
        if is_law_citation(cite):
            # Statute → top citing court decisions
            citing_cases = graph_retriever.articles_to_cases(
                [cite], top_k_per_article=top_cases_per_article
            )
            for case_cite, score in citing_cases[:top_cases_per_article]:
                if case_cite not in expanded:
                    cases_found += 1
                expanded.add(case_cite)
                src.setdefault(case_cite, set()).add("citation_graph")
            if citing_cases:
                articles_expanded += 1

        elif is_case_citation(cite):
            # Court decision → cited statutes
            cited_articles = graph_retriever.cases_to_articles([cite])
            for art in cited_articles:
                if art not in expanded:
                    reverse_articles_found += 1
                expanded.add(art)
                src.setdefault(art, set()).add("citation_graph")

            # Court decision → related cases (case-to-case graph)
            related = graph_retriever.graph.get_related_cases(
                cite, top_n=top_related_cases
            )
            for rel_cite in related:
                if rel_cite not in expanded:
                    related_cases_found += 1
                expanded.add(rel_cite)
                src.setdefault(rel_cite, set()).add("citation_graph")

    # ── Targeted sibling Abs. expansion ─────────────────────────────────
    # For articles with strong retrieval signals, add all sibling Abs.
    # paragraphs from the same base article. E.g., if Art. 277 Abs. 2 ZGB
    # has a bm25_initial hit, also add Art. 277 Abs. 1 ZGB.
    sibling_found = 0
    if a2a_lower:
        pool_lower = {c.lower() for c in expanded}
        for cite in list(candidate_pool):
            if not cite.startswith("Art."):
                continue
            cite_sources = src.get(cite, set())
            if not cite_sources & _STRONG_SOURCES:
                continue
            base = _extract_base_article(cite)
            if not base:
                continue
            children = a2a_lower.get(base.lower(), [])
            for sib in children:
                if sib.lower() not in pool_lower:
                    expanded.add(sib)
                    pool_lower.add(sib.lower())
                    src.setdefault(sib, set()).add("sibling_expansion")
                    sibling_found += 1

    # ── Text-mined statute→BGE expansion ────────────────────────────────
    # Cell 37 showed 0 statute→BGE edges in the precomputed graph, so BGE
    # gold was unreachable via graph expansion. We text-mine edges from
    # court_considerations.csv in a separate builder cell; here we just
    # inject the top-K most-co-occurring BGEs for each statute candidate
    # that has a strong retrieval signal.
    text_mined_bges_found = 0
    if statute_to_bge:
        for cite in list(candidate_pool):
            if not is_law_citation(cite):
                continue
            cite_sources = src.get(cite, set())
            if not cite_sources & _STRONG_SOURCES:
                continue
            # Prefer the base article key (Art. 221 StPO) but also try
            # the full citation in case it was indexed verbatim.
            base = _extract_base_article(cite) or cite
            bges = statute_to_bge.get(base) or statute_to_bge.get(cite) or []
            for bge in bges[:statute_to_bge_topk]:
                if bge not in expanded:
                    text_mined_bges_found += 1
                expanded.add(bge)
                src.setdefault(bge, set()).add("citation_graph_text_mined")

    # ── Text-mined statute → numbered case expansion ────────────────────
    text_mined_numbered_found = 0
    if statute_to_numbered:
        for cite in list(candidate_pool):
            if not is_law_citation(cite):
                continue
            cite_sources = src.get(cite, set())
            if not cite_sources & _STRONG_SOURCES:
                continue
            base = _extract_base_article(cite) or cite
            cases = statute_to_numbered.get(base) or statute_to_numbered.get(cite) or []
            for case in cases[:statute_to_numbered_topk]:
                if case not in expanded:
                    text_mined_numbered_found += 1
                expanded.add(case)
                src.setdefault(case, set()).add("citation_graph_text_mined")

    # ── Numbered case → statutes / BGE / co-citation ────────────────────
    # Iterate over ALL numbered cases in expanded (not just candidate_pool)
    # so that cases added by statute_to_numbered also get second-hop expansions.
    numbered_reverse_statutes_found = 0
    numbered_to_bge_found = 0
    numbered_cocit_found = 0
    # STRONG-SIGNAL GATE: only second-hop-expand numbered cases that
    # came from retrieval/graph (not from statute_to_numbered itself).
    # Prior version cascaded — each text-mined numbered case seeded
    # 20 more statutes + 3 BGEs + 3 co-cited numbered, multiplying
    # the pool ~3×. Gating roughly halves total expansion with no
    # measurable recall loss on val.
    _all_numbered = [
        c for c in expanded
        if is_case_citation(c) and not c.startswith("BGE")
        and (src.get(c, set()) & _STRONG_SOURCES)
    ]
    for cite in _all_numbered:
        base = cite.split(" E.")[0].strip()
        if numbered_to_statutes:
            for stat in numbered_to_statutes.get(base, [])[:5]:
                if stat not in expanded:
                    numbered_reverse_statutes_found += 1
                expanded.add(stat)
                src.setdefault(stat, set()).add("citation_graph_text_mined")
        if numbered_to_bge:
            for bge in numbered_to_bge.get(base, [])[:numbered_to_bge_topk]:
                if bge not in expanded:
                    numbered_to_bge_found += 1
                expanded.add(bge)
                src.setdefault(bge, set()).add("citation_graph_text_mined")
        if numbered_cocitation:
            for rel in numbered_cocitation.get(base, [])[:numbered_cocitation_topk]:
                if rel not in expanded:
                    numbered_cocit_found += 1
                expanded.add(rel)
                src.setdefault(rel, set()).add("citation_graph_text_mined")

    # ── Co-citation expansion: use PMI-ranked co-citations ──────────────
    # PMI filters out ubiquitous articles (Art. 66 BGG) that co-occur with
    # everything but carry no relevance signal. Tighter params than before:
    # top_n=3 (was 5), only expand from Stage 2 original pool (not graph-found).
    co_cited_found = 0
    _original_pool = set(candidate_pool)  # only expand from retrieval hits
    for cite in list(_original_pool):
        companions = graph_retriever.graph.get_co_cited_pmi(
            cite, top_n=3
        )
        for companion, pmi_score in companions:
            if pmi_score < 2.0:  # skip low-PMI (generic) co-citations
                continue
            if companion not in expanded:
                co_cited_found += 1
            expanded.add(companion)
            src.setdefault(companion, set()).add("co_citation")

    # ── Train-only gold co-citation prior expansion ──────────────────────
    # This uses train supervision only, so it is safe for runtime. We keep it
    # conservative and pair-type-aware because court-court co-occurrences are
    # much sparser and noisier than statute bundles.
    gold_prior_found = 0
    gold_prior_rules = gold_prior_rules or _GOLD_PRIOR_RULES
    if gold_prior:
        for cite in list(_original_pool):
            cite_sources = src.get(cite, set())
            if not cite_sources & _STRONG_SOURCES:
                continue

            pair_types = {"law-law", "law-court"} if is_law_citation(cite) else {"law-court", "court-court"}
            for pair_type in pair_types:
                rule = gold_prior_rules[pair_type]
                companions = gold_prior.get_companions(
                    cite,
                    pair_types={pair_type},
                    min_support=rule["min_support"],
                    min_npmi=rule["min_npmi"],
                    top_n=rule["top_n"],
                )
                for item in companions:
                    companion = item["citation"]
                    if companion not in expanded:
                        gold_prior_found += 1
                    expanded.add(companion)
                    src.setdefault(companion, set()).add(f"gold_cocitation_prior:{pair_type}")

    return {
        "expanded_pool": sorted(expanded),
        "sources": {c: sorted(s) for c, s in src.items()},
        "gold_prior_rules": gold_prior_rules if gold_prior else None,
        "graph_stats": {
            "original_pool":           len(candidate_pool),
            "expanded_pool":           len(expanded),
            "articles_expanded":       articles_expanded,
            "cases_found_via_graph":   cases_found,
            "reverse_articles_found":  reverse_articles_found,
            "related_cases_found":     related_cases_found,
            "co_cited_found":          co_cited_found,
            "gold_prior_found":        gold_prior_found,
            "sibling_found":           sibling_found,
            "text_mined_bges_found":          text_mined_bges_found,
        "text_mined_numbered_found":      text_mined_numbered_found,
        "numbered_reverse_statutes_found": numbered_reverse_statutes_found,
        "numbered_to_bge_found":           numbered_to_bge_found,
        "numbered_cocit_found":            numbered_cocit_found,
        },
    }


def _load_gold_prior(prior_mode: str):
    if prior_mode == "off":
        print("  Gold co-citation prior disabled")
        return None

    if prior_mode == "trainval":
        prior_path = GOLD_COCITATION_PRIOR_TRAINVAL_PKL
        label = "train+val"
    else:
        prior_path = GOLD_COCITATION_PRIOR_TRAIN_PKL
        label = "train-only"

    if not prior_path.exists():
        print(f"  Gold co-citation prior not found at {prior_path}; skipping that expansion channel")
        return None

    print(f"Loading {label} gold co-citation prior...")
    prior = GoldCoCitationPrior(prior_path)
    print(f"  Loaded prior '{prior.prior_name}' for {len(prior.neighbors):,} citations")
    return prior


def run_batch_stage3(
    split: str,
    resume: bool = True,
    gold_prior_mode: str = "train",
    gold_prior_rules: dict[str, dict[str, float | int]] | None = None,
) -> dict:
    """Run Stage 3 on all queries in a split."""
    # Load Stage 2 results
    stage2_path = CHECKPOINTS_DIR / f"stage2_{split}.json"
    if not stage2_path.exists():
        print(f"ERROR: Stage 2 results not found: {stage2_path}")
        print("Run first: python stage2_mas_retrieval.py")
        sys.exit(1)
    with open(stage2_path, encoding="utf-8") as f:
        stage2_results = json.load(f)

    # Load citation graph
    if not CITATION_GRAPH_PKL.exists():
        print(f"ERROR: Citation graph not found: {CITATION_GRAPH_PKL}")
        print("Run first: python stage0_build_index.py --graph-only")
        sys.exit(1)
    print("Loading citation graph...")
    graph = GraphRetriever()

    gold_prior = _load_gold_prior(gold_prior_mode)

    # Load article_to_abs lookup for sibling expansion
    a2a_lower = {}
    if LOOKUP_PKL.exists():
        with open(LOOKUP_PKL, "rb") as f:
            lookup = pickle.load(f)
        a2a = lookup.get("article_to_abs", {})
        a2a_lower = {k.lower(): v for k, v in a2a.items()}
        print(f"  Loaded article_to_abs: {len(a2a_lower)} entries for sibling expansion")

    out_path = CHECKPOINTS_DIR / f"stage3_{split}.json"
    results = {}
    if resume and out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            results = json.load(f)
        print(f"  Resuming: {len(results)} queries already done")

    total_original = 0
    total_expanded = 0

    for qid, stage2 in stage2_results.items():
        if qid in results:
            stats = results[qid].get("graph_stats", {})
            total_original += stats.get("original_pool", 0)
            total_expanded += stats.get("expanded_pool", 0)
            continue

        t0 = time.time()
        result = expand_single_query(
            candidate_pool=stage2["candidate_pool"],
            sources=stage2.get("sources", {}),
            graph_retriever=graph,
            gold_prior=gold_prior,
            gold_prior_rules=gold_prior_rules,
            a2a_lower=a2a_lower,
            statute_to_bge=STATUTE_TO_BGE,
            statute_to_bge_topk=STAGE3_STATUTE_TO_BGE_TOPK,
            statute_to_numbered=STATUTE_TO_NUMBERED,
            statute_to_numbered_topk=STAGE3_STATUTE_TO_NUMBERED_TOPK,
            numbered_to_statutes=NUMBERED_TO_STATUTES,
            numbered_to_bge=NUMBERED_TO_BGE,
            numbered_to_bge_topk=STAGE3_NUMBERED_TO_BGE_TOPK,
            numbered_cocitation=NUMBERED_COCITATION,
            numbered_cocitation_topk=STAGE3_NUMBERED_COCITATION_TOPK,
        )
        elapsed = time.time() - t0

        stats = result["graph_stats"]
        total_original += stats["original_pool"]
        total_expanded += stats["expanded_pool"]

        print(f"  {qid}: {stats['original_pool']} → {stats['expanded_pool']} "
              f"(+{stats['cases_found_via_graph']} graph-cases, "
              f"+{stats.get('text_mined_bges_found', 0)} tm-BGE, "
              f"+{stats.get('text_mined_numbered_found', 0)} tm-numbered, "
              f"+{stats.get('numbered_to_bge_found', 0)} num->BGE, "
              f"+{stats.get('numbered_cocit_found', 0)} num-cocit) "
              f"[{elapsed:.2f}s]")

        results[qid] = result

    # Save
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nStage 3 complete. Saved: {out_path}")
    print(f"  Total: {total_original} → {total_expanded} citations "
          f"(+{total_expanded - total_original} from graph)")
    return results


### Stage 3 runner

In [51]:
stage3_outputs = run_stage3(split=SPLIT, gold_prior_mode=STAGE3_GOLD_PRIOR_MODE)

Loading citation graph...
  Loading citation graph ...
    103,125 articles with case links
    914,658 cases with article links
  Gold co-citation prior disabled
  Loaded article_to_abs: 43066 entries for sibling expansion
  val_001: 976 → 3721 (+222 graph-cases, +178 tm-BGE, +171 tm-numbered, +531 num->BGE, +379 num-cocit) [0.47s]
  val_002: 1122 → 3983 (+423 graph-cases, +209 tm-BGE, +207 tm-numbered, +420 num->BGE, +279 num-cocit) [0.06s]
  val_003: 1102 → 4431 (+313 graph-cases, +249 tm-BGE, +231 tm-numbered, +608 num->BGE, +385 num-cocit) [0.06s]
  val_004: 1066 → 4568 (+357 graph-cases, +229 tm-BGE, +299 tm-numbered, +658 num->BGE, +343 num-cocit) [1.35s]
  val_005: 1229 → 4703 (+342 graph-cases, +196 tm-BGE, +257 tm-numbered, +663 num->BGE, +396 num-cocit) [0.06s]
  val_006: 1105 → 4969 (+568 graph-cases, +286 tm-BGE, +368 tm-numbered, +507 num->BGE, +320 num-cocit) [0.06s]
  val_007: 1430 → 5792 (+406 graph-cases, +237 tm-BGE, +282 tm-numbered, +821 num->BGE, +465 num-cocit) [

### Stage 3 diagnosis (5 cells) + BGE safety-net + Abs-level audit

1. Candidate counts, per-type recall post-expansion
2. Graph reachability: val-gold BGE at 1-hop / 2-hop from Stage-2 seeds
3. Procedural-cite coverage per query (BGG / ZPO procedural gold)
4. BGE safety-net (read-only): what BGE citations *could* be added
5. Abs-level coverage audit (exact / parent-Art / missing)


In [52]:

# ══ Stage 3 diag 1/5: candidate counts + per-type recall ═══════════════
import json
from statistics import mean

s3_path = get_checkpoint_path("stage3", SPLIT)
if not s3_path.exists():
    print(f"not found: {s3_path}")
else:
    s3 = json.load(open(s3_path, encoding="utf-8"))

    def _cands(rec):
        v = rec.get("expanded_pool")          # Stage 3 primary key
        if isinstance(v, list): return v
        for k in ("expanded_candidates", "graph_candidates", "candidates", "all_candidates"):
            v = rec.get(k)
            if isinstance(v, list): return v
            if isinstance(v, dict):
                flat = [x for lst in v.values() if isinstance(lst, list) for x in lst]
                if flat: return flat
        return []

    counts    = [len(_cands(r)) for r in s3.values()]
    bge_per_q = [sum(1 for c in _cands(r) if c.startswith("BGE")) for r in s3.values()]
    num_per_q = [sum(1 for c in _cands(r) if "_" in c and "/" in c) for r in s3.values()]
    print(f"Stage 3 checkpoint: {s3_path}   queries: {len(s3)}")
    print(f"  cands/q:      min={min(counts)} max={max(counts)} mean={mean(counts):.1f}")
    print(f"  BGE/q:        min={min(bge_per_q)} max={max(bge_per_q)} mean={mean(bge_per_q):.1f}")
    print(f"  Numbered/q:   min={min(num_per_q)} max={max(num_per_q)} mean={mean(num_per_q):.1f}")

    if SPLIT == "val":
        import pandas as pd
        vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
        per = {"Law":[0,0], "BGE":[0,0], "Numbered":[0,0]}
        for _, r in vdf.iterrows():
            gold = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
            cs = set(_cands(s3.get(r["query_id"], {})))
            for c in gold:
                t = "BGE" if c.startswith("BGE") else ("Law" if c.startswith("Art") else "Numbered")
                per[t][1] += 1
                if c in cs: per[t][0] += 1
        for k, (h, t) in per.items():
            if t:
                flag = "  <-- FIX (graph expansion isn't producing BGEs)" if k == "BGE" and h/t < 0.3 else ""
                print(f"  {k}: {h}/{t} = {h/t*100:.1f}%{flag}")


Stage 3 checkpoint: /content/drive/MyDrive/swiss_law/checkpoints/stage3_val.json   queries: 10
  cands/q:      min=3721 max=6200 mean=4860.4
  BGE/q:        min=1083 max=1818 mean=1439.0
  Numbered/q:   min=1292 max=1998 mean=1661.2
  Law: 135/149 = 90.6%
  BGE: 58/69 = 84.1%
  Numbered: 26/33 = 78.8%


In [53]:

# ══ Stage 3 diag 2/5: graph reachability (1-hop / 2-hop) ═══════════════
import json, pickle
from pathlib import Path

s2_path = get_checkpoint_path("stage2", SPLIT)
if SPLIT == "val" and s2_path.exists():
    g = None
    for p in ["index/citation_graph.pkl", "index/citation_graph_v2.pkl", "index/edges.pkl"]:
        if Path(p).exists():
            try: g = pickle.load(open(p, "rb")); break
            except Exception: pass
    if g is None:
        print("  skipped (no graph file)")
    else:
        def _nbrs(x):
            try:
                if hasattr(g, "neighbors"): return set(str(n) for n in g.neighbors(str(x)))
            except Exception: pass
            if isinstance(g, dict):
                v = g.get(str(x))
                if isinstance(v, (list, set, tuple)): return set(str(n) for n in v)
                if isinstance(v, dict): return set(str(n) for n in v)
            return set()

        import pandas as pd
        vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
        s2 = json.load(open(s2_path, encoding="utf-8"))

        def _cands_s2(rec):
            v = rec.get("candidate_pool")     # Stage 2 primary key
            if isinstance(v, list): return v
            for k in ("bm25_candidates", "candidates", "all_candidates"):
                v = rec.get(k)
                if isinstance(v, list): return v
                if isinstance(v, dict):
                    return [x for lst in v.values() if isinstance(lst, list) for x in lst]
            return []

        r1 = [0,0]; r2 = [0,0]
        for _, row in vdf.iterrows():
            gold_bge = {c.strip().split(" E.")[0] for c in str(row["gold_citations"]).split(";") if c.strip().startswith("BGE")}
            if not gold_bge: continue
            seeds = set(_cands_s2(s2.get(row["query_id"], {})))
            hop1 = set()
            for s in seeds: hop1 |= _nbrs(s)
            hop1_stems = {c.split(" E.")[0] for c in hop1 if c.startswith("BGE")}
            hop2 = set()
            for s in list(hop1)[:500]: hop2 |= _nbrs(s)
            hop2_stems = {c.split(" E.")[0] for c in hop2 if c.startswith("BGE")}
            for g_ in gold_bge:
                r1[1] += 1; r2[1] += 1
                if g_ in hop1_stems: r1[0] += 1
                if g_ in hop1_stems or g_ in hop2_stems: r2[0] += 1
        if r2[1]:
            print(f"  val gold BGE reachable at 1-hop: {r1[0]}/{r1[1]} = {r1[0]/max(r1[1],1)*100:.1f}%")
            print(f"  val gold BGE reachable at 2-hop: {r2[0]}/{r2[1]} = {r2[0]/max(r2[1],1)*100:.1f}%")
            if r2[0] == 0:
                print("  ** graph has no path to val-gold BGEs — statute\u2192BGE edges are missing")
                print("     Fix: run build_statute_to_bge.py and build_numbered_indexes.py")


# ── v2 patch: also count BGE reachability via the NEW statute_to_bge.pkl
# (the original block above only consulted citation_graph.pkl, which has
# 0 statute→BGE edges — hence the misleading 0/56 printout).
try:
    if SPLIT == "val" and STATUTE_TO_BGE:
        import pandas as _p_tm, re as _r_tm
        vdf_tm = _p_tm.read_csv(VAL_CSV)
        _BGE_STEM = _r_tm.compile(r"^(BGE\s+\d+\s+[IVX]+\s+\d+)")
        reach_1hop = 0
        total = 0
        for _, row in vdf_tm.iterrows():
            qid = row["query_id"]
            gold = [c.strip() for c in str(row.get("gold_citations", "")).split(";") if c.strip()]
            bge_stems = set()
            for c in gold:
                m = _BGE_STEM.match(c)
                if m:
                    bge_stems.add(m.group(1))
            if not bge_stems:
                continue
            # Find BGEs reachable from any statute in this query's Stage-3 pool.
            try:
                s3 = _json.load(open(get_checkpoint_path("stage3", SPLIT), encoding="utf-8"))
            except Exception:
                s3 = {}
            pool = set(s3.get(qid, {}).get("expanded_pool", []))
            reachable_bges = set()
            for cite in pool:
                if not cite.startswith("Art."):
                    continue
                parts = cite.split()
                base = f"Art. {parts[1]} {parts[-1]}" if len(parts) >= 3 else cite
                for bge in STATUTE_TO_BGE.get(base, [])[:50]:
                    # Reduce BGE to stem for matching against gold stems
                    m = _BGE_STEM.match(bge)
                    if m:
                        reachable_bges.add(m.group(1))
            for stem in bge_stems:
                total += 1
                if stem in reachable_bges:
                    reach_1hop += 1
        if total:
            print(f"  val gold BGE reachable via text-mined statute_to_bge: "
                  f"{reach_1hop}/{total} = {reach_1hop/total:.1%}")
except Exception as _e_tm:
    print(f"  [v2-patch] text-mined BGE reachability check skipped: {_e_tm!r}")


  val gold BGE reachable at 1-hop: 0/56 = 0.0%
  val gold BGE reachable at 2-hop: 0/56 = 0.0%
  ** graph has no path to val-gold BGEs — statute→BGE edges are missing
     Fix: run build_statute_to_bge.py and build_numbered_indexes.py
  val gold BGE reachable via text-mined statute_to_bge: 26/56 = 46.4%


In [54]:

# ══ Stage 3 diag 3/5: procedural-cite coverage per query ═══════════════
import json
from collections import Counter

PROC = [
    "Art. 72 BGG", "Art. 74 BGG", "Art. 75 BGG", "Art. 76 BGG",
    "Art. 82 BGG", "Art. 90 BGG", "Art. 95 BGG", "Art. 97 BGG",
    "Art. 99 BGG", "Art. 100 BGG", "Art. 105 BGG", "Art. 106 BGG",
    "Art. 113 BGG", "Art. 66 BGG", "Art. 68 BGG",
]

s3_path = get_checkpoint_path("stage3", SPLIT)
if s3_path.exists() and SPLIT == "val":
    import pandas as pd
    s3 = json.load(open(s3_path, encoding="utf-8"))
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")

    def _cands(rec):
        v = rec.get("expanded_pool")          # Stage 3 primary key
        if isinstance(v, list): return v
        for k in ("expanded_candidates", "graph_candidates", "candidates", "all_candidates"):
            v = rec.get(k)
            if isinstance(v, list): return v
            if isinstance(v, dict):
                flat = [x for lst in v.values() if isinstance(lst, list) for x in lst]
                if flat: return flat
        return []

    in_gold_count = Counter()
    in_s3_count   = Counter()
    for _, r in vdf.iterrows():
        gold  = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
        cands = set(_cands(s3.get(r["query_id"], {})))
        for p in PROC:
            if any(c.startswith(p) or c == p for c in gold):
                in_gold_count[p] += 1
                if any(c.startswith(p) or c == p for c in cands):
                    in_s3_count[p] += 1

    print("  proc cite            gold-queries   in-stage3   coverage")
    for p in PROC:
        g = in_gold_count[p]; s = in_s3_count[p]
        if g: print(f"  {p:22s} {g:4d}           {s:4d}        {s/g*100:.0f}%")
    if sum(in_gold_count.values()):
        total_g = sum(in_gold_count.values()); total_s = sum(in_s3_count.values())
        print(f"\n  Total procedural gold events: {total_g}  covered by Stage 3: {total_s}  = {total_s/total_g*100:.0f}%")
        print(f"  Gap: {total_g - total_s} procedural cites that a small domain-prior could catch")


  proc cite            gold-queries   in-stage3   coverage
  Art. 82 BGG               1              1        100%
  Art. 113 BGG              1              1        100%

  Total procedural gold events: 2  covered by Stage 3: 2  = 100%
  Gap: 0 procedural cites that a small domain-prior could catch


In [55]:

# ══ Stage 3 diag 4/5: BGE safety-net (read-only) ═══════════════════════
import csv, re, json
from collections import Counter, defaultdict
from pathlib import Path

if SPLIT != "val":
    print("  skipped (val-only)")
else:
    s3_path = get_checkpoint_path("stage3", SPLIT)
    court_csv = COURT_CSV if 'COURT_CSV' in dir() else "data/court_considerations.csv"
    if not s3_path.exists() or not Path(court_csv).exists():
        print("  skipped (missing checkpoint or corpus)")
    else:
        import pandas as pd
        s3 = json.load(open(s3_path, encoding="utf-8"))
        vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")

        def _cands(rec):
            v = rec.get("expanded_pool")
            if isinstance(v, list): return v
            for k in ("expanded_candidates", "graph_candidates", "candidates"):
                v = rec.get(k)
                if isinstance(v, list): return v
                if isinstance(v, dict):
                    return [x for lst in v.values() if isinstance(lst, list) for x in lst]
            return []

        gold_bge = {r["query_id"]: {c.strip().split(" E.")[0] for c in str(r["gold_citations"]).split(";") if c.strip().startswith("BGE")}
                    for _, r in vdf.iterrows()}
        per_q_statutes = {}
        for qid, rec in s3.items():
            stats = [c for c in _cands(rec) if c.startswith("Art.")][:20]
            shorts = []
            for s in stats:
                m = re.match(r"Art\. *(\d+\w*) +([A-Z]{2,8})", s)
                if m: shorts.append((m.group(1), m.group(2)))
            per_q_statutes[qid] = shorts

        print("  streaming court_considerations.csv (1-2 min)...")
        bge_hits = defaultdict(Counter)
        pat = re.compile(r"Art\.\s*(\d+\w*)[^A-Za-z]+([A-Z]{2,8})")
        with open(court_csv, encoding="utf-8") as f:
            rdr = csv.reader(f); hdr = next(rdr)
            ci = hdr.index("citation") if "citation" in hdr else 0
            ti = hdr.index("text")     if "text"     in hdr else 1
            for i, row in enumerate(rdr):
                if i and i % 500_000 == 0: print(f"    {i:,} rows...")
                if len(row) <= max(ci, ti): continue
                cite = row[ci]
                if not cite.startswith("BGE"): continue
                stem = cite.split(" E.")[0]
                text = row[ti] or ""
                ms = set(pat.findall(text))
                for qid, wanted in per_q_statutes.items():
                    for art, code_ in wanted:
                        if (art, code_) in ms:
                            bge_hits[qid][stem] += 1
                            break
        mt = 0; ft = 0
        print("\n  MISSED gold BGEs the safety-net WOULD surface:")
        for qid, wants in gold_bge.items():
            in_s3 = {c.split(" E.")[0] for c in _cands(s3.get(qid, {})) if c.startswith("BGE")}
            missing = wants - in_s3
            caught = [b for b in missing if b in bge_hits[qid]]
            mt += len(missing); ft += len(caught)
            if missing:
                print(f"    {qid}: missing {len(missing)}  safety-net catches {len(caught)}")
                for b in caught[:5]:
                    print(f"       + {b} (mentions={bge_hits[qid][b]})")
        if mt:
            print(f"\n  Total: {ft}/{mt} missed BGEs recoverable via statute-text mining = {ft/mt*100:.0f}%")


  streaming court_considerations.csv (1-2 min)...
    500,000 rows...
    1,000,000 rows...
    1,500,000 rows...
    2,000,000 rows...

  MISSED gold BGEs the safety-net WOULD surface:
    val_003: missing 3  safety-net catches 0
    val_006: missing 2  safety-net catches 0

  Total: 0/5 missed BGEs recoverable via statute-text mining = 0%


In [56]:

# ══ Stage 3 diag 5/5: Abs-level coverage audit ═════════════════════════
import json, re
from collections import Counter

s3_path = get_checkpoint_path("stage3", SPLIT)
if s3_path.exists() and SPLIT == "val":
    import pandas as pd
    s3 = json.load(open(s3_path, encoding="utf-8"))
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")

    def _cands(rec):
        v = rec.get("expanded_pool")          # Stage 3 primary key
        if isinstance(v, list): return v
        for k in ("expanded_candidates", "graph_candidates", "candidates", "all_candidates"):
            v = rec.get(k)
            if isinstance(v, list): return v
            if isinstance(v, dict):
                flat = [x for lst in v.values() if isinstance(lst, list) for x in lst]
                if flat: return flat
        return []

    def _root(c):
        c = re.sub(r"\s+Abs\.?\s*\d+\w*", "", c)
        c = re.sub(r"\s+lit\.?\s*[a-z]", "", c)
        return c

    counter = Counter()
    flood_by_q = Counter()
    flood_examples = []
    for _, r in vdf.iterrows():
        qid = r["query_id"]
        gold  = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
        cands = set(_cands(s3.get(qid, {})))
        cand_roots = {_root(c) for c in cands if c.startswith("Art")}
        for g in gold:
            if not g.startswith("Art"): continue
            if " Abs. " not in g and " lit. " not in g:
                counter["bare_art_gold"] += 1; continue
            counter["abs_level_gold"] += 1
            if g in cands:               counter["abs_exact_hit"] += 1
            elif _root(g) in cand_roots: counter["abs_parent_only"] += 1
            else:                        counter["abs_missed"] += 1
        from collections import defaultdict as _dd
        by_root = _dd(list)
        for c in cands:
            if c.startswith("Art") and (" Abs. " in c or " lit. " in c):
                by_root[_root(c)].append(c)
        for root, vs in by_root.items():
            if len(vs) >= 3:
                flood_by_q[qid] += 1
                if len(flood_examples) < 10:
                    flood_examples.append((qid, root, vs[:5]))

    print("  Abs-level coverage on val gold:")
    tot = counter["abs_level_gold"]
    if tot:
        for k in ("abs_exact_hit", "abs_parent_only", "abs_missed"):
            v = counter[k]
            print(f"    {k:20s} {v}/{tot} = {v/tot*100:.0f}%")
    print(f"\n  Abs-flooding (same Art. with >=3 Abs. variants in candidates):")
    print(f"    {sum(flood_by_q.values())} events across {len(flood_by_q)} queries")
    for qid, root, vs in flood_examples[:5]:
        print(f"    {qid}: {root} -> {vs}")


  Abs-level coverage on val gold:
    abs_exact_hit        111/121 = 92%
    abs_parent_only      8/121 = 7%
    abs_missed           2/121 = 2%

  Abs-flooding (same Art. with >=3 Abs. variants in candidates):
    1284 events across 10 queries
    val_001: Art. 227 StPO -> ['Art. 227 Abs. 1 StPO', 'Art. 227 Abs. 5 StPO', 'Art. 227 Abs. 4 StPO', 'Art. 227 Abs. 2 StPO', 'Art. 227 Abs. 7 StPO']
    val_001: Art. 158 OR -> ['Art. 158 Abs. 2 OR', 'Art. 158 Abs. 3 OR', 'Art. 158 Abs. 1 OR']
    val_001: Art. 100 BGG -> ['Art. 100 Abs. 3 BGG', 'Art. 100 Abs. 6 BGG', 'Art. 100 Abs. 1 BGG', 'Art. 100 Abs. 5 BGG', 'Art. 100 Abs. 4 BGG']
    val_001: Art. 307 STPO -> ['Art. 307 Abs. 1 STPO', 'Art. 307 Abs. 2 STPO', 'Art. 307 Abs. 3 STPO']
    val_001: Art. 15 748.225.1 -> ['Art. 15 Abs. 3 748.225.1', 'Art. 15 Abs. 2 748.225.1', 'Art. 15 Abs. 4 748.225.1', 'Art. 15 Abs. 5 748.225.1', 'Art. 15 Abs. 1 748.225.1']


## Stage 4 — Reranking and Direct Generation

LLM reranking and cross-encoder scoring.

### Ranker prompt (inlined in new `stage4_llm_reranker.py`)

```text
You are a Swiss Federal Tribunal clerk drafting the citation list for a decision.

SCENARIO (English):
{query_text}

CANDIDATE CITATIONS (numbered; each with its German text snippet when available):
{numbered}

TASK: Score EACH numbered candidate by how likely a Swiss judge writing this
decision would cite it. Consider:

  - Directly applicable statutory provisions (score 8-10)
  - Procedural articles for this type of appeal/proceeding (score 6-9)
  - Constitutional rights invoked by the facts (score 6-9)
  - Leading BGE or case decisions on this specific issue (score 6-9)
  - Same-area but indirectly related provisions (score 3-5)
  - Clearly off-topic / different legal domain (score 0-2)

Output — ONE JSON object, no prose, no <think> blocks:
{"scores": {"<citation>": <0-10 float>, ...}}
```

**Key change vs. legacy prompt:** continuous 0-10 scores per candidate in a single pass, instead of 40 batches of tier-{0,1,2,3} judgments. The new Stage 5 already handles 0-10 scores (same tier-threshold code path).


### Direct-gen prompt (OFF by default)

Direct generation added noise on val. It's off in the default config; flip `STAGE4_SKIP_DIRECT_GEN = False` to enable. Kept in the code for debugging.


### `stage4_llm_reranker.py`

In [57]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage4_llm_reranker.py"

"""
stage4_llm_reranker.py -- STAGE 4: MINIMAL LLM RANKER (single-pass rewrite)

Replaces the old multi-batch Qwen3-32B reranker + direct-gen stack that ran
for 6+ hours on the val set and underperformed pure BM25 top-20.

Key changes vs. previous design
  * ONE prompt per query (not ~40 batches of 30). Runtime: ~60s/query on H100.
  * Candidate pool capped at RANK_POOL_SIZE (~120) drawn from Stage 2 BM25
    top-N + Stage 1 procedural + explicit + Stage 3 case candidates. Matches
    the research finding: small, focused pool beats a 1200-candidate sweep.
  * Output is a CONTINUOUS 0-10 score per candidate, not a {2,3} tier. This
    lets Stage 5 rank instead of threshold-on-tier, which was the main
    precision leak. Stage 5 already handles 0-10 scores (see lines 230-236).
  * Direct generation is OFF by default (it added noise on val). Enable via
    STAGE4_DIRECT_GEN = True only if you see missing categories on shadow-val.

Output schema is UNCHANGED so Stage 5 still works:
  {qid: {reranked, reranked_tiers, direct_gen, explicit, procedural,
         multi_signal, stage1_passthrough, all_citations, sources, stats}}
"""

sys.path.insert(0, str(Path(__file__).parent))

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Tunables (mirrored in config cell as STAGE4_* knobs) ─────────────────
# NOTE on max_new_tokens: A 120-item JSON like {"scores": {"cite": 8.5, ...}}
# costs ~25-35 output tokens per item = 3000-4200 tokens. We observed
# truncation + empty JSON parses when max_new_tokens was 2000 (9/10 val
# queries failed). Bumped to 4500 for safety; with pool_size=80 the typical
# output is ~2800 tokens leaving headroom.
_DEFAULT_RANK_POOL_SIZE = 150   # v2: was 80; enlarged now that Stage-3 is 8K wide
_DEFAULT_BM25_TOP_N     = 110   # v2: pool grew to 150, keep BM25 as the primary feeder
_DEFAULT_CASE_TOP_N     = 20    # from Stage 3 graph expansion (case citations)
_DEFAULT_LAW_TOP_N      = 20    # from Stage 3 graph expansion (law citations)
_DEFAULT_MAX_NEW_TOKENS = 4500  # 120 items * ~35 tok/item + safety margin
_DEFAULT_SNIPPET_LEN    = 140   # chars of German article text per candidate

# Signals that mean "this candidate has real retrieval backing"
_SIGNAL_SOURCES = {
    "explicit_from_query", "bm25_top10", "bm25_initial",
    "citation_graph", "llm_stage1", "llm_stage1_procedural",
    "mas_rewrite", "mas_supplement", "mas_decompose",
    "mas_supportive", "mas_crossref",
}

_SWISS_ART_RE  = re.compile(r"^Art\.\s+\d+")
_SWISS_CASE_RE = re.compile(r"^(BGE\s+\d+|\d+[A-Z]_\d+/\d{4})")
_CITE_RE       = re.compile(r"^(Art\.\s+\d+|BGE\s+\d+|\d+[A-Z]_\d+/\d{4})")

# ── Ranker prompt ────────────────────────────────────────────────────────
_RANK_PROMPT = """You are a Swiss Federal Tribunal clerk drafting the citation list for a decision.

SCENARIO (English):
{query_text}

CANDIDATE CITATIONS (numbered; each shown with its German text snippet when available):
{numbered}

TASK: Score EACH numbered candidate by how likely a Swiss judge writing this
decision would cite it. Consider:

  - Directly applicable statutory provisions for the facts (score 8-10)
  - Procedural articles for this type of appeal/proceeding (score 6-9)
  - Constitutional rights invoked by the facts (score 6-9)
  - Leading BGE or case decisions on this specific issue (score 6-9)
  - Same-area but indirectly related provisions (score 3-5)
  - Clearly off-topic / different legal domain (score 0-2)

A typical Federal Tribunal decision cites 15-40 provisions. Be GENEROUS with
procedural and constitutional citations (they almost always appear).

Output format — ONE JSON object, no prose, no <think> blocks:
{{"scores": {{"<citation #1 exactly as shown>": <0-10 float>,
             "<citation #2 exactly as shown>": <0-10 float>,
             ...}}}}

Include EVERY candidate with a score >= 1. Omit only clearly irrelevant (0)
candidates. Use the EXACT citation string from the numbered list."""

# Fallback direct-gen prompt (OFF by default; kept for debugging)
_DIRECT_GEN_PROMPT = """You are drafting the citation list for a Swiss Federal Tribunal decision.

SCENARIO:
{query_text}

CITATIONS ALREADY FOUND (top 20):
{selected_citations}

A typical Federal Tribunal decision cites 15-40 provisions. Generate UP TO 15
additional citations that a court would cite but are MISSING from the list.

Focus on:
  * Procedural: Art. 82/90/95/100/105/106 BGG; Art. 66/68 BGG; Art. 422/428 StPO
  * Constitutional: Art. 29 BV, Art. 10 BV, Art. 31 BV when fundamental rights apply
  * Leading BGE decisions on the specific legal issue

Output ONLY:
{{"additional": ["Art. 100 Abs. 1 BGG", "Art. 66 Abs. 1 BGG", ...]}}"""


# ── Citation text cache (same implementation as legacy Stage 4) ─────────
_CITATION_TEXTS: "dict[str, str] | None" = None


def _load_citation_texts() -> "dict[str, str]":
    """Load canonical citation -> text_snippet map once from KB JSONL."""
    global _CITATION_TEXTS
    if _CITATION_TEXTS is not None:
        return _CITATION_TEXTS

    texts: "dict[str, str]" = {}
    base_texts: "dict[str, str]" = {}

    print("  Loading citation texts from KB...")
    try:
        with open(KB_JSONL, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                canon = rec.get("citation_canon", "")
                if not canon:
                    continue
                text = (rec.get("search", {}).get("search_text_de", "")
                        or rec.get("content", {}).get("text_clean_de", "") or "")
                if not text:
                    continue
                texts[canon] = text[:_DEFAULT_SNIPPET_LEN]
                if canon.startswith("Art."):
                    parts = canon.split()
                    if len(parts) >= 3:
                        base = f"Art. {parts[1]} {parts[-1]}"
                        if base not in base_texts:
                            base_texts[base] = text[:_DEFAULT_SNIPPET_LEN]
    except Exception as e:
        print(f"  WARNING: could not load citation texts: {e}")

    for base, text in base_texts.items():
        if base not in texts:
            texts[base] = text

    _CITATION_TEXTS = texts
    print(f"  Loaded {len(_CITATION_TEXTS):,} citation texts")
    return _CITATION_TEXTS


def _get_text(cite: str, texts: "dict[str, str]") -> str:
    """Look up text with Abs/lit fallback to base article."""
    if cite in texts:
        return texts[cite]
    if cite.startswith("Art."):
        parts = cite.split()
        if len(parts) >= 3:
            base = f"Art. {parts[1]} {parts[-1]}"
            if base in texts:
                return texts[base]
    return ""


# ── Pool construction ────────────────────────────────────────────────────
def _build_rank_pool(
    stage1_analysis: dict,
    stage2_result: dict,
    stage3_result: dict,
    pool_size: int = _DEFAULT_RANK_POOL_SIZE,
    bm25_top_n: int = _DEFAULT_BM25_TOP_N,
    case_top_n: int = _DEFAULT_CASE_TOP_N,
    law_top_n:  int = _DEFAULT_LAW_TOP_N,
) -> "tuple[list[str], dict[str, set[str]]]":
    """Compose the ranking pool with deterministic ordering.

    Priority (added until pool_size is hit):
      1. Explicit citations from query regex (always in)
      2. Stage 1 procedural articles (always in)
      3. Stage 1 LLM-proposed candidate_articles + candidate_cases
      4. Top BM25 (from Stage 2 bm25_scores)
      5. Top case citations from Stage 3 expansion not already present
      6. Top law citations from Stage 3 expansion not already present

    Returns (pool_list, sources_dict) where sources carry provenance tags.
    """
    bm25_scores = stage2_result.get("bm25_scores", {}) if stage2_result else {}
    expanded = stage3_result.get("expanded_pool", []) if stage3_result else []
    sources_raw = stage3_result.get("sources", {}) if stage3_result else {}

    pool: list = []
    seen: set = set()
    sources: "dict[str, set[str]]" = {c: set(s) for c, s in sources_raw.items()}

    def _add(cite: str, tag: str):
        if not cite or not _CITE_RE.match(cite):
            return False
        if cite in seen:
            sources.setdefault(cite, set()).add(tag)
            return False
        if len(pool) >= pool_size:
            return False
        pool.append(cite)
        seen.add(cite)
        sources.setdefault(cite, set()).add(tag)
        return True

    # 1. Explicit citations from query
    for c in stage1_analysis.get("explicit_citations", []):
        _add(c, "explicit_from_query")

    # 2. Procedural articles (universal BGG, costs, admissibility)
    for c in stage1_analysis.get("procedural_articles", []):
        _add(c, "llm_stage1_procedural")

    # 3. LLM stage1 suggestions
    for c in stage1_analysis.get("candidate_articles", []) + stage1_analysis.get("candidate_cases", []):
        _add(c, "llm_stage1")

    # 4. BM25 top-N (includes both laws + cases, already fused)
    bm_sorted = sorted(bm25_scores.items(), key=lambda x: -x[1])
    for c, _ in bm_sorted[:bm25_top_n]:
        _add(c, "bm25_top10")

    # 5. Stage-3 case citations not yet in pool
    stage3_cases = [c for c in expanded if not c.startswith("Art") and _CITE_RE.match(c)]
    for c in stage3_cases[:case_top_n * 3]:  # look ahead, many may already be in pool
        _add(c, "citation_graph")

    # 6. Stage-3 law citations not yet in pool
    stage3_laws = [c for c in expanded if c.startswith("Art")]
    for c in stage3_laws[:law_top_n * 3]:
        _add(c, "citation_graph")

    return pool, sources


# ── LLM ranker ───────────────────────────────────────────────────────────
def _build_ranking_prompt(query_text: str, pool: list, citation_texts: dict) -> str:
    lines = []
    for i, c in enumerate(pool, 1):
        txt = _get_text(c, citation_texts)
        if txt:
            lines.append(f"{i}. {c} — {txt.replace(chr(10), ' ')[:_DEFAULT_SNIPPET_LEN]}")
        else:
            lines.append(f"{i}. {c}")
    numbered = "\n".join(lines)
    return _RANK_PROMPT.format(query_text=query_text, numbered=numbered)


def _parse_partial_json_pairs(raw_text: str) -> "dict[str, float]":
    """Salvage "key": number pairs from TRUNCATED JSON output.

    When the LLM hits max_new_tokens mid-response, json.loads fails and
    generate_json returns {}. This function scans the raw text with a regex
    and recovers every complete "key": number pair — order-preserving.

    Handles:
      * Unclosed outer {} (common truncation case)
      * Trailing comma / incomplete pair at the end (ignored)
      * Escaped quotes inside keys
    """
    out: "dict[str, float]" = {}
    # Match '"<key-without-unescaped-quote>"\s*:\s*<number>'
    pattern = re.compile(r'"((?:[^"\\]|\\.)+)"\s*:\s*(-?\d+(?:\.\d+)?)')
    for m in pattern.finditer(raw_text or ""):
        key = m.group(1).strip()
        try:
            score = float(m.group(2))
        except ValueError:
            continue
        if score < 0: score = 0.0
        if score > 10: score = 10.0
        out[key] = max(out.get(key, 0.0), score)
    return out


def _parse_ranker_output(response: dict, pool: list,
                         raw_text: str = "") -> "dict[str, float]":
    """Extract {cite: 0-10 score} from LLM output.

    Tries the parsed JSON first; if empty and raw_text is provided, falls
    back to the partial-pair regex parser (handles truncated output).
    """
    scores_raw = response.get("scores", {}) if isinstance(response, dict) else {}
    if not isinstance(scores_raw, dict) or not scores_raw:
        if raw_text:
            scores_raw = _parse_partial_json_pairs(raw_text)
        if not scores_raw:
            return {}

    # Build lowercase lookup for robust matching to pool
    pool_lookup = {c.lower(): c for c in pool}
    out: "dict[str, float]" = {}
    for key, val in scores_raw.items():
        if not isinstance(key, str):
            continue
        key_s = key.strip()
        try:
            score = float(val)
        except (TypeError, ValueError):
            continue
        if score < 0: score = 0.0
        if score > 10: score = 10.0
        canonical = pool_lookup.get(key_s.lower(), key_s)
        out[canonical] = max(out.get(canonical, 0.0), score)
    return out


def _generate_ranker_once(
    query_text: str,
    pool: list,
    llm_backend_mod,
    citation_texts: dict,
    max_new_tokens: int,
) -> "dict[str, float]":
    """One ranker call. Captures raw text so the partial-pair parser can
    salvage scores if json.loads fails due to truncation.
    """
    prompt = _build_ranking_prompt(query_text, pool, citation_texts)
    raw_text = ""
    try:
        raw_text = llm_backend_mod.generate(
            system_prompt="You are a Swiss law expert. Output only JSON.",
            user_prompt=prompt,
            max_new_tokens=max_new_tokens,
            enable_thinking=False,
        )
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            try: torch.cuda.empty_cache()
            except Exception: pass
            raw_text = llm_backend_mod.generate(
                system_prompt="You are a Swiss law expert. Output only JSON.",
                user_prompt=prompt,
                max_new_tokens=max_new_tokens // 2,
                enable_thinking=False,
            )
        else:
            raise

    # Try clean JSON first, then partial-pair salvage on the raw text
    try:
        parsed = llm_backend_mod.parse_json_response(raw_text)
    except Exception:
        parsed = {}
    return _parse_ranker_output(parsed, pool, raw_text=raw_text)


def rank_candidates_single_pass(
    query_text: str,
    pool: list,
    llm_backend_mod,
    citation_texts: dict,
    max_new_tokens: int = _DEFAULT_MAX_NEW_TOKENS,
) -> "dict[str, float]":
    """ONE LLM call that ranks the entire pool.

    If the pool is too large for a single prompt (> 160 items), falls back
    to 2 chunks. If the first call returns 0 scores (model refused / format
    failure), automatically retries once with a shorter explicit prompt.
    """
    CHUNK = 160
    if len(pool) <= CHUNK:
        scored = _generate_ranker_once(query_text, pool, llm_backend_mod,
                                       citation_texts, max_new_tokens)
        if not scored:
            print("      ranker returned 0 scores — retrying with shorter pool "
                  f"(top-{min(60, len(pool))})", flush=True)
            short_pool = pool[:min(60, len(pool))]
            scored = _generate_ranker_once(query_text, short_pool,
                                           llm_backend_mod, citation_texts,
                                           max_new_tokens)
        return scored

    # Chunked fallback for oversized pools (keeps one ranker call per chunk)
    scored: "dict[str, float]" = {}
    for start in range(0, len(pool), CHUNK):
        chunk = pool[start:start + CHUNK]
        for c, s in _generate_ranker_once(query_text, chunk, llm_backend_mod,
                                          citation_texts, max_new_tokens).items():
            scored[c] = max(scored.get(c, 0.0), s)
    return scored


# ── Direct generation (disabled by default) ──────────────────────────────
def generate_direct_citations(query_text: str, already_selected: list, llm_backend_mod) -> list:
    prompt = _DIRECT_GEN_PROMPT.format(
        query_text=query_text,
        selected_citations="\n".join(already_selected[:20]),
    )
    resp = llm_backend_mod.generate_json(
        system_prompt="You are a Swiss law expert. Output only JSON.",
        user_prompt=prompt,
        max_new_tokens=400,
        enable_thinking=False,
    )
    additional = resp.get("additional", [])
    if not isinstance(additional, list):
        return []
    out = []
    for c in additional:
        if not isinstance(c, str): continue
        c = c.strip()
        if " ff." in c or " ff " in c: continue
        if _SWISS_ART_RE.match(c) or _SWISS_CASE_RE.match(c):
            out.append(c)
    return out


# ── Per-query assembly (keeps legacy output schema) ─────────────────────
def _assemble_output(
    query_text: str,
    stage1_analysis: dict,
    stage3_result: dict,
    pool: list,
    ranker_scores: "dict[str, float]",
    direct_gen: list,
    sources: "dict[str, set[str]]",
) -> dict:
    # Tag sources with llm_reranker / llm_reranker_tier3 for Stage 5 compat
    for cite, score in ranker_scores.items():
        s = sources.setdefault(cite, set())
        s.add("llm_reranker")
        if score >= 6.0:
            s.add("llm_reranker_tier3")

    for cite in direct_gen:
        sources.setdefault(cite, set()).add("llm_direct_gen")

    reranked_list = sorted(ranker_scores.keys())

    explicit_set = {c for c in stage1_analysis.get("explicit_citations", [])
                    if _CITE_RE.match(c)}
    procedural_set = set(stage1_analysis.get("procedural_articles", []))

    # multi_signal = any Stage-3 candidate with independent retrieval
    # signal that the LLM didn't score. Previously restricted to the
    # 80-item rank pool, which silently dropped 25 pp of Law gold
    # between Stage 3 and Stage 4 (diag cell 100). Expanding to the
    # full Stage-3 expanded_pool preserves recall; Stage 6's BM25+LLM
    # rescoring handles ranking without relying on the LLM to score
    # every candidate.
    stage3_full = stage3_result.get("expanded_pool", []) if stage3_result else []
    # Fallback for callers that didn't pass stage3_result: still use pool.
    _passthrough_universe = stage3_full if stage3_full else pool
    multi_signal = []
    for c in _passthrough_universe:
        if c in ranker_scores:
            continue
        c_sources = sources.get(c, set())
        if len(c_sources & _SIGNAL_SOURCES) >= 1:
            multi_signal.append(c)

    stage1_candidates = set(
        stage1_analysis.get("candidate_articles", [])
        + stage1_analysis.get("candidate_cases", [])
    )
    stage1_passthrough = [
        c for c in _passthrough_universe
        if c in stage1_candidates and c not in ranker_scores
    ]

    all_citations = list(dict.fromkeys(
        reranked_list
        + sorted(multi_signal)
        + sorted(stage1_passthrough)
        + sorted(explicit_set)
        + sorted(procedural_set)
        + list(direct_gen)
    ))

    return {
        "reranked": reranked_list,
        # Continuous float scores 0-10 (Stage 5 already handles this format):
        "reranked_tiers": {c: float(s) for c, s in ranker_scores.items()},
        "direct_gen": list(direct_gen),
        "explicit": sorted(explicit_set),
        "procedural": sorted(procedural_set),
        "multi_signal": sorted(multi_signal),
        "stage1_passthrough": sorted(stage1_passthrough),
        "all_citations": all_citations,
        "sources": {c: sorted(s) for c, s in sources.items() if c in set(all_citations)},
        "stats": {
            "pool_size": len(pool),
            "ranker_scored": len(ranker_scores),
            "ranker_high": sum(1 for v in ranker_scores.values() if v >= 6.0),
            "ranker_med":  sum(1 for v in ranker_scores.values() if 3.0 <= v < 6.0),
            "multi_signal": len(multi_signal),
            "stage1_pass": len(stage1_passthrough),
            "direct_gen": len(direct_gen),
            "explicit": len(explicit_set),
            "procedural": len(procedural_set),
            "total_out": len(all_citations),
        },
    }


def process_single_query(
    query_text: str,
    stage1_analysis: dict,
    stage2_result: dict,
    stage3_result: dict,
    llm_backend_mod,
    pool_size: int = _DEFAULT_RANK_POOL_SIZE,
    do_direct_gen: bool = False,
) -> dict:
    """Single-pass rank for one query. ~60s on H100."""
    pool, sources = _build_rank_pool(stage1_analysis, stage2_result, stage3_result,
                                     pool_size=pool_size)
    print(f"    Pool size: {len(pool)}  (bm25 + procedural + explicit + graph)")

    citation_texts = _load_citation_texts()

    t0 = time.time()
    ranker_scores = rank_candidates_single_pass(
        query_text, pool, llm_backend_mod, citation_texts,
    )
    print(f"    Ranker: {len(ranker_scores)} scored "
          f"(high={sum(1 for v in ranker_scores.values() if v>=6)} "
          f"med={sum(1 for v in ranker_scores.values() if 3<=v<6)}) "
          f"in {time.time()-t0:.1f}s")

    direct_gen = []
    if do_direct_gen:
        ranked_sorted = [c for c, _ in sorted(ranker_scores.items(), key=lambda x:-x[1])[:20]]
        direct_gen = generate_direct_citations(query_text, ranked_sorted, llm_backend_mod)
        print(f"    Direct-gen: {len(direct_gen)} added")

    return _assemble_output(query_text, stage1_analysis, stage3_result,
                            pool, ranker_scores, direct_gen, sources)


# ── Batch runner ────────────────────────────────────────────────────────
def run_batch_stage4(split: str, backend: str = "local",
                     skip_direct_gen: bool = True, resume: bool = True,
                     pool_size: int = _DEFAULT_RANK_POOL_SIZE) -> dict:
    """Run the minimal ranker over all queries in a split."""
    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    df = pd.read_csv(csv_map[split])

    stage1_path = CHECKPOINTS_DIR / f"stage1_{split}.json"
    stage2_path = CHECKPOINTS_DIR / f"stage2_{split}.json"
    stage3_path = CHECKPOINTS_DIR / f"stage3_{split}.json"
    for p, n in [(stage1_path, 1), (stage2_path, 2), (stage3_path, 3)]:
        if not p.exists():
            print(f"ERROR: Stage {n} checkpoint missing: {p}")
            sys.exit(1)

    with open(stage1_path, encoding="utf-8") as f: stage1 = json.load(f)
    with open(stage2_path, encoding="utf-8") as f: stage2 = json.load(f)
    with open(stage3_path, encoding="utf-8") as f: stage3 = json.load(f)

    if backend == "local":
        print("Loading Qwen3-32B...")
        get_model_and_tokenizer()

    out_path = CHECKPOINTS_DIR / f"stage4_{split}.json"
    results: dict = {}
    if resume and out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            results = json.load(f)
        # Drop any legacy-format entries that lack the new continuous scores.
        # (Legacy entries have tiered {2,3} values instead of floats.)
        to_redo = []
        for qid, r in results.items():
            tiers = r.get("reranked_tiers", {})
            if tiers and all(isinstance(v, int) and v <= 3 for v in tiers.values()):
                to_redo.append(qid)
        for qid in to_redo:
            del results[qid]
        if to_redo:
            print(f"  Resume: discarded {len(to_redo)} legacy-format entries "
                  f"(will re-rank): {to_redo[:5]}{'...' if len(to_redo)>5 else ''}")
        else:
            print(f"  Resume: {len(results)} queries already done")

    pending = []
    for _, row in df.iterrows():
        qid = row["query_id"]
        if qid in results:
            continue
        pending.append((qid, row["query"],
                        stage1.get(qid, {}),
                        stage2.get(qid, {}),
                        stage3.get(qid, {"expanded_pool": [], "sources": {}})))

    print(f"  {len(pending)} queries pending")

    llm_mod = sys.modules[__name__]  # use this cell's generate_json
    for i, (qid, qtext, s1, s2, s3) in enumerate(pending, 1):
        print(f"\n[{i}/{len(pending)}] {qid}")
        try:
            result = process_single_query(
                qtext, s1, s2, s3, llm_mod,
                pool_size=pool_size,
                do_direct_gen=(not skip_direct_gen),
            )
        except Exception as e:
            print(f"  ERROR: {e!r}")
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            result = {"all_citations": [], "sources": {},
                      "reranked": [], "reranked_tiers": {},
                      "direct_gen": [], "explicit": [],
                      "procedural": [], "multi_signal": [],
                      "stage1_passthrough": [],
                      "stats": {"error": str(e)}}

        results[qid] = result
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nStage 4 done. Saved: {out_path}")
    return results


### Stage 4 runner

In [58]:
stage4_outputs = run_stage4(split=SPLIT, backend=BACKEND, skip_direct_gen=STAGE4_SKIP_DIRECT_GEN)

Loading Qwen3-32B...
  10 queries pending

[1/10] val_001
    Pool size: 150  (bm25 + procedural + explicit + graph)
  Loading citation texts from KB...
  Loaded 220,490 citation texts
    Ranker: 78 scored (high=47 med=31) in 107.3s

[2/10] val_002
    Pool size: 150  (bm25 + procedural + explicit + graph)
    Ranker: 99 scored (high=45 med=54) in 145.9s

[3/10] val_003
    Pool size: 150  (bm25 + procedural + explicit + graph)
    Ranker: 125 scored (high=125 med=0) in 177.0s

[4/10] val_004
    Pool size: 150  (bm25 + procedural + explicit + graph)
    Ranker: 66 scored (high=58 med=8) in 82.5s

[5/10] val_005
    Pool size: 150  (bm25 + procedural + explicit + graph)
    Ranker: 129 scored (high=84 med=31) in 173.3s

[6/10] val_006
    Pool size: 150  (bm25 + procedural + explicit + graph)
    Ranker: 148 scored (high=146 med=2) in 89.3s

[7/10] val_007
    Pool size: 150  (bm25 + procedural + explicit + graph)
    Ranker: 134 scored (high=134 med=0) in 255.9s

[8/10] val_008
    P

### Stage 4 diagnosis (5 cells)

1. Counts (rerank list size, direct-gen size)
2. Direct-gen hallucination rate (generated cites not in KB)
3. Reranker tier collapse (score/tier variance)
4. Gold dropout from Stage 3 → Stage 4
5. Top-K recall on val (10, 20, 50)


In [59]:

# ══ Stage 4 diag 1/5: counts ═══════════════════════════════════════════
import json
from statistics import mean

s4_path = get_checkpoint_path("stage4", SPLIT)
if not s4_path.exists():
    print(f"not found: {s4_path}")
else:
    s4 = json.load(open(s4_path, encoding="utf-8"))
    def _rerank(r):
        for k in ("reranked", "llm_reranked", "top_citations", "reranker_output"):
            v = r.get(k)
            if isinstance(v, list): return v
        return []
    def _dgen(r):
        for k in ("direct_gen", "direct_generated", "llm_direct_gen", "generated"):
            v = r.get(k)
            if isinstance(v, list): return v
        return []
    rc = [len(_rerank(r)) for r in s4.values()]
    dc = [len(_dgen(r))   for r in s4.values()]
    print(f"  reranked/q:   min={min(rc)} max={max(rc)} mean={mean(rc):.1f}")
    print(f"  direct-gen/q: min={min(dc)} max={max(dc)} mean={mean(dc):.1f}")


  reranked/q:   min=66 max=149 mean=116.0
  direct-gen/q: min=0 max=0 mean=0.0


In [60]:

# ══ Stage 4 diag 2/5: direct-gen hallucination rate ════════════════════
import json
from pathlib import Path

s4_path = get_checkpoint_path("stage4", SPLIT)
laws_csv = LAWS_CSV if 'LAWS_CSV' in dir() else "data/laws_de.csv"
court_csv = COURT_CSV if 'COURT_CSV' in dir() else "data/court_considerations.csv"
if s4_path.exists() and Path(laws_csv).exists():
    import pandas as pd, csv as _csv
    s4 = json.load(open(s4_path, encoding="utf-8"))
    valid = set(pd.read_csv(laws_csv)["citation"].dropna().astype(str))
    court_stems = set()
    if Path(court_csv).exists():
        print("  loading court citation stems (1 pass)...")
        with open(court_csv, encoding="utf-8") as f:
            rdr = _csv.reader(f); hdr = next(rdr)
            ci = hdr.index("citation") if "citation" in hdr else 0
            for row in rdr:
                if row and len(row) > ci:
                    court_stems.add(row[ci])
                    if row[ci].startswith("BGE"):
                        court_stems.add(row[ci].split(" E.")[0])

    def _dgen(r):
        for k in ("direct_gen", "direct_generated", "llm_direct_gen", "generated"):
            v = r.get(k)
            if isinstance(v, list): return v
        return []

    total = 0; bad = 0; ex = []
    for qid, rec in s4.items():
        for c in _dgen(rec):
            total += 1
            ok = (c in valid) or (c in court_stems) or (c.split(" E.")[0] in court_stems)
            if not ok:
                bad += 1
                if len(ex) < 15: ex.append((qid, c))
    print(f"  direct-gen validity: {total-bad}/{total} = {(total-bad)/max(total,1)*100:.1f}%")
    for qid, c in ex: print(f"    HALLUC {qid}: {c}")


  loading court citation stems (1 pass)...
  direct-gen validity: 0/0 = 0.0%


In [61]:

# ══ Stage 4 diag 3/5: reranker tier distribution ═══════════════════════
# reranked_tiers = {citation: tier_int} where T3 = most relevant, T2 = relevant.
# If most citations land in T2, the reranker isn't discriminating well.
import json
from collections import Counter
from statistics import mean

s4_path = get_checkpoint_path("stage4", SPLIT)
if s4_path.exists():
    s4 = json.load(open(s4_path, encoding="utf-8"))
    tiers = Counter()
    for rec in s4.values():
        td = rec.get("reranked_tiers") or {}
        if isinstance(td, dict):
            for v in td.values():
                try: tiers[int(v)] += 1
                except Exception: pass
    if tiers:
        total = sum(tiers.values())
        print("  Tier distribution across all queries:")
        for t in sorted(tiers, reverse=True):
            n = tiers[t]
            print(f"    T{t}: {n:6d} ({n/total*100:.1f}%)")
        t3 = tiers.get(3, 0); t2 = tiers.get(2, 0)
        if t2 > 0 and t3 / max(t2, 1) < 0.2:
            print("  ** LOW T3/T2 ratio — reranker is accepting most candidates as T2 without strong discrimination")
    else:
        print("  (reranked_tiers not available)")


  Tier distribution across all queries:
    T10:      6 (0.5%)
    T9:     26 (2.2%)
    T8:     44 (3.8%)
    T7:    387 (33.4%)
    T6:    390 (33.6%)
    T5:    261 (22.5%)
    T4:     24 (2.1%)
    T3:      8 (0.7%)
    T2:     14 (1.2%)


In [62]:

# ══ Stage 4 diag 4/5: gold dropout Stage 3 → Stage 4 ═══════════════════
import json

s3_path = get_checkpoint_path("stage3", SPLIT)
s4_path = get_checkpoint_path("stage4", SPLIT)
if s3_path.exists() and s4_path.exists() and SPLIT == "val":
    import pandas as pd
    s3 = json.load(open(s3_path, encoding="utf-8"))
    s4 = json.load(open(s4_path, encoding="utf-8"))
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")

    def _c3(r):
        v = r.get("expanded_pool")           # Stage 3 primary key
        if isinstance(v, list): return v
        for k in ("expanded_candidates", "graph_candidates", "candidates"):
            v = r.get(k)
            if isinstance(v, list): return v
            if isinstance(v, dict):
                return [x for lst in v.values() if isinstance(lst, list) for x in lst]
        return []

    def _c4_all(r):
        """All citations Stage 4 kept (union of all output lists)."""
        seen = set()
        for k in ("all_citations", "reranked", "explicit", "procedural", "direct_gen",
                  "multi_signal", "stage1_passthrough"):
            for c in (r.get(k) or []):
                seen.add(c)
        tiers = r.get("reranked_tiers") or {}
        if isinstance(tiers, dict): seen.update(tiers.keys())
        return seen

    dropped_total = 0; gold_total = 0; dropped_ex = []
    for _, r in vdf.iterrows():
        qid = r["query_id"]
        gold = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
        in3  = set(_c3(s3.get(qid, {})))
        in4  = _c4_all(s4.get(qid, {}))
        for c in gold:
            if c in in3:
                gold_total += 1
                if c not in in4:
                    dropped_total += 1
                    dropped_ex.append((qid, c))
    print(f"  Gold in Stage 3 but DROPPED at Stage 4: {dropped_total}/{gold_total}")
    for qid, c in dropped_ex[:15]:
        print(f"    {qid}: {c}")


  Gold in Stage 3 but DROPPED at Stage 4: 6/219
    val_002: Art. 56 Abs. 1 ATSG
    val_003: BGE 143 IV 330 E. 2.1
    val_003: BGE 145 IV 99 E. 3.1
    val_007: Art. 292 StGB
    val_008: BGE 149 IV 42 E. 3.5
    val_010: BGE 134 III 151 E. 2.4


In [63]:

# ══ Stage 4 diag 5/5: pool recall + tier analysis ══════════════════════
# Stage 4 keeps gold in all_citations via passthrough — tier rank is an
# intermediate signal; Stage 5 multi-signal confidence does the final cut.
# Key question: Is gold in the pool? What tier is it assigned?
import json, re
from collections import Counter

s4_path = get_checkpoint_path("stage4", SPLIT)
if s4_path.exists() and SPLIT == "val":
    import pandas as pd
    s4 = json.load(open(s4_path, encoding="utf-8"))
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")

    def _all_cands(r):
        """Full pool Stage 4 kept — union of all output lists."""
        seen = []
        seen_set = set()
        for k in ("all_citations", "reranked", "explicit", "procedural",
                  "direct_gen", "multi_signal", "stage1_passthrough"):
            for c in (r.get(k) or []):
                if c not in seen_set:
                    seen.append(c); seen_set.add(c)
        tiers = r.get("reranked_tiers") or {}
        if isinstance(tiers, dict):
            for c in tiers:
                if c not in seen_set:
                    seen.append(c); seen_set.add(c)
        return seen

    # ── 1. Full-pool recall (ceiling before Stage 5) ──────────────────────
    per_pool = {"Law":[0,0], "BGE":[0,0], "Numbered":[0,0]}
    for _, r in vdf.iterrows():
        gold  = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
        pool  = set(_all_cands(s4.get(r["query_id"], {})))
        for c in gold:
            t = "BGE" if c.startswith("BGE") else ("Law" if c.startswith("Art") else "Numbered")
            per_pool[t][1] += 1
            if c in pool: per_pool[t][0] += 1
    print("  Stage 4 full-pool recall (ceiling for Stage 5):")
    for t, (h, tot) in per_pool.items():
        if tot: print(f"    {t:9s}: {h}/{tot} = {h/tot*100:.1f}%")
    print()

    # ── 2. Tier distribution: gold T3 vs T2 vs absent, by citation type ───
    gold_tier = {"Law": Counter(), "BGE": Counter(), "Numbered": Counter()}
    for _, r in vdf.iterrows():
        gold  = {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()}
        tiers = s4.get(r["query_id"], {}).get("reranked_tiers") or {}
        pool  = set(_all_cands(s4.get(r["query_id"], {})))
        for c in gold:
            t = "BGE" if c.startswith("BGE") else ("Law" if c.startswith("Art") else "Numbered")
            if c in tiers:
                try: gold_tier[t][int(tiers[c])] += 1
                except Exception: gold_tier[t]["?"] += 1
            elif c in pool:
                gold_tier[t]["passthru"] += 1
            else:
                gold_tier[t]["absent"] += 1
    print("  Gold tier assignment (T3=best, T2=ok, passthru=kept via other signal, absent=lost):")
    for t in ("Law", "BGE", "Numbered"):
        ct = gold_tier[t]
        total = sum(ct.values())
        if total:
            parts = ", ".join(f"T{k}={v}" if isinstance(k,int) else f"{k}={v}"
                              for k, v in sorted(ct.items(), key=lambda x: str(x[0])))
            print(f"    {t:9s} ({total} gold): {parts}")
    print()

    # ── 3. Non-gold tier distribution (noise) ─────────────────────────────
    all_tier = Counter()
    gold_all = set()
    for _, r in vdf.iterrows():
        gold_all.update(c.strip() for c in str(r["gold_citations"]).split(";") if c.strip())
    for rec in s4.values():
        tiers = rec.get("reranked_tiers") or {}
        if isinstance(tiers, dict):
            for c, v in tiers.items():
                if c not in gold_all:
                    try: all_tier[int(v)] += 1
                    except Exception: pass
    print("  Non-gold tier distribution (noise the reranker accepted):")
    total_ng = sum(all_tier.values())
    for t in sorted(all_tier, reverse=True):
        n = all_tier[t]
        print(f"    T{t}: {n:6d} ({n/max(total_ng,1)*100:.1f}%)")
    print()
    print("  NOTE: Law articles systemically land in T2 (abstract provisions).")
    print("  Case citations land in T3 (directly match facts). This is expected.")
    print("  Stage 5 multi-signal confidence — not tier rank alone — does the final cut.")


  Stage 4 full-pool recall (ceiling for Stage 5):
    Law      : 133/149 = 89.3%
    BGE      : 54/69 = 78.3%
    Numbered : 26/33 = 78.8%

  Gold tier assignment (T3=best, T2=ok, passthru=kept via other signal, absent=lost):
    Law       (149 gold): T10=4, T4=2, T5=6, T6=10, T7=12, T8=12, T9=11, absent=16, passthru=76
    BGE       (69 gold): T6=6, T7=12, T8=5, T9=1, absent=15, passthru=30
    Numbered  (33 gold): T6=6, T7=4, T8=1, T9=1, absent=7, passthru=14

  Non-gold tier distribution (noise the reranker accepted):
    T10:      2 (0.2%)
    T9:     13 (1.2%)
    T8:     25 (2.4%)
    T7:    354 (33.9%)
    T6:    357 (34.2%)
    T5:    251 (24.0%)
    T4:     21 (2.0%)
    T3:      7 (0.7%)
    T2:     14 (1.3%)

  NOTE: Law articles systemically land in T2 (abstract provisions).
  Case citations land in T3 (directly match facts). This is expected.
  Stage 5 multi-signal confidence — not tier rank alone — does the final cut.


## Stage 5 — Verification, Confidence, and Thresholding

Final normalization, confidence scoring, and submission writing.

### `stage5_verify_and_score.py`

In [64]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage5_verify_and_score.py"

"""
stage5_verify_and_score.py -- STAGE 5: Verification + Confidence Scoring + F1 Tuning

Three sub-stages:

  5A. VERIFICATION: Normalize citation formatting and check existence against
      the corpus. Drop hallucinated citations. Expand Art.-level to Abs.-level.

  5B. CONFIDENCE SCORING: Each citation gets a composite score combining:
      - Binary signal weights (explicit, graph, reranker tier, etc.)
      - Continuous BM25 relevance score (per-query normalized, 0-0.20)
      - Graduated multi-signal bonus (2+, 3+, 4+ independent signals)

  5C. THRESHOLD TUNING (val only): Sweep thresholds to maximize macro-F1.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage5_verify_and_score.py                    # val: verify + tune threshold
#   python stage5_verify_and_score.py --split test       # test: verify + apply threshold
#   python stage5_verify_and_score.py --threshold 0.20   # override threshold
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   index/citation_lookup.pkl        (from stage0)
#   index/bm25_v2_index.pkl         (from stage0, for continuous BM25 scoring)
#   checkpoints/stage4_{split}.json  (from stage4)
#   For val: data/val.csv with gold_citations column
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage5_{split}.json  (scored predictions)
#   submissions/submission_{split}.csv  (final submission file)
#
# ─── EXPECTED TIMING ────────────────────────────────────────────────────────
#   ~30s index load + <1 second per query + ~5 seconds for threshold sweep
"""



sys.path.insert(0, str(Path(__file__).parent))


CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _compute_bm25_scores(
    qid: str,
    query_text: str,
    all_citations: list[str],
    stage1_analysis: dict,
    sparse: SparseRetriever,
    verbose: bool = False,
) -> dict[str, float]:
    """
    Compute per-query normalized BM25 scores for candidate citations ONLY.

    FAST approach:
      1. Combine all query texts into one merged token list (no PMI enhancement)
      2. Call bm25.get_scores() ONCE (single pass over 2.15M docs)
      3. Look up only the indices of our candidate citations
      4. Normalize to [0, 1]

    Skipping PMI enhancement (which adds 50+ tokens) makes this ~6x faster.
    Using a single combined call instead of per-query calls makes it ~5x faster.
    Net: ~30x faster than the naive approach.
    """
    # Gather query texts: original + top DE translations
    de_queries = stage1_analysis.get("search_queries_de", [])[:3]
    en_queries = [query_text] + stage1_analysis.get("search_queries_en", [])[:2]
    all_queries = en_queries + de_queries

    # Merge all query tokens into one set (deduplicated), NO PMI enhancement
    merged_tokens = []
    seen = set()
    for q in all_queries:
        for t in tokenise(q):
            if t not in seen:
                merged_tokens.append(t)
                seen.add(t)

    if not merged_tokens:
        return {}

    if verbose:
        print(f"      {len(all_queries)} queries → {len(merged_tokens)} unique tokens")

    # Pre-resolve citation indices
    cite_indices: dict[str, int] = {}
    for cite in all_citations:
        idx = sparse._id_to_idx.get(cite, -1)
        if idx >= 0:
            cite_indices[cite] = idx
        else:
            cl = cite.lower()
            for canon_cite, canon_idx in sparse._id_to_idx.items():
                if canon_cite.lower() == cl:
                    cite_indices[cite] = canon_idx
                    break

    if not cite_indices:
        return {}

    # Single BM25 call with merged tokens — one pass over 2.15M docs
    if verbose:
        print(f"      Scoring {len(cite_indices)} citations (single BM25 call)...")

    raw = sparse._bm25.get_scores(merged_tokens)

    # Extract only our candidate citations' scores
    cite_names = list(cite_indices.keys())
    idx_array = np.array([cite_indices[c] for c in cite_names])
    scores = raw[idx_array]

    if verbose:
        print(f"      BM25 call done")

    # Normalize by max score → [0, 1]
    max_val = scores.max()
    if max_val <= 0:
        return {}

    normalized = {}
    for i, cite in enumerate(cite_names):
        if scores[i] > 0:
            normalized[cite] = float(scores[i] / max_val)

    return normalized


def _compute_and_cache_all_bm25_scores(
    split: str,
    stage4_results: dict,
    stage1_results: dict,
    query_texts: dict[str, str],
    sparse: SparseRetriever,
) -> dict[str, dict[str, float]]:
    """
    Compute BM25 scores for ALL queries and cache to disk.
    On subsequent runs, loads from cache instantly.

    Returns dict: qid -> {citation: normalized_score}
    """
    cache_path = CHECKPOINTS_DIR / f"bm25_scores_{split}.json"

    # Try loading from cache
    if cache_path.exists():
        print(f"  Loading cached BM25 scores from {cache_path}")
        t0 = time.time()
        with open(cache_path, encoding="utf-8") as f:
            cached = json.load(f)
        print(f"  Loaded {len(cached)} queries in {time.time()-t0:.1f}s")
        # Verify cache covers all queries
        missing = set(stage4_results.keys()) - set(cached.keys())
        if not missing:
            return cached
        print(f"  Cache missing {len(missing)} queries, computing them...")
    else:
        cached = {}

    # Compute missing queries
    total = len(stage4_results)
    for i, (qid, stage4) in enumerate(stage4_results.items()):
        if qid in cached:
            continue
        t0 = time.time()
        bm25_scores = _compute_bm25_scores(
            qid, query_texts.get(qid, ""),
            stage4.get("all_citations", []),
            stage1_results.get(qid, {}),
            sparse,
            verbose=False,
        )
        elapsed = time.time() - t0
        cached[qid] = bm25_scores
        n_scored = len(bm25_scores)
        n_total = len(stage4.get("all_citations", []))
        print(f"    [{i+1}/{total}] {qid}: {n_scored}/{n_total} scored in {elapsed:.1f}s")

        # Save incrementally
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(cached, f, ensure_ascii=False)

    print(f"  Cached to {cache_path}")
    return cached


def verify_and_score_single(
    stage4_result: dict,
    verifier: Verifier | None,
    confidence_weights: dict[str, float] | None = None,
    bm25_scores: dict[str, float] | None = None,
) -> dict:
    """
    Verify, normalize, and score all citations for a single query.

    Returns {
        'scored': [(citation, score), ...],  # sorted by score desc
        'verified': [...],
        'dropped': [...],
        'stats': {...}
    }
    """
    all_citations = stage4_result.get("all_citations", [])
    sources_raw = stage4_result.get("sources", {})
    reranked_tiers = stage4_result.get("reranked_tiers", {})

    # Convert source lists to sets
    source_sets: dict[str, set[str]] = {}
    for source_name in ["explicit_from_query", "bm25_top10", "bm25_initial",
                        "citation_graph", "co_citation",
                        "llm_reranker", "llm_reranker_tier3", "llm_direct_gen",
                        "llm_stage1", "llm_stage1_procedural",
                        "mas_rewrite", "mas_supplement", "mas_decompose",
                        "mas_supportive", "mas_crossref"]:
        source_sets[source_name] = set()

    for cite, cite_sources in sources_raw.items():
        for s in cite_sources:
            if s in source_sets:
                source_sets[s].add(cite)
            # Map MAS sources to generic "bm25_initial" for scoring
            if s.startswith("mas_"):
                source_sets.setdefault("bm25_initial", set()).add(cite)

    # Inject reranker tier data from Stage 4's tiered output.
    #
    # Handles legacy integer {0,1,2,3} tiers AND the new continuous 0-10
    # scores from the rewritten Stage 4 ranker.
    #
    # IMPORTANT: keep tier3 SELECTIVE. The previous code used `>=3 -> tier3`
    # which tagged every LLM-favored item (40-80 per query); the uniform
    # +0.25 tier3 boost drowned out continuous BM25 signal and collapsed F1
    # from 0.58 (BM25 alone) to 0.46. New thresholds:
    #   score >= 8       -> tier3 (directly applicable)
    #   5 <= score < 8   -> tier2 (closely related)
    #   score < 5        -> no boost (let BM25 drive)
    for cite, score_val in reranked_tiers.items():
        if not isinstance(score_val, (int, float)):
            continue
        sv = float(score_val)
        if sv == int(sv) and sv <= 3.0:
            # Legacy integer-tier path
            if sv >= 3:
                source_sets.setdefault("llm_reranker_tier3", set()).add(cite)
            if sv >= 2:
                source_sets.setdefault("llm_reranker", set()).add(cite)
        else:
            # New continuous 0-10 scores: selective thresholds
            if sv >= 8.0:
                source_sets.setdefault("llm_reranker_tier3", set()).add(cite)
            elif sv >= 5.0:
                source_sets.setdefault("llm_reranker", set()).add(cite)

    # 5A: Verification
    if verifier:
        verified, dropped = verifier.verify_and_normalize(all_citations)
    else:
        verified = all_citations
        dropped = []

    # Remap source sets: verifier may normalize citation strings (e.g. STPO→StPO),
    # so we need to ensure the sources dict uses the post-verification forms.
    _verified_lower_to_verified = {c.lower(): c for c in verified}
    for source_name, cite_set in source_sets.items():
        remapped = set()
        for c in cite_set:
            cl = c.lower()
            if cl in _verified_lower_to_verified:
                remapped.add(_verified_lower_to_verified[cl])
            else:
                remapped.add(c)
        source_sets[source_name] = remapped

    # Remap BM25 scores to use post-verification citation forms
    bm25_remapped = None
    if bm25_scores:
        bm25_remapped = {}
        for c, s in bm25_scores.items():
            cl = c.lower()
            if cl in _verified_lower_to_verified:
                bm25_remapped[_verified_lower_to_verified[cl]] = max(
                    bm25_remapped.get(_verified_lower_to_verified[cl], 0.0), s
                )
            else:
                bm25_remapped[c] = max(bm25_remapped.get(c, 0.0), s)

    # 5B: Confidence scoring
    scored = score_all_citations(verified, source_sets, weights=confidence_weights,
                                 bm25_scores=bm25_remapped)

    return {
        "scored": scored,
        "verified": verified,
        "dropped": dropped,
        "stats": {
            "input_citations": len(all_citations),
            "verified": len(verified),
            "dropped": len(dropped),
        },
    }


def run_batch_stage5(
    split: str,
    threshold: float | None = None,
    confidence_weights: dict[str, float] | None = None,
) -> pd.DataFrame:
    """
    Run Stage 5 on all queries in a split.

    For val: tunes threshold on gold data.
    For test: applies provided threshold (or default).
    """
    # Load Stage 4 results
    stage4_path = CHECKPOINTS_DIR / f"stage4_{split}.json"
    if not stage4_path.exists():
        print(f"ERROR: Stage 4 results not found: {stage4_path}")
        print("Run first: python stage4_llm_reranker.py")
        sys.exit(1)
    with open(stage4_path, encoding="utf-8") as f:
        stage4_results = json.load(f)

    # Load verifier
    verifier = None
    if LOOKUP_PKL.exists():
        print("Loading citation verifier...")
        verifier = Verifier()
    else:
        print("WARNING: Lookup tables not found. Skipping verification.")

    # Load BM25 index for continuous scoring
    sparse = None
    if STATUTORY_BM25_PKL.exists():
        print("Loading BM25 index for continuous scoring...")
        sparse = SparseRetriever()
    else:
        print("WARNING: BM25 index not found. Using binary scoring only.")

    # Load query data
    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    df = pd.read_csv(csv_map[split])

    # Load Stage 1 results (for query text → BM25 scoring)
    stage1_path = CHECKPOINTS_DIR / f"stage1_{split}.json"
    stage1_results = {}
    if stage1_path.exists():
        with open(stage1_path, encoding="utf-8") as f:
            stage1_results = json.load(f)

    # Build query_id -> query_text mapping
    query_texts = {row["query_id"]: row["query"] for _, row in df.iterrows()}

    # ── Precompute BM25 scores (cached to disk) ─────────────────────────
    all_bm25_scores = {}
    if sparse:
        print("\nPrecomputing BM25 scores (cached to disk)...")
        all_bm25_scores = _compute_and_cache_all_bm25_scores(
            split, stage4_results, stage1_results, query_texts, sparse,
        )

    # ── Load cross-encoder scores from Stage 4b (Strategy C) ──────────
    all_ce_scores: dict[str, dict[str, float]] = {}
    ce_path = CHECKPOINTS_DIR / f"reranker_scores_{split}.json"
    if ce_path.exists():
        print(f"\nLoading cross-encoder scores from {ce_path}...")
        with open(ce_path, encoding="utf-8") as f:
            all_ce_scores = json.load(f)
        n_scored = sum(len(v) for v in all_ce_scores.values())
        print(f"  Loaded CE scores for {len(all_ce_scores)} queries ({n_scored} citations)")
    else:
        print(f"\nNo cross-encoder scores found at {ce_path} — using binary reranker only.")

    # ── Process all queries ──────────────────────────────────────────────
    print(f"\nScoring {len(stage4_results)} queries...")
    all_scored: dict[str, list[tuple[str, float]]] = {}
    all_results: dict[str, dict] = {}

    total_verified = 0
    total_dropped = 0

    for qid, stage4 in stage4_results.items():
        bm25_scores = all_bm25_scores.get(qid)
        result = verify_and_score_single(stage4, verifier, confidence_weights,
                                          bm25_scores=bm25_scores)
        all_scored[qid] = result["scored"]
        all_results[qid] = result
        total_verified += result["stats"]["verified"]
        total_dropped += result["stats"]["dropped"]

    print(f"  Total verified: {total_verified}, dropped: {total_dropped}")

    # ── 5B+: Cross-encoder Strategy C (post-processing) ────────────────
    # Exactly replicates the validated test (test_reranker_all10.py Strategy C):
    #   1. Subtract binary reranker weight (0.25 tier3, 0.15 tier2)
    #   2. Add cross-encoder P(yes) * weight
    # Uses sources_raw (pre-verification keys) for subtraction — this matches
    # the test behavior where key normalization mismatches are expected.
    if all_ce_scores:
        # Build case-insensitive CE lookup per query
        ce_lower: dict[str, dict[str, float]] = {}
        for qid, ce_dict in all_ce_scores.items():
            ce_lower[qid] = {k.lower(): v for k, v in ce_dict.items()}

        def _apply_strategy_c(base_scored, stage4_data, ce_scores_lower, w_ce):
            """Apply Strategy C: subtract binary reranker, add CE * weight."""
            adjusted_scored = {}
            for qid, scored_list in base_scored.items():
                sources_raw = stage4_data[qid].get("sources", {})
                ce_q = ce_scores_lower.get(qid, {})
                new_list = []
                for c, base_s in scored_list:
                    cite_sources = set(sources_raw.get(c, []))
                    adjusted = base_s
                    if "llm_reranker_tier3" in cite_sources:
                        adjusted -= DEFAULT_WEIGHTS.get("llm_reranker_tier3", 0.25)
                    elif "llm_reranker" in cite_sources:
                        adjusted -= DEFAULT_WEIGHTS.get("llm_reranker", 0.15)
                    adjusted = max(0.0, adjusted)
                    rk_s = ce_q.get(c.lower(), 0.0)
                    adjusted += rk_s * w_ce
                    new_list.append((c, min(adjusted, 1.0)))
                new_list.sort(key=lambda x: -x[1])
                adjusted_scored[qid] = new_list
            return adjusted_scored

        # Save base scores for sweeping (don't mutate yet)
        base_scored_copy = {qid: list(scored) for qid, scored in all_scored.items()}

        if split == "val":
            print("\nSweeping cross-encoder Strategy C weight...")

            val_gold_sweep: dict[str, set[str]] = {}
            for _, row in df.iterrows():
                qid_r = row["query_id"]
                raw = str(row.get("gold_citations", "") or "")
                gold_s = {c.strip() for c in raw.split(";") if c.strip()}
                if gold_s:
                    val_gold_sweep[qid_r] = gold_s

            base_thresh, base_f1 = optimize_threshold(all_scored, val_gold_sweep)
            print(f"  Base (no CE): macro-F1={base_f1:.4f} @ thresh={base_thresh:.2f}")

            best_ce_w, best_ce_f1, best_ce_thresh = 0.0, base_f1, base_thresh
            for w_int in range(5, 61):
                w_ce = w_int / 100.0
                adjusted = _apply_strategy_c(
                    base_scored_copy, stage4_results, ce_lower, w_ce)
                t, f1 = optimize_threshold(adjusted, val_gold_sweep)
                if f1 > best_ce_f1:
                    best_ce_f1 = f1
                    best_ce_w = w_ce
                    best_ce_thresh = t

            print(f"  Best CE weight: {best_ce_w:.2f}, macro-F1={best_ce_f1:.4f} "
                  f"@ thresh={best_ce_thresh:.2f} (delta={best_ce_f1 - base_f1:+.4f})")

            if best_ce_w > 0:
                all_scored = _apply_strategy_c(
                    base_scored_copy, stage4_results, ce_lower, best_ce_w)

                # Diagnostic: count how many citations got CE boost vs subtracted
                n_ce_hit = 0
                n_sub = 0
                for qid in all_scored:
                    sources_raw = stage4_results[qid].get("sources", {})
                    ce_q = ce_lower.get(qid, {})
                    for c, _ in base_scored_copy[qid]:
                        if ce_q.get(c.lower(), 0.0) > 0:
                            n_ce_hit += 1
                        cite_src = set(sources_raw.get(c, []))
                        if "llm_reranker_tier3" in cite_src or "llm_reranker" in cite_src:
                            n_sub += 1
                print(f"  Diagnostic: {n_ce_hit} citations got CE boost, "
                      f"{n_sub} had reranker subtracted (via sources_raw key match)")

                ce_config = {
                    "ce_weight": best_ce_w, "ce_threshold": best_ce_thresh,
                    "strategy": "C",
                }
                ce_config_path = CHECKPOINTS_DIR / "ce_config.json"
                with open(ce_config_path, "w") as f:
                    json.dump(ce_config, f)
                print(f"  Saved CE config to {ce_config_path}")
        else:
            ce_config_path = CHECKPOINTS_DIR / "ce_config.json"
            if ce_config_path.exists():
                with open(ce_config_path) as f:
                    ce_config = json.load(f)
                w_ce = ce_config["ce_weight"]
                print(f"\nApplying CE Strategy C with weight={w_ce:.2f} (from val tuning)")
                all_scored = _apply_strategy_c(
                    base_scored_copy, stage4_results, ce_lower, w_ce)
            else:
                print(f"\nWARNING: CE scores found but no ce_config.json — "
                      f"run val split first to tune CE weight.")

    # ── 5C: Threshold tuning (val only) ─────────────────────────────────
    if split == "val" and threshold is None:
        print("\nTuning threshold on val gold...")
        val_gold: dict[str, set[str]] = {}
        for _, row in df.iterrows():
            qid = row["query_id"]
            raw = str(row.get("gold_citations", "") or "")
            gold = {c.strip() for c in raw.split(";") if c.strip()}
            if gold:
                val_gold[qid] = gold

        best_thresh, best_f1 = optimize_threshold(all_scored, val_gold)
        print(f"  Best threshold: {best_thresh:.2f}")
        print(f"  Best macro-F1:  {best_f1:.4f}")
        threshold = best_thresh

        # Persist so test-split runs reuse the tuned value (there is no
        # val gold on test, and the default 0.15 is way too low).
        _thr_cfg = PROJECT_ROOT / "configs" / "threshold.json"
        _thr_cfg.parent.mkdir(parents=True, exist_ok=True)
        with open(_thr_cfg, "w", encoding="utf-8") as _thr_f:
            json.dump({"threshold": float(best_thresh),
                       "macro_f1_val": float(best_f1),
                       "tuned_on_split": "val"}, _thr_f, indent=2)
        print(f"  Saved tuned threshold to {_thr_cfg}")

        # Also show per-query F1 at best threshold
        print(f"\n  Per-query F1 @ threshold={threshold:.2f}:")
        f1s = []
        for qid, scored in all_scored.items():
            preds = [c for c, s in scored if s >= threshold]
            gold = val_gold.get(qid, set())
            if gold:
                f = compute_f1(preds, gold)
                f1s.append(f)
                print(f"    {qid}: F1={f:.4f} (pred={len(preds)}, gold={len(gold)})")
        macro = sum(f1s) / len(f1s) if f1s else 0.0
        print(f"\n  Macro F1: {macro:.4f}")

    if threshold is None:
        _thr_cfg = PROJECT_ROOT / "configs" / "threshold.json"
        if _thr_cfg.exists() and split != "val":
            with open(_thr_cfg, encoding="utf-8") as _thr_f:
                _thr_data = json.load(_thr_f)
            threshold = float(_thr_data.get("threshold", 0.15))
            print(f"\nLoaded tuned threshold from {_thr_cfg}: "
                  f"{threshold:.2f} (val macro-F1={_thr_data.get('macro_f1_val', '?')})")
        else:
            threshold = 0.15
            print(f"\nUsing default threshold: {threshold:.2f}")

    # ── Generate submission ──────────────────────────────────────────────
    print(f"\nGenerating submission with threshold={threshold:.2f}...")
    rows = []
    for _, row in df.iterrows():
        qid = row["query_id"]
        scored = all_scored.get(qid, [])
        preds = [c for c, s in scored if s >= threshold]
        rows.append({
            "query_id": qid,
            "predicted_citations": ";".join(preds),
        })

    out_df = pd.DataFrame(rows)
    out_csv = OUT_DIR / f"submission_{split}.csv"
    out_df.to_csv(out_csv, index=False)
    print(f"  Saved: {out_csv}")

    # Save detailed results
    out_detail = CHECKPOINTS_DIR / f"stage5_{split}.json"
    detail = {
        "threshold": threshold,
        "results": {qid: {
            "scored": r["scored"],
            "stats": r["stats"],
        } for qid, r in all_results.items()},
    }
    with open(out_detail, "w", encoding="utf-8") as f:
        json.dump(detail, f, ensure_ascii=False, indent=2)
    print(f"  Saved: {out_detail}")

    # Summary stats
    n_preds = out_df["predicted_citations"].apply(
        lambda x: len([c for c in str(x).split(";") if c.strip()]) if pd.notna(x) else 0
    )
    print(f"\n  Submission summary:")
    print(f"    Queries: {len(out_df)}")
    print(f"    Avg citations: {n_preds.mean():.1f}")
    print(f"    Max citations: {n_preds.max()}")
    print(f"    Queries with 0: {(n_preds == 0).sum()}")

    return out_df


### Stage 5 runner

In [65]:
stage5_outputs = run_stage5(split=SPLIT, threshold=STAGE5_THRESHOLD)

submission_path = get_submission_path(SPLIT)
print("Generated submission:", submission_path)


Loading citation verifier...
Loading BM25 index for continuous scoring...
  Loading combined BM25 index ...


  Loading BM25 chunks: 100%|##################| 216/216 [00:19<00:00, 11.22it/s]


  Converting BM25 doc_freqs to sparse matrix ...
    Sparse matrix: 2,156,832 docs x 822,251 terms (212,228,366 non-zero) in 36.4s
    171,654 law  +  1,985,178 court  =  2,156,832 total

Precomputing BM25 scores (cached to disk)...
    [1/10] val_001: 1507/1884 scored in 44.8s
    [2/10] val_002: 1733/2135 scored in 48.2s
    [3/10] val_003: 1745/2178 scored in 52.4s
    [4/10] val_004: 1597/2184 scored in 53.3s
    [5/10] val_005: 1871/2517 scored in 68.6s
    [6/10] val_006: 1967/2560 scored in 70.9s
    [7/10] val_007: 2273/2965 scored in 80.4s
    [8/10] val_008: 2255/2831 scored in 65.8s
    [9/10] val_009: 1609/2160 scored in 51.1s
    [10/10] val_010: 2319/3142 scored in 76.3s
  Cached to /content/drive/MyDrive/swiss_law/checkpoints/bm25_scores_val.json

No cross-encoder scores found at /content/drive/MyDrive/swiss_law/checkpoints/reranker_scores_val.json — using binary reranker only.

Scoring 10 queries...
  Total verified: 22816, dropped: 2646

Tuning threshold on val gold...

### Stage 5 diagnosis (6 cells)

1. Global-threshold macro-F1 + per-type P/R/F1 breakdown
2. Per-type calibrated thresholds (LOO CV on val) + alternate submission
3. Query-level F1 breakdown (one row per val query)
4. Citation-level error analysis (misses and false positives)
5. Abs-level flooding detection in final predictions
6. Procedural-prior potential (gold procedural cites not in predictions)


In [66]:

# ══ Stage 5 diag 1/6: global-threshold macro-F1 + per-type ═════════════
import json
from collections import defaultdict
from statistics import mean

def _scored(rec):
    for k in ("scored", "citations_scored", "final_scored"):
        v = rec.get(k)
        if isinstance(v, list): return v
    return []
def _ctype(c):
    if c.startswith("BGE"): return "BGE"
    if c.startswith("Art"): return "Law"
    if "_" in c and "/" in c: return "Numbered"
    return "Other"
def _f1(p, g):
    p, g = set(p), set(g)
    if not p and not g: return 1.0
    if not p or not g:  return 0.0
    tp = len(p & g)
    if tp == 0: return 0.0
    P = tp/len(p); R = tp/len(g)
    return 2*P*R/(P+R)

s5_path = get_checkpoint_path("stage5", SPLIT)
if s5_path.exists() and SPLIT == "val":
    import pandas as pd
    s5 = json.load(open(s5_path, encoding="utf-8"))
    thr = s5.get("threshold", 0.15)
    results = s5.get("results", {})
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
    gold = {r["query_id"]: {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()} for _, r in vdf.iterrows()}

    scored = {qid: _scored(rec) for qid, rec in results.items()}
    f1s = []; per_type = defaultdict(lambda: {"tp":0, "fp":0, "fn":0})
    for qid, s in scored.items():
        g = gold.get(qid, set())
        preds = {c for c, sc in s if sc >= thr}
        f1s.append(_f1(preds, g))
        for c in preds | g:
            t = _ctype(c)
            if c in preds and c in g: per_type[t]["tp"] += 1
            elif c in preds: per_type[t]["fp"] += 1
            else: per_type[t]["fn"] += 1
    print(f"  Global threshold: {thr}")
    print(f"  Macro-F1: {mean(f1s):.4f}")
    print(f"  Per-type:")
    for t, d in per_type.items():
        if d["tp"]+d["fp"] and d["tp"]+d["fn"]:
            P = d["tp"]/(d["tp"]+d["fp"]); R = d["tp"]/(d["tp"]+d["fn"])
            F = 2*P*R/(P+R) if (P+R) else 0
            print(f"    {t:10s} P={P:.3f} R={R:.3f} F1={F:.3f}  (tp={d['tp']} fp={d['fp']} fn={d['fn']})")


  Global threshold: 0.6500000000000001
  Macro-F1: 0.4820
  Per-type:
    Law        P=0.946 R=0.470 F1=0.628  (tp=70 fp=4 fn=79)
    Numbered   P=0.179 R=0.364 F1=0.240  (tp=12 fp=55 fn=21)
    BGE        P=0.650 R=0.377 F1=0.477  (tp=26 fp=14 fn=43)


In [67]:

# ══ Stage 5 diag 3/6: query-level F1 breakdown ═════════════════════════
import json

def _scored(r):
    for k in ("scored", "citations_scored", "final_scored"):
        v = r.get(k)
        if isinstance(v, list): return v
    return []
def _f1(p, g):
    p, g = set(p), set(g)
    if not p and not g: return 1.0, 1.0, 1.0
    if not p or not g:  return 0.0, 0.0, 0.0
    tp = len(p & g)
    if tp == 0: return 0.0, 0.0, 0.0
    P = tp/len(p); R = tp/len(g); F = 2*P*R/(P+R) if (P+R) else 0.0
    return P, R, F

s5_path = get_checkpoint_path("stage5", SPLIT)
if s5_path.exists() and SPLIT == "val":
    import pandas as pd
    s5 = json.load(open(s5_path, encoding="utf-8"))
    thr = s5.get("threshold", 0.15)
    results = s5.get("results", {})
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
    gold = {r["query_id"]: {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()} for _, r in vdf.iterrows()}

    print(f"  threshold={thr}")
    print(f"  {'qid':10s} {'pred':>5} {'gold':>5} {'TP':>4} {'P':>6} {'R':>6} {'F1':>6}")
    for qid in sorted(results):
        s = _scored(results[qid])
        preds = {c for c, sc in s if sc >= thr}
        g = gold.get(qid, set())
        P, R, F = _f1(preds, g)
        print(f"  {qid:10s} {len(preds):5d} {len(g):5d} {len(preds&g):4d} {P:.3f}  {R:.3f}  {F:.3f}")


  threshold=0.6500000000000001
  qid         pred  gold   TP      P      R     F1
  val_001       26    42   21 0.808  0.500  0.618
  val_002       10    36   10 1.000  0.278  0.435
  val_003       18    47   13 0.722  0.277  0.400
  val_004       13    10    4 0.308  0.400  0.348
  val_005       20    11    6 0.300  0.545  0.387
  val_006        5    18    5 1.000  0.278  0.435
  val_007       17    19    8 0.471  0.421  0.444
  val_008       23    29   15 0.652  0.517  0.577
  val_009       25    14   11 0.440  0.786  0.564
  val_010       24    25   15 0.625  0.600  0.612


In [68]:

# ══ Stage 5 diag 4/6: citation-level error analysis ════════════════════
import json
from collections import Counter

def _scored(r):
    for k in ("scored", "citations_scored", "final_scored"):
        v = r.get(k)
        if isinstance(v, list): return v
    return []
def _t(c):
    if c.startswith("BGE"): return "BGE"
    if c.startswith("Art"): return "Law"
    if "_" in c and "/" in c: return "Numbered"
    return "Other"

s5_path = get_checkpoint_path("stage5", SPLIT)
if s5_path.exists() and SPLIT == "val":
    import pandas as pd
    s5 = json.load(open(s5_path, encoding="utf-8"))
    thr = s5.get("threshold", 0.15)
    results = s5.get("results", {})
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
    gold = {r["query_id"]: {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()} for _, r in vdf.iterrows()}

    miss_by_type = Counter(); fp_by_type = Counter()
    miss_ex = []; fp_ex = []
    for qid, rec in results.items():
        s = _scored(rec)
        preds = {c for c, sc in s if sc >= thr}
        g = gold.get(qid, set())
        for c in g - preds:
            miss_by_type[_t(c)] += 1
            if len(miss_ex) < 30: miss_ex.append((qid, c))
        for c in preds - g:
            fp_by_type[_t(c)] += 1
            if len(fp_ex) < 30: fp_ex.append((qid, c))
    print("  MISSES by type:");
    for t, n in miss_by_type.most_common(): print(f"    {t}: {n}")
    print("  MISSES examples:")
    for qid, c in miss_ex: print(f"    {qid}: {c}")
    print("  FALSE POSITIVES by type:")
    for t, n in fp_by_type.most_common(): print(f"    {t}: {n}")
    print("  FP examples:")
    for qid, c in fp_ex[:15]: print(f"    {qid}: {c}")


  MISSES by type:
    Law: 79
    BGE: 43
    Numbered: 21
  MISSES examples:
    val_001: Art. 135 Abs. 4 StPO
    val_001: 7B_301/2024 E. 2.4
    val_001: BGE 139 IV 270 E. 3.1
    val_001: Art. 422 Abs. 1 StPO
    val_001: BGE 137 IV 122 E. 6.2
    val_001: 1B_90/2021 E. 2.1
    val_001: Art. 100 Abs. 1 BGG
    val_001: Art. 212 Abs. 3 StPO
    val_001: 7B_496/2025 E. 3.2
    val_001: Art. 221 Abs. 2 StPO
    val_001: 1B_28/2022 E. 4.1
    val_001: 7B_69/2024 E. 3.3.2
    val_001: 7B_231/2025 E. 4.1
    val_001: BGE 143 IV 168 E. 5.1
    val_001: BGE 133 I 270 E. 3.4.2
    val_001: 7B_12/2025 E. 2.2
    val_001: BGE 133 I 168 E. 4.1
    val_001: Art. 385 Abs. 1 StPO
    val_001: Art. 221 Abs. 1 StPO
    val_001: Art. 422 Abs. 2 StPO
    val_001: Art. 428 Abs. 1 StPO
    val_002: Art. 61 ATSG
    val_002: 8C_421/2023 E. 2.2
    val_002: BGE 135 V 39 E. 6.1
    val_002: Art. 56 Abs. 1 ATSG
    val_002: BGE 124 V 108 E. 2b
    val_002: Art. 6 ATSG
    val_002: 8C_160/2016 E. 4.1
    va

In [69]:

# ══ Stage 5 diag 5/6: Abs-level flooding in predictions ════════════════
import json, re
from collections import defaultdict, Counter

def _scored(r):
    for k in ("scored", "citations_scored", "final_scored"):
        v = r.get(k)
        if isinstance(v, list): return v
    return []
def _root(c):
    c = re.sub(r"\s+Abs\.?\s*\d+\w*", "", c)
    c = re.sub(r"\s+lit\.?\s*[a-z]", "", c)
    return c

s5_path = get_checkpoint_path("stage5", SPLIT)
if s5_path.exists():
    s5 = json.load(open(s5_path, encoding="utf-8"))
    thr = s5.get("threshold", 0.15)
    results = s5.get("results", {})
    events = 0; qf = Counter()
    samples = []
    for qid, rec in results.items():
        preds = [c for c, sc in _scored(rec) if sc >= thr]
        by_root = defaultdict(list)
        for c in preds:
            if c.startswith("Art") and (" Abs. " in c or " lit. " in c):
                by_root[_root(c)].append(c)
        for root, vs in by_root.items():
            if len(vs) >= 3:
                events += 1; qf[qid] += 1
                if len(samples) < 10: samples.append((qid, root, vs[:5]))
    print(f"  Abs-flooding events (same Art ≥3 Abs variants): {events} across {len(qf)} queries")
    for qid, root, vs in samples[:8]:
        print(f"    {qid}: {root} → {vs}")
    if events > 20:
        print("  ** flooding likely tanks precision; targeted Abs-selection would help")


  Abs-flooding events (same Art ≥3 Abs variants): 0 across 0 queries


In [70]:

# ══ Stage 5 diag 6/6: procedural-prior potential ═══════════════════════
# Common procedural cites in val gold that are NOT predicted.  Size of
# the recall gain a domain→procedural lookup could deliver.
import json
from collections import Counter

PROC = [
    "Art. 72 BGG", "Art. 74 BGG", "Art. 75 BGG", "Art. 76 BGG",
    "Art. 82 BGG", "Art. 90 BGG", "Art. 95 BGG", "Art. 97 BGG",
    "Art. 99 BGG", "Art. 100 BGG", "Art. 105 BGG", "Art. 106 BGG",
    "Art. 113 BGG", "Art. 66 BGG", "Art. 68 BGG",
]

def _scored(r):
    for k in ("scored", "citations_scored", "final_scored"):
        v = r.get(k)
        if isinstance(v, list): return v
    return []

s5_path = get_checkpoint_path("stage5", SPLIT)
if s5_path.exists() and SPLIT == "val":
    import pandas as pd
    s5 = json.load(open(s5_path, encoding="utf-8"))
    thr = s5.get("threshold", 0.15)
    results = s5.get("results", {})
    vdf = pd.read_csv(VAL_CSV if 'VAL_CSV' in dir() else "data/val.csv")
    gold = {r["query_id"]: {c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()} for _, r in vdf.iterrows()}

    catchable_tp = 0; gold_proc_total = 0; would_be_fp = 0
    for qid, rec in results.items():
        preds = {c for c, sc in _scored(rec) if sc >= thr}
        g = gold.get(qid, set())
        # Proc cites in gold for this query
        gp = {c for c in g if any(c.startswith(p) for p in PROC)}
        gold_proc_total += len(gp)
        # Proc cites we'd ADD (via a naive "always add them" prior)
        for p in PROC:
            if not any(c.startswith(p) for c in preds):
                if any(c.startswith(p) for c in g):
                    catchable_tp += 1
                else:
                    would_be_fp += 1
    if gold_proc_total:
        print(f"  Gold procedural cites total:         {gold_proc_total}")
        print(f"  Catchable by 'always add' prior:     +{catchable_tp} true positives")
        print(f"  But introduces:                      {would_be_fp} false positives")
        print(f"  → a SELECTIVE prior (domain-conditioned) is required")


  Gold procedural cites total:         2
  Catchable by 'always add' prior:     +2 true positives
  But introduces:                      148 false positives
  → a SELECTIVE prior (domain-conditioned) is required


## Submission Creation

Submission utilities and validation.

### `submit.py`

In [71]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/submit.py"

"""
submit.py -- Generate and validate a competition submission CSV.

# HOW TO RUN:
#   cd E:\\swiss-law-pipeline
#   python submit.py                                    # run pipeline on test
#   python submit.py --validate submissions/submission_val.csv  # validate only
#   python submit.py --from-checkpoint                  # build from stage5 checkpoint
"""



sys.path.insert(0, str(Path(__file__).parent))

OUT_DIR.mkdir(parents=True, exist_ok=True)


def validate_submission(submission_path: Path) -> bool:
    """Validate submission format against test query IDs."""
    print(f"Validating: {submission_path}")

    try:
        sub = pd.read_csv(submission_path)
    except Exception as e:
        print(f"  ERROR: Cannot read CSV: {e}")
        return False

    required_cols = {"query_id", "predicted_citations"}
    missing = required_cols - set(sub.columns)
    if missing:
        print(f"  ERROR: Missing columns: {missing}")
        return False

    test_df = pd.read_csv(TEST_CSV)
    test_ids = set(test_df["query_id"].astype(str))
    sub_ids = set(sub["query_id"].astype(str))

    missing_ids = test_ids - sub_ids
    extra_ids = sub_ids - test_ids

    if missing_ids:
        print(f"  ERROR: Missing {len(missing_ids)} test query IDs")
        return False
    if extra_ids:
        print(f"  WARNING: {len(extra_ids)} extra query IDs")

    sub["n_citations"] = sub["predicted_citations"].apply(
        lambda x: len([c for c in str(x).split(";") if c.strip()]) if pd.notna(x) else 0
    )
    print(f"  Queries:        {len(sub)}")
    print(f"  Avg citations:  {sub['n_citations'].mean():.1f}")
    print(f"  Max citations:  {sub['n_citations'].max()}")
    print(f"  Queries with 0: {(sub['n_citations'] == 0).sum()}")
    print("  VALID")
    return True


### Submission validation

In [72]:
submission_path = get_submission_path(SPLIT)
print("Submission path:", submission_path)

if submission_path.exists():
    submission_valid = validate_submission(get_submission_path(SPLIT))
    print("Submission valid:", submission_valid)
else:
    print("Submission file does not exist yet.")


Submission path: /content/drive/MyDrive/swiss_law/submissions/submission_val.csv
Validating: /content/drive/MyDrive/swiss_law/submissions/submission_val.csv
  ERROR: Missing 40 test query IDs
Submission valid: False


## Rollup dashboard

One cell to show where gold is lost. Reads every stage checkpoint and prints a
stage-by-stage recall table (Law / BGE / Numbered). The stage with the biggest
drop is where to focus next. Run this last.


---

## Diagnostics — run before/after any change

Each diagnostic answers one "will this work?" question using on-disk checkpoints (no pipeline rerun needed). Safe to rerun after any Stage 4/5 change.

- **D1 — Ranking ceiling**: is the gold already in our pool?
- **D2 — Adaptive-K oracle**: what F1 would we get if we knew K per query?
- **D3 — BGE-M3 recall**: does a proper multilingual dense retriever help?
- **D4 — CE calibration**: is the cross-encoder well calibrated (if used)?
- **D5 — LLM ranker smoke**: new Stage 4 ranker on 3 queries — does it beat BM25?
- **D6 — Shadow-val builder**: construct a 50-query val-like subset of train for tuning (so we don't overfit to the 10 real val queries).


In [73]:
# =================================================================
# DIAGNOSTICS — run these BEFORE or AFTER pipeline changes to confirm
# (a) the gold is findable, (b) a change actually helps, and
# (c) the pipeline will generalise to hidden test.
#
# Each function is standalone and uses only on-disk checkpoints — you can
# rerun a diagnostic without re-running the pipeline.
# =================================================================

import csv as _csv_diag
import json as _json_diag
import statistics as _stats_diag
import re as _re_diag


def _load_gold_map(split: str) -> dict:
    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    gold = {}
    with open(csv_map[split], encoding="utf-8") as f:
        for row in _csv_diag.DictReader(f):
            cites = {c.strip() for c in str(row.get("gold_citations", "")).split(";") if c.strip()}
            if cites:
                gold[row["query_id"]] = cites
    return gold


def _load_query_map(split: str) -> dict:
    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    out = {}
    with open(csv_map[split], encoding="utf-8") as f:
        for row in _csv_diag.DictReader(f):
            out[row["query_id"]] = row["query"]
    return out


# ── D1. Ranking ceiling ────────────────────────────────────────────────
def diag_d1_ranking_ceiling(split: str = "val"):
    """Is the gold in the pool? If yes, the problem is ranking, not retrieval.

    Prints per-query: pool recall, how many gold are findable in BM25 top-K,
    median rank of findable gold in BM25.
    """
    print("=" * 60)
    print(f"D1 — Ranking ceiling on {split}")
    print("=" * 60)

    gold = _load_gold_map(split)
    with open(CHECKPOINTS_DIR / f"stage3_{split}.json", encoding="utf-8") as f:
        s3 = _json_diag.load(f)
    with open(CHECKPOINTS_DIR / f"bm25_scores_{split}.json", encoding="utf-8") as f:
        bm = _json_diag.load(f)

    for qid, g in gold.items():
        pool = set(s3.get(qid, {}).get("expanded_pool", []))
        in_pool = g & pool
        bm_sorted = [c for c, _ in sorted(bm.get(qid, {}).items(), key=lambda x: -x[1])]
        ranks = []
        for c in g:
            try:
                ranks.append(bm_sorted.index(c) + 1)
            except ValueError:
                pass
        med = _stats_diag.median(ranks) if ranks else None
        print(f"  {qid}: gold={len(g)} in_pool={len(in_pool)}  in_bm25={len(ranks)}  "
              f"med_rank={med}  max_rank={max(ranks) if ranks else None}")

    print()
    print("Interpretation:")
    print("  * If in_pool ~= len(gold), recall is fine; focus on ranker.")
    print("  * If med_rank < 50 for most queries, BM25 top-50 already contains them.")
    print("  * If many gold are NOT in BM25 at all, consider dense retriever (D3).")


# ── D2. Adaptive-K oracle ─────────────────────────────────────────────
def diag_d2_adaptive_k_oracle(split: str = "val"):
    """If we knew K for each query, what would BM25 top-K give? And Stage 5 scored top-K?"""
    print("=" * 60)
    print(f"D2 — Adaptive-K oracle on {split}")
    print("=" * 60)

    gold = _load_gold_map(split)
    with open(CHECKPOINTS_DIR / f"bm25_scores_{split}.json", encoding="utf-8") as f:
        bm = _json_diag.load(f)

    stage5_path = CHECKPOINTS_DIR / f"stage5_{split}.json"
    s5 = None
    if stage5_path.exists():
        with open(stage5_path, encoding="utf-8") as f:
            s5 = _json_diag.load(f).get("results", {})

    def _f1(preds, g):
        if not preds or not g: return 0
        tp = len(set(preds) & g)
        p = tp/len(preds); r = tp/len(g)
        return 2*p*r/(p+r) if p+r else 0

    f1s_bm = []
    f1s_s5 = []
    for qid, g in gold.items():
        K = len(g)
        bm_preds = [c for c, _ in sorted(bm.get(qid, {}).items(), key=lambda x: -x[1])[:K]]
        f1s_bm.append(_f1(bm_preds, g))
        if s5:
            s5_preds = [c for c, _ in s5.get(qid, {}).get("scored", [])[:K]]
            f1s_s5.append(_f1(s5_preds, g))

    print(f"  BM25 oracle-K macro-F1:    {_stats_diag.mean(f1s_bm):.4f}")
    if f1s_s5:
        print(f"  Stage-5 oracle-K macro-F1: {_stats_diag.mean(f1s_s5):.4f}")
    print()
    print("Interpretation:")
    print("  * Oracle-K is the ceiling for any ranker that emits exactly K predictions.")
    print("  * If BM25 oracle is already ~0.65, a decent K predictor + ranker can hit 0.7+.")


# ── D3. BGE-M3 recall sanity check ──────────────────────────────────────
def diag_d3_bge_recall(split: str = "val", top_k: int = 50, max_queries: int = 3):
    """Does BGE-M3 dense retrieval beat BM25 recall@50? Expensive, so runs on 3 queries."""
    print("=" * 60)
    print(f"D3 — BGE-M3 recall sanity on {split} (first {max_queries} queries)")
    print("=" * 60)

    try:
        from FlagEmbedding import BGEM3FlagModel
    except ImportError:
        print("  FlagEmbedding not installed. Install with:")
        print("    pip install -U FlagEmbedding")
        print("  Skipping D3.")
        return

    gold = _load_gold_map(split)
    qmap = _load_query_map(split)

    # Use Stage 3 pool as the candidate universe (recall-bound by 0.886)
    with open(CHECKPOINTS_DIR / f"stage3_{split}.json", encoding="utf-8") as f:
        s3 = _json_diag.load(f)

    # Load cite -> text map
    texts = _load_citation_texts() if "_load_citation_texts" in globals() else {}
    if not texts:
        print("  No citation_texts loaded (Stage 4 must have been run once). Skipping D3.")
        return

    print(f"  Loading BGE-M3 (this takes ~30s)...")
    model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

    with open(CHECKPOINTS_DIR / f"bm25_scores_{split}.json", encoding="utf-8") as f:
        bm = _json_diag.load(f)

    for i, (qid, g) in enumerate(gold.items()):
        if i >= max_queries:
            break
        pool = list(set(s3.get(qid, {}).get("expanded_pool", [])))
        if not pool:
            continue
        docs = [texts.get(c, "")[:400] or c for c in pool]

        qemb = model.encode([qmap[qid]], return_dense=True)["dense_vecs"]
        demb = model.encode(docs, return_dense=True)["dense_vecs"]
        import numpy as _np
        scores = (qemb @ demb.T)[0]
        top = [pool[j] for j in _np.argsort(-scores)[:top_k]]
        bge_recall = len(set(top) & g) / len(g) if g else 0

        bm_top = [c for c, _ in sorted(bm.get(qid, {}).items(), key=lambda x: -x[1])[:top_k]]
        bm_recall = len(set(bm_top) & g) / len(g) if g else 0
        print(f"  {qid}: BGE-M3 R@{top_k}={bge_recall:.3f}  BM25 R@{top_k}={bm_recall:.3f}  "
              f"diff={bge_recall-bm_recall:+.3f}")

    print()
    print("Interpretation:")
    print("  * If BGE-M3 R@50 >= BM25 R@50 + 5pp consistently -> add to fusion pool.")
    print("  * If BGE-M3 is flat or worse -> skip dense retrieval entirely.")


# ── D4. Reranker calibration (only run if CE scores exist) ───────────────
def diag_d4_ce_calibration(split: str = "val"):
    """Plot-free histogram of CE scores: gold vs non-gold."""
    print("=" * 60)
    print(f"D4 — CE calibration on {split}")
    print("=" * 60)

    ce_path = CHECKPOINTS_DIR / f"reranker_scores_{split}.json"
    if not ce_path.exists():
        print(f"  No CE scores at {ce_path} (Stage 4b was disabled). Skipping D4.")
        return

    gold = _load_gold_map(split)
    with open(ce_path, encoding="utf-8") as f:
        ce = _json_diag.load(f)

    bins = {(0.0, 0.2): [0, 0], (0.2, 0.4): [0, 0], (0.4, 0.6): [0, 0],
            (0.6, 0.8): [0, 0], (0.8, 1.01): [0, 0]}
    for qid, g in gold.items():
        ce_q = ce.get(qid, {})
        for c, s in ce_q.items():
            for (lo, hi), pair in bins.items():
                if lo <= s < hi:
                    pair[0] += 1            # total
                    if c in g:
                        pair[1] += 1        # positive
                    break

    print(f"{'bin':<12} {'n':>6} {'n_gold':>8} {'frac_gold':>10}")
    for (lo, hi), (n, ng) in bins.items():
        frac = ng/n if n else 0
        print(f"{lo:.1f}–{hi:.1f}    {n:>6} {ng:>8} {frac:>10.3f}")
    print()
    print("Interpretation:")
    print("  * If frac_gold MONOTONICALLY rises with bin, CE is well calibrated.")
    print("  * If frac_gold is ~flat across bins, CE is broken (wrong prompt template / truncation).")


# ── D5. LLM-ranker smoke test on 3 queries ────────────────────────────
def diag_d5_llm_ranker_smoke(split: str = "val", n_queries: int = 3, pool_size: int = 100):
    """Runs the new Stage 4 ranker on 3 queries, reports F1 vs BM25 top-K."""
    print("=" * 60)
    print(f"D5 — LLM ranker smoke test on {split} ({n_queries} queries)")
    print("=" * 60)

    gold = _load_gold_map(split)
    qmap = _load_query_map(split)

    with open(CHECKPOINTS_DIR / f"stage1_{split}.json", encoding="utf-8") as f: s1 = _json_diag.load(f)
    with open(CHECKPOINTS_DIR / f"stage2_{split}.json", encoding="utf-8") as f: s2 = _json_diag.load(f)
    with open(CHECKPOINTS_DIR / f"stage3_{split}.json", encoding="utf-8") as f: s3 = _json_diag.load(f)
    with open(CHECKPOINTS_DIR / f"bm25_scores_{split}.json", encoding="utf-8") as f: bm = _json_diag.load(f)

    import sys as _sys_d5
    llm_mod = _sys_d5.modules[__name__]
    # `_build_rank_pool`, `rank_candidates_single_pass`, `_load_citation_texts`
    # are brought into this module by the Stage 4 cell earlier.
    if "_build_rank_pool" not in globals():
        print("  Stage 4 code not yet executed — run the Stage 4 cell first. Skipping D5.")
        return

    get_model_and_tokenizer()

    qids = list(gold.keys())[:n_queries]
    for qid in qids:
        pool, _ = _build_rank_pool(s1.get(qid, {}), s2.get(qid, {}), s3.get(qid, {}),
                                   pool_size=pool_size)
        scores = rank_candidates_single_pass(qmap[qid], pool, llm_mod,
                                             _load_citation_texts())
        top_k = sorted(scores.items(), key=lambda x: -x[1])[:20]
        preds = [c for c, _ in top_k]
        g = gold[qid]
        tp = len(set(preds) & g)
        p = tp/max(len(preds), 1); r = tp/max(len(g), 1)
        f1 = 2*p*r/(p+r) if p+r else 0

        bm_preds = [c for c, _ in sorted(bm.get(qid, {}).items(), key=lambda x: -x[1])[:20]]
        tp_b = len(set(bm_preds) & g)
        p_b = tp_b/max(len(bm_preds), 1); r_b = tp_b/max(len(g), 1)
        f1_b = 2*p_b*r_b/(p_b+r_b) if p_b+r_b else 0

        print(f"  {qid}: LLM-ranker top-20 F1={f1:.3f} (P={p:.2f} R={r:.2f})  |  "
              f"BM25 top-20 F1={f1_b:.3f} (P={p_b:.2f} R={r_b:.2f})  "
              f"delta={f1 - f1_b:+.3f}")

    print()
    print("Interpretation:")
    print("  * If LLM-ranker delta is consistently positive on 3 queries, promote to full val.")
    print("  * If delta is flat/negative, the prompt or pool composition needs work.")


# ── D6. Shadow-val builder (val-like subset of train) ─────────────────────
def diag_d6_build_shadow_val(n_shadow: int = 150, min_gold: int = 6):
    """Build checkpoints/shadow_val.csv — train queries with richer gold lists.

    Reality check: train has mean=4.1 gold cites/query with ~0% cases, while
    val has mean=25 with 30-40% cases. We CANNOT reproduce val's profile from
    train — only 1-2 train queries qualify. What we CAN do is pick the top
    ~150 train queries by gold-list size, giving the K regressor signal on
    which query features predict LARGER vs smaller gold lists. A downstream
    calibration step in Stage 6 then rescales the output to val's range.

    Defaults:
      * n_shadow=150 queries (enough for a stable regressor)
      * min_gold=6 (top ~22% of train by gold size)

    Known limitation: the case-vs-law mixture in shadow_val won't match val.
    That's handled at submission time by the procedural floor (Stage 6).
    """
    print("=" * 60)
    print(f"D6 — Building shadow-val (val-like subset of train)")
    print("=" * 60)

    import random as _random_d6
    rows = []
    with open(TRAIN_CSV, encoding="utf-8") as f:
        for row in _csv_diag.DictReader(f):
            cites = [c.strip() for c in str(row.get("gold_citations", "")).split(";") if c.strip()]
            if len(cites) >= min_gold:
                rows.append((row, len(cites)))

    # Sort by gold size descending so we pick the richest examples first.
    rows.sort(key=lambda x: -x[1])
    print(f"  Train rows with gold>={min_gold}: {len(rows)}")
    if len(rows) < n_shadow:
        print(f"  WARNING: only {len(rows)} qualify (wanted {n_shadow}); using all.")
        n_shadow = len(rows)

    # Balance: take top-half richest + random sample of the rest, for spread
    half = n_shadow // 2
    top = [r for r, _ in rows[:half]]
    _random_d6.seed(0)
    rest = [r for r, _ in rows[half:]]
    _random_d6.shuffle(rest)
    shadow = top + rest[:n_shadow - half]

    out_path = CHECKPOINTS_DIR / "shadow_val.csv"
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = _csv_diag.DictWriter(f, fieldnames=["query_id", "query", "gold_citations"])
        w.writeheader()
        for r in shadow:
            w.writerow({"query_id": r["query_id"], "query": r["query"],
                        "gold_citations": r["gold_citations"]})

    import statistics as _stats_d6
    sizes = [len(str(r["gold_citations"]).split(";")) for r in shadow]
    print(f"  Wrote {len(shadow)} rows to {out_path}")
    print(f"  Gold-size distribution: mean={_stats_d6.mean(sizes):.1f}  "
          f"median={_stats_d6.median(sizes):.0f}  min={min(sizes)}  max={max(sizes)}")
    print()
    print("Next steps:")
    print("  1. Use shadow_val.csv to fit the Stage 6 adaptive-K regressor.")
    print("  2. Stage 6 auto-calibrates the output range using val's real gold sizes.")
    print("  3. Do your ITERATIVE tuning on shadow-val, NOT on the 10 real val queries.")
    print("  4. Use real val only as a final smoke check (accept/reject config).")


def run_all_diagnostics(split: str = "val"):
    """Run D1, D2, D4, D6 (skipping D3 and D5 which need GPU loading)."""
    diag_d1_ranking_ceiling(split); print()
    diag_d2_adaptive_k_oracle(split); print()
    diag_d4_ce_calibration(split); print()
    diag_d6_build_shadow_val(); print()
    print("Tip: run diag_d3_bge_recall() and diag_d5_llm_ranker_smoke() separately — "
          "each requires loading a GPU model (~30-60s).")


### Run the fast diagnostics (D1, D2, D4, D6 — no GPU needed)


In [74]:
run_all_diagnostics(split=SPLIT)


D1 — Ranking ceiling on val
  val_001: gold=42 in_pool=42  in_bm25=42  med_rank=21.5  max_rank=684
  val_002: gold=36 in_pool=32  in_bm25=31  med_rank=16  max_rank=1593
  val_003: gold=47 in_pool=35  in_bm25=33  med_rank=17  max_rank=189
  val_004: gold=10 in_pool=8  in_bm25=8  med_rank=10.0  max_rank=122
  val_005: gold=11 in_pool=10  in_bm25=10  med_rank=5.5  max_rank=76
  val_006: gold=18 in_pool=14  in_bm25=14  med_rank=7.5  max_rank=876
  val_007: gold=19 in_pool=16  in_bm25=15  med_rank=8  max_rank=906
  val_008: gold=29 in_pool=26  in_bm25=25  med_rank=13  max_rank=772
  val_009: gold=14 in_pool=14  in_bm25=14  med_rank=7.5  max_rank=148
  val_010: gold=25 in_pool=22  in_bm25=21  med_rank=11  max_rank=666

Interpretation:
  * If in_pool ~= len(gold), recall is fine; focus on ranker.
  * If med_rank < 50 for most queries, BM25 top-50 already contains them.
  * If many gold are NOT in BM25 at all, consider dense retriever (D3).

D2 — Adaptive-K oracle on val
  BM25 oracle-K macro-

### GPU diagnostics (D3 BGE-M3, D5 LLM ranker smoke)

Run only after the fast diagnostics. Each loads a separate model.


---

## Stage 6 — Adaptive-K + procedural floor + final submission

Stage 5 emits a scored list; a single global threshold over-predicts for short-gold queries and under-predicts for long-gold ones. Stage 6:

1. Predicts **K per query** from query features (regressor trained on shadow-val).
2. Applies a hard **floor** (`STAGE6_MIN_K = 10`) so no query gets 0 predictions.
3. Applies a hard **ceiling** (`STAGE6_MAX_K = 45`) to cap precision collapse.
4. Force-includes **universal procedural citations** when the query mentions appeal/cost/constitutional triggers (gated on the Stage 5 verified set).
5. Writes the final `submissions/submission_{split}.csv`.

**Safety note on val generalization**: the K predictor is fitted on **shadow-val** (50 val-like train queries built in D6), *not* on the 10 real val queries. If shadow-val doesn't exist yet, Stage 6 falls back to a constant K = 22 (roughly the median val gold size).


In [75]:
# =================================================================
# STAGE 6 — Adaptive K + Procedural floor + Final submission
# =================================================================
#
# Stage 5 produces `checkpoints/stage5_{split}.json` which contains a
# `scored: [(cite, score), ...]` list per query. A single global threshold
# over-predicts for short-gold queries and under-predicts for long-gold
# ones. This cell:
#
#   1. Fits a per-query K predictor (regression on shadow-val labels) that
#      estimates how many citations to emit.
#   2. Enforces a hard floor (STAGE6_MIN_K) so no query gets 0 predictions.
#   3. Enforces a ceiling (STAGE6_MAX_K) to cap precision collapse.
#   4. Force-includes universal procedural citations when query mentions
#      appeal / Beschwerde / costs triggers.
#   5. Writes the final submission CSV (submissions/submission_{split}.csv).
#
# Runs fast (~1s). Idempotent — rerun after any Stage 5 change.

import re as _re_adaptk
import json as _json_adaptk
import csv as _csv_adaptk
from pathlib import Path as _Path_adaptk

STAGE6_MIN_K = 10              # per-query floor
STAGE6_MAX_K = 45              # per-query ceiling
STAGE6_USE_ADAPTIVE_K = True
STAGE6_PROCEDURAL_FLOOR = True

# Score source for final ranking. Options:
#   "bm25_plus_llm" — BM25 normalized + STAGE6_LLM_BOOST * (llm_score/10).
#                     Bypasses Stage 5's composite scoring (which was too
#                     noisy for this task). Best on val: F1=0.71 with w=0.5.
#                     RECOMMENDED DEFAULT.
#   "stage5"        — use Stage 5's composite scored list verbatim.
#                     Keeps the original pipeline but underperforms — Stage 5
#                     adds too many discrete source-boosts that drown the
#                     continuous BM25 signal.
STAGE6_SCORE_SOURCE = "bm25_plus_llm"
STAGE6_LLM_BOOST    = 0.5   # weight on (llm_score/10) added to BM25 normalized

# Strategy for per-query K. Options:
#   "score_gap"   — decide K from the score distribution (default). Robust
#                   to distribution shift, no training needed.
#   "regressor"   — Ridge regressor on shadow-val features. Often
#                   extrapolates badly, producing K=10 clamp everywhere.
#                   Keep only as a diagnostic fallback.
#   "fixed"       — constant K = STAGE6_FIXED_K for every query.
STAGE6_K_STRATEGY = "score_gap"
STAGE6_FIXED_K    = 15

# score-gap tuning knobs:
#   frac_of_max   — keep items whose score >= frac_of_max * max_score_in_query
#   elbow_drop    — stop when the score drop between ranks i and i+1 exceeds
#                   elbow_drop * max_score (detects a clear "cliff")
# Tuned on val initially (3 hyperparameters — low overfit risk on 10
# queries). Preferred: override via configs/stage6_knobs.json which can
# be written by the shadow-val tuning helper below.
STAGE6_FRAC_OF_MAX = 0.55
STAGE6_ELBOW_DROP  = 0.12
try:
    _s6_cfg = PROJECT_ROOT / "configs" / "stage6_knobs.json"
    if _s6_cfg.exists():
        import json as _s6_json
        _s6_data = _s6_json.loads(_s6_cfg.read_text(encoding="utf-8"))
        STAGE6_FRAC_OF_MAX = float(_s6_data.get("frac_of_max", STAGE6_FRAC_OF_MAX))
        STAGE6_ELBOW_DROP  = float(_s6_data.get("elbow_drop",  STAGE6_ELBOW_DROP))
        print(f"  Loaded Stage 6 knobs from {_s6_cfg}: "
              f"frac_of_max={STAGE6_FRAC_OF_MAX}, elbow_drop={STAGE6_ELBOW_DROP}")
except Exception as _e:
    print(f"  [stage6] knob load skipped ({_e!r})")

# Universal procedural citations by trigger category. These appear in ~95%
# of BGE decisions of the matching type. They ARE checked for existence by
# Stage 5 earlier — here we only ensure they survive the threshold.
_PROCEDURAL_BY_TRIGGER = {
    "appeal_bgg": [
        "Art. 42 BGG", "Art. 66 Abs. 1 BGG", "Art. 68 BGG",
        "Art. 95 BGG", "Art. 100 Abs. 1 BGG", "Art. 105 BGG", "Art. 106 BGG",
    ],
    "criminal": [
        "Art. 81 BGG", "Art. 80 BGG",
    ],
    "costs": [
        "Art. 422 StPO", "Art. 428 StPO",
    ],
    "constitutional": [
        "Art. 29 Abs. 2 BV", "Art. 9 BV",
    ],
}

_TRIGGERS = {
    "appeal_bgg":     _re_adaptk.compile(r"\b(appeal|Beschwerde|contest|challenge|before the Federal)\b", _re_adaptk.I),
    "criminal":       _re_adaptk.compile(r"\b(criminal|Strafverfahren|Strafprozess|detention|Untersuchungshaft|prosecut)\b", _re_adaptk.I),
    "costs":          _re_adaptk.compile(r"\b(cost|Gebühr|fee|compensation|Entschädigung|indemnif)\b", _re_adaptk.I),
    "constitutional": _re_adaptk.compile(r"\b(right to be heard|hearing|fair trial|fundamental right|Anspruch auf rechtliches Gehör|Grundrecht)\b", _re_adaptk.I),
}


def _features_for_k(query: str) -> list:
    q = query or ""
    ql = q.lower()
    return [
        len(q),
        len(q.split()),
        q.count("?"),
        len(_re_adaptk.findall(r"Art\.\s*\d+|BGE\s+\d+|[0-9]+[A-Z]_\d+/\d{4}", q)),
        ql.count(" and "),
        ql.count(" or "),
        ql.count("whether"),
        ql.count("issue"),
        ql.count("court"),
        len(_re_adaptk.findall(r"\b(StPO|StGB|ZGB|OR|BGG|BV|ZPO|SchKG|IPRG|ATSG|IVG|UVG|AVIG|BVG)\b", q)),
    ]


def _fit_k_predictor_from_shadow(shadow_val_path: _Path_adaptk, val_csv_path=None):
    """Fit a regressor query_features -> n_gold_citations on shadow-val.

    Because train gold-size distribution (mean 4.1) is much smaller than val's
    (mean 25), we fit the regressor on shadow-val to learn RELATIVE size (which
    queries need more vs fewer citations), then apply a ONE-PARAMETER affine
    calibration using val's gold-size statistics: K_cal = a * K_raw + b chosen
    to match val's (min, median, max). Only 2 numbers leak from val into the
    predictor — too few to overfit on 10 queries.

    Returns a callable predict(query) -> int. Falls back to a constant
    median of val if shadow-val is missing.
    """
    try:
        import numpy as _np
        from sklearn.linear_model import Ridge
    except ImportError:
        print("  sklearn missing — using constant fallback (median K=22)")
        return lambda q: 22

    # Gather val's gold sizes (2 numbers used for calibration)
    val_sizes = []
    if val_csv_path and _Path_adaptk(val_csv_path).exists():
        with open(val_csv_path, encoding="utf-8") as f:
            for row in _csv_adaptk.DictReader(f):
                cites = [c.strip() for c in str(row.get("gold_citations", "")).split(";") if c.strip()]
                if cites: val_sizes.append(len(cites))

    if not shadow_val_path.exists():
        med_k = int(_np.median(val_sizes)) if val_sizes else 22
        print(f"  Shadow-val not found — using constant K={med_k}")
        return lambda q: med_k

    X, y = [], []
    with open(shadow_val_path, encoding="utf-8") as f:
        for row in _csv_adaptk.DictReader(f):
            X.append(_features_for_k(row["query"]))
            cites = [c.strip() for c in str(row.get("gold_citations", "")).split(";") if c.strip()]
            y.append(len(cites))
    if not X:
        med_k = int(_np.median(val_sizes)) if val_sizes else 22
        return lambda q: med_k

    reg = Ridge(alpha=1.0).fit(X, y)
    train_med = float(_np.median(y))
    train_iqr = float(_np.percentile(y, 75) - _np.percentile(y, 25)) or 1.0

    # Calibration to val's scale (2 val-derived numbers: median + IQR)
    if val_sizes and len(val_sizes) >= 5:
        val_med = float(_np.median(val_sizes))
        val_iqr = float(_np.percentile(val_sizes, 75) - _np.percentile(val_sizes, 25)) or 1.0
        # Affine map: centre and scale
        a = val_iqr / train_iqr
        b = val_med - a * train_med
        print(f"  K predictor fitted on {len(X)} shadow-val rows "
              f"(train med={train_med:.0f}, IQR={train_iqr:.1f})")
        print(f"  Calibrated to val range (val med={val_med:.0f}, IQR={val_iqr:.1f})  "
              f"map: K_cal = {a:.2f} * K_raw + {b:.1f}")
    else:
        a, b = 1.0, 0.0
        print(f"  K predictor fitted on {len(X)} shadow-val rows — no val calibration")

    def _predict(q):
        x = _np.array(_features_for_k(q)).reshape(1, -1)
        raw = float(reg.predict(x)[0])
        cal = a * raw + b
        return int(round(max(STAGE6_MIN_K, min(STAGE6_MAX_K, cal))))
    return _predict


def _k_from_score_gap(scored_list, frac_of_max: float = STAGE6_FRAC_OF_MAX,
                      elbow_drop: float = STAGE6_ELBOW_DROP,
                      min_k: int = STAGE6_MIN_K, max_k: int = STAGE6_MAX_K) -> int:
    """Per-query K from the score distribution.

    Two termination signals (whichever triggers first):
      1. fraction-of-max: stop when score drops below `frac_of_max * max_score`.
      2. elbow drop:      stop when the drop between ranks i and i+1 exceeds
                          `elbow_drop * max_score` — a clear "cliff".

    Distribution-free: no training, no assumptions about the absolute score
    scale. Each query decides its own K from its own scored list, so this
    generalises from val to test without re-tuning.
    """
    if not scored_list:
        return min_k
    scores = [float(s) for _, s in scored_list]
    top = scores[0]
    if top <= 0:
        return min_k
    cutoff = frac_of_max * top
    K = 1
    for i in range(1, len(scores)):
        # elbow check
        if (scores[i-1] - scores[i]) > elbow_drop * top and K >= min_k:
            break
        # fraction-of-max check
        if scores[i] < cutoff:
            break
        K = i + 1
        if K >= max_k:
            break
    return max(min_k, min(max_k, K))


def _make_k_fn(strategy: str, query_map: dict, scored_map: dict):
    """Return callable qid -> int K, parameterised by STAGE6_K_STRATEGY."""
    if strategy == "fixed":
        print(f"  K strategy: fixed (K = {STAGE6_FIXED_K})")
        def _fn(qid): return STAGE6_FIXED_K
        return _fn

    if strategy == "regressor":
        shadow_path = CHECKPOINTS_DIR / "shadow_val.csv"
        reg_fn = _fit_k_predictor_from_shadow(shadow_path, val_csv_path=VAL_CSV)
        def _fn(qid): return reg_fn(query_map.get(qid, ""))
        return _fn

    # Default: score_gap
    print(f"  K strategy: score_gap "
          f"(frac_of_max={STAGE6_FRAC_OF_MAX}, elbow_drop={STAGE6_ELBOW_DROP})")
    def _fn(qid):
        return _k_from_score_gap(scored_map.get(qid, []))
    return _fn


def _applicable_procedural(query: str) -> list:
    if not STAGE6_PROCEDURAL_FLOOR:
        return []
    out = []
    for tag, pat in _TRIGGERS.items():
        if pat.search(query or ""):
            out.extend(_PROCEDURAL_BY_TRIGGER.get(tag, []))
    # Dedupe preserving order
    seen = set(); dedup = []
    for c in out:
        if c not in seen:
            seen.add(c); dedup.append(c)
    return dedup


def _macro_f1(all_scored: dict, gold_map: dict, k_fn, procedural_fn, verified_sets: dict):
    f1s, precs, recs = [], [], []
    for qid, g in gold_map.items():
        preds = _select_preds(all_scored.get(qid, []), k_fn(qid), procedural_fn(qid), verified_sets.get(qid, set()))
        if not preds and not g:
            f1s.append(1.0); precs.append(1.0); recs.append(1.0); continue
        if not preds or not g:
            f1s.append(0.0); precs.append(0.0); recs.append(0.0); continue
        tp = len(set(preds) & g)
        p = tp / len(preds); r = tp / len(g)
        f1 = 2*p*r/(p+r) if p+r else 0
        f1s.append(f1); precs.append(p); recs.append(r)
    return sum(f1s)/len(f1s), sum(precs)/len(precs), sum(recs)/len(recs)


def _select_preds(scored_list, K, procedural_floor, verified):
    """Take top-K by score, then union with verified procedural floor."""
    top = [c for c, _ in scored_list[:K]]
    if procedural_floor:
        # Only include procedural if it appears in the verified set (i.e. it
        # was actually found in the corpus during Stage 5 verification).
        for c in procedural_floor:
            if c in verified and c not in top:
                top.append(c)
    return top


def _build_bm25_plus_llm_scored(split: str,
                                 llm_boost: float = STAGE6_LLM_BOOST) -> dict:
    """Build per-query scored lists from BM25 + small LLM boost.

    score(c) = bm25_normalized(c) + llm_boost * (llm_score(c) / 10.0)

    Bypasses Stage 5's composite scoring (too noisy on this task — adds too
    many discrete source-boosts that drown the continuous BM25 signal).
    Falls back to whatever Stage 5 produces if BM25 scores are missing.

    Returns {qid: [(cite, score), ...]} sorted descending.
    """
    bm_path = CHECKPOINTS_DIR / f"bm25_scores_{split}.json"
    s4_path = CHECKPOINTS_DIR / f"stage4_{split}.json"
    if not bm_path.exists():
        print(f"  bm25_plus_llm mode needs {bm_path} — falling back to stage5")
        return None
    with open(bm_path, encoding="utf-8") as f:
        bm = _json_adaptk.load(f)
    tiers_by_qid = {}
    if s4_path.exists():
        with open(s4_path, encoding="utf-8") as f:
            s4 = _json_adaptk.load(f)
        for qid, entry in s4.items():
            tiers_by_qid[qid] = entry.get("reranked_tiers", {}) or {}

    out = {}
    for qid, bm_q in bm.items():
        tiers = tiers_by_qid.get(qid, {})
        scored = []
        for c, bm_s in bm_q.items():
            llm_s = tiers.get(c, 0.0)
            if not isinstance(llm_s, (int, float)):
                llm_s = 0.0
            scored.append((c, float(bm_s) + llm_boost * (float(llm_s) / 10.0)))
        scored.sort(key=lambda x: -x[1])
        out[qid] = scored
    return out


def run_stage6_adaptive_k(split: str = None):
    split = SPLIT if split is None else split

    # Auto-retune Stage-6 knobs on shadow-val if it's fresher than the
    # saved config. No-ops cleanly when shadow-val artifacts are absent.
    _sv_stage5 = CHECKPOINTS_DIR / "stage5_shadow_val.json"
    _s6_cfg    = PROJECT_ROOT / "configs" / "stage6_knobs.json"
    if _sv_stage5.exists() and (
        not _s6_cfg.exists()
        or _sv_stage5.stat().st_mtime > _s6_cfg.stat().st_mtime
    ):
        print("  [shadow-val retune] stage5_shadow_val.json is newer than "
              "configs/stage6_knobs.json — re-tuning knobs ...")
        try:
            best = tune_stage6_on_shadow_val(stage5_path=_sv_stage5)
            if best is not None:
                global STAGE6_FRAC_OF_MAX, STAGE6_ELBOW_DROP
                STAGE6_FRAC_OF_MAX, STAGE6_ELBOW_DROP = best[0], best[1]
                print(f"  [shadow-val retune] applied: "
                      f"frac_of_max={STAGE6_FRAC_OF_MAX}, "
                      f"elbow_drop={STAGE6_ELBOW_DROP}")
        except Exception as _e:
            print(f"  [shadow-val retune] skipped ({_e!r})")

    stage5_path = CHECKPOINTS_DIR / f"stage5_{split}.json"
    if not stage5_path.exists():
        print(f"ERROR: Stage 5 checkpoint missing: {stage5_path}")
        return None
    with open(stage5_path, encoding="utf-8") as f:
        stage5 = _json_adaptk.load(f)

    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    query_map = {}
    gold_map = {}
    with open(csv_map[split], encoding="utf-8") as f:
        for row in _csv_adaptk.DictReader(f):
            query_map[row["query_id"]] = row["query"]
            if "gold_citations" in row and row.get("gold_citations"):
                gold_map[row["query_id"]] = {c.strip() for c in str(row["gold_citations"]).split(";") if c.strip()}

    # Pick scoring source
    if STAGE6_SCORE_SOURCE == "bm25_plus_llm":
        alt = _build_bm25_plus_llm_scored(split)
        if alt is not None:
            scored_map = alt
            print(f"  Score source: bm25_plus_llm (w_llm={STAGE6_LLM_BOOST})")
        else:
            scored_map = {qid: r["scored"] for qid, r in stage5["results"].items()}
            print("  Score source: stage5 (bm25_plus_llm fallback)")
    else:
        scored_map = {qid: r["scored"] for qid, r in stage5["results"].items()}
        print("  Score source: stage5 composite")

    # Verified-in-corpus set per query — union Stage 5 scored + current scored
    verified_sets = {qid: {c for c, _ in r["scored"]} for qid, r in stage5["results"].items()}
    for qid, lst in scored_map.items():
        verified_sets.setdefault(qid, set()).update(c for c, _ in lst)

    # ── Precision post-processing on scored_map ──────────────────
    # Abs-collapse + per-type Numbered suppression. Runs BEFORE K
    # selection so the elbow/frac heuristics see a cleaner list.
    explicit_by_q = _s6_collect_explicit_abs(split)
    bm_cache_path = CHECKPOINTS_DIR / f"bm25_scores_{split}.json"
    bm_by_q = {}
    if bm_cache_path.exists():
        with open(bm_cache_path, encoding="utf-8") as _bm_f:
            bm_by_q = _json_adaptk.load(_bm_f)
    # Load per-query source maps from Stage 4 for the source-gate
    # branch of Numbered suppression.
    sources_by_q: dict[str, dict[str, list[str]]] = {}
    _s4_path = CHECKPOINTS_DIR / f"stage4_{split}.json"
    if _s4_path.exists():
        with open(_s4_path, encoding="utf-8") as _s4_f:
            _s4_data = _json_adaptk.load(_s4_f)
        for qid, entry in _s4_data.items():
            sources_by_q[qid] = entry.get("sources", {}) or {}
    _n_before = sum(len(v) for v in scored_map.values())
    for qid, scored in list(scored_map.items()):
        scored = _s6_suppress_numbered(
            scored,
            bm_by_q.get(qid, {}),
            sources_by_cite=sources_by_q.get(qid, {}),
        )
        scored = _s6_abs_collapse(scored, explicit_by_q.get(qid, set()))
        scored_map[qid] = scored
    _n_after = sum(len(v) for v in scored_map.values())
    print(f"  Post-proc: {_n_before} → {_n_after} candidates "
          f"(Abs-collapse + Numbered suppression, quantile={STAGE6_NUMBERED_BM25_QUANTILE}, "
          f"require_graph={STAGE6_NUMBERED_REQUIRE_GRAPH_SOURCE})")

    # Build per-query K function according to STAGE6_K_STRATEGY
    if STAGE6_USE_ADAPTIVE_K:
        k_fn = _make_k_fn(STAGE6_K_STRATEGY, query_map, scored_map)
    else:
        k_fn = lambda qid: STAGE6_FIXED_K

    per_q_k = {qid: k_fn(qid) for qid in query_map}
    per_q_proc = {qid: _applicable_procedural(q) for qid, q in query_map.items()}

    # Build predictions
    pred_map = {}
    sizes = []
    for qid, scored in scored_map.items():
        preds = _select_preds(scored, per_q_k[qid], per_q_proc[qid], verified_sets.get(qid, set()))
        preds = preds[:STAGE6_MAX_K]                 # ceiling
        if len(preds) < STAGE6_MIN_K:                # floor
            extras = [c for c, _ in scored if c not in preds]
            preds += extras[:STAGE6_MIN_K - len(preds)]
        pred_map[qid] = preds
        sizes.append(len(preds))

    # Report metrics if gold available
    if gold_map:
        import statistics
        f1s=[]; prs=[]; rcs=[]
        for qid, g in gold_map.items():
            preds = set(pred_map.get(qid, []))
            if not preds and not g:
                f1, p, r = 1.0, 1.0, 1.0
            elif not preds or not g:
                f1, p, r = 0.0, 0.0, 0.0
            else:
                tp = len(preds & g)
                p = tp / len(preds); r = tp / len(g)
                f1 = 2*p*r/(p+r) if p+r else 0
            f1s.append(f1); prs.append(p); rcs.append(r)
            print(f"    {qid}: K={per_q_k[qid]} preds={len(preds)} gold={len(g)} P={p:.3f} R={r:.3f} F1={f1:.3f}")
        print(f"\n  === MACRO F1 (adaptive-K): {statistics.mean(f1s):.4f}  "
              f"P={statistics.mean(prs):.4f}  R={statistics.mean(rcs):.4f} ===")

    # Write submission
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
    out_path = SUBMISSIONS_DIR / f"submission_{split}.csv"
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = _csv_adaptk.writer(f)
        w.writerow(["query_id", "predicted_citations"])
        for qid in query_map:
            preds = pred_map.get(qid, [])
            w.writerow([qid, ";".join(preds)])
    print(f"\n  Saved adaptive-K submission: {out_path}")
    import statistics
    print(f"  Mean predictions per query: {statistics.mean(sizes):.1f}  "
          f"(min={min(sizes)}, max={max(sizes)})")
    return pred_map


# ── [NEW] Stage-6 precision post-processing ─────────────────────────
_S6_ABS_RE = _re_adaptk.compile(r"^(Art\.\s+\d+[a-z]?)\s+(?:Abs\.?\s*\d+[a-z]?\s*)?(?:lit\.?\s*[a-z]\s*)?(.*)$")


def _s6_art_root(c: str):
    """Return (base_art, law_code) if `c` is 'Art. N [Abs. X [lit. y]] LAWCODE'."""
    if not c.startswith("Art."):
        return None
    parts = c.split()
    if len(parts) < 3:
        return None
    return (f"Art. {parts[1]}", parts[-1])


def _s6_abs_collapse(scored_list, explicit_abs: set,
                     max_variants: int = STAGE6_ABS_MAX_VARIANTS):
    """Reduce Abs-flooding.

    For each (base article, law code) group, keep at most `max_variants`
    top-scoring Abs/lit variants. Exception: any variant explicitly
    named in the query text (`explicit_abs`) is always kept.
    Diagnostic cell 87 showed 1228 flooding events on val — 5+ Abs
    variants of the same article bloating the candidate list and
    hurting precision.
    """
    if not STAGE6_ABS_COLLAPSE:
        return scored_list
    kept = []
    seen_counts = {}
    for c, s in scored_list:
        root = _s6_art_root(c)
        if root is None or c in explicit_abs:
            kept.append((c, s))
            continue
        n = seen_counts.get(root, 0)
        if n >= max_variants:
            continue
        kept.append((c, s))
        seen_counts[root] = n + 1
    return kept


# Non-text-mined sources: a Numbered citation with one of these is a
# first-hop retrieval/graph hit, not an N-hop co-citation cascade.
_S6_STRONG_NUMBERED_SOURCES = {
    "explicit_from_query", "bm25_top10", "bm25_initial",
    "citation_graph",
    "llm_stage1", "llm_stage1_procedural",
    "mas_rewrite", "mas_supplement", "mas_decompose",
    "mas_supportive", "mas_crossref",
}


def _s6_suppress_numbered(scored_list, bm25_scores: dict,
                          sources_by_cite: dict | None = None,
                          quantile: float = STAGE6_NUMBERED_BM25_QUANTILE,
                          require_graph: bool = STAGE6_NUMBERED_REQUIRE_GRAPH_SOURCE):
    """Per-type precision gate for Numbered case citations.

    Drops a Numbered candidate unless BOTH:
      (a) its BM25 score exceeds the `quantile`-th percentile of the
          per-query BM25 scores of NON-Numbered candidates, AND
      (b) at least one of its sources is a first-hop retrieval/graph
          signal (not `citation_graph_text_mined` alone).

    Diagnostic cell 109 on the v1 run still showed 45 Numbered FP
    vs 6 TP — the 0.75-quantile alone wasn't enough because the
    numbered_cocitation cascade adds thousands of case numbers per
    query that happen to hit a few query tokens.
    """
    import numpy as _s6_np
    def _is_num(c):
        return ("_" in c and "/" in c
                and not c.startswith("Art.")
                and not c.startswith("BGE"))
    if not bm25_scores and not require_graph:
        return scored_list
    non_num_scores = [bm25_scores.get(c, 0.0)
                      for c, _ in scored_list if not _is_num(c)]
    gate = (float(_s6_np.quantile(non_num_scores, quantile))
            if non_num_scores else 0.0)
    kept = []
    for c, s in scored_list:
        if not _is_num(c):
            kept.append((c, s))
            continue
        # Gate (a): BM25 percentile
        if bm25_scores and bm25_scores.get(c, 0.0) < gate:
            continue
        # Gate (b): require at least one non-text-mined source
        if require_graph and sources_by_cite is not None:
            srcs = set(sources_by_cite.get(c, ()))
            if not (srcs & _S6_STRONG_NUMBERED_SOURCES):
                continue
        kept.append((c, s))
    return kept


def _s6_collect_explicit_abs(split: str) -> dict:
    """Read Stage 1 checkpoint and return {qid: set(explicit_citations)}
    — Abs variants named verbatim in the query are always kept.
    """
    path = CHECKPOINTS_DIR / f"stage1_{split}.json"
    if not path.exists():
        return {}
    with open(path, encoding="utf-8") as f:
        s1 = _json_adaptk.load(f)
    return {qid: set(r.get("explicit_citations", []) or []) for qid, r in s1.items()}


def tune_stage6_on_shadow_val(stage5_path=None, shadow_csv=None,
                              frac_range=None, elbow_range=None):
    """Optional: sweep (frac_of_max, elbow_drop) on shadow-val F1.

    Requires a stage5 checkpoint computed on shadow-val queries. We
    don't run the full pipeline on shadow-val here — this is a
    post-hoc tuner. Writes configs/stage6_knobs.json if a better
    setting is found.
    """
    stage5_path = stage5_path or (CHECKPOINTS_DIR / "stage5_shadow_val.json")
    shadow_csv  = shadow_csv  or (CHECKPOINTS_DIR / "shadow_val.csv")
    if not _Path_adaptk(stage5_path).exists() or not _Path_adaptk(shadow_csv).exists():
        print(f"  shadow-val stage5 or csv missing — skipping Stage 6 retune")
        return None
    import itertools as _s6_it
    with open(stage5_path, encoding="utf-8") as f:
        sv = _json_adaptk.load(f)
    gold = {}
    with open(shadow_csv, encoding="utf-8") as f:
        for row in _csv_adaptk.DictReader(f):
            cs = [c.strip() for c in str(row.get("gold_citations", "")).split(";") if c.strip()]
            if cs:
                gold[row["query_id"]] = set(cs)
    scored_map = {qid: r["scored"] for qid, r in sv.get("results", {}).items() if qid in gold}
    frac_range  = frac_range  or [0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
    elbow_range = elbow_range or [0.06, 0.09, 0.12, 0.15, 0.18, 0.22]
    best = (STAGE6_FRAC_OF_MAX, STAGE6_ELBOW_DROP, -1.0)
    for frac, elbow in _s6_it.product(frac_range, elbow_range):
        f1s = []
        for qid, scored in scored_map.items():
            k = _k_from_score_gap(scored, frac_of_max=frac, elbow_drop=elbow)
            preds = set(c for c, _ in scored[:k])
            g = gold[qid]
            tp = len(preds & g)
            p = tp/len(preds) if preds else 0.0
            r = tp/len(g) if g else 0.0
            f1s.append(2*p*r/(p+r) if p+r else 0.0)
        m = sum(f1s)/len(f1s) if f1s else 0.0
        if m > best[2]:
            best = (frac, elbow, m)
    print(f"  shadow-val best: frac_of_max={best[0]}, elbow_drop={best[1]}, F1={best[2]:.4f}")
    out = PROJECT_ROOT / "configs" / "stage6_knobs.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(_json_adaptk.dumps({
        "frac_of_max": best[0],
        "elbow_drop":  best[1],
        "shadow_val_f1": best[2],
    }, indent=2), encoding="utf-8")
    print(f"  Wrote {out}")
    return best


### Run Stage 6


In [76]:
# Requires shadow_val.csv — run diag_d6_build_shadow_val() once before this.
stage6_predictions = run_stage6_adaptive_k(split=SPLIT)


  Score source: bm25_plus_llm (w_llm=0.5)
  Post-proc: 18876 → 11422 candidates (Abs-collapse + Numbered suppression, quantile=0.9, require_graph=True)
  K strategy: score_gap (frac_of_max=0.55, elbow_drop=0.12)
    val_001: K=24 preds=24 gold=42 P=1.000 R=0.571 F1=0.727
    val_002: K=25 preds=25 gold=36 P=1.000 R=0.694 F1=0.820
    val_003: K=18 preds=23 gold=47 P=0.870 R=0.426 F1=0.571
    val_004: K=10 preds=13 gold=10 P=0.462 R=0.600 F1=0.522
    val_005: K=10 preds=10 gold=11 P=0.600 R=0.545 F1=0.571
    val_006: K=10 preds=10 gold=18 P=0.900 R=0.500 F1=0.643
    val_007: K=10 preds=10 gold=19 P=1.000 R=0.526 F1=0.690
    val_008: K=15 preds=15 gold=29 P=1.000 R=0.517 F1=0.682
    val_009: K=10 preds=10 gold=14 P=1.000 R=0.714 F1=0.833
    val_010: K=14 preds=14 gold=25 P=1.000 R=0.560 F1=0.718

  === MACRO F1 (adaptive-K): 0.6777  P=0.8831  R=0.5655 ===

  Saved adaptive-K submission: /content/drive/MyDrive/swiss_law/submissions/submission_val.csv
  Mean predictions per query: 1

---

## How to run the new pipeline end-to-end

```
1. Run diag_d6_build_shadow_val()   # one-time, creates shadow_val.csv
2. Run Stages 1-3 (unchanged)
3. Run the new Stage 4 (minimal LLM ranker) — ~10 min/val, ~40 min/test
4. Run the new Stage 4b (no-op, cleans stale CE scores) — <1s
5. Run Stage 5 (verify + score + threshold tune) — unchanged
6. Run Stage 6 (adaptive-K + procedural floor) — <1s
7. Submission is in submissions/submission_test.csv
```

**Timing target (test, 40 queries)**: <1h total vs. >24h previously.

**F1 target**: validate with D2 oracle-K first. If oracle-K > 0.65 on val, the new Stage 4 + Stage 6 should land 0.65-0.75. If oracle-K < 0.55, the bottleneck is retrieval (not ranking) — revisit Stages 2-3.


## HyDE post-mortem — why DE/HyDE query variants were dead

Diagnostic data (diag cells 74, 75):

| Variant | Gold hits | Avg/run |
|---------|-----------|---------|
| `EN_raw` (original English query) | 121 | 12.10 |
| `EN_0` (reformulated English)     |  70 |  7.00 |
| `DE_2` (best German)              |  11 |  1.10 |
| `DE_0`, `DE_3`, `DE_4`, `HYDE_1`  |  ≤5 |  ≤0.5 |

**Root cause.** HyDE generates narrative German prose describing what a
relevant statutory article *would say* (Stage 1 Step 5, cell 54). BM25
over statute text is driven by **exact legal tokens** — article numbers,
law-code abbreviations (`StPO`, `ZGB`), and structural markers (`Art.`,
`Abs.`). HyDE prose carries topic signal but almost no citation-level
signal, so it behaves like fuzzy semantic search — the exact failure
mode documented in `research/why_bge-m3_failed.txt` for dense models:
*semantic similarity over-retrieves narrative matches while under-
retrieving exact legal authorities.*

**Why EN_raw wins.** The English query already contains the litigable
facts plus any literal citation strings (detection of `Art. 221 StPO`
etc.). Explicit-extraction gives us those verbatim, and BM25 over the
already-mixed EN+DE law corpus (with DE terminology injected via
Stage 1's cross-lingual term expansion) retrieves enough signal by
itself. The translation→retrieve chain adds noise without adding
precision.

**Decision.** Keep HyDE in Stage 1 outputs for debugging but *do not
route it into BM25 retrieval*. Future experiment: replace HyDE prose
with *citation-rich* LLM output (ask for `Art. N LAWCODE; …` not
narrative text) — closer to LegalMALR's decomposition agent.
